# Healthcare Challenge 2 - Baseline Submission

This notebook provides a simple baseline for **Healthcare Challenge 2: ED Cost Prediction**.

**Goal**: Predict `ed_cost_next3y_usd` for each patient
**Metric**: Mean Absolute Error (MAE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular ED cost data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your_api_key",        # Get from your team dashboard
    team_name="your_team_name"     # Your exact team name
)

# Iteration 1 
- MAE 493.8977

In [2]:
# -*- coding: utf-8 -*-
"""
AgentDS Healthcare Challenge 2 — ED Cost Forecasting (MAE)
Multimodal Specialist (PDF receipts + tabular) — LightGBM Quantile (Median) Regression

===========================================================
INSTALL (run once)
====:contentReference[oaicite:4]{index=4}==================================
pip install -U pandas numpy scipy scikit-learn lightgbm pymupdf joblib tqdm

Notes:
- This script avoids heavy OCR and uses fast text extraction with PyMuPDF.
- It parses CPT/HCPCS-like codes, quantities, and line totals from receipts to create predictive features.
- It trains MAE-aligned models using median (quantile=0.5) regression.
===========================================================
"""

from __future__ import annotations

import os
import re
import math
import json
import logging
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple

import numpy as np
import pandas as pd

from tqdm import tqdm
import joblib

import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction import FeatureHasher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import lightgbm as lgb
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# CONFIG (EDIT THESE)
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"          # optional (if present)
PDF_DIR = BASE_DIR / "receipts_pdf"               # receipt_{patient_id}.pdf
SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

# Caching receipts is strongly recommended (parsing 4k PDFs repeatedly is wasteful).
CACHE_PATH = BASE_DIR / "cache_receipts_multimodal_v1.joblib"
CACHE_VERSION = 1

# PDF parsing behavior
PARSE_WORKERS = 1  # keep 1 for maximum safety; increase if you want (read-only PDFs)
PDF_Y_TOL = 1.5    # y-axis tolerance when clustering words into lines

# Feature blocks
USE_PATIENTS_TABLE_IF_AVAILABLE = True
USE_TOP_CODE_FEATURES = True
TOP_K_CODES = 250
USE_HASHED_CODE_FEATURES = True
HASH_DIM_COUNT = 2**12
HASH_DIM_SPEND = 2**13
HASH_DIM_QTY = 2**12
USE_TFIDF_SVD = True
TFIDF_MAX_FEATURES = 20000
TFIDF_NGRAM_RANGE = (1, 2)
SVD_COMPONENTS = 64

# Modeling / CV
N_FOLDS = 5
SEEDS = [2024, 2025, 2026]  # small bagging ensemble

EARLY_STOPPING_ROUNDS = 200
LGBM_BASE_PARAMS = dict(
    objective="quantile",
    alpha=0.5,                 # median regression -> MAE-aligned
    boosting_type="gbdt",
    learning_rate=0.03,
    n_estimators=6000,         # large upper bound; early stopping will pick best iteration
    num_leaves=64,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    max_bin=255,
    n_jobs=-1,
    verbose=-1,
)

# Blend raw-target + log-target models (often robust on skewed costs)
TRAIN_BOTH_RAW_AND_LOG = True

# Final prediction post-processing
CLIP_PREDICTIONS_AT_ZERO = True


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
    )


# =========================
# UTIL
# =========================
_money_re = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d{2})$|^\d+(?:\.\d{2})$")
_code_re = re.compile(r"^(?:[A-Z]\d{4}|\d{4,5})$")  # examples: G0378, 99284, 85025, 36556

def is_money(token: str) -> bool:
    return bool(_money_re.match(token))

def money_to_float(token: str) -> float:
    return float(token.replace(",", ""))

def is_code(token: str) -> bool:
    return bool(_code_re.match(token))

def safe_log1p(x: float) -> float:
    # avoids tiny negative due to float noise
    return math.log1p(max(0.0, float(x)))

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


# =========================
# PDF PARSING
# =========================
def parse_header_fields(text: str) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    """
    Extract patient_id, zip3, insurance from receipt header if present.
    """
    pid = None
    zip3 = None
    insurance = None

    m = re.search(r"Patient ID:\s*(\d+)", text)
    if m:
        pid = int(m.group(1))

    m = re.search(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)", text)
    if m:
        zip3 = str(m.group(1))
        insurance = str(m.group(2)).lower()

    return pid, zip3, insurance


def cluster_words_into_lines(words: List[Tuple], y_tol: float = PDF_Y_TOL) -> List[List[str]]:
    """
    Cluster PyMuPDF 'words' into lines by y coordinate, then sort by x coordinate.
    Each word tuple: (x0, y0, x1, y1, word, block_no, line_no, word_no)
    Returns lines as list of tokens (strings) in reading order.
    """
    if not words:
        return []

    words_sorted = sorted(words, key=lambda w: (w[1], w[0]))  # sort by y0 then x0
    lines: List[List[str]] = []
    current: List[Tuple[float, str]] = []
    current_y: Optional[float] = None

    for w in words_sorted:
        x0, y0, *_rest = w
        token = str(w[4])

        if current_y is None or abs(y0 - current_y) <= y_tol:
            current.append((float(x0), token))
            # smooth y to reduce micro-jitter
            current_y = y0 if current_y is None else (0.8 * current_y + 0.2 * y0)
        else:
            lines.append([t for _, t in sorted(current, key=lambda z: z[0])])
            current = [(float(x0), token)]
            current_y = y0

    if current:
        lines.append([t for _, t in sorted(current, key=lambda z: z[0])])

    return lines


def parse_line_items_from_words(words: List[Tuple]) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    """
    Primary parser: uses word positions -> lines -> row parsing.
    Returns list of items: {code, desc, qty, unit, line_total}, and receipt total if found.
    """
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None

    lines = cluster_words_into_lines(words, y_tol=PDF_Y_TOL)
    if not lines:
        return items, total

    # Find the header line (contains Code / Description / Qty)
    header_idx = None
    for i, toks in enumerate(lines):
        low = [t.lower() for t in toks]
        if ("code" in low) and ("description" in low) and ("qty" in low):
            header_idx = i
            break

    if header_idx is None:
        return items, total

    for toks in lines[header_idx + 1 :]:
        if not toks:
            continue

        # TOTAL line can appear as ["TOTAL", "7,978.28"] or similar
        if toks[0].upper() == "TOTAL":
            # find last money token
            for t in reversed(toks[1:]):
                if is_money(t):
                    total = money_to_float(t)
                    break
            break

        # Typical row pattern: [code, desc..., qty, unit, line_total]
        if len(toks) >= 5 and is_code(toks[0]) and toks[-3].isdigit() and is_money(toks[-2]) and is_money(toks[-1]):
            code = toks[0]
            qty = int(toks[-3])
            unit = money_to_float(toks[-2])
            line_total = money_to_float(toks[-1])
            desc = " ".join(toks[1:-3])
            items.append({"code": code, "desc": desc, "qty": qty, "unit": unit, "line_total": line_total})

    return items, total


def parse_line_items_fallback_text(text: str) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    """
    Fallback parser: works if PyMuPDF text extraction returns one cell per line in row-major order.
    Example sequence after header:
        CODE
        DESC
        QTY
        UNIT
        LINE_TOTAL
        ...
        TOTAL
        50.00
    """
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None

    lines = [ln.strip() for ln in (text.splitlines() if text else []) if ln.strip()]
    if not lines:
        return items, total

    # Find where the table header begins
    # Look for a line that is exactly "Code" and next lines contain "Description", "Qty", etc.
    start_idx = None
    for i in range(len(lines) - 4):
        if lines[i].lower() == "code" and lines[i + 1].lower() == "description" and lines[i + 2].lower() == "qty":
            start_idx = i + 5  # data begins after 5 header lines
            break
        # Alternate: "Code Description Qty Unit ($) Line Total ($)" in one line
        if "code" in lines[i].lower() and "description" in lines[i].lower() and "qty" in lines[i].lower():
            start_idx = i + 1
            break

    if start_idx is None:
        return items, total

    i = start_idx
    while i < len(lines):
        if lines[i].upper().startswith("TOTAL"):
            # TOTAL may be in same line ("TOTAL 7,978.28") or next line ("TOTAL" then "50.00")
            parts = lines[i].split()
            if len(parts) >= 2 and is_money(parts[-1]):
                total = money_to_float(parts[-1])
            elif i + 1 < len(lines) and is_money(lines[i + 1]):
                total = money_to_float(lines[i + 1])
            break

        # Row-major 5-line pattern
        if i + 4 < len(lines) and is_code(lines[i]) and lines[i + 2].isdigit() and is_money(lines[i + 3]) and is_money(lines[i + 4]):
            code = lines[i]
            desc = lines[i + 1]
            qty = int(lines[i + 2])
            unit = money_to_float(lines[i + 3])
            line_total = money_to_float(lines[i + 4])
            items.append({"code": code, "desc": desc, "qty": qty, "unit": unit, "line_total": line_total})
            i += 5
        else:
            # If not matching, advance by 1 to re-sync
            i += 1

    return items, total


@dataclass
class ReceiptEntry:
    meta: Dict[str, Any]
    tok_count: Dict[str, float]
    tok_spend: Dict[str, float]
    tok_qty: Dict[str, float]
    doc_text: str
    items: List[Dict[str, Any]]


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptEntry:
    """
    Read receipt_{patient_id}.pdf and produce:
      - meta numeric/categorical features
      - hashed-token dicts (count/spend/qty) for long-tail codes
      - doc_text for TF-IDF
      - items list for explicit top-code pivots
    """
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"

    # Defaults for missing / parse failure
    empty_meta = dict(
        pdf_ok=0,
        pdf_n_lines=0,
        pdf_n_unique_codes=0,
        pdf_sum_qty=0.0,
        pdf_mean_qty=0.0,
        pdf_max_qty=0.0,
        pdf_sum_line_total=0.0,
        pdf_mean_line_total=0.0,
        pdf_max_line_total=0.0,
        pdf_std_line_total=0.0,
        pdf_top1_cost_share=0.0,
        pdf_top3_cost_share=0.0,
        pdf_repeat_cost_share=0.0,
        pdf_sum_unit=0.0,
        pdf_mean_unit=0.0,
        pdf_max_unit=0.0,
        pdf_std_unit=0.0,
        pdf_total=0.0,
        pdf_total_diff=0.0,
        pdf_insurance=None,
        pdf_zip3=None,
        pdf_parse_notes="missing_pdf",
    )

    if not pdf_path.exists():
        return ReceiptEntry(meta=empty_meta, tok_count={}, tok_spend={}, tok_qty={}, doc_text="", items=[])

    try:
        with fitz.open(pdf_path) as doc:
            if doc.page_count < 1:
                empty_meta["pdf_parse_notes"] = "empty_pdf"
                return ReceiptEntry(meta=empty_meta, tok_count={}, tok_spend={}, tok_qty={}, doc_text="", items=[])

            page = doc[0]
            text = page.get_text("text") or ""
            words = page.get_text("words") or []

    except Exception as e:
        empty_meta["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return ReceiptEntry(meta=empty_meta, tok_count={}, tok_spend={}, tok_qty={}, doc_text="", items=[])

    header_pid, zip3, insurance = parse_header_fields(text)

    # Primary parse via words; fallback via text
    items, total = parse_line_items_from_words(words)
    if not items:
        items, total = parse_line_items_fallback_text(text)

    if not items:
        empty_meta["pdf_parse_notes"] = "no_items_parsed"
        empty_meta["pdf_zip3"] = zip3
        empty_meta["pdf_insurance"] = insurance
        return ReceiptEntry(meta=empty_meta, tok_count={}, tok_spend={}, tok_qty={}, doc_text="", items=[])

    # Aggregate stats
    line_totals = np.array([float(it["line_total"]) for it in items], dtype=float)
    qtys = np.array([float(it["qty"]) for it in items], dtype=float)
    units = np.array([float(it["unit"]) for it in items], dtype=float)

    sum_line_total = float(line_totals.sum())
    pdf_total = float(total) if total is not None else sum_line_total

    codes = [str(it["code"]) for it in items]
    n_lines = len(items)
    n_unique = len(set(codes))

    sorted_cost = np.sort(line_totals)[::-1]
    top1 = float(sorted_cost[0] / sum_line_total) if sum_line_total > 0 else 0.0
    top3 = float(sorted_cost[:3].sum() / sum_line_total) if sum_line_total > 0 else 0.0

    # repeat spend share
    spend_by_code: Dict[str, float] = {}
    cnt_by_code: Dict[str, int] = {}
    for it in items:
        c = str(it["code"])
        spend_by_code[c] = spend_by_code.get(c, 0.0) + float(it["line_total"])
        cnt_by_code[c] = cnt_by_code.get(c, 0) + 1
    repeat_spend = sum(v for c, v in spend_by_code.items() if cnt_by_code.get(c, 0) >= 2)
    repeat_share = float(repeat_spend / sum_line_total) if sum_line_total > 0 else 0.0

    meta = dict(
        pdf_ok=1,
        pdf_n_lines=n_lines,
        pdf_n_unique_codes=n_unique,
        pdf_sum_qty=float(qtys.sum()),
        pdf_mean_qty=float(qtys.mean()) if n_lines else 0.0,
        pdf_max_qty=float(qtys.max()) if n_lines else 0.0,
        pdf_sum_line_total=sum_line_total,
        pdf_mean_line_total=float(line_totals.mean()) if n_lines else 0.0,
        pdf_max_line_total=float(line_totals.max()) if n_lines else 0.0,
        pdf_std_line_total=float(line_totals.std()) if n_lines else 0.0,
        pdf_top1_cost_share=top1,
        pdf_top3_cost_share=top3,
        pdf_repeat_cost_share=repeat_share,
        pdf_sum_unit=float(units.sum()),
        pdf_mean_unit=float(units.mean()) if n_lines else 0.0,
        pdf_max_unit=float(units.max()) if n_lines else 0.0,
        pdf_std_unit=float(units.std()) if n_lines else 0.0,
        pdf_total=pdf_total,
        pdf_total_diff=float(abs(sum_line_total - pdf_total)),
        pdf_insurance=(insurance.lower() if isinstance(insurance, str) else None),
        pdf_zip3=(str(zip3) if zip3 is not None else None),
        pdf_parse_notes="ok",
    )

    # Token dicts for hashing (long-tail code signals)
    tok_count: Dict[str, float] = {}
    tok_spend: Dict[str, float] = {}
    tok_qty: Dict[str, float] = {}

    doc_tokens: List[str] = []

    for it in items:
        code = str(it["code"])
        qty = float(it["qty"])
        spend = float(it["line_total"])

        # tokens: explicit code + prefixes + type
        toks = [
            f"code={code}",
            f"p1={code[0]}",
            f"p2={code[:2]}",
            f"p3={code[:3]}",
            f"type={'hcpcs' if code[0].isalpha() else 'cpt'}",
        ]
        for t in toks:
            tok_count[t] = tok_count.get(t, 0.0) + 1.0
            tok_qty[t] = tok_qty.get(t, 0.0) + qty
            tok_spend[t] = tok_spend.get(t, 0.0) + spend

        # doc text: include code tokens + cleaned description words
        doc_tokens.append(f"code_{code}")
        desc = str(it.get("desc", ""))
        desc_clean = re.sub(r"[^a-z0-9]+", " ", desc.lower()).strip()
        if desc_clean:
            doc_tokens.extend(desc_clean.split())

    # compress values (helps stability with trees + hashing)
    tok_count = {k: safe_log1p(v) for k, v in tok_count.items()}
    tok_qty = {k: safe_log1p(v) for k, v in tok_qty.items()}
    tok_spend = {k: safe_log1p(v) for k, v in tok_spend.items()}

    doc_text = " ".join(doc_tokens)

    return ReceiptEntry(meta=meta, tok_count=tok_count, tok_spend=tok_spend, tok_qty=tok_qty, doc_text=doc_text, items=items)


def load_receipt_cache(cache_path: Path) -> Dict[str, Any]:
    if cache_path.exists():
        try:
            obj = joblib.load(cache_path)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "data" in obj:
                return obj
        except Exception:
            pass
    return {"version": CACHE_VERSION, "data": {}}


def save_receipt_cache(cache_path: Path, cache_obj: Dict[str, Any]) -> None:
    try:
        joblib.dump(cache_obj, cache_path)
    except Exception as e:
        logging.warning(f"Failed to save cache to {cache_path}: {e}")


def build_receipt_artifacts(patient_ids: List[int], pdf_dir: Path, cache_path: Path) -> Tuple[pd.DataFrame, Dict[int, Dict[str, float]], Dict[int, Dict[str, float]], Dict[int, Dict[str, float]], Dict[int, str], pd.DataFrame]:
    """
    Returns:
      - receipt_meta_df: one row per patient_id with meta features + pdf_zip3/pdf_insurance
      - tok_count_map: patient_id -> token-count dict
      - tok_spend_map: patient_id -> token-spend dict
      - tok_qty_map: patient_id -> token-qty dict
      - doc_text_map: patient_id -> doc_text
      - line_items_df: long table of (patient_id, code, qty, unit, line_total, desc)
    """
    cache_obj = load_receipt_cache(cache_path)
    cache_data: Dict[str, Any] = cache_obj.get("data", {})

    # Parse missing entries
    to_parse = [pid for pid in patient_ids if str(pid) not in cache_data]
    if to_parse:
        logging.info(f"Parsing {len(to_parse):,} receipts (cache miss).")
    else:
        logging.info("All receipts loaded from cache.")

    for pid in tqdm(to_parse, desc="Parsing receipts", ncols=100):
        entry = featurize_receipt(pid, pdf_dir)
        cache_data[str(pid)] = {
            "meta": entry.meta,
            "tok_count": entry.tok_count,
            "tok_spend": entry.tok_spend,
            "tok_qty": entry.tok_qty,
            "doc_text": entry.doc_text,
            "items": entry.items,
        }

    cache_obj["data"] = cache_data
    if to_parse:
        save_receipt_cache(cache_path, cache_obj)

    # Build artifacts
    meta_rows: List[Dict[str, Any]] = []
    tok_count_map: Dict[int, Dict[str, float]] = {}
    tok_spend_map: Dict[int, Dict[str, float]] = {}
    tok_qty_map: Dict[int, Dict[str, float]] = {}
    doc_text_map: Dict[int, str] = {}

    item_rows: List[Dict[str, Any]] = []
    for pid in patient_ids:
        obj = cache_data.get(str(pid), None)
        if obj is None:
            entry = featurize_receipt(pid, pdf_dir)
            obj = {
                "meta": entry.meta,
                "tok_count": entry.tok_count,
                "tok_spend": entry.tok_spend,
                "tok_qty": entry.tok_qty,
                "doc_text": entry.doc_text,
                "items": entry.items,
            }

        meta = dict(patient_id=pid, **obj["meta"])
        meta_rows.append(meta)

        tok_count_map[pid] = obj.get("tok_count", {}) or {}
        tok_spend_map[pid] = obj.get("tok_spend", {}) or {}
        tok_qty_map[pid] = obj.get("tok_qty", {}) or {}
        doc_text_map[pid] = obj.get("doc_text", "") or ""

        for it in (obj.get("items", []) or []):
            item_rows.append(
                {
                    "patient_id": pid,
                    "code": str(it.get("code", "")),
                    "qty": float(it.get("qty", 0.0)),
                    "unit": float(it.get("unit", 0.0)),
                    "line_total": float(it.get("line_total", 0.0)),
                    "desc": str(it.get("desc", "")),
                }
            )

    receipt_meta_df = pd.DataFrame(meta_rows)
    line_items_df = pd.DataFrame(item_rows)
    return receipt_meta_df, tok_count_map, tok_spend_map, tok_qty_map, doc_text_map, line_items_df


# =========================
# FEATURE ENGINEERING
# =========================
def make_ohe() -> OneHotEncoder:
    """
    Compatible across sklearn versions:
      - newer: sparse_output
      - older: sparse
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        try:
            return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse=True)
        except TypeError:
            return OneHotEncoder(handle_unknown="ignore", sparse=True)


def add_top_code_features(
    df_all: pd.DataFrame,
    line_items_df: pd.DataFrame,
    train_patient_ids: np.ndarray,
    top_k: int = TOP_K_CODES,
) -> pd.DataFrame:
    """
    Adds explicit (non-hashed) CPT/HCPCS composition features:
      - cpt_cnt_{code}
      - cpt_share_{code}  (spend share relative to prior_ed_cost_5y_usd)

    Uses top codes by frequency in TRAIN receipts only.
    """
    if line_items_df.empty:
        logging.warning("line_items_df is empty: skipping top-code features.")
        return df_all

    train_items = line_items_df[line_items_df["patient_id"].isin(train_patient_ids)]
    vc = train_items["code"].value_counts()
    top_codes = vc.head(top_k).index.tolist()

    if not top_codes:
        logging.warning("No codes found in receipts: skipping top-code features.")
        return df_all

    logging.info(f"Adding top-code features for TOP_K_CODES={len(top_codes)}")

    # Aggregate per patient+code
    agg = (
        line_items_df[line_items_df["code"].isin(top_codes)]
        .groupby(["patient_id", "code"], as_index=False)
        .agg(cnt=("code", "size"), spend=("line_total", "sum"))
    )

    cnt_wide = agg.pivot(index="patient_id", columns="code", values="cnt").fillna(0.0)
    spend_wide = agg.pivot(index="patient_id", columns="code", values="spend").fillna(0.0)

    # Compute spend share relative to prior cost
    denom = df_all.set_index("patient_id")["prior_ed_cost_5y_usd"].astype(float)
    # Align denom index to spend_wide rows
    spend_share = spend_wide.div(denom.reindex(spend_wide.index).values.reshape(-1, 1) + 1e-6).fillna(0.0)

    cnt_wide.columns = [f"cpt_cnt_{c}" for c in cnt_wide.columns]
    spend_share.columns = [f"cpt_share_{c}" for c in spend_share.columns]

    # Merge
    df_all = df_all.merge(cnt_wide.reset_index(), on="patient_id", how="left")
    df_all = df_all.merge(spend_share.reset_index(), on="patient_id", how="left")

    # Missing -> 0
    new_cols = [c for c in df_all.columns if c.startswith("cpt_cnt_") or c.startswith("cpt_share_")]
    df_all[new_cols] = df_all[new_cols].fillna(0.0)

    return df_all


def build_tfidf_svd(doc_texts: List[str], max_features: int, ngram_range: Tuple[int, int], n_components: int, seed: int) -> Tuple[np.ndarray, TfidfVectorizer, Optional[TruncatedSVD]]:
    """
    Fit TF-IDF on all docs (train+test) and then SVD to get dense embedding features.
    (Unsupervised on test is usually acceptable in competitions.)
    """
    vectorizer = TfidfVectorizer(
        min_df=2,
        max_features=max_features,
        ngram_range=ngram_range,
        token_pattern=r"(?u)\b\w+\b",
    )
    X_tfidf = vectorizer.fit_transform(doc_texts)

    if X_tfidf.shape[1] <= 2:
        # not enough features for SVD
        X_dense = X_tfidf.toarray().astype(np.float32)
        return X_dense, vectorizer, None

    k = int(min(n_components, X_tfidf.shape[1] - 1))
    svd = TruncatedSVD(n_components=k, random_state=seed)
    X_svd = svd.fit_transform(X_tfidf).astype(np.float32)
    return X_svd, vectorizer, svd


def build_feature_matrix(
    df_all: pd.DataFrame,
    tok_count_map: Dict[int, Dict[str, float]],
    tok_spend_map: Dict[int, Dict[str, float]],
    tok_qty_map: Dict[int, Dict[str, float]],
    doc_text_map: Dict[int, str],
) -> Tuple[csr_matrix, Dict[str, Any]]:
    """
    Build a combined sparse feature matrix:
      - tabular numeric + categorical (OHE)
      - hashed long-tail code dicts
      - optional TF-IDF+SVD dense embeddings
    """
    df_all = df_all.copy()

    # Basic derived features (tabular)
    df_all["prior_cost_per_visit"] = df_all["prior_ed_cost_5y_usd"] / (df_all["prior_ed_visits_5y"] + 1.0)
    df_all["log_prior_cost"] = np.log1p(df_all["prior_ed_cost_5y_usd"].astype(float))
    df_all["log_prior_visits"] = np.log1p(df_all["prior_ed_visits_5y"].astype(float))

    # Normalize pdf categoricals
    if "pdf_insurance" in df_all.columns:
        df_all["pdf_insurance"] = df_all["pdf_insurance"].fillna("missing").astype(str)
    if "pdf_zip3" in df_all.columns:
        df_all["pdf_zip3"] = df_all["pdf_zip3"].fillna("missing").astype(str)

    # Decide categorical columns present
    categorical_cols = []
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3"]:
        if c in df_all.columns:
            categorical_cols.append(c)

    # Numeric columns: everything numeric excluding ids/targets; keep receipt meta + engineered
    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train"}
    numeric_cols = [
        c for c in df_all.columns
        if (c not in exclude)
        and (c not in categorical_cols)
        and (pd.api.types.is_numeric_dtype(df_all[c]))
    ]

    # Tabular preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), categorical_cols),
        ],
        remainder="drop",
        sparse_threshold=0.5,
    )

    X_tab = preprocessor.fit_transform(df_all)
    if not hasattr(X_tab, "tocsr"):
        X_tab = csr_matrix(X_tab)
    else:
        X_tab = X_tab.tocsr()

    blocks = [X_tab]
    artifacts: Dict[str, Any] = {"preprocessor": preprocessor, "numeric_cols": numeric_cols, "categorical_cols": categorical_cols}

    # Hashed code features
    if USE_HASHED_CODE_FEATURES:
        ids = df_all["patient_id"].astype(int).tolist()
        tok_counts = [tok_count_map.get(pid, {}) for pid in ids]
        tok_spends = [tok_spend_map.get(pid, {}) for pid in ids]
        tok_qtys = [tok_qty_map.get(pid, {}) for pid in ids]

        hasher_count = FeatureHasher(n_features=HASH_DIM_COUNT, input_type="dict", dtype=np.float32)
        hasher_spend = FeatureHasher(n_features=HASH_DIM_SPEND, input_type="dict", dtype=np.float32)
        hasher_qty = FeatureHasher(n_features=HASH_DIM_QTY, input_type="dict", dtype=np.float32)

        X_count = hasher_count.transform(tok_counts)
        X_spend = hasher_spend.transform(tok_spends)
        X_qty = hasher_qty.transform(tok_qtys)

        blocks.extend([X_count, X_spend, X_qty])
        artifacts.update({"hasher_count": hasher_count, "hasher_spend": hasher_spend, "hasher_qty": hasher_qty})

    # TF-IDF + SVD embeddings
    if USE_TFIDF_SVD:
        ids = df_all["patient_id"].astype(int).tolist()
        doc_texts = [doc_text_map.get(pid, "") for pid in ids]
        X_svd, vectorizer, svd = build_tfidf_svd(
            doc_texts=doc_texts,
            max_features=TFIDF_MAX_FEATURES,
            ngram_range=TFIDF_NGRAM_RANGE,
            n_components=SVD_COMPONENTS,
            seed=SEEDS[0],
        )
        blocks.append(csr_matrix(X_svd))
        artifacts.update({"tfidf": vectorizer, "svd": svd, "svd_dim": X_svd.shape[1]})

    X = hstack(blocks, format="csr")
    artifacts["n_features"] = X.shape[1]
    return X, artifacts


# =========================
# MODELING
# =========================
def mae_original_from_log(y_true_log: np.ndarray, y_pred_log: np.ndarray) -> Tuple[str, float, bool]:
    """
    Custom eval metric for log1p(target) training:
      - y_true_log, y_pred_log are log1p values
      - compute MAE in original USD scale
    """
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return "mae_usd", float(np.mean(np.abs(y_true - y_pred))), False


def make_strat_labels(train_df: pd.DataFrame) -> np.ndarray:
    """
    Stratify by (primary_chronic + prior_cost decile) to keep severity mix stable across folds.
    """
    chronic = train_df["primary_chronic"].astype(str)
    # qcut returns categorical bins; duplicates drop if ties
    cost_bin = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    return (chronic + "_" + cost_bin).values


def cv_oof_and_best_iters(
    X_train: csr_matrix,
    y: np.ndarray,
    strat_labels: np.ndarray,
    params: Dict[str, Any],
    seeds: List[int],
    use_log_target: bool,
) -> Tuple[np.ndarray, List[int], float]:
    """
    CV loop to:
      - produce out-of-fold predictions (OOF) for MAE estimation
      - collect best_iteration_ for later full-data fitting
    """
    oof_preds_per_seed: List[np.ndarray] = []
    best_iters: List[int] = []

    for seed in seeds:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        oof_seed = np.zeros(len(y), dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, strat_labels)):
            X_tr, X_va = X_train[tr_idx], X_train[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            model_params = dict(params)
            model_params["random_state"] = seed * 100 + fold

            model = LGBMRegressor(**model_params)

            if use_log_target:
                y_tr_fit = np.log1p(y_tr)
                y_va_fit = np.log1p(y_va)
                model.fit(
                    X_tr, y_tr_fit,
                    eval_set=[(X_va, y_va_fit)],
                    eval_metric=mae_original_from_log,
                    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
                )
                best_it = int(getattr(model, "best_iteration_", model_params["n_estimators"]) or model_params["n_estimators"])
                pred_va = np.expm1(model.predict(X_va, num_iteration=best_it))
            else:
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    eval_metric="mae",
                    callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
                )
                best_it = int(getattr(model, "best_iteration_", model_params["n_estimators"]) or model_params["n_estimators"])
                pred_va = model.predict(X_va, num_iteration=best_it)

            if CLIP_PREDICTIONS_AT_ZERO:
                pred_va = np.maximum(pred_va, 0.0)

            oof_seed[va_idx] = pred_va.astype(np.float32)
            best_iters.append(best_it)

        oof_preds_per_seed.append(oof_seed)

    # MAE likes medians; median across seeds tends to be robust
    oof = np.median(np.vstack(oof_preds_per_seed), axis=0)
    score = mae(y, oof)
    return oof, best_iters, score


def fit_full_and_predict(
    X_train: csr_matrix,
    y: np.ndarray,
    X_test: csr_matrix,
    params: Dict[str, Any],
    seeds: List[int],
    n_estimators: int,
    use_log_target: bool,
) -> np.ndarray:
    """
    Fit on full training data for each seed, then take median prediction across seeds.
    """
    preds_seed: List[np.ndarray] = []
    for seed in seeds:
        model_params = dict(params)
        model_params["random_state"] = seed
        model_params["n_estimators"] = int(n_estimators)

        model = LGBMRegressor(**model_params)

        if use_log_target:
            y_fit = np.log1p(y)
            model.fit(X_train, y_fit)
            pred = np.expm1(model.predict(X_test))
        else:
            model.fit(X_train, y)
            pred = model.predict(X_test)

        if CLIP_PREDICTIONS_AT_ZERO:
            pred = np.maximum(pred, 0.0)

        preds_seed.append(pred.astype(np.float32))

    return np.median(np.vstack(preds_seed), axis=0)


def optimize_blend_weight(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray) -> Tuple[float, float]:
    """
    Find w in [0,1] minimizing MAE of blend = w*pred_a + (1-w)*pred_b.
    Simple grid search is fine for 2k rows.
    """
    best_w = 0.5
    best_mae = float("inf")
    for w in np.linspace(0, 1, 101):
        blend = w * pred_a + (1.0 - w) * pred_b
        m = mae(y_true, blend)
        if m < best_mae:
            best_mae = m
            best_w = float(w)
    return best_w, best_mae


# =========================
# MAIN
# =========================
def main() -> None:
    setup_logging()
    logging.info("=== AgentDS Challenge 2: ED Cost Forecasting (Multimodal Specialist) ===")

    # Load train/test
    if not TRAIN_CSV.exists():
        raise FileNotFoundError(f"Train CSV not found: {TRAIN_CSV}")
    if not TEST_CSV.exists():
        raise FileNotFoundError(f"Test CSV not found: {TEST_CSV}")
    if not PDF_DIR.exists():
        raise FileNotFoundError(f"PDF directory not found: {PDF_DIR}")

    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)

    required_train_cols = {"patient_id", "primary_chronic", "prior_ed_visits_5y", "prior_ed_cost_5y_usd", "ed_cost_next3y_usd"}
    required_test_cols = {"patient_id", "primary_chronic", "prior_ed_visits_5y", "prior_ed_cost_5y_usd"}
    if not required_train_cols.issubset(train_df.columns):
        raise ValueError(f"Train CSV missing columns: {required_train_cols - set(train_df.columns)}")
    if not required_test_cols.issubset(test_df.columns):
        raise ValueError(f"Test CSV missing columns: {required_test_cols - set(test_df.columns)}")

    logging.info(f"Train shape: {train_df.shape} | Test shape: {test_df.shape}")

    # Optional patients.csv
    if USE_PATIENTS_TABLE_IF_AVAILABLE and PATIENTS_CSV.exists():
        patients = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in patients.columns:
            logging.info("Merging patients.csv features (age/sex/insurance/zip3).")
            train_df = train_df.merge(patients, on="patient_id", how="left")
            test_df = test_df.merge(patients, on="patient_id", how="left")
        else:
            logging.warning("patients.csv exists but has no patient_id column; skipping.")
    else:
        logging.info("patients.csv not used (not found or disabled). Will rely on receipt header features if present.")

    # Build receipt cache/artifacts for all patients
    all_patient_ids = pd.concat([train_df["patient_id"], test_df["patient_id"]]).astype(int).unique().tolist()
    all_patient_ids.sort()

    receipt_meta_df, tok_count_map, tok_spend_map, tok_qty_map, doc_text_map, line_items_df = build_receipt_artifacts(
        patient_ids=all_patient_ids,
        pdf_dir=PDF_DIR,
        cache_path=CACHE_PATH,
    )

    # Merge receipt meta
    train_df = train_df.merge(receipt_meta_df, on="patient_id", how="left")
    test_df = test_df.merge(receipt_meta_df, on="patient_id", how="left")

    # If patients.csv missing zip3/insurance, receipt header might provide pdf_zip3/pdf_insurance
    # (We keep both; model will decide. If both exist, they're usually identical.)
    for col in ["pdf_zip3", "pdf_insurance"]:
        if col in train_df.columns:
            train_df[col] = train_df[col].astype("string")
            test_df[col] = test_df[col].astype("string")

    # Combine train+test for feature building
    train_df["is_train"] = 1
    test_df["is_train"] = 0
    df_all = pd.concat([train_df, test_df], ignore_index=True)

    # Add explicit top-code features (counts + spend shares)
    if USE_TOP_CODE_FEATURES:
        df_all = add_top_code_features(
            df_all=df_all,
            line_items_df=line_items_df,
            train_patient_ids=train_df["patient_id"].astype(int).values,
            top_k=TOP_K_CODES,
        )

    # Build combined feature matrix
    logging.info("Building feature matrix...")
    X_all, artifacts = build_feature_matrix(
        df_all=df_all,
        tok_count_map=tok_count_map,
        tok_spend_map=tok_spend_map,
        tok_qty_map=tok_qty_map,
        doc_text_map=doc_text_map,
    )
    logging.info(f"Feature matrix built: shape={X_all.shape} | n_features={artifacts.get('n_features')}")

    # Split back
    is_train_mask = df_all["is_train"].values.astype(int) == 1
    X_train = X_all[is_train_mask]
    X_test = X_all[~is_train_mask]
    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)

    # Strat labels for CV
    strat_labels = make_strat_labels(train_df)

    # CV OOF + best iterations
    logging.info("Running CV (raw target)...")
    oof_raw, best_iters_raw, mae_raw = cv_oof_and_best_iters(
        X_train=X_train,
        y=y,
        strat_labels=strat_labels,
        params=LGBM_BASE_PARAMS,
        seeds=SEEDS,
        use_log_target=False,
    )
    logging.info(f"OOF MAE (raw target): {mae_raw:.4f}")

    if TRAIN_BOTH_RAW_AND_LOG:
        logging.info("Running CV (log1p target)...")
        oof_log, best_iters_log, mae_log = cv_oof_and_best_iters(
            X_train=X_train,
            y=y,
            strat_labels=strat_labels,
            params=LGBM_BASE_PARAMS,
            seeds=SEEDS,
            use_log_target=True,
        )
        logging.info(f"OOF MAE (log1p target, evaluated in USD): {mae_log:.4f}")

        # Blend OOF
        w, mae_blend = optimize_blend_weight(y_true=y, pred_a=oof_raw, pred_b=oof_log)
        oof_blend = w * oof_raw + (1.0 - w) * oof_log

        # Bias calibration (MAE-optimal constant shift is median residual)
        bias = float(np.median(y - oof_blend))
        oof_blend_cal = oof_blend + bias
        mae_blend_cal = mae(y, oof_blend_cal)

        logging.info(f"Blend weight w(raw)={w:.2f} | OOF MAE (blend): {mae_blend:.4f} | bias={bias:.4f} | OOF MAE (blend+bias): {mae_blend_cal:.4f}")

        # Full-data training iterations
        n_est_raw = int(np.median(best_iters_raw)) if best_iters_raw else int(LGBM_BASE_PARAMS["n_estimators"])
        n_est_log = int(np.median(best_iters_log)) if best_iters_log else int(LGBM_BASE_PARAMS["n_estimators"])
        logging.info(f"Full fit n_estimators: raw={n_est_raw} | log={n_est_log}")

        # Fit full and predict test
        logging.info("Fitting full-data models and predicting test (raw)...")
        test_pred_raw = fit_full_and_predict(
            X_train=X_train, y=y, X_test=X_test,
            params=LGBM_BASE_PARAMS, seeds=SEEDS,
            n_estimators=n_est_raw, use_log_target=False,
        )

        logging.info("Fitting full-data models and predicting test (log1p)...")
        test_pred_log = fit_full_and_predict(
            X_train=X_train, y=y, X_test=X_test,
            params=LGBM_BASE_PARAMS, seeds=SEEDS,
            n_estimators=n_est_log, use_log_target=True,
        )

        test_pred = w * test_pred_raw + (1.0 - w) * test_pred_log + bias

    else:
        # Use raw only
        bias = float(np.median(y - oof_raw))
        logging.info(f"Bias calibration (raw-only): bias={bias:.4f}")

        n_est_raw = int(np.median(best_iters_raw)) if best_iters_raw else int(LGBM_BASE_PARAMS["n_estimators"])
        logging.info(f"Full fit n_estimators (raw-only): {n_est_raw}")

        logging.info("Fitting full-data models and predicting test (raw-only)...")
        test_pred_raw = fit_full_and_predict(
            X_train=X_train, y=y, X_test=X_test,
            params=LGBM_BASE_PARAMS, seeds=SEEDS,
            n_estimators=n_est_raw, use_log_target=False,
        )
        test_pred = test_pred_raw + bias

    if CLIP_PREDICTIONS_AT_ZERO:
        test_pred = np.maximum(test_pred, 0.0)

    # Write submission
    submission = pd.DataFrame(
        {
            "patient_id": test_df["patient_id"].values,
            "ed_cost_next3y_usd": test_pred.astype(np.float32),
        }
    )

    # Ensure exact patient_id order as test
    submission["patient_id"] = submission["patient_id"].astype(int)

    SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(SUBMISSION_PATH, index=False)

    logging.info(f"Saved submission to: {SUBMISSION_PATH}")
    logging.info("Done.")


if __name__ == "__main__":
    main()


2026-02-13 01:00:58,541 | INFO | === AgentDS Challenge 2: ED Cost Forecasting (Multimodal Specialist) ===
2026-02-13 01:00:58,579 | INFO | Train shape: (2000, 5) | Test shape: (2000, 4)
2026-02-13 01:00:58,599 | INFO | Merging patients.csv features (age/sex/insurance/zip3).
2026-02-13 01:00:58,617 | INFO | Parsing 4,000 receipts (cache miss).
Parsing receipts: 100%|█████████████████████████████████████████| 4000/4000 [00:41<00:00, 95.30it/s]
2026-02-13 01:01:41,538 | INFO | Adding top-code features for TOP_K_CODES=18
2026-02-13 01:01:41,580 | INFO | Building feature matrix...
2026-02-13 01:01:42,084 | INFO | Feature matrix built: shape=(4000, 16539) | n_features=16539
2026-02-13 01:01:42,100 | INFO | Running CV (raw target)...
2026-02-13 01:02:06,078 | INFO | OOF MAE (raw target): 474.2093
2026-02-13 01:02:06,080 | INFO | Running CV (log1p target)...
2026-02-13 01:02:32,265 | INFO | OOF MAE (log1p target, evaluated in USD): 472.3089
2026-02-13 01:02:32,271 | INFO | Blend weight w(raw)=

# Iteration 2
-  478.3755

In [10]:
# -*- coding: utf-8 -*-
r"""
ITERATION 2 (GPU-Fast) — Observability / Heartbeats Edition
===========================================================

Changes requested:
1) Fix Regex Warnings: all regex patterns use raw strings r"..."
2) Training Heartbeats:
   - XGBoost: verbosity=1 and print every 100 rounds via fit(verbose=100) (or verbose_eval if supported)
   - CatBoost: verbose=100 in params (prints every 100 iterations)
   - logging.info before/after each model.fit() per fold
3) Preserve Logic: NO changes to model architecture, hyperparameters (except verbosity), or seed logic.

===========================================================
INSTALL (run once)
===========================================================
pip install -U pandas numpy scipy scikit-learn xgboost catboost pymupdf joblib tqdm

Paths:
D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv
D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv
D:\AgentDs\agent_ds_healthcare\patients.csv (optional)
D:\AgentDs\agent_ds_healthcare\receipts_pdf\receipt_{patient_id}.pdf

Output:
D:\AgentDs\agent_ds_healthcare\submission.csv
===========================================================
"""

from __future__ import annotations

import re
import math
import time
import logging
import warnings
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

import xgboost as xgb
from xgboost import XGBRegressor

from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# CONFIG (PRESERVED)
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"          # optional
PDF_DIR = BASE_DIR / "receipts_pdf"
SUBMISSION_PATH = BASE_DIR / "submission.csv"

CACHE_PATH = BASE_DIR / "cache_receipts_iter2_gpu_fast.joblib"
CACHE_VERSION = 21

# Speed / rigor tradeoff (PRESERVED)
FAST_MODE = True   # True -> 2 repeats (10 folds). False -> 3 repeats (15 folds)

N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026] if FAST_MODE else [2024, 2025, 2026]
EARLY_STOPPING = 200

# Target encoding (PRESERVED)
TE_SMOOTH = 20.0
TE_MIN_COUNT_MEDIAN = 5

# Use GPU (PRESERVED)
XGB_DEVICE = "cuda"       # use "cuda:0" if you want explicit device
CAT_TASK_TYPE = "GPU"     # will auto-fallback to CPU if GPU fails

# Prediction post-process (PRESERVED)
CLIP_AT_ZERO = True

# Blending (PRESERVED)
BLEND_STEP = 0.02


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


# =========================
# HELPERS
# =========================
_money_re = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d{2})$|^\d+(?:\.\d{2})$")
_code_re = re.compile(r"^(?:[A-Z]\d{4}|\d{4,5})$")

def is_money(tok: str) -> bool:
    return bool(_money_re.match(tok))

def money_to_float(tok: str) -> float:
    return float(tok.replace(",", ""))

def is_code(tok: str) -> bool:
    return bool(_code_re.match(tok))

def safe_log1p(x: float) -> float:
    return math.log1p(max(0.0, float(x)))

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))

def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


# =========================
# PDF PARSING
# =========================
def cluster_words_into_lines(words: List[Tuple], y_tol: float = 1.5) -> List[List[str]]:
    if not words:
        return []
    words_sorted = sorted(words, key=lambda w: (w[1], w[0]))
    lines: List[List[str]] = []
    cur: List[Tuple[float, str]] = []
    cur_y: Optional[float] = None

    for w in words_sorted:
        x0, y0 = float(w[0]), float(w[1])
        tok = str(w[4])
        if cur_y is None or abs(y0 - cur_y) <= y_tol:
            cur.append((x0, tok))
            cur_y = y0 if cur_y is None else (0.85 * cur_y + 0.15 * y0)
        else:
            lines.append([t for _, t in sorted(cur, key=lambda z: z[0])])
            cur = [(x0, tok)]
            cur_y = y0
    if cur:
        lines.append([t for _, t in sorted(cur, key=lambda z: z[0])])
    return lines


def parse_header_fields(text: str) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    pid = None
    zip3 = None
    insurance = None

    m = re.search(r"Patient ID:\s*(\d+)", text)
    if m:
        pid = int(m.group(1))

    m = re.search(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)", text)
    if m:
        zip3 = str(m.group(1))
        insurance = str(m.group(2)).lower()

    return pid, zip3, insurance


def parse_items_from_words(words: List[Tuple]) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None
    lines = cluster_words_into_lines(words, y_tol=1.5)
    if not lines:
        return items, total

    header_idx = None
    for i, toks in enumerate(lines):
        low = [t.lower() for t in toks]
        if ("code" in low) and ("description" in low) and ("qty" in low):
            header_idx = i
            break
    if header_idx is None:
        return items, total

    for toks in lines[header_idx + 1:]:
        if not toks:
            continue
        if toks[0].upper() == "TOTAL":
            for t in reversed(toks[1:]):
                if is_money(t):
                    total = money_to_float(t)
                    break
            break

        if len(toks) >= 5 and is_code(toks[0]) and toks[-3].isdigit() and is_money(toks[-2]) and is_money(toks[-1]):
            items.append({
                "code": toks[0],
                "qty": int(toks[-3]),
                "unit": money_to_float(toks[-2]),
                "line_total": money_to_float(toks[-1]),
                "desc": " ".join(toks[1:-3]),
            })
    return items, total


def parse_items_fallback_text(text: str) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None
    lines = [ln.strip() for ln in (text.splitlines() if text else []) if ln.strip()]
    if not lines:
        return items, total

    start_idx = None
    for i in range(len(lines) - 4):
        if lines[i].lower() == "code" and lines[i+1].lower() == "description" and lines[i+2].lower() == "qty":
            start_idx = i + 5
            break
        if "code" in lines[i].lower() and "description" in lines[i].lower() and "qty" in lines[i].lower():
            start_idx = i + 1
            break
    if start_idx is None:
        return items, total

    i = start_idx
    while i < len(lines):
        if lines[i].upper().startswith("TOTAL"):
            parts = lines[i].split()
            if len(parts) >= 2 and is_money(parts[-1]):
                total = money_to_float(parts[-1])
            elif i+1 < len(lines) and is_money(lines[i+1]):
                total = money_to_float(lines[i+1])
            break

        if i+4 < len(lines) and is_code(lines[i]) and lines[i+2].isdigit() and is_money(lines[i+3]) and is_money(lines[i+4]):
            items.append({
                "code": lines[i],
                "desc": lines[i+1],
                "qty": int(lines[i+2]),
                "unit": money_to_float(lines[i+3]),
                "line_total": money_to_float(lines[i+4]),
            })
            i += 5
        else:
            i += 1
    return items, total


@dataclass
class ReceiptProfile:
    meta: Dict[str, Any]
    code_spend: Dict[str, float]


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptProfile:
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"
    empty_meta = dict(
        pdf_ok=0, pdf_n_lines=0, pdf_n_unique_codes=0,
        pdf_sum_line_total=0.0, pdf_top1_cost_share=0.0, pdf_top3_cost_share=0.0,
        pdf_repeat_cost_share=0.0, pdf_total=0.0, pdf_total_diff=0.0,
        pdf_insurance=None, pdf_zip3=None, pdf_parse_notes="missing_pdf",
    )

    if not pdf_path.exists():
        return ReceiptProfile(meta=empty_meta, code_spend={})

    try:
        with fitz.open(pdf_path) as doc:
            full_text = ""
            items: List[Dict[str, Any]] = []
            total: Optional[float] = None
            zip3 = None
            insurance = None

            for page in doc:
                text = page.get_text("text") or ""
                full_text += text + "\n"
                words = page.get_text("words") or []

                items_p, total_p = parse_items_from_words(words)
                if not items_p:
                    items_p, total_p = parse_items_fallback_text(text)

                if items_p:
                    items.extend(items_p)
                if total_p is not None:
                    total = total_p

            _, zip3, insurance = parse_header_fields(full_text)

    except Exception as e:
        empty_meta["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return ReceiptProfile(meta=empty_meta, code_spend={})

    if not items:
        empty_meta["pdf_parse_notes"] = "no_items_parsed"
        empty_meta["pdf_zip3"] = zip3
        empty_meta["pdf_insurance"] = insurance
        return ReceiptProfile(meta=empty_meta, code_spend={})

    code_spend: Dict[str, float] = {}
    cnt_by_code: Dict[str, int] = {}
    for it in items:
        c = str(it["code"])
        code_spend[c] = code_spend.get(c, 0.0) + float(it["line_total"])
        cnt_by_code[c] = cnt_by_code.get(c, 0) + 1

    line_totals = np.array([float(it["line_total"]) for it in items], dtype=float)
    sum_line_total = float(line_totals.sum())
    pdf_total = float(total) if total is not None else sum_line_total

    sorted_cost = np.sort(line_totals)[::-1]
    top1 = float(sorted_cost[0] / sum_line_total) if sum_line_total > 0 else 0.0
    top3 = float(sorted_cost[:3].sum() / sum_line_total) if sum_line_total > 0 else 0.0

    repeat_spend = sum(v for c, v in code_spend.items() if cnt_by_code.get(c, 0) >= 2)
    repeat_share = float(repeat_spend / sum_line_total) if sum_line_total > 0 else 0.0

    meta = dict(
        pdf_ok=1,
        pdf_n_lines=len(items),
        pdf_n_unique_codes=len(code_spend),
        pdf_sum_line_total=sum_line_total,
        pdf_top1_cost_share=top1,
        pdf_top3_cost_share=top3,
        pdf_repeat_cost_share=repeat_share,
        pdf_total=pdf_total,
        pdf_total_diff=float(abs(sum_line_total - pdf_total)),
        pdf_insurance=(insurance.lower() if isinstance(insurance, str) else None),
        pdf_zip3=(str(zip3) if zip3 is not None else None),
        pdf_parse_notes="ok",
    )

    return ReceiptProfile(meta=meta, code_spend=code_spend)


def load_cache(cache_path: Path) -> Dict[str, Any]:
    if cache_path.exists():
        try:
            obj = joblib.load(cache_path)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "data" in obj:
                return obj
        except Exception:
            pass
    return {"version": CACHE_VERSION, "data": {}}


def save_cache(cache_path: Path, cache_obj: Dict[str, Any]) -> None:
    try:
        joblib.dump(cache_obj, cache_path)
    except Exception as e:
        logging.warning(f"Cache save failed: {e}")


def build_receipt_artifacts(patient_ids: List[int], pdf_dir: Path, cache_path: Path):
    cache_obj = load_cache(cache_path)
    data: Dict[str, Any] = cache_obj.get("data", {})

    to_parse = [pid for pid in patient_ids if str(pid) not in data]
    if to_parse:
        logging.info(f"Parsing {len(to_parse):,} receipts (cache miss).")

    for pid in tqdm(to_parse, desc="Parsing receipts", ncols=100):
        prof = featurize_receipt(pid, pdf_dir)
        data[str(pid)] = {"meta": prof.meta, "code_spend": prof.code_spend}

    cache_obj["data"] = data
    if to_parse:
        save_cache(cache_path, cache_obj)

    meta_rows = []
    profiles: Dict[int, Dict[str, Any]] = {}
    for pid in patient_ids:
        obj = data.get(str(pid), None)
        if obj is None:
            prof = featurize_receipt(pid, pdf_dir)
            obj = {"meta": prof.meta, "code_spend": prof.code_spend}
        meta_rows.append({"patient_id": pid, **obj["meta"]})
        profiles[pid] = obj

    return pd.DataFrame(meta_rows), profiles


# =========================
# BASE FEATURES (OHE for XGB)
# =========================
def build_base_sparse_matrix(df_all: pd.DataFrame) -> Tuple[csr_matrix, Dict[str, Any]]:
    df = df_all.copy()

    # derived tabular
    df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
    df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
    df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("missing")

    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train"}
    cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                if c in df.columns]
    num_cols = [c for c in df.columns if c not in exclude and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    X = preprocessor.fit_transform(df)
    X = X.tocsr() if hasattr(X, "tocsr") else csr_matrix(X)
    return X, {"preprocessor": preprocessor, "num_cols": num_cols, "cat_cols": cat_cols}


# =========================
# TARGET ENCODING (CV-SAFE)
# =========================
@dataclass
class TokenStats:
    cnt: int
    mean_smooth: float
    median_adj: float

def code_to_prefix2(code: str) -> str:
    return code[:2] if len(code) >= 2 else code

def build_token_stats_for_fold(
    train_pids: List[int],
    y_log: np.ndarray,
    pid_to_row: Dict[int, int],
    profiles: Dict[int, Dict[str, Any]],
) -> Tuple[Dict[str, TokenStats], Dict[str, TokenStats], float, float]:
    vals_code: Dict[str, List[float]] = {}
    vals_p2: Dict[str, List[float]] = {}
    ys = []

    for pid in train_pids:
        r = pid_to_row.get(pid, None)
        if r is None:
            continue
        v = float(y_log[r])
        ys.append(v)
        code_spend = profiles.get(pid, {}).get("code_spend", {}) or {}
        for c in code_spend.keys():
            vals_code.setdefault(c, []).append(v)
            vals_p2.setdefault(code_to_prefix2(c), []).append(v)

    gmean = float(np.mean(ys)) if ys else 0.0
    gmed = float(np.median(ys)) if ys else 0.0

    def to_stats(m: Dict[str, List[float]]) -> Dict[str, TokenStats]:
        out = {}
        for tok, arr in m.items():
            a = np.asarray(arr, dtype=float)
            cnt = int(a.size)
            mean = float(np.mean(a))
            med = float(np.median(a))
            mean_smooth = float((cnt * mean + TE_SMOOTH * gmean) / (cnt + TE_SMOOTH))
            median_adj = med if cnt >= TE_MIN_COUNT_MEDIAN else gmed
            out[tok] = TokenStats(cnt=cnt, mean_smooth=mean_smooth, median_adj=median_adj)
        return out

    return to_stats(vals_code), to_stats(vals_p2), gmean, gmed


def te_features_for_patients(
    pids: List[int],
    profiles: Dict[int, Dict[str, Any]],
    stats_code: Dict[str, TokenStats],
    stats_p2: Dict[str, TokenStats],
    gmean: float,
    gmed: float,
) -> np.ndarray:
    """
    16 TE features: exact code (8) + prefix2 (8), spend-weighted.
    """
    feats = np.zeros((len(pids), 16), dtype=np.float32)

    for i, pid in enumerate(pids):
        code_spend = profiles.get(pid, {}).get("code_spend", {}) or {}
        if not code_spend:
            base = np.array([gmean, gmed, gmean, gmean, gmean, gmed, 0.0, 0.0], dtype=np.float32)
            feats[i, 0:8] = base
            feats[i, 8:16] = base
            continue

        codes = list(code_spend.keys())
        spends = np.array([float(code_spend[c]) for c in codes], dtype=float)
        ssum = float(spends.sum())
        w = (np.ones_like(spends) / max(1, spends.size)) if ssum <= 0 else (spends / (ssum + 1e-9))

        # exact
        means = []
        meds = []
        cnts = []
        known = 0
        for c in codes:
            st = stats_code.get(c)
            if st is None:
                means.append(gmean); meds.append(gmed); cnts.append(0)
            else:
                means.append(st.mean_smooth); meds.append(st.median_adj); cnts.append(st.cnt); known += 1
        means = np.asarray(means, dtype=float)
        meds = np.asarray(meds, dtype=float)
        cnts = np.asarray(cnts, dtype=float)
        top = int(np.argmax(w)) if w.size else 0

        feats[i, 0:8] = np.array([
            float(np.sum(w * means)),
            float(np.sum(w * meds)),
            float(np.mean(means)),
            float(np.max(means)),
            float(np.min(means)),
            float(meds[top] if meds.size else gmed),
            float(known / max(1, len(codes))),
            float(np.mean(np.log1p(cnts))) if cnts.size else 0.0,
        ], dtype=np.float32)

        # prefix2
        p2_map: Dict[str, float] = {}
        for c, sp in code_spend.items():
            p2 = code_to_prefix2(c)
            p2_map[p2] = p2_map.get(p2, 0.0) + float(sp)
        p2s = list(p2_map.keys())
        p2sp = np.asarray([p2_map[p] for p in p2s], dtype=float)
        p2sum = float(p2sp.sum())
        w2 = (np.ones_like(p2sp) / max(1, p2sp.size)) if p2sum <= 0 else (p2sp / (p2sum + 1e-9))

        means2 = []
        meds2 = []
        cnts2 = []
        known2 = 0
        for p in p2s:
            st = stats_p2.get(p)
            if st is None:
                means2.append(gmean); meds2.append(gmed); cnts2.append(0)
            else:
                means2.append(st.mean_smooth); meds2.append(st.median_adj); cnts2.append(st.cnt); known2 += 1
        means2 = np.asarray(means2, dtype=float)
        meds2 = np.asarray(meds2, dtype=float)
        cnts2 = np.asarray(cnts2, dtype=float)
        top2 = int(np.argmax(w2)) if w2.size else 0

        feats[i, 8:16] = np.array([
            float(np.sum(w2 * means2)),
            float(np.sum(w2 * meds2)),
            float(np.mean(means2)),
            float(np.max(means2)),
            float(np.min(means2)),
            float(meds2[top2] if meds2.size else gmed),
            float(known2 / max(1, len(p2s))),
            float(np.mean(np.log1p(cnts2))) if cnts2.size else 0.0,
        ], dtype=np.float32)

    return feats


def make_strat_labels(train_df: pd.DataFrame) -> np.ndarray:
    chronic = train_df["primary_chronic"].astype(str)
    cost_bin = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    return (chronic + "_" + cost_bin).values


# =========================
# BLEND
# =========================
def best_blend_weight(y_true: np.ndarray, p1: np.ndarray, p2: np.ndarray, step: float) -> Tuple[float, float]:
    best_w = 0.5
    best_m = 1e18
    for w in np.arange(0.0, 1.0 + 1e-9, step):
        blend = w * p1 + (1.0 - w) * p2
        m = mae(y_true, blend)
        if m < best_m:
            best_m = m
            best_w = float(w)
    return best_w, best_m


# =========================
# MAIN
# =========================
def main() -> None:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb, '__version__', 'unknown')}")
    logging.info(f"FAST_MODE={FAST_MODE} | repeats={len(REPEAT_SEEDS)} | folds={N_FOLDS}")

    # load
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)

    # optional patients
    if PATIENTS_CSV.exists():
        patients = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in patients.columns:
            train_df = train_df.merge(patients, on="patient_id", how="left")
            test_df = test_df.merge(patients, on="patient_id", how="left")
            logging.info("Merged patients.csv")

    train_df["is_train"] = 1
    test_df["is_train"] = 0
    df_all = pd.concat([train_df, test_df], ignore_index=True)

    # receipts (cached)
    all_pids = df_all["patient_id"].astype(int).unique().tolist()
    all_pids.sort()
    receipt_meta_df, profiles = build_receipt_artifacts(all_pids, PDF_DIR, CACHE_PATH)
    df_all = df_all.merge(receipt_meta_df, on="patient_id", how="left")

    # base sparse (OHE) for XGB
    X_all_base, _ = build_base_sparse_matrix(df_all)
    is_train = df_all["is_train"].values.astype(int) == 1
    X_train_base = X_all_base[is_train]
    X_test_base = X_all_base[~is_train]

    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)
    y_log = np.log1p(y)

    train_pids = train_df["patient_id"].astype(int).tolist()
    test_pids = test_df["patient_id"].astype(int).tolist()
    pid_to_row = {pid: i for i, pid in enumerate(train_pids)}

    # CatBoost base frames
    df_cb_all = df_all.drop(columns=["ed_cost_next3y_usd"], errors="ignore").copy()
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df_cb_all.columns:
            df_cb_all[c] = df_cb_all[c].astype(str).fillna("missing")

    df_cb_all["prior_cost_per_visit"] = df_cb_all["prior_ed_cost_5y_usd"] / (df_cb_all["prior_ed_visits_5y"] + 1.0)
    df_cb_all["log_prior_cost"] = np.log1p(df_cb_all["prior_ed_cost_5y_usd"].astype(float))
    df_cb_all["log_prior_visits"] = np.log1p(df_cb_all["prior_ed_visits_5y"].astype(float))

    df_train_cb = df_cb_all[is_train].reset_index(drop=True)
    df_test_cb = df_cb_all[~is_train].reset_index(drop=True)

    cb_cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                   if c in df_train_cb.columns]
    cb_cat_idx = [df_train_cb.columns.get_loc(c) for c in cb_cat_cols]

    # CV splits
    strat = make_strat_labels(train_df)
    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), strat)):
            splits.append((rep, fold, tr_idx, va_idx))
    logging.info(f"Total splits: {len(splits)}")

    # storage
    oof_xgb = np.zeros(len(y), dtype=np.float64)
    oof_cat = np.zeros(len(y), dtype=np.float64)
    cnt_xgb = np.zeros(len(y), dtype=np.float64)
    cnt_cat = np.zeros(len(y), dtype=np.float64)

    test_xgb_sum = np.zeros(len(test_df), dtype=np.float64)
    test_cat_sum = np.zeros(len(test_df), dtype=np.float64)
    n_xgb = 0
    n_cat = 0

    # ===== Model params (PRESERVED except verbosity/verbose) =====
    xgb_params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=XGB_DEVICE,     # GPU ON
        n_estimators=6000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_lambda=1.0,
        min_child_weight=1.0,
        n_jobs=-1,
        verbosity=1,  # CHANGED for heartbeat / observability (was 0)
        early_stopping_rounds=EARLY_STOPPING,
    )

    cat_params = dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=8000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=6.0,
        bagging_temperature=0.8,
        random_strength=0.5,
        od_type="Iter",
        od_wait=EARLY_STOPPING,
        task_type=CAT_TASK_TYPE,
        devices="0",
        verbose=100,  # CHANGED for heartbeat / observability (was False)
    )

    te_cols = [f"te_{i:02d}" for i in range(16)]
    verbose_eval = 100  # requested "print every 100 rounds"

    # CV loop
    for rep, fold, tr_idx, va_idx in tqdm(splits, desc="CV folds", ncols=110):
        t0 = time.time()
        fold_tag = f"rep={rep} fold={fold}"

        logging.info(f"{fold_tag} | Starting fold | train={len(tr_idx)} val={len(va_idx)}")

        tr_pids = [train_pids[i] for i in tr_idx.tolist()]
        va_pids = [train_pids[i] for i in va_idx.tolist()]

        logging.info(f"{fold_tag} | Building TE stats...")
        stats_code, stats_p2, gmean, gmed = build_token_stats_for_fold(tr_pids, y_log, pid_to_row, profiles)

        logging.info(f"{fold_tag} | Building TE feature blocks (train/val/test)...")
        te_tr = te_features_for_patients(tr_pids, profiles, stats_code, stats_p2, gmean, gmed)
        te_va = te_features_for_patients(va_pids, profiles, stats_code, stats_p2, gmean, gmed)
        te_te = te_features_for_patients(test_pids, profiles, stats_code, stats_p2, gmean, gmed)

        # XGB matrices (sparse)
        X_tr = hstack([X_train_base[tr_idx], csr_matrix(te_tr)], format="csr")
        X_va = hstack([X_train_base[va_idx], csr_matrix(te_va)], format="csr")
        X_te = hstack([X_test_base, csr_matrix(te_te)], format="csr")

        y_tr = y[tr_idx]
        y_va = y[va_idx]

        # ---- XGBoost GPU ----
        logging.info(f"{fold_tag} | Starting XGBoost training | device={xgb_params.get('device')} | verbose_eval={verbose_eval}")
        try:
            xgb_model = XGBRegressor(**xgb_params, random_state=1000 * rep + fold)

            # "verbose_eval" is not always a sklearn-fit arg; use it if present, else use verbose=int
            fit_params = inspect.signature(xgb_model.fit).parameters
            fit_kwargs = {"eval_set": [(X_va, y_va)]}

            if "verbose_eval" in fit_params:
                fit_kwargs["verbose_eval"] = verbose_eval
            else:
                # XGBoost sklearn API: verbose=int prints evaluation metric every n boosting stages
                fit_kwargs["verbose"] = verbose_eval

            xgb_model.fit(X_tr, y_tr, **fit_kwargs)

            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_te)

            best_it = getattr(xgb_model, "best_iteration", None)
            logging.info(f"{fold_tag} | Finished XGBoost training | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | XGBoost GPU failed: {e} -> fallback CPU")
            xgb_cpu = dict(xgb_params)
            xgb_cpu["device"] = "cpu"
            xgb_model = XGBRegressor(**xgb_cpu, random_state=1000 * rep + fold)

            fit_params = inspect.signature(xgb_model.fit).parameters
            fit_kwargs = {"eval_set": [(X_va, y_va)]}
            if "verbose_eval" in fit_params:
                fit_kwargs["verbose_eval"] = verbose_eval
            else:
                fit_kwargs["verbose"] = verbose_eval

            logging.info(f"{fold_tag} | Starting XGBoost training (CPU fallback) | verbose_eval={verbose_eval}")
            xgb_model.fit(X_tr, y_tr, **fit_kwargs)
            logging.info(f"{fold_tag} | Finished XGBoost training (CPU fallback)")

            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_te)

        if CLIP_AT_ZERO:
            p_va_x = np.maximum(p_va_x, 0.0)
            p_te_x = np.maximum(p_te_x, 0.0)

        oof_xgb[va_idx] += p_va_x
        cnt_xgb[va_idx] += 1.0
        test_xgb_sum += p_te_x
        n_xgb += 1

        # ---- CatBoost GPU ----
        logging.info(f"{fold_tag} | Preparing CatBoost pools...")
        te_train_full = np.zeros((len(train_df), 16), dtype=np.float32)
        te_train_full[tr_idx, :] = te_tr
        te_train_full[va_idx, :] = te_va

        df_tr = df_train_cb.copy()
        df_te = df_test_cb.copy()
        for j, col in enumerate(te_cols):
            df_tr[col] = te_train_full[:, j]
            df_te[col] = te_te[:, j]

        tr_pool = Pool(df_tr.iloc[tr_idx], label=y_tr, cat_features=cb_cat_idx)
        va_pool = Pool(df_tr.iloc[va_idx], label=y_va, cat_features=cb_cat_idx)
        te_pool = Pool(df_te, cat_features=cb_cat_idx)

        logging.info(f"{fold_tag} | Starting CatBoost training | task_type={cat_params.get('task_type')} | verbose={cat_params.get('verbose')}")
        try:
            cat_model = CatBoostRegressor(**cat_params, random_seed=2000 * rep + fold)
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)

            best_it = None
            try:
                best_it = cat_model.get_best_iteration()
            except Exception:
                best_it = None
            logging.info(f"{fold_tag} | Finished CatBoost training | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | CatBoost GPU failed: {e} -> fallback CPU")
            cat_cpu = dict(cat_params)
            cat_cpu["task_type"] = "CPU"
            cat_cpu.pop("devices", None)
            cat_model = CatBoostRegressor(**cat_cpu, random_seed=2000 * rep + fold)

            logging.info(f"{fold_tag} | Starting CatBoost training (CPU fallback) | verbose={cat_cpu.get('verbose')}")
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
            logging.info(f"{fold_tag} | Finished CatBoost training (CPU fallback)")

        p_va_c = cat_model.predict(va_pool)
        p_te_c = cat_model.predict(te_pool)

        if CLIP_AT_ZERO:
            p_va_c = np.maximum(p_va_c, 0.0)
            p_te_c = np.maximum(p_te_c, 0.0)

        oof_cat[va_idx] += p_va_c
        cnt_cat[va_idx] += 1.0
        test_cat_sum += p_te_c
        n_cat += 1

        dt = time.time() - t0
        logging.info(f"{fold_tag} | Fold complete in {dt:.1f}s")

    # finalize oof/test
    oof_x = (oof_xgb / np.maximum(cnt_xgb, 1.0)).astype(np.float32)
    oof_c = (oof_cat / np.maximum(cnt_cat, 1.0)).astype(np.float32)
    te_x = (test_xgb_sum / max(1, n_xgb)).astype(np.float32)
    te_c = (test_cat_sum / max(1, n_cat)).astype(np.float32)

    mae_x = mae(y, oof_x)
    mae_c = mae(y, oof_c)
    logging.info(f"OOF MAE | XGB: {mae_x:.4f}")
    logging.info(f"OOF MAE | CAT: {mae_c:.4f}")

    # blend weight
    w, blend_mae = best_blend_weight(y, oof_x, oof_c, step=BLEND_STEP)
    oof_blend = w * oof_x + (1.0 - w) * oof_c
    bias = float(np.median(y - oof_blend))  # MAE-optimal constant shift
    oof_final = oof_blend + bias
    test_final = w * te_x + (1.0 - w) * te_c + bias

    if CLIP_AT_ZERO:
        oof_final = np.maximum(oof_final, 0.0)
        test_final = np.maximum(test_final, 0.0)

    logging.info(f"Blend w(XGB)={w:.2f} | OOF MAE pre-bias={blend_mae:.4f} | bias={bias:.4f} | OOF MAE final={mae(y, oof_final):.4f}")

    # save submission
    sub = pd.DataFrame({
        "patient_id": test_df["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": test_final.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")


if __name__ == "__main__":
    main()


2026-02-13 03:51:29,701 | INFO | XGBoost version: 3.0.0
2026-02-13 03:51:29,703 | INFO | FAST_MODE=True | repeats=2 | folds=5
2026-02-13 03:51:29,715 | INFO | Merged patients.csv
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 03:51:29,942 | INFO | Total splits: 10
CV folds:   0%|                                                                        | 0/10 [00:00<?, ?it/s]2026-02-13 03:51:29,942 | INFO | rep=0 fold=0 | Starting fold | train=1600 val=400
2026-02-13 03:51:29,945 | INFO | rep=0 fold=0 | Building TE stats...
2026-02-13 03:51:29,948 | INFO | rep=0 fold=0 | Building TE feature blocks (train/val/test)...
2026-02-13 03:51:30,122 | INFO | rep=0 fold=0 | Starting XGBoost training | device=cuda | verbose_eval=100


[0]	validation_0-mae:1331.25133
[100]	validation_0-mae:456.89574
[200]	validation_0-mae:452.24041
[300]	validation_0-mae:449.68873
[400]	validation_0-mae:448.29009
[500]	validation_0-mae:447.97311
[600]	validation_0-mae:448.44757
[700]	validation_0-mae:448.19332
[720]	validation_0-mae:448.49859


2026-02-13 03:51:33,402 | INFO | rep=0 fold=0 | Finished XGBoost training | best_iteration=520
2026-02-13 03:51:33,405 | INFO | rep=0 fold=0 | Preparing CatBoost pools...
2026-02-13 03:51:33,439 | INFO | rep=0 fold=0 | Starting CatBoost training | task_type=GPU | verbose=100
Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1444.3684375	test: 1388.4384375	best: 1388.4384375 (0)	total: 25.6ms	remaining: 3m 24s
100:	learn: 1442.7423437	test: 1386.8368750	best: 1386.8368750 (100)	total: 3.53s	remaining: 4m 35s
200:	learn: 1441.1075000	test: 1385.2296875	best: 1385.2296875 (200)	total: 7.47s	remaining: 4m 49s
300:	learn: 1439.4790625	test: 1383.6256250	best: 1383.6256250 (300)	total: 10.6s	remaining: 4m 31s
400:	learn: 1437.8462500	test: 1382.0178125	best: 1382.0178125 (400)	total: 13.7s	remaining: 4m 20s
500:	learn: 1436.2153125	test: 1380.4115625	best: 1380.4115625 (500)	total: 16.8s	remaining: 4m 10s
600:	learn: 1434.5871875	test: 1378.8068750	best: 1378.8068750 (600)	total: 19.7s	remaining: 4m 2s
700:	learn: 1432.9648438	test: 1377.2065625	best: 1377.2065625 (700)	total: 22.7s	remaining: 3m 55s
800:	learn: 1431.3481250	test: 1375.6103125	best: 1375.6103125 (800)	total: 25.6s	remaining: 3m 50s
900:	learn: 1429.7375000	test: 1374.0162500	best: 1374.0162500 (900)	total: 28.8s	remaining: 3m 46s
1000

2026-02-13 03:55:59,079 | INFO | rep=0 fold=0 | Finished CatBoost training | best_iteration=7999
2026-02-13 03:55:59,096 | INFO | rep=0 fold=0 | Fold complete in 269.2s
CV folds:  10%|██████▎                                                        | 1/10 [04:29<40:22, 269.16s/it]2026-02-13 03:55:59,100 | INFO | rep=0 fold=1 | Starting fold | train=1600 val=400
2026-02-13 03:55:59,103 | INFO | rep=0 fold=1 | Building TE stats...
2026-02-13 03:55:59,109 | INFO | rep=0 fold=1 | Building TE feature blocks (train/val/test)...
2026-02-13 03:55:59,262 | INFO | rep=0 fold=1 | Starting XGBoost training | device=cuda | verbose_eval=100


...

2026-02-13 04:33:16,993 | INFO | rep=1 fold=3 | Finished CatBoost training | best_iteration=7999
2026-02-13 04:33:17,009 | INFO | rep=1 fold=3 | Fold complete in 286.4s
CV folds:  90%|████████████████████████████████████████████████████████▋      | 9/10 [41:47<04:40, 280.92s/it]2026-02-13 04:33:17,009 | INFO | rep=1 fold=4 | Starting fold | train=1600 val=400
2026-02-13 04:33:17,013 | INFO | rep=1 fold=4 | Building TE stats...
2026-02-13 04:33:17,018 | INFO | rep=1 fold=4 | Building TE feature blocks (train/val/test)...
2026-02-13 04:33:17,188 | INFO | rep=1 fold=4 | Starting XGBoost training | device=cuda | verbose_eval=100


[0]	validation_0-mae:1351.80266
[100]	validation_0-mae:449.12291
[200]	validation_0-mae:443.74695
[300]	validation_0-mae:444.30873
[400]	validation_0-mae:444.37693
[413]	validation_0-mae:444.48535


2026-02-13 04:33:18,789 | INFO | rep=1 fold=4 | Finished XGBoost training | best_iteration=213
2026-02-13 04:33:18,790 | INFO | rep=1 fold=4 | Preparing CatBoost pools...
2026-02-13 04:33:18,820 | INFO | rep=1 fold=4 | Starting CatBoost training | task_type=GPU | verbose=100
Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1440.0231250	test: 1406.4243750	best: 1406.4243750 (0)	total: 27.9ms	remaining: 3m 42s
100:	learn: 1438.3764062	test: 1404.8046875	best: 1404.8046875 (100)	total: 3.58s	remaining: 4m 39s
200:	learn: 1436.7168750	test: 1403.1717188	best: 1403.1717188 (200)	total: 7.23s	remaining: 4m 40s
300:	learn: 1435.0437500	test: 1401.5196875	best: 1401.5196875 (300)	total: 10.8s	remaining: 4m 35s
400:	learn: 1433.3729687	test: 1399.8765625	best: 1399.8765625 (400)	total: 14.2s	remaining: 4m 29s
500:	learn: 1431.7035937	test: 1398.2375000	best: 1398.2375000 (500)	total: 17.7s	remaining: 4m 25s
600:	learn: 1430.0331250	test: 1396.5921875	best: 1396.5921875 (600)	total: 21.2s	remaining: 4m 20s
700:	learn: 1428.3621875	test: 1394.9476562	best: 1394.9476562 (700)	total: 24.5s	remaining: 4m 15s
800:	learn: 1426.6923437	test: 1393.3142188	best: 1393.3142188 (800)	total: 27.9s	remaining: 4m 10s
900:	learn: 1425.0223437	test: 1391.6889062	best: 1391.6889062 (900)	total: 31.3s	remaining: 4m 6s
1000

2026-02-13 04:38:01,196 | INFO | rep=1 fold=4 | Finished CatBoost training | best_iteration=7999
2026-02-13 04:38:01,211 | INFO | rep=1 fold=4 | Fold complete in 284.2s
CV folds: 100%|██████████████████████████████████████████████████████████████| 10/10 [46:31<00:00, 279.13s/it]
2026-02-13 04:38:01,214 | INFO | OOF MAE | XGB: 457.6140
2026-02-13 04:38:01,214 | INFO | OOF MAE | CAT: 1308.0355
2026-02-13 04:38:01,225 | INFO | Blend w(XGB)=1.00 | OOF MAE pre-bias=457.6140 | bias=-3.8372 | OOF MAE final=457.6035
2026-02-13 04:38:01,234 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission.csv


# Iteration 3
- 480.9038

In [12]:
# -*- coding: utf-8 -*-
r"""
ITERATION 3 — Domain Expert CPT/HCPCS Feature Engineering (ED Cost Forecasting)
===============================================================================

What’s new vs Iteration 2:
- Replaces “blind TF-IDF” with structured clinical features:
  * ED E/M visit acuity from CPT 99281–99285 (levels 1–5)
  * Critical care intensity from CPT 99291/99292
  * Observation hours from HCPCS G0378
  * CPT major section spend mix using standard CPT range groupings
  * Surgical subrange buckets (10000–69999 broken into coarse 10k bands)
  * Keyword-derived procedure signals from the line-item descriptions (rule-based, not TF-IDF)
- Keeps GPU XGBoost + GPU CatBoost ensemble with rich logging.
- Keeps CV-safe target encoding but upgrades hierarchy backoff: exact code + 3-char prefix
  (e.g., 992xx family, 850xx family, G03x family).

===========================================================
INSTALL (run once)
===========================================================
pip install -U pandas numpy scipy scikit-learn xgboost catboost pymupdf joblib tqdm

===========================================================
PATHS (Windows)
===========================================================
Train: D:\AgentDs\agent_ds_healthcare\ed_cost_train.csv
Test : D:\AgentDs\agent_ds_healthcare\ed_cost_test.csv
PDFs : D:\AgentDs\agent_ds_healthcare\receipts_pdf\receipt_{patient_id}.pdf
Optional: D:\AgentDs\agent_ds_healthcare\patients.csv

Output:
submission_ICHI_V1.csv (in the base dir)
===========================================================
"""

from __future__ import annotations

import re
import math
import time
import logging
import warnings
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

import xgboost as xgb
from xgboost import XGBRegressor

from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"          # optional
PDF_DIR = BASE_DIR / "receipts_pdf"

SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

CACHE_PATH = BASE_DIR / "cache_receipts_iter3_domain.joblib"
CACHE_VERSION = 31

# CV (keep practical; set FAST_MODE=False if you want more rigor)
FAST_MODE = True
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026] if FAST_MODE else [2024, 2025, 2026]
EARLY_STOPPING = 200

# Target encoding
TE_SMOOTH = 20.0
TE_MIN_COUNT_MEDIAN = 5

# GPU
XGB_DEVICE = "cuda"       # "cuda" or "cuda:0"
CAT_TASK_TYPE = "GPU"     # will auto-fallback to CPU if GPU fails

# Post-process
CLIP_AT_ZERO = True

# Blending
BLEND_STEP = 0.02

# Verbose heartbeats
XGB_VERBOSE_EVERY = 100
CAT_VERBOSE_EVERY = 100


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


# =========================
# REGEX (RAW STRINGS)
# =========================
_money_re = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d{2})$|^\d+(?:\.\d{2})$")
_code_re = re.compile(r"^(?:[A-Z]\d{4}|\d{4,5})$")


def is_money(tok: str) -> bool:
    return bool(_money_re.match(tok))


def money_to_float(tok: str) -> float:
    return float(tok.replace(",", ""))


def is_code(tok: str) -> bool:
    return bool(_code_re.match(tok))


def safe_log1p(x: float) -> float:
    return math.log1p(max(0.0, float(x)))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


# =========================
# CPT/HCPCS DOMAIN HELPERS
# =========================
def code_prefix3(code: str) -> str:
    # Hierarchical family (992xx, 850xx, G03x, etc.)
    return code[:3] if len(code) >= 3 else code


def parse_cpt_int(code: str) -> Optional[int]:
    # numeric CPT codes may have leading zeros; int() will drop them which is fine for range checks
    if code.isdigit():
        try:
            return int(code)
        except Exception:
            return None
    return None


def cpt_major_section(code: str) -> Tuple[str, str]:
    """
    Returns (section, detail)
    section in:
      em, anesthesia, surgery, radiology, lab, medicine, hcpcs, other
    detail: for hcpcs -> letter, for numeric -> ""
    CPT major range groupings used:
      - Anesthesia: 00100–01999 and 99100–99140
      - Surgery: 10021–69990
      - Radiology: 70010–79999
      - Path/Lab: 80047–89398
      - Medicine: 90281–99607
      - E/M: 99201–99499
    """
    if not code:
        return ("other", "")

    if code[0].isalpha():
        return ("hcpcs", code[0])

    n = parse_cpt_int(code)
    if n is None:
        return ("other", "")

    # anesthesia (00100-01999) => int 100-1999 after stripping leading zeros
    if (100 <= n <= 1999) or (99100 <= n <= 99140):
        return ("anesthesia", "")

    if 99201 <= n <= 99499:
        return ("em", "")

    if 10021 <= n <= 69990:
        return ("surgery", "")

    if 70010 <= n <= 79999:
        return ("radiology", "")

    if 80047 <= n <= 89398:
        return ("lab", "")

    if 90281 <= n <= 99607:
        return ("medicine", "")

    return ("other", "")


def ed_em_level(code: str) -> int:
    """
    ED E/M visit levels: 99281–99285 => level 1..5
    """
    n = parse_cpt_int(code) if code and code[0].isdigit() else None
    if n is None:
        return 0
    if 99281 <= n <= 99285:
        return int(n - 99280)
    return 0


def is_critical_care(code: str) -> bool:
    # 99291, 99292
    n = parse_cpt_int(code) if code and code[0].isdigit() else None
    return bool(n in (99291, 99292))


# =========================
# PDF PARSING
# =========================
def cluster_words_into_lines(words: List[Tuple], y_tol: float = 1.5) -> List[List[str]]:
    if not words:
        return []
    words_sorted = sorted(words, key=lambda w: (w[1], w[0]))
    lines: List[List[str]] = []
    cur: List[Tuple[float, str]] = []
    cur_y: Optional[float] = None

    for w in words_sorted:
        x0, y0 = float(w[0]), float(w[1])
        tok = str(w[4])

        if cur_y is None or abs(y0 - cur_y) <= y_tol:
            cur.append((x0, tok))
            cur_y = y0 if cur_y is None else (0.85 * cur_y + 0.15 * y0)
        else:
            lines.append([t for _, t in sorted(cur, key=lambda z: z[0])])
            cur = [(x0, tok)]
            cur_y = y0

    if cur:
        lines.append([t for _, t in sorted(cur, key=lambda z: z[0])])

    return lines


def parse_header_fields(text: str) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    pid = None
    zip3 = None
    insurance = None

    m = re.search(r"Patient ID:\s*(\d+)", text)
    if m:
        pid = int(m.group(1))

    m = re.search(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)", text)
    if m:
        zip3 = str(m.group(1))
        insurance = str(m.group(2)).lower()

    return pid, zip3, insurance


def parse_items_from_words(words: List[Tuple]) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None

    lines = cluster_words_into_lines(words, y_tol=1.5)
    if not lines:
        return items, total

    header_idx = None
    for i, toks in enumerate(lines):
        low = [t.lower() for t in toks]
        if ("code" in low) and ("description" in low) and ("qty" in low):
            header_idx = i
            break
    if header_idx is None:
        return items, total

    for toks in lines[header_idx + 1:]:
        if not toks:
            continue

        if toks[0].upper() == "TOTAL":
            for t in reversed(toks[1:]):
                if is_money(t):
                    total = money_to_float(t)
                    break
            break

        if len(toks) >= 5 and is_code(toks[0]) and toks[-3].isdigit() and is_money(toks[-2]) and is_money(toks[-1]):
            items.append({
                "code": toks[0],
                "qty": int(toks[-3]),
                "unit": money_to_float(toks[-2]),
                "line_total": money_to_float(toks[-1]),
                "desc": " ".join(toks[1:-3]),
            })

    return items, total


def parse_items_fallback_text(text: str) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None

    lines = [ln.strip() for ln in (text.splitlines() if text else []) if ln.strip()]
    if not lines:
        return items, total

    start_idx = None
    for i in range(len(lines) - 4):
        if lines[i].lower() == "code" and lines[i + 1].lower() == "description" and lines[i + 2].lower() == "qty":
            start_idx = i + 5
            break
        if "code" in lines[i].lower() and "description" in lines[i].lower() and "qty" in lines[i].lower():
            start_idx = i + 1
            break

    if start_idx is None:
        return items, total

    i = start_idx
    while i < len(lines):
        if lines[i].upper().startswith("TOTAL"):
            parts = lines[i].split()
            if len(parts) >= 2 and is_money(parts[-1]):
                total = money_to_float(parts[-1])
            elif i + 1 < len(lines) and is_money(lines[i + 1]):
                total = money_to_float(lines[i + 1])
            break

        if i + 4 < len(lines) and is_code(lines[i]) and lines[i + 2].isdigit() and is_money(lines[i + 3]) and is_money(lines[i + 4]):
            items.append({
                "code": lines[i],
                "desc": lines[i + 1],
                "qty": int(lines[i + 2]),
                "unit": money_to_float(lines[i + 3]),
                "line_total": money_to_float(lines[i + 4]),
            })
            i += 5
        else:
            i += 1

    return items, total


@dataclass
class ReceiptProfile:
    meta: Dict[str, Any]
    code_spend: Dict[str, float]
    code_qty: Dict[str, float]
    code_lines: Dict[str, int]
    kw_spend: Dict[str, float]
    kw_lines: Dict[str, int]


KW_GROUPS = {
    "kw_ed_visit": [r"ed visit"],
    "kw_critical": [r"critical care"],
    "kw_observation": [r"observation"],
    "kw_intubation": [r"intubation"],
    "kw_catheter": [r"catheter"],
    "kw_lab": [r"\bcvc\b", r"\bcbc\b", r"metabolic", r"panel", r"differential", r"chem", r"culture", r"blood"],
    "kw_imaging": [r"\bct\b", r"\bmri\b", r"x-?ray", r"ultrasound", r"radiograph"],
}


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptProfile:
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"

    empty_meta = dict(
        pdf_ok=0, pdf_n_lines=0, pdf_n_unique_codes=0,
        pdf_sum_qty=0.0,
        pdf_sum_line_total=0.0, pdf_mean_line_total=0.0, pdf_max_line_total=0.0, pdf_std_line_total=0.0,
        pdf_top1_cost_share=0.0, pdf_top3_cost_share=0.0,
        pdf_total=0.0, pdf_total_diff=0.0,
        pdf_insurance=None, pdf_zip3=None, pdf_parse_notes="missing_pdf",
    )

    if not pdf_path.exists():
        return ReceiptProfile(meta=empty_meta, code_spend={}, code_qty={}, code_lines={}, kw_spend={}, kw_lines={})

    try:
        with fitz.open(pdf_path) as doc:
            full_text = ""
            items: List[Dict[str, Any]] = []
            total: Optional[float] = None
            zip3 = None
            insurance = None

            for page in doc:
                text = page.get_text("text") or ""
                full_text += text + "\n"
                words = page.get_text("words") or []

                items_p, total_p = parse_items_from_words(words)
                if not items_p:
                    items_p, total_p = parse_items_fallback_text(text)

                if items_p:
                    items.extend(items_p)
                if total_p is not None:
                    total = total_p

            _, zip3, insurance = parse_header_fields(full_text)

    except Exception as e:
        empty_meta["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return ReceiptProfile(meta=empty_meta, code_spend={}, code_qty={}, code_lines={}, kw_spend={}, kw_lines={})

    if not items:
        empty_meta["pdf_parse_notes"] = "no_items_parsed"
        empty_meta["pdf_zip3"] = zip3
        empty_meta["pdf_insurance"] = insurance
        return ReceiptProfile(meta=empty_meta, code_spend={}, code_qty={}, code_lines={}, kw_spend={}, kw_lines={})

    code_spend: Dict[str, float] = {}
    code_qty: Dict[str, float] = {}
    code_lines: Dict[str, int] = {}

    kw_spend: Dict[str, float] = {k: 0.0 for k in KW_GROUPS.keys()}
    kw_lines: Dict[str, int] = {k: 0 for k in KW_GROUPS.keys()}

    for it in items:
        c = str(it["code"])
        q = float(it.get("qty", 0))
        lt = float(it.get("line_total", 0.0))
        desc = str(it.get("desc", "")).lower()

        code_spend[c] = code_spend.get(c, 0.0) + lt
        code_qty[c] = code_qty.get(c, 0.0) + q
        code_lines[c] = code_lines.get(c, 0) + 1

        for group, pats in KW_GROUPS.items():
            for pat in pats:
                if re.search(pat, desc):
                    kw_spend[group] += lt
                    kw_lines[group] += 1
                    break

    line_totals = np.array([float(it.get("line_total", 0.0)) for it in items], dtype=float)
    qtys = np.array([float(it.get("qty", 0.0)) for it in items], dtype=float)

    sum_line_total = float(line_totals.sum())
    pdf_total = float(total) if total is not None else sum_line_total

    sorted_cost = np.sort(line_totals)[::-1]
    top1 = float(sorted_cost[0] / sum_line_total) if sum_line_total > 0 else 0.0
    top3 = float(sorted_cost[:3].sum() / sum_line_total) if sum_line_total > 0 else 0.0

    meta = dict(
        pdf_ok=1,
        pdf_n_lines=int(len(items)),
        pdf_n_unique_codes=int(len(code_spend)),
        pdf_sum_qty=float(qtys.sum()),
        pdf_sum_line_total=float(sum_line_total),
        pdf_mean_line_total=float(line_totals.mean()) if len(items) else 0.0,
        pdf_max_line_total=float(line_totals.max()) if len(items) else 0.0,
        pdf_std_line_total=float(line_totals.std()) if len(items) else 0.0,
        pdf_top1_cost_share=float(top1),
        pdf_top3_cost_share=float(top3),
        pdf_total=float(pdf_total),
        pdf_total_diff=float(abs(sum_line_total - pdf_total)),
        pdf_insurance=(insurance.lower() if isinstance(insurance, str) else None),
        pdf_zip3=(str(zip3) if zip3 is not None else None),
        pdf_parse_notes="ok",
    )

    return ReceiptProfile(meta=meta, code_spend=code_spend, code_qty=code_qty, code_lines=code_lines,
                          kw_spend=kw_spend, kw_lines=kw_lines)


def load_cache(cache_path: Path) -> Dict[str, Any]:
    if cache_path.exists():
        try:
            obj = joblib.load(cache_path)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "data" in obj:
                return obj
        except Exception:
            pass
    return {"version": CACHE_VERSION, "data": {}}


def save_cache(cache_path: Path, cache_obj: Dict[str, Any]) -> None:
    try:
        joblib.dump(cache_obj, cache_path)
    except Exception as e:
        logging.warning(f"Cache save failed: {e}")


def build_receipt_artifacts(patient_ids: List[int], pdf_dir: Path, cache_path: Path):
    cache_obj = load_cache(cache_path)
    data: Dict[str, Any] = cache_obj.get("data", {})

    to_parse = [pid for pid in patient_ids if str(pid) not in data]
    if to_parse:
        logging.info(f"Receipt parsing: {len(to_parse):,} cache-miss PDFs to parse.")
    else:
        logging.info("Receipt parsing: all PDFs loaded from cache.")

    for pid in tqdm(to_parse, desc="Parsing receipts", ncols=100):
        prof = featurize_receipt(pid, pdf_dir)
        data[str(pid)] = {
            "meta": prof.meta,
            "code_spend": prof.code_spend,
            "code_qty": prof.code_qty,
            "code_lines": prof.code_lines,
            "kw_spend": prof.kw_spend,
            "kw_lines": prof.kw_lines,
        }

    cache_obj["data"] = data
    if to_parse:
        save_cache(cache_path, cache_obj)

    meta_rows = []
    profiles: Dict[int, Dict[str, Any]] = {}
    for pid in patient_ids:
        obj = data.get(str(pid), None)
        if obj is None:
            prof = featurize_receipt(pid, pdf_dir)
            obj = {
                "meta": prof.meta,
                "code_spend": prof.code_spend,
                "code_qty": prof.code_qty,
                "code_lines": prof.code_lines,
                "kw_spend": prof.kw_spend,
                "kw_lines": prof.kw_lines,
            }
        meta_rows.append({"patient_id": pid, **obj["meta"]})
        profiles[pid] = obj

    receipt_meta_df = pd.DataFrame(meta_rows)
    return receipt_meta_df, profiles


# =========================
# DOMAIN FEATURE ENGINEERING
# =========================
def compute_domain_features_for_patient(pid: int, prof: Dict[str, Any]) -> Dict[str, Any]:
    meta = prof.get("meta", {}) or {}
    code_spend: Dict[str, float] = prof.get("code_spend", {}) or {}
    code_qty: Dict[str, float] = prof.get("code_qty", {}) or {}
    code_lines: Dict[str, int] = prof.get("code_lines", {}) or {}
    kw_spend: Dict[str, float] = prof.get("kw_spend", {}) or {}
    kw_lines: Dict[str, int] = prof.get("kw_lines", {}) or {}

    total_spend = float(meta.get("pdf_total", 0.0)) if meta.get("pdf_ok", 0) == 1 else float(sum(code_spend.values()))
    if total_spend <= 0:
        total_spend = float(sum(code_spend.values()))

    feats: Dict[str, Any] = {}

    # Overall
    feats["clin_total_spend_pdf"] = float(total_spend)
    feats["clin_total_unique_codes"] = float(len(code_spend))
    feats["clin_total_lines"] = float(sum(code_lines.values())) if code_lines else 0.0
    feats["clin_total_qty"] = float(sum(code_qty.values())) if code_qty else 0.0

    # Major sections
    sections = ["em", "anesthesia", "surgery", "radiology", "lab", "medicine", "hcpcs", "other"]
    sec_spend = {s: 0.0 for s in sections}
    sec_lines = {s: 0.0 for s in sections}
    sec_unique = {s: 0.0 for s in sections}
    sec_qty = {s: 0.0 for s in sections}

    # HCPCS letter breakout (small fixed set)
    hcpcs_letters = ["G", "J", "A", "Q", "S", "T", "V", "C", "K", "L", "M", "P"]
    hcpcs_spend = {f"hcpcs_{L}_spend": 0.0 for L in hcpcs_letters}
    hcpcs_lines = {f"hcpcs_{L}_lines": 0.0 for L in hcpcs_letters}

    # ED E/M acuity
    ed_cnt = {lvl: 0.0 for lvl in range(1, 6)}
    ed_sp = {lvl: 0.0 for lvl in range(1, 6)}

    # Critical care
    crit_99291_cnt = 0.0
    crit_99292_cnt = 0.0
    crit_spend = 0.0

    # Observation
    obs_hours = 0.0
    obs_spend = 0.0
    obs_lines = 0.0

    # Surgery coarse 10k buckets (domain-ish hierarchy)
    surg_buckets = [
        (10000, 19999, "surg_10_19"),
        (20000, 29999, "surg_20_29"),
        (30000, 39999, "surg_30_39"),
        (40000, 49999, "surg_40_49"),
        (50000, 59999, "surg_50_59"),
        (60000, 69999, "surg_60_69"),
    ]
    for _, _, name in surg_buckets:
        feats[f"{name}_spend"] = 0.0
        feats[f"{name}_lines"] = 0.0
        feats[f"{name}_unique"] = 0.0

    # Iterate codes
    for code, spend in code_spend.items():
        spend_f = float(spend)
        lines_f = float(code_lines.get(code, 1))
        qty_f = float(code_qty.get(code, 0.0))

        sec, detail = cpt_major_section(code)
        if sec not in sec_spend:
            sec = "other"
        sec_spend[sec] += spend_f
        sec_lines[sec] += lines_f
        sec_qty[sec] += qty_f
        sec_unique[sec] += 1.0

        # HCPCS letter detail
        if sec == "hcpcs" and detail in hcpcs_letters:
            hcpcs_spend[f"hcpcs_{detail}_spend"] += spend_f
            hcpcs_lines[f"hcpcs_{detail}_lines"] += lines_f

        # ED E/M levels
        lvl = ed_em_level(code)
        if lvl in ed_cnt:
            ed_cnt[lvl] += lines_f
            ed_sp[lvl] += spend_f

        # Critical care
        if code == "99291":
            crit_99291_cnt += lines_f
            crit_spend += spend_f
        elif code == "99292":
            crit_99292_cnt += lines_f
            crit_spend += spend_f

        # Observation (facility, per hour)
        if code == "G0378":
            obs_lines += lines_f
            obs_spend += spend_f
            obs_hours += qty_f

        # Surgery buckets
        n = parse_cpt_int(code)
        if n is not None and 10021 <= n <= 69990:
            for lo, hi, name in surg_buckets:
                if lo <= n <= hi:
                    feats[f"{name}_spend"] += spend_f
                    feats[f"{name}_lines"] += lines_f
                    feats[f"{name}_unique"] += 1.0
                    break

    # Major section features
    denom = total_spend + 1e-9
    for s in sections:
        feats[f"sec_{s}_spend"] = float(sec_spend[s])
        feats[f"sec_{s}_share"] = float(sec_spend[s] / denom)
        feats[f"sec_{s}_lines"] = float(sec_lines[s])
        feats[f"sec_{s}_unique"] = float(sec_unique[s])
        feats[f"sec_{s}_qty"] = float(sec_qty[s])

    # HCPCS letter spend/lines + shares
    for k, v in hcpcs_spend.items():
        feats[k] = float(v)
        feats[k.replace("_spend", "_share")] = float(v / denom)
    for k, v in hcpcs_lines.items():
        feats[k] = float(v)

    # ED E/M derived stats
    ed_total_cnt = float(sum(ed_cnt.values()))
    ed_total_spend = float(sum(ed_sp.values()))
    feats["ed_total_cnt"] = ed_total_cnt
    feats["ed_total_spend"] = ed_total_spend
    feats["ed_total_share"] = float(ed_total_spend / denom)

    max_lvl = 0
    for lvl in range(5, 0, -1):
        if ed_cnt[lvl] > 0 or ed_sp[lvl] > 0:
            max_lvl = lvl
            break
    feats["ed_max_level"] = float(max_lvl)

    # mean level weighted by count and by spend
    if ed_total_cnt > 0:
        feats["ed_mean_level_cntw"] = float(sum(lvl * ed_cnt[lvl] for lvl in range(1, 6)) / (ed_total_cnt + 1e-9))
    else:
        feats["ed_mean_level_cntw"] = 0.0

    if ed_total_spend > 0:
        feats["ed_mean_level_spw"] = float(sum(lvl * ed_sp[lvl] for lvl in range(1, 6)) / (ed_total_spend + 1e-9))
    else:
        feats["ed_mean_level_spw"] = 0.0

    # per-level features
    for lvl in range(1, 6):
        feats[f"ed_lvl{lvl}_cnt"] = float(ed_cnt[lvl])
        feats[f"ed_lvl{lvl}_spend"] = float(ed_sp[lvl])
        feats[f"ed_lvl{lvl}_share"] = float(ed_sp[lvl] / denom)

    feats["ed_high_cnt_45"] = float(ed_cnt[4] + ed_cnt[5])
    feats["ed_high_share_45"] = float((ed_sp[4] + ed_sp[5]) / denom)

    # Critical care features
    feats["crit_99291_cnt"] = float(crit_99291_cnt)
    feats["crit_99292_cnt"] = float(crit_99292_cnt)
    feats["crit_blocks_cnt"] = float(crit_99291_cnt + crit_99292_cnt)
    feats["crit_spend"] = float(crit_spend)
    feats["crit_share"] = float(crit_spend / denom)

    # Observation features
    feats["obs_g0378_lines"] = float(obs_lines)
    feats["obs_g0378_spend"] = float(obs_spend)
    feats["obs_g0378_share"] = float(obs_spend / denom)
    feats["obs_g0378_hours"] = float(obs_hours)

    # Keyword-rule features (spend + share + line counts)
    for k in KW_GROUPS.keys():
        s = float(kw_spend.get(k, 0.0))
        c = float(kw_lines.get(k, 0))
        feats[f"{k}_spend"] = s
        feats[f"{k}_share"] = float(s / denom)
        feats[f"{k}_lines"] = c

    # Ratios capturing “E/M vs Procedures” intensity
    proc_spend = sec_spend["surgery"] + sec_spend["radiology"] + sec_spend["lab"] + sec_spend["medicine"]
    feats["em_to_proc_spend_ratio"] = float(sec_spend["em"] / (proc_spend + 1e-6))
    feats["lab_to_total_ratio"] = float(sec_spend["lab"] / denom)
    feats["surg_to_total_ratio"] = float(sec_spend["surgery"] / denom)

    # Simple composite acuity score (let model learn; this is just a compressed signal)
    feats["acuity_score"] = float(
        feats["ed_mean_level_spw"]
        + 1.5 * math.log1p(feats["crit_blocks_cnt"])
        + 0.05 * feats["obs_g0378_hours"]
        + 0.25 * math.log1p(sec_lines["surgery"])
        + 0.10 * feats["sec_radiology_share"] * 10.0
    )

    return feats


def build_domain_feature_df(patient_ids: List[int], profiles: Dict[int, Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for pid in tqdm(patient_ids, desc="Domain features", ncols=100):
        prof = profiles.get(pid, {}) or {}
        feats = compute_domain_features_for_patient(pid, prof)
        rows.append({"patient_id": pid, **feats})
    return pd.DataFrame(rows)


# =========================
# BASE MATRIX (XGB)
# =========================
def build_base_sparse_matrix(df_all: pd.DataFrame) -> Tuple[csr_matrix, Dict[str, Any]]:
    df = df_all.copy()

    # derived tabular
    df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
    df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
    df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

    # normalize cats
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("missing")

    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train"}
    cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                if c in df.columns]
    num_cols = [c for c in df.columns
                if c not in exclude and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    X = preprocessor.fit_transform(df)
    X = X.tocsr() if hasattr(X, "tocsr") else csr_matrix(X)
    return X, {"preprocessor": preprocessor, "num_cols": num_cols, "cat_cols": cat_cols}


# =========================
# CV-SAFE TARGET ENCODING (exact code + prefix3)
# =========================
@dataclass
class TokenStats:
    cnt: int
    mean_smooth: float
    median_adj: float


def build_token_stats_for_fold(
    train_pids: List[int],
    y_log: np.ndarray,
    pid_to_row: Dict[int, int],
    profiles: Dict[int, Dict[str, Any]],
) -> Tuple[Dict[str, TokenStats], Dict[str, TokenStats], float, float]:
    vals_code: Dict[str, List[float]] = {}
    vals_p3: Dict[str, List[float]] = {}
    ys = []

    for pid in train_pids:
        r = pid_to_row.get(pid, None)
        if r is None:
            continue
        v = float(y_log[r])
        ys.append(v)

        code_spend = profiles.get(pid, {}).get("code_spend", {}) or {}
        for c in code_spend.keys():
            vals_code.setdefault(c, []).append(v)
            vals_p3.setdefault(code_prefix3(c), []).append(v)

    gmean = float(np.mean(ys)) if ys else 0.0
    gmed = float(np.median(ys)) if ys else 0.0

    def to_stats(m: Dict[str, List[float]]) -> Dict[str, TokenStats]:
        out: Dict[str, TokenStats] = {}
        for tok, arr in m.items():
            a = np.asarray(arr, dtype=float)
            cnt = int(a.size)
            mean = float(np.mean(a))
            med = float(np.median(a))
            mean_smooth = float((cnt * mean + TE_SMOOTH * gmean) / (cnt + TE_SMOOTH))
            median_adj = med if cnt >= TE_MIN_COUNT_MEDIAN else gmed
            out[tok] = TokenStats(cnt=cnt, mean_smooth=mean_smooth, median_adj=median_adj)
        return out

    return to_stats(vals_code), to_stats(vals_p3), gmean, gmed


def te_features_for_patients(
    pids: List[int],
    profiles: Dict[int, Dict[str, Any]],
    stats_code: Dict[str, TokenStats],
    stats_p3: Dict[str, TokenStats],
    gmean: float,
    gmed: float,
) -> np.ndarray:
    """
    16 TE features:
      exact-code (8) + prefix3 (8), spend-weighted.
    """
    feats = np.zeros((len(pids), 16), dtype=np.float32)

    for i, pid in enumerate(pids):
        code_spend = profiles.get(pid, {}).get("code_spend", {}) or {}
        if not code_spend:
            base = np.array([gmean, gmed, gmean, gmean, gmean, gmed, 0.0, 0.0], dtype=np.float32)
            feats[i, 0:8] = base
            feats[i, 8:16] = base
            continue

        codes = list(code_spend.keys())
        spends = np.array([float(code_spend[c]) for c in codes], dtype=float)
        ssum = float(spends.sum())
        w = (np.ones_like(spends) / max(1, spends.size)) if ssum <= 0 else (spends / (ssum + 1e-9))

        # exact code
        means, meds, cnts = [], [], []
        known = 0
        for c in codes:
            st = stats_code.get(c)
            if st is None:
                means.append(gmean); meds.append(gmed); cnts.append(0)
            else:
                means.append(st.mean_smooth); meds.append(st.median_adj); cnts.append(st.cnt); known += 1
        means = np.asarray(means, dtype=float)
        meds = np.asarray(meds, dtype=float)
        cnts = np.asarray(cnts, dtype=float)
        top = int(np.argmax(w)) if w.size else 0

        feats[i, 0:8] = np.array([
            float(np.sum(w * means)),
            float(np.sum(w * meds)),
            float(np.mean(means)),
            float(np.max(means)),
            float(np.min(means)),
            float(meds[top] if meds.size else gmed),
            float(known / max(1, len(codes))),
            float(np.mean(np.log1p(cnts))) if cnts.size else 0.0,
        ], dtype=np.float32)

        # prefix3
        p3_map: Dict[str, float] = {}
        for c, sp in code_spend.items():
            p3 = code_prefix3(c)
            p3_map[p3] = p3_map.get(p3, 0.0) + float(sp)
        p3s = list(p3_map.keys())
        p3sp = np.asarray([p3_map[p] for p in p3s], dtype=float)
        p3sum = float(p3sp.sum())
        w2 = (np.ones_like(p3sp) / max(1, p3sp.size)) if p3sum <= 0 else (p3sp / (p3sum + 1e-9))

        means2, meds2, cnts2 = [], [], []
        known2 = 0
        for p in p3s:
            st = stats_p3.get(p)
            if st is None:
                means2.append(gmean); meds2.append(gmed); cnts2.append(0)
            else:
                means2.append(st.mean_smooth); meds2.append(st.median_adj); cnts2.append(st.cnt); known2 += 1
        means2 = np.asarray(means2, dtype=float)
        meds2 = np.asarray(meds2, dtype=float)
        cnts2 = np.asarray(cnts2, dtype=float)
        top2 = int(np.argmax(w2)) if w2.size else 0

        feats[i, 8:16] = np.array([
            float(np.sum(w2 * means2)),
            float(np.sum(w2 * meds2)),
            float(np.mean(means2)),
            float(np.max(means2)),
            float(np.min(means2)),
            float(meds2[top2] if meds2.size else gmed),
            float(known2 / max(1, len(p3s))),
            float(np.mean(np.log1p(cnts2))) if cnts2.size else 0.0,
        ], dtype=np.float32)

    return feats


def make_strat_labels(train_df: pd.DataFrame) -> np.ndarray:
    chronic = train_df["primary_chronic"].astype(str)
    cost_bin = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    return (chronic + "_" + cost_bin).values


# =========================
# BLEND
# =========================
def best_blend_weight(y_true: np.ndarray, p1: np.ndarray, p2: np.ndarray, step: float) -> Tuple[float, float]:
    best_w = 0.5
    best_m = 1e18
    for w in np.arange(0.0, 1.0 + 1e-9, step):
        blend = w * p1 + (1.0 - w) * p2
        m = mae(y_true, blend)
        if m < best_m:
            best_m = m
            best_w = float(w)
    return best_w, best_m


# =========================
# MAIN
# =========================
def main() -> None:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb, '__version__', 'unknown')}")
    logging.info(f"FAST_MODE={FAST_MODE} | repeats={len(REPEAT_SEEDS)} | folds={N_FOLDS}")
    logging.info(f"GPU config: XGB_DEVICE={XGB_DEVICE}, CAT_TASK_TYPE={CAT_TASK_TYPE}")

    # load
    if not TRAIN_CSV.exists():
        raise FileNotFoundError(f"Missing: {TRAIN_CSV}")
    if not TEST_CSV.exists():
        raise FileNotFoundError(f"Missing: {TEST_CSV}")
    if not PDF_DIR.exists():
        raise FileNotFoundError(f"Missing PDF dir: {PDF_DIR}")

    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train_df.shape}, test={test_df.shape}")

    # optional patients.csv
    if PATIENTS_CSV.exists():
        patients = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in patients.columns:
            train_df = train_df.merge(patients, on="patient_id", how="left")
            test_df = test_df.merge(patients, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train_df.shape}, test={test_df.shape}")
        else:
            logging.warning("patients.csv exists but has no patient_id; skipping merge.")
    else:
        logging.info("patients.csv not found; continuing without it.")

    train_df["is_train"] = 1
    test_df["is_train"] = 0
    df_all = pd.concat([train_df, test_df], ignore_index=True)

    # receipts parsing (cached)
    all_pids = df_all["patient_id"].astype(int).unique().tolist()
    all_pids.sort()
    receipt_meta_df, profiles = build_receipt_artifacts(all_pids, PDF_DIR, CACHE_PATH)
    df_all = df_all.merge(receipt_meta_df, on="patient_id", how="left")

    # domain features
    logging.info("Building structured clinical domain features from parsed codes...")
    domain_df = build_domain_feature_df(all_pids, profiles)
    df_all = df_all.merge(domain_df, on="patient_id", how="left")

    # Tabular interactions (domain-driven)
    logging.info("Adding chronic/utilization interaction features...")
    df_all["prior_cost_per_visit"] = df_all["prior_ed_cost_5y_usd"] / (df_all["prior_ed_visits_5y"] + 1.0)
    df_all["log_prior_cost"] = np.log1p(df_all["prior_ed_cost_5y_usd"].astype(float))
    df_all["log_prior_visits"] = np.log1p(df_all["prior_ed_visits_5y"].astype(float))

    # Chronic flags + interactions with acuity
    for chronic in ["HF", "DiabetesComp", "Pneumonia"]:
        col = f"is_{chronic}"
        df_all[col] = (df_all["primary_chronic"].astype(str) == chronic).astype(np.float32)
        df_all[f"acuity_x_{chronic}"] = df_all["acuity_score"].astype(float) * df_all[col].astype(float)
        df_all[f"edmax_x_{chronic}"] = df_all["ed_max_level"].astype(float) * df_all[col].astype(float)
        df_all[f"crit_x_{chronic}"] = df_all["crit_blocks_cnt"].astype(float) * df_all[col].astype(float)

    # Utilization x severity
    df_all["edmax_x_logvis"] = df_all["ed_max_level"].astype(float) * df_all["log_prior_visits"].astype(float)
    df_all["edmean_spw_x_logcost"] = df_all["ed_mean_level_spw"].astype(float) * df_all["log_prior_cost"].astype(float)
    df_all["critblocks_x_logcost"] = df_all["crit_blocks_cnt"].astype(float) * df_all["log_prior_cost"].astype(float)
    df_all["obshrs_x_logvis"] = df_all["obs_g0378_hours"].astype(float) * df_all["log_prior_visits"].astype(float)

    # Base sparse for XGB
    logging.info("Building XGB base sparse matrix (OHE cats + numeric clinical features)...")
    X_all_base, base_art = build_base_sparse_matrix(df_all)
    logging.info(f"X_all_base shape: {X_all_base.shape}")

    is_train = df_all["is_train"].values.astype(int) == 1
    X_train_base = X_all_base[is_train]
    X_test_base = X_all_base[~is_train]

    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)
    y_log = np.log1p(y)

    train_pids = train_df["patient_id"].astype(int).tolist()
    test_pids = test_df["patient_id"].astype(int).tolist()
    pid_to_row = {pid: i for i, pid in enumerate(train_pids)}

    # CatBoost frames: use df_all but drop target & keys
    logging.info("Preparing CatBoost base frames...")
    df_cb_all = df_all.drop(columns=["ed_cost_next3y_usd"], errors="ignore").copy()
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df_cb_all.columns:
            df_cb_all[c] = df_cb_all[c].astype(str).fillna("missing")

    # Remove keys from CB features
    drop_cols = [c for c in ["patient_id", "is_train"] if c in df_cb_all.columns]
    df_cb_all = df_cb_all.drop(columns=drop_cols, errors="ignore")

    df_train_cb = df_cb_all[is_train].reset_index(drop=True)
    df_test_cb = df_cb_all[~is_train].reset_index(drop=True)

    cb_cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                   if c in df_train_cb.columns]
    cb_cat_idx = [df_train_cb.columns.get_loc(c) for c in cb_cat_cols]
    logging.info(f"CatBoost cat_features={cb_cat_cols}")

    # CV splits
    strat = make_strat_labels(train_df)
    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), strat)):
            splits.append((rep, fold, tr_idx, va_idx))
    logging.info(f"Total CV splits: {len(splits)}")

    # Storage
    oof_xgb = np.zeros(len(y), dtype=np.float64)
    oof_cat = np.zeros(len(y), dtype=np.float64)
    cnt_xgb = np.zeros(len(y), dtype=np.float64)
    cnt_cat = np.zeros(len(y), dtype=np.float64)

    test_xgb_sum = np.zeros(len(test_df), dtype=np.float64)
    test_cat_sum = np.zeros(len(test_df), dtype=np.float64)
    n_xgb = 0
    n_cat = 0

    # Model params (same core as Iter2; only verbosity increased)
    xgb_params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=XGB_DEVICE,
        n_estimators=6000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_lambda=1.0,
        min_child_weight=1.0,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    cat_params = dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=8000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=6.0,
        bagging_temperature=0.8,
        random_strength=0.5,
        od_type="Iter",
        od_wait=EARLY_STOPPING,
        task_type=CAT_TASK_TYPE,
        devices="0",
        verbose=CAT_VERBOSE_EVERY,
    )

    te_cols = [f"te_{i:02d}" for i in range(16)]
    fit_sig_xgb = None

    # CV loop
    for rep, fold, tr_idx, va_idx in tqdm(splits, desc="CV folds", ncols=110):
        t0 = time.time()
        fold_tag = f"rep={rep} fold={fold}"
        logging.info(f"{fold_tag} | START | train={len(tr_idx)} val={len(va_idx)}")

        # Build TE stats for fold
        tr_pids = [train_pids[i] for i in tr_idx.tolist()]
        va_pids = [train_pids[i] for i in va_idx.tolist()]

        logging.info(f"{fold_tag} | TE stats build...")
        stats_code, stats_p3, gmean, gmed = build_token_stats_for_fold(tr_pids, y_log, pid_to_row, profiles)

        logging.info(f"{fold_tag} | TE feature build (train/val/test)...")
        te_tr = te_features_for_patients(tr_pids, profiles, stats_code, stats_p3, gmean, gmed)
        te_va = te_features_for_patients(va_pids, profiles, stats_code, stats_p3, gmean, gmed)
        te_te = te_features_for_patients(test_pids, profiles, stats_code, stats_p3, gmean, gmed)

        # XGB matrices
        X_tr = hstack([X_train_base[tr_idx], csr_matrix(te_tr)], format="csr")
        X_va = hstack([X_train_base[va_idx], csr_matrix(te_va)], format="csr")
        X_te = hstack([X_test_base, csr_matrix(te_te)], format="csr")

        y_tr = y[tr_idx]
        y_va = y[va_idx]

        # ----- XGBoost -----
        logging.info(f"{fold_tag} | XGB fit START | device={xgb_params['device']} | verbose_every={XGB_VERBOSE_EVERY}")
        try:
            xgb_model = XGBRegressor(**xgb_params, random_state=1000 * rep + fold)

            # Use verbose_eval if supported; else use verbose=int (prints every n rounds)
            if fit_sig_xgb is None:
                fit_sig_xgb = inspect.signature(xgb_model.fit).parameters

            fit_kwargs = {"eval_set": [(X_va, y_va)]}
            if "verbose_eval" in fit_sig_xgb:
                fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
            else:
                fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

            xgb_model.fit(X_tr, y_tr, **fit_kwargs)
            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_te)

            best_it = getattr(xgb_model, "best_iteration", None)
            logging.info(f"{fold_tag} | XGB fit END | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | XGB GPU failed: {e} -> fallback CPU")
            xgb_cpu = dict(xgb_params)
            xgb_cpu["device"] = "cpu"
            xgb_model = XGBRegressor(**xgb_cpu, random_state=1000 * rep + fold)

            fit_sig_xgb = inspect.signature(xgb_model.fit).parameters
            fit_kwargs = {"eval_set": [(X_va, y_va)]}
            if "verbose_eval" in fit_sig_xgb:
                fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
            else:
                fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

            logging.info(f"{fold_tag} | XGB (CPU fallback) fit START")
            xgb_model.fit(X_tr, y_tr, **fit_kwargs)
            logging.info(f"{fold_tag} | XGB (CPU fallback) fit END")
            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_te)

        if CLIP_AT_ZERO:
            p_va_x = np.maximum(p_va_x, 0.0)
            p_te_x = np.maximum(p_te_x, 0.0)

        oof_xgb[va_idx] += p_va_x
        cnt_xgb[va_idx] += 1.0
        test_xgb_sum += p_te_x
        n_xgb += 1

        # ----- CatBoost -----
        logging.info(f"{fold_tag} | CatBoost pools build...")
        te_train_full = np.zeros((len(train_df), 16), dtype=np.float32)
        te_train_full[tr_idx, :] = te_tr
        te_train_full[va_idx, :] = te_va

        df_tr = df_train_cb.copy()
        df_te = df_test_cb.copy()
        for j, col in enumerate(te_cols):
            df_tr[col] = te_train_full[:, j]
            df_te[col] = te_te[:, j]

        tr_pool = Pool(df_tr.iloc[tr_idx], label=y_tr, cat_features=cb_cat_idx)
        va_pool = Pool(df_tr.iloc[va_idx], label=y_va, cat_features=cb_cat_idx)
        te_pool = Pool(df_te, cat_features=cb_cat_idx)

        logging.info(f"{fold_tag} | CatBoost fit START | task_type={cat_params['task_type']} | verbose={cat_params['verbose']}")
        try:
            cat_model = CatBoostRegressor(**cat_params, random_seed=2000 * rep + fold)
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
            best_it = None
            try:
                best_it = cat_model.get_best_iteration()
            except Exception:
                best_it = None
            logging.info(f"{fold_tag} | CatBoost fit END | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | CatBoost GPU failed: {e} -> fallback CPU")
            cat_cpu = dict(cat_params)
            cat_cpu["task_type"] = "CPU"
            cat_cpu.pop("devices", None)
            cat_model = CatBoostRegressor(**cat_cpu, random_seed=2000 * rep + fold)

            logging.info(f"{fold_tag} | CatBoost (CPU fallback) fit START | verbose={cat_cpu['verbose']}")
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
            logging.info(f"{fold_tag} | CatBoost (CPU fallback) fit END")

        p_va_c = cat_model.predict(va_pool)
        p_te_c = cat_model.predict(te_pool)

        if CLIP_AT_ZERO:
            p_va_c = np.maximum(p_va_c, 0.0)
            p_te_c = np.maximum(p_te_c, 0.0)

        oof_cat[va_idx] += p_va_c
        cnt_cat[va_idx] += 1.0
        test_cat_sum += p_te_c
        n_cat += 1

        dt = time.time() - t0
        logging.info(f"{fold_tag} | END | fold_time={dt:.1f}s")

    # finalize
    oof_x = (oof_xgb / np.maximum(cnt_xgb, 1.0)).astype(np.float32)
    oof_c = (oof_cat / np.maximum(cnt_cat, 1.0)).astype(np.float32)
    te_x = (test_xgb_sum / max(1, n_xgb)).astype(np.float32)
    te_c = (test_cat_sum / max(1, n_cat)).astype(np.float32)

    logging.info(f"OOF MAE | XGB: {mae(y, oof_x):.4f}")
    logging.info(f"OOF MAE | CAT: {mae(y, oof_c):.4f}")

    # blend + median bias
    w, blend_mae = best_blend_weight(y, oof_x, oof_c, step=BLEND_STEP)
    oof_blend = w * oof_x + (1.0 - w) * oof_c
    bias = float(np.median(y - oof_blend))  # MAE-optimal constant shift
    oof_final = oof_blend + bias
    test_final = w * te_x + (1.0 - w) * te_c + bias

    if CLIP_AT_ZERO:
        oof_final = np.maximum(oof_final, 0.0)
        test_final = np.maximum(test_final, 0.0)

    logging.info(f"Blend w(XGB)={w:.2f} | OOF MAE pre-bias={blend_mae:.4f} | bias={bias:.4f} | OOF MAE final={mae(y, oof_final):.4f}")

    # Save submission
    sub = pd.DataFrame({
        "patient_id": test_df["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": test_final.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")


if __name__ == "__main__":
    main()


2026-02-13 08:46:17,534 | INFO | XGBoost version: 3.0.0
2026-02-13 08:46:17,534 | INFO | FAST_MODE=True | repeats=2 | folds=5
2026-02-13 08:46:17,534 | INFO | GPU config: XGB_DEVICE=cuda, CAT_TASK_TYPE=GPU
2026-02-13 08:46:17,553 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 08:46:17,565 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 08:46:17,573 | INFO | Receipt parsing: 4,000 cache-miss PDFs to parse.
Parsing receipts: 100%|████████████████████████████████████████| 4000/4000 [00:06<00:00, 647.88it/s]
2026-02-13 08:46:24,030 | INFO | Building structured clinical domain features from parsed codes...
Domain features: 100%|███████████████████████████████████████| 4000/4000 [00:00<00:00, 28042.37it/s]
2026-02-13 08:46:24,257 | INFO | Adding chronic/utilization interaction features...
2026-02-13 08:46:24,266 | INFO | Building XGB base sparse matrix (OHE cats + numeric clinical features)...
2026-02-13 08:46:24,374 | INFO | X_all_base shape: (4000, 221)

[0]	validation_0-mae:1330.36615
[100]	validation_0-mae:465.47704
[200]	validation_0-mae:463.31340
[300]	validation_0-mae:461.54353
[400]	validation_0-mae:460.79161
[500]	validation_0-mae:459.46066
[600]	validation_0-mae:459.74673
[700]	validation_0-mae:459.85582
[702]	validation_0-mae:459.87016


2026-02-13 08:46:27,862 | INFO | rep=0 fold=0 | XGB fit END | best_iteration=503
2026-02-13 08:46:27,864 | INFO | rep=0 fold=0 | CatBoost pools build...
2026-02-13 08:46:27,924 | INFO | rep=0 fold=0 | CatBoost fit START | task_type=GPU | verbose=100
Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1444.3682813	test: 1388.4387500	best: 1388.4387500 (0)	total: 44.8ms	remaining: 5m 58s
100:	learn: 1442.7225000	test: 1386.8337500	best: 1386.8337500 (100)	total: 3.92s	remaining: 5m 6s
200:	learn: 1441.0681250	test: 1385.2210937	best: 1385.2210937 (200)	total: 8.22s	remaining: 5m 18s
300:	learn: 1439.4034375	test: 1383.6034375	best: 1383.6034375 (300)	total: 12.9s	remaining: 5m 31s
400:	learn: 1437.7389062	test: 1381.9839063	best: 1381.9839063 (400)	total: 17.9s	remaining: 5m 39s
500:	learn: 1436.0745313	test: 1380.3657813	best: 1380.3657813 (500)	total: 23.1s	remaining: 5m 45s
600:	learn: 1434.4140625	test: 1378.7493750	best: 1378.7493750 (600)	total: 28.2s	remaining: 5m 47s
700:	learn: 1432.7581250	test: 1377.1353125	best: 1377.1353125 (700)	total: 33.3s	remaining: 5m 46s
800:	learn: 1431.1075000	test: 1375.5234375	best: 1375.5234375 (800)	total: 38.5s	remaining: 5m 45s
900:	learn: 1429.4589062	test: 1373.9126563	best: 1373.9126563 (900)	total: 43.6s	remaining: 5m 43s
1000

2026-02-13 08:53:03,762 | INFO | rep=0 fold=0 | CatBoost fit END | best_iteration=7999
2026-02-13 08:53:03,778 | INFO | rep=0 fold=0 | END | fold_time=399.3s
CV folds:  10%|██████▎                                                        | 1/10 [06:39<59:53, 399.32s/it]2026-02-13 08:53:03,780 | INFO | rep=0 fold=1 | START | train=1600 val=400
2026-02-13 08:53:03,782 | INFO | rep=0 fold=1 | TE stats build...
2026-02-13 08:53:03,790 | INFO | rep=0 fold=1 | TE feature build (train/val/test)...
2026-02-13 08:53:03,972 | INFO | rep=0 fold=1 | XGB fit START | device=cuda | verbose_every=100


...

2026-02-13 10:14:37,017 | INFO | rep=1 fold=3 | CatBoost fit END | best_iteration=7999
2026-02-13 10:14:37,035 | INFO | rep=1 fold=3 | END | fold_time=2011.5s
CV folds:  90%|██████████████████████████████████████████████████████▉      | 9/10 [1:28:12<15:17, 917.29s/it]2026-02-13 10:14:37,042 | INFO | rep=1 fold=4 | START | train=1600 val=400
2026-02-13 10:14:37,043 | INFO | rep=1 fold=4 | TE stats build...
2026-02-13 10:14:37,050 | INFO | rep=1 fold=4 | TE feature build (train/val/test)...
2026-02-13 10:14:37,204 | INFO | rep=1 fold=4 | XGB fit START | device=cuda | verbose_every=100


[0]	validation_0-mae:1349.45498
[100]	validation_0-mae:457.59493
[200]	validation_0-mae:459.40299
[300]	validation_0-mae:463.04227
[352]	validation_0-mae:464.01757


2026-02-13 10:14:39,012 | INFO | rep=1 fold=4 | XGB fit END | best_iteration=153
2026-02-13 10:14:39,013 | INFO | rep=1 fold=4 | CatBoost pools build...
2026-02-13 10:14:39,126 | INFO | rep=1 fold=4 | CatBoost fit START | task_type=GPU | verbose=100
Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1440.0226562	test: 1406.4240625	best: 1406.4240625 (0)	total: 41ms	remaining: 5m 28s
100:	learn: 1438.3706250	test: 1404.7967188	best: 1404.7967188 (100)	total: 4.59s	remaining: 5m 59s
200:	learn: 1436.7059375	test: 1403.1542187	best: 1403.1542187 (200)	total: 9.01s	remaining: 5m 49s
300:	learn: 1435.0296875	test: 1401.5065625	best: 1401.5065625 (300)	total: 13.6s	remaining: 5m 48s
400:	learn: 1433.3559375	test: 1399.8648438	best: 1399.8648438 (400)	total: 18.1s	remaining: 5m 43s
500:	learn: 1431.6842187	test: 1398.2251563	best: 1398.2251563 (500)	total: 22.7s	remaining: 5m 40s
600:	learn: 1430.0106250	test: 1396.5837500	best: 1396.5837500 (600)	total: 27.1s	remaining: 5m 33s
700:	learn: 1428.3371875	test: 1394.9435937	best: 1394.9435937 (700)	total: 31.6s	remaining: 5m 29s
800:	learn: 1426.6646875	test: 1393.3031250	best: 1393.3031250 (800)	total: 36.3s	remaining: 5m 26s
900:	learn: 1424.9917188	test: 1391.6721875	best: 1391.6721875 (900)	total: 40.8s	remaining: 5m 21s
1000:

2026-02-13 10:21:00,155 | INFO | rep=1 fold=4 | CatBoost fit END | best_iteration=7999
2026-02-13 10:21:00,176 | INFO | rep=1 fold=4 | END | fold_time=383.1s
CV folds: 100%|████████████████████████████████████████████████████████████| 10/10 [1:34:35<00:00, 567.57s/it]
2026-02-13 10:21:00,179 | INFO | OOF MAE | XGB: 466.0066
2026-02-13 10:21:00,180 | INFO | OOF MAE | CAT: 1308.0187
2026-02-13 10:21:00,184 | INFO | Blend w(XGB)=1.00 | OOF MAE pre-bias=466.0066 | bias=1.8523 | OOF MAE final=466.0052
2026-02-13 10:21:00,189 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv


# Iteration 4
- 483.8662

In [16]:
# -*- coding: utf-8 -*-
r"""
ITERATION 4 — "Return to the Math" (No manual CPT grouping)
==========================================================

Goal:
- Represent high-dimensional sparse code data (thousands of CPT/HCPCS-like codes) in a way
  XGBoost/CatBoost can digest efficiently, WITHOUT manual clinical hierarchies.

Core idea (Third Path):
1) Build sparse patient×code matrices from receipts (binary presence + spend weights).
2) Learn dense embeddings via TruncatedSVD (LSA) on the sparse matrix (unsupervised).
3) Cluster code embeddings (MiniBatchKMeans) => statistically similar code groups.
4) Create patient "bag-of-clusters" features (counts + spend shares + entropy).
5) Add CV-safe target encoding (OOF) for exact codes + learned clusters (no manual grouping).

Models:
- XGBoost GPU (device=cuda, reg:absoluteerror)
- CatBoost GPU (MAE)

Output:
- submission_ICHI_V1.csv with columns: patient_id, ed_cost_next3y_usd

===========================================================
INSTALL (run once)
===========================================================
pip install -U pandas numpy scipy scikit-learn xgboost catboost pymupdf joblib tqdm
"""

from __future__ import annotations

import re
import math
import time
import logging
import warnings
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import MiniBatchKMeans

import xgboost as xgb
from xgboost import XGBRegressor

from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"          # optional
PDF_DIR = BASE_DIR / "receipts_pdf"

SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

CACHE_PATH = BASE_DIR / "cache_receipts_iter4_math.joblib"
CACHE_VERSION = 41

# CV
FAST_MODE = True
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026] if FAST_MODE else [2024, 2025, 2026]
EARLY_STOPPING = 200

# Target encoding
TE_SMOOTH = 20.0
TE_MIN_COUNT_MEDIAN = 5

# Unsupervised representation
N_SVD_TFIDF = 64
N_SVD_SPEND = 32
N_CODE_CLUSTERS = 48

# GPU
XGB_DEVICE = "cuda"
CAT_TASK_TYPE = "GPU"

# Post-process
CLIP_AT_ZERO = True

# Blending
BLEND_STEP = 0.02

# Verbose heartbeats
XGB_VERBOSE_EVERY = 100
CAT_VERBOSE_EVERY = 100


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


# =========================
# REGEX (RAW STRINGS)
# =========================
_money_re = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d{2})$|^\d+(?:\.\d{2})$")
_code_re = re.compile(r"^(?:[A-Z]\d{4}|\d{4,5})$")


def is_money(tok: str) -> bool:
    return bool(_money_re.match(tok))


def money_to_float(tok: str) -> float:
    return float(tok.replace(",", ""))


def is_code(tok: str) -> bool:
    return bool(_code_re.match(tok))


def safe_log1p(x: float) -> float:
    return math.log1p(max(0.0, float(x)))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


# =========================
# PDF PARSING
# =========================
def cluster_words_into_lines(words: List[Tuple], y_tol: float = 1.5) -> List[List[str]]:
    if not words:
        return []
    words_sorted = sorted(words, key=lambda w: (w[1], w[0]))
    lines: List[List[str]] = []
    cur: List[Tuple[float, str]] = []
    cur_y: Optional[float] = None

    for w in words_sorted:
        x0, y0 = float(w[0]), float(w[1])
        tok = str(w[4])

        if cur_y is None or abs(y0 - cur_y) <= y_tol:
            cur.append((x0, tok))
            cur_y = y0 if cur_y is None else (0.85 * cur_y + 0.15 * y0)
        else:
            lines.append([t for _, t in sorted(cur, key=lambda z: z[0])])
            cur = [(x0, tok)]
            cur_y = y0

    if cur:
        lines.append([t for _, t in sorted(cur, key=lambda z: z[0])])

    return lines


def parse_header_fields(text: str) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    pid = None
    zip3 = None
    insurance = None

    m = re.search(r"Patient ID:\s*(\d+)", text)
    if m:
        pid = int(m.group(1))

    m = re.search(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)", text)
    if m:
        zip3 = str(m.group(1))
        insurance = str(m.group(2)).lower()

    return pid, zip3, insurance


def parse_items_from_words(words: List[Tuple]) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None

    lines = cluster_words_into_lines(words, y_tol=1.5)
    if not lines:
        return items, total

    header_idx = None
    for i, toks in enumerate(lines):
        low = [t.lower() for t in toks]
        if ("code" in low) and ("description" in low) and ("qty" in low):
            header_idx = i
            break
    if header_idx is None:
        return items, total

    for toks in lines[header_idx + 1:]:
        if not toks:
            continue

        if toks[0].upper() == "TOTAL":
            for t in reversed(toks[1:]):
                if is_money(t):
                    total = money_to_float(t)
                    break
            break

        if len(toks) >= 5 and is_code(toks[0]) and toks[-3].isdigit() and is_money(toks[-2]) and is_money(toks[-1]):
            items.append({
                "code": toks[0],
                "qty": int(toks[-3]),
                "unit": money_to_float(toks[-2]),
                "line_total": money_to_float(toks[-1]),
                "desc": " ".join(toks[1:-3]),
            })

    return items, total


def parse_items_fallback_text(text: str) -> Tuple[List[Dict[str, Any]], Optional[float]]:
    items: List[Dict[str, Any]] = []
    total: Optional[float] = None

    lines = [ln.strip() for ln in (text.splitlines() if text else []) if ln.strip()]
    if not lines:
        return items, total

    start_idx = None
    for i in range(len(lines) - 4):
        if lines[i].lower() == "code" and lines[i + 1].lower() == "description" and lines[i + 2].lower() == "qty":
            start_idx = i + 5
            break
        if "code" in lines[i].lower() and "description" in lines[i].lower() and "qty" in lines[i].lower():
            start_idx = i + 1
            break

    if start_idx is None:
        return items, total

    i = start_idx
    while i < len(lines):
        if lines[i].upper().startswith("TOTAL"):
            parts = lines[i].split()
            if len(parts) >= 2 and is_money(parts[-1]):
                total = money_to_float(parts[-1])
            elif i + 1 < len(lines) and is_money(lines[i + 1]):
                total = money_to_float(lines[i + 1])
            break

        if i + 4 < len(lines) and is_code(lines[i]) and lines[i + 2].isdigit() and is_money(lines[i + 3]) and is_money(lines[i + 4]):
            items.append({
                "code": lines[i],
                "desc": lines[i + 1],
                "qty": int(lines[i + 2]),
                "unit": money_to_float(lines[i + 3]),
                "line_total": money_to_float(lines[i + 4]),
            })
            i += 5
        else:
            i += 1

    return items, total


@dataclass
class ReceiptProfile:
    meta: Dict[str, Any]
    code_spend: Dict[str, float]
    code_qty: Dict[str, float]
    code_lines: Dict[str, int]


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptProfile:
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"

    empty_meta = dict(
        pdf_ok=0, pdf_n_lines=0, pdf_n_unique_codes=0,
        pdf_sum_qty=0.0,
        pdf_sum_line_total=0.0, pdf_mean_line_total=0.0, pdf_max_line_total=0.0, pdf_std_line_total=0.0,
        pdf_top1_cost_share=0.0, pdf_top3_cost_share=0.0,
        pdf_total=0.0, pdf_total_diff=0.0,
        pdf_insurance=None, pdf_zip3=None, pdf_parse_notes="missing_pdf",
    )

    if not pdf_path.exists():
        return ReceiptProfile(meta=empty_meta, code_spend={}, code_qty={}, code_lines={})

    try:
        with fitz.open(pdf_path) as doc:
            full_text = ""
            items: List[Dict[str, Any]] = []
            total: Optional[float] = None
            zip3 = None
            insurance = None

            for page in doc:
                text = page.get_text("text") or ""
                full_text += text + "\n"
                words = page.get_text("words") or []

                items_p, total_p = parse_items_from_words(words)
                if not items_p:
                    items_p, total_p = parse_items_fallback_text(text)

                if items_p:
                    items.extend(items_p)
                if total_p is not None:
                    total = total_p

            _, zip3, insurance = parse_header_fields(full_text)

    except Exception as e:
        empty_meta["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return ReceiptProfile(meta=empty_meta, code_spend={}, code_qty={}, code_lines={})

    if not items:
        empty_meta["pdf_parse_notes"] = "no_items_parsed"
        empty_meta["pdf_zip3"] = zip3
        empty_meta["pdf_insurance"] = insurance
        return ReceiptProfile(meta=empty_meta, code_spend={}, code_qty={}, code_lines={})

    code_spend: Dict[str, float] = {}
    code_qty: Dict[str, float] = {}
    code_lines: Dict[str, int] = {}

    for it in items:
        c = str(it["code"])
        q = float(it.get("qty", 0.0))
        lt = float(it.get("line_total", 0.0))
        code_spend[c] = code_spend.get(c, 0.0) + lt
        code_qty[c] = code_qty.get(c, 0.0) + q
        code_lines[c] = code_lines.get(c, 0) + 1

    line_totals = np.array([float(it.get("line_total", 0.0)) for it in items], dtype=float)
    qtys = np.array([float(it.get("qty", 0.0)) for it in items], dtype=float)
    sum_line_total = float(line_totals.sum())
    pdf_total = float(total) if total is not None else sum_line_total

    sorted_cost = np.sort(line_totals)[::-1]
    top1 = float(sorted_cost[0] / sum_line_total) if sum_line_total > 0 else 0.0
    top3 = float(sorted_cost[:3].sum() / sum_line_total) if sum_line_total > 0 else 0.0

    meta = dict(
        pdf_ok=1,
        pdf_n_lines=int(len(items)),
        pdf_n_unique_codes=int(len(code_spend)),
        pdf_sum_qty=float(qtys.sum()),
        pdf_sum_line_total=float(sum_line_total),
        pdf_mean_line_total=float(line_totals.mean()) if len(items) else 0.0,
        pdf_max_line_total=float(line_totals.max()) if len(items) else 0.0,
        pdf_std_line_total=float(line_totals.std()) if len(items) else 0.0,
        pdf_top1_cost_share=float(top1),
        pdf_top3_cost_share=float(top3),
        pdf_total=float(pdf_total),
        pdf_total_diff=float(abs(sum_line_total - pdf_total)),
        pdf_insurance=(insurance.lower() if isinstance(insurance, str) else None),
        pdf_zip3=(str(zip3) if zip3 is not None else None),
        pdf_parse_notes="ok",
    )

    return ReceiptProfile(meta=meta, code_spend=code_spend, code_qty=code_qty, code_lines=code_lines)


def load_cache(cache_path: Path) -> Dict[str, Any]:
    if cache_path.exists():
        try:
            obj = joblib.load(cache_path)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "data" in obj:
                return obj
        except Exception:
            pass
    return {"version": CACHE_VERSION, "data": {}}


def save_cache(cache_path: Path, cache_obj: Dict[str, Any]) -> None:
    try:
        joblib.dump(cache_obj, cache_path)
    except Exception as e:
        logging.warning(f"Cache save failed: {e}")


def build_receipt_artifacts(patient_ids: List[int], pdf_dir: Path, cache_path: Path):
    cache_obj = load_cache(cache_path)
    data: Dict[str, Any] = cache_obj.get("data", {})

    to_parse = [pid for pid in patient_ids if str(pid) not in data]
    if to_parse:
        logging.info(f"Receipt parsing: {len(to_parse):,} cache-miss PDFs to parse.")
    else:
        logging.info("Receipt parsing: all PDFs loaded from cache.")

    for pid in tqdm(to_parse, desc="Parsing receipts", ncols=100):
        prof = featurize_receipt(pid, pdf_dir)
        data[str(pid)] = {
            "meta": prof.meta,
            "code_spend": prof.code_spend,
            "code_qty": prof.code_qty,
            "code_lines": prof.code_lines,
        }

    cache_obj["data"] = data
    if to_parse:
        save_cache(cache_path, cache_obj)

    meta_rows = []
    profiles: Dict[int, Dict[str, Any]] = {}
    for pid in patient_ids:
        obj = data.get(str(pid), None)
        if obj is None:
            prof = featurize_receipt(pid, pdf_dir)
            obj = {
                "meta": prof.meta,
                "code_spend": prof.code_spend,
                "code_qty": prof.code_qty,
                "code_lines": prof.code_lines,
            }
        meta_rows.append({"patient_id": pid, **(obj["meta"] or {})})
        profiles[pid] = obj

    receipt_meta_df = pd.DataFrame(meta_rows)
    return receipt_meta_df, profiles


# =========================
# UNSUPERVISED REPRESENTATION (SVD + clustering)
# =========================
def build_code_vocab(profiles: Dict[int, Dict[str, Any]], patient_ids: List[int]) -> List[str]:
    vocab = set()
    for pid in patient_ids:
        d = profiles.get(pid, {}).get("code_spend", {}) or {}
        for c in d.keys():
            vocab.add(str(c))
    return sorted(vocab)


def build_sparse_code_matrices(
    patient_ids: List[int],
    profiles: Dict[int, Dict[str, Any]],
    code_to_idx: Dict[str, int],
) -> Tuple[csr_matrix, csr_matrix, csr_matrix]:
    """
    Returns:
      X_bin   (rows=patients, cols=codes): 1 if code present
      X_spend (rows=patients, cols=codes): raw spend
      X_qty   (rows=patients, cols=codes): raw qty
    """
    n_pat = len(patient_ids)
    n_code = len(code_to_idx)

    rb, cb, db = [], [], []
    rs, cs, ds = [], [], []
    rq, cq, dq = [], [], []

    for i, pid in enumerate(patient_ids):
        obj = profiles.get(pid, {}) or {}
        spend = obj.get("code_spend", {}) or {}
        qty = obj.get("code_qty", {}) or {}

        for c, sp in spend.items():
            c = str(c)
            j = code_to_idx.get(c, None)
            if j is None:
                continue
            rb.append(i); cb.append(j); db.append(1.0)
            rs.append(i); cs.append(j); ds.append(float(sp))
            rq.append(i); cq.append(j); dq.append(float(qty.get(c, 0.0)))

    X_bin = csr_matrix((np.asarray(db, dtype=np.float32), (np.asarray(rb), np.asarray(cb))), shape=(n_pat, n_code))
    X_spend = csr_matrix((np.asarray(ds, dtype=np.float32), (np.asarray(rs), np.asarray(cs))), shape=(n_pat, n_code))
    X_qty = csr_matrix((np.asarray(dq, dtype=np.float32), (np.asarray(rq), np.asarray(cq))), shape=(n_pat, n_code))
    return X_bin, X_spend, X_qty


def build_unsupervised_features(
    patient_ids: List[int],
    profiles: Dict[int, Dict[str, Any]],
    n_svd_tfidf: int,
    n_svd_spend: int,
    n_clusters: int,
    random_state: int = 2027,
) -> Tuple[pd.DataFrame, Dict[str, int], Dict[str, int]]:
    """
    Produces:
      - patient-level dense SVD features (TFIDF-LSA and SPEND-SVD)
      - patient-level bag-of-clusters features (counts + spend shares + entropy)
      - code_to_cluster mapping for TE/cluster encoding
    """
    logging.info("Building code vocabulary from receipts...")
    vocab = build_code_vocab(profiles, patient_ids)
    code_to_idx = {c: i for i, c in enumerate(vocab)}
    n_pat = len(patient_ids)
    n_code = len(vocab)
    logging.info(f"Vocabulary size: {n_code:,} codes | patients: {n_pat:,}")

    if n_code == 0:
        # No codes at all -> return empty features
        df = pd.DataFrame({"patient_id": patient_ids})
        return df, code_to_idx, {}

    logging.info("Building sparse code matrices (binary/spend/qty)...")
    X_bin, X_spend, X_qty = build_sparse_code_matrices(patient_ids, profiles, code_to_idx)

    # TF-IDF on binary presence
    logging.info("TF-IDF transform on binary matrix...")
    tfidf = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True, sublinear_tf=True)
    X_tfidf = tfidf.fit_transform(X_bin)

    # SVD on TF-IDF (LSA)
    k1 = int(max(2, min(n_svd_tfidf, n_pat - 1, n_code - 1)))
    logging.info(f"Fitting TruncatedSVD on TF-IDF: n_components={k1} ...")
    svd_tfidf = TruncatedSVD(n_components=k1, random_state=random_state, algorithm="randomized")
    Z_tfidf = svd_tfidf.fit_transform(X_tfidf).astype(np.float32)

    # Spend composition SVD (normalize rows to shares, then SVD)
    logging.info("Building spend-composition matrix (L1 normalize raw spend)...")
    X_sp_share = normalize(X_spend, norm="l1", axis=1, copy=True)
    k2 = int(max(2, min(n_svd_spend, n_pat - 1, n_code - 1)))
    logging.info(f"Fitting TruncatedSVD on spend-share: n_components={k2} ...")
    svd_spend = TruncatedSVD(n_components=k2, random_state=random_state + 1, algorithm="randomized")
    Z_spend = svd_spend.fit_transform(X_sp_share).astype(np.float32)

    # Cluster codes by their TF-IDF SVD embedding (statistical similarity)
    code_emb = svd_tfidf.components_.T  # (n_code, k1)
    k_clu = int(max(2, min(n_clusters, n_code)))
    logging.info(f"Clustering codes in embedding space: n_clusters={k_clu} ...")
    try:
        kmeans = MiniBatchKMeans(
            n_clusters=k_clu,
            random_state=random_state + 2,
            batch_size=2048,
            n_init="auto",
            max_iter=200,
        )
    except TypeError:
        kmeans = MiniBatchKMeans(
            n_clusters=k_clu,
            random_state=random_state + 2,
            batch_size=2048,
            n_init=10,
            max_iter=200,
        )

    code_cluster = kmeans.fit_predict(code_emb)
    idx_to_code = {i: c for c, i in code_to_idx.items()}
    code_to_cluster = {idx_to_code[i]: int(code_cluster[i]) for i in range(n_code)}

    # Patient bag-of-clusters features
    logging.info("Building patient bag-of-clusters features (count/share/entropy)...")
    clu_cnt = np.zeros((n_pat, k_clu), dtype=np.float32)
    clu_sp = np.zeros((n_pat, k_clu), dtype=np.float32)

    for i, pid in enumerate(patient_ids):
        obj = profiles.get(pid, {}) or {}
        spend = obj.get("code_spend", {}) or {}
        if not spend:
            continue
        total = float(sum(float(v) for v in spend.values()))
        total = max(total, 1e-9)

        for c, sp in spend.items():
            c = str(c)
            clu = code_to_cluster.get(c, None)
            if clu is None:
                continue
            clu_cnt[i, clu] += 1.0
            clu_sp[i, clu] += float(sp) / total  # spend share

    # Entropy + active clusters
    eps = 1e-12
    sp_p = np.clip(clu_sp, eps, 1.0)
    sp_p = sp_p / np.sum(sp_p, axis=1, keepdims=True)
    sp_entropy = -np.sum(sp_p * np.log(sp_p), axis=1).astype(np.float32)
    active_sp = (clu_sp > 0).sum(axis=1).astype(np.float32)

    cnt_sum = np.sum(clu_cnt, axis=1, keepdims=True)
    cnt_p = np.where(cnt_sum > 0, clu_cnt / (cnt_sum + eps), 0.0)
    cnt_p2 = np.clip(cnt_p, eps, 1.0)
    cnt_entropy = -np.sum(cnt_p2 * np.log(cnt_p2), axis=1).astype(np.float32)
    active_cnt = (clu_cnt > 0).sum(axis=1).astype(np.float32)

    # Build dataframe
    rows = {"patient_id": patient_ids}
    for j in range(k1):
        rows[f"lsa_tfidf_{j:03d}"] = Z_tfidf[:, j]
    for j in range(k2):
        rows[f"svd_spend_{j:03d}"] = Z_spend[:, j]

    for k in range(k_clu):
        rows[f"clu_cnt_{k:02d}"] = clu_cnt[:, k]
        rows[f"clu_sp_share_{k:02d}"] = clu_sp[:, k]

    rows["clu_sp_entropy"] = sp_entropy
    rows["clu_cnt_entropy"] = cnt_entropy
    rows["clu_active_sp"] = active_sp
    rows["clu_active_cnt"] = active_cnt

    feat_df = pd.DataFrame(rows)
    return feat_df, code_to_idx, code_to_cluster


# =========================
# BASE MATRIX (XGB)
# =========================
def build_base_sparse_matrix(df_all: pd.DataFrame) -> Tuple[csr_matrix, Dict[str, Any]]:
    df = df_all.copy()

    # derived tabular
    df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
    df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
    df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("missing")

    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train"}
    cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                if c in df.columns]
    num_cols = [c for c in df.columns
                if c not in exclude and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    X = preprocessor.fit_transform(df)
    X = X.tocsr() if hasattr(X, "tocsr") else csr_matrix(X)
    return X, {"preprocessor": preprocessor, "num_cols": num_cols, "cat_cols": cat_cols}


# =========================
# CV-SAFE TARGET ENCODING (exact code + learned cluster)
# =========================
@dataclass
class TokenStats:
    cnt: int
    mean_smooth: float
    median_adj: float


def build_te_stats_for_fold(
    train_pids: List[int],
    y_log: np.ndarray,
    pid_to_row: Dict[int, int],
    profiles: Dict[int, Dict[str, Any]],
    code_to_cluster: Dict[str, int],
) -> Tuple[Dict[str, TokenStats], Dict[int, TokenStats], float, float]:
    """
    Build stats for:
      - exact code tokens (string) => TokenStats
      - code clusters (int) => TokenStats
    """
    vals_code: Dict[str, List[float]] = {}
    vals_clu: Dict[int, List[float]] = {}
    ys = []

    for pid in train_pids:
        r = pid_to_row.get(pid, None)
        if r is None:
            continue
        v = float(y_log[r])
        ys.append(v)

        spend = profiles.get(pid, {}).get("code_spend", {}) or {}
        for c in spend.keys():
            c = str(c)
            vals_code.setdefault(c, []).append(v)
            clu = code_to_cluster.get(c, None)
            if clu is not None:
                vals_clu.setdefault(int(clu), []).append(v)

    gmean = float(np.mean(ys)) if ys else 0.0
    gmed = float(np.median(ys)) if ys else 0.0

    def to_stats_str(m: Dict[Any, List[float]]) -> Dict[Any, TokenStats]:
        out: Dict[Any, TokenStats] = {}
        for tok, arr in m.items():
            a = np.asarray(arr, dtype=float)
            cnt = int(a.size)
            mean = float(np.mean(a))
            med = float(np.median(a))
            mean_smooth = float((cnt * mean + TE_SMOOTH * gmean) / (cnt + TE_SMOOTH))
            median_adj = med if cnt >= TE_MIN_COUNT_MEDIAN else gmed
            out[tok] = TokenStats(cnt=cnt, mean_smooth=mean_smooth, median_adj=median_adj)
        return out

    return to_stats_str(vals_code), to_stats_str(vals_clu), gmean, gmed


def te_features_for_patients(
    pids: List[int],
    profiles: Dict[int, Dict[str, Any]],
    stats_code: Dict[str, TokenStats],
    stats_clu: Dict[int, TokenStats],
    code_to_cluster: Dict[str, int],
    gmean: float,
    gmed: float,
) -> np.ndarray:
    """
    16 TE features:
      exact-code (8) + cluster (8), spend-weighted.
    """
    feats = np.zeros((len(pids), 16), dtype=np.float32)

    for i, pid in enumerate(pids):
        spend = profiles.get(pid, {}).get("code_spend", {}) or {}
        if not spend:
            base = np.array([gmean, gmed, gmean, gmean, gmean, gmed, 0.0, 0.0], dtype=np.float32)
            feats[i, 0:8] = base
            feats[i, 8:16] = base
            continue

        codes = list(spend.keys())
        sp = np.asarray([float(spend[c]) for c in codes], dtype=float)
        ssum = float(sp.sum())
        w = (np.ones_like(sp) / max(1, sp.size)) if ssum <= 0 else (sp / (ssum + 1e-9))
        top = int(np.argmax(w)) if w.size else 0

        # ----- exact-code -----
        means, meds, cnts = [], [], []
        known = 0
        for c in codes:
            st = stats_code.get(str(c))
            if st is None:
                means.append(gmean); meds.append(gmed); cnts.append(0)
            else:
                means.append(st.mean_smooth); meds.append(st.median_adj); cnts.append(st.cnt); known += 1
        means = np.asarray(means, dtype=float)
        meds = np.asarray(meds, dtype=float)
        cnts = np.asarray(cnts, dtype=float)

        feats[i, 0:8] = np.array([
            float(np.sum(w * means)),
            float(np.sum(w * meds)),
            float(np.mean(means)),
            float(np.max(means)),
            float(np.min(means)),
            float(meds[top] if meds.size else gmed),
            float(known / max(1, len(codes))),
            float(np.mean(np.log1p(cnts))) if cnts.size else 0.0,
        ], dtype=np.float32)

        # ----- cluster -----
        clu_spend: Dict[int, float] = {}
        for c, v in spend.items():
            clu = code_to_cluster.get(str(c), None)
            if clu is None:
                continue
            clu_spend[int(clu)] = clu_spend.get(int(clu), 0.0) + float(v)

        if not clu_spend:
            base = np.array([gmean, gmed, gmean, gmean, gmean, gmed, 0.0, 0.0], dtype=np.float32)
            feats[i, 8:16] = base
            continue

        clus = list(clu_spend.keys())
        sp2 = np.asarray([float(clu_spend[k]) for k in clus], dtype=float)
        ssum2 = float(sp2.sum())
        w2 = (np.ones_like(sp2) / max(1, sp2.size)) if ssum2 <= 0 else (sp2 / (ssum2 + 1e-9))
        top2 = int(np.argmax(w2)) if w2.size else 0

        means2, meds2, cnts2 = [], [], []
        known2 = 0
        for k in clus:
            st = stats_clu.get(int(k))
            if st is None:
                means2.append(gmean); meds2.append(gmed); cnts2.append(0)
            else:
                means2.append(st.mean_smooth); meds2.append(st.median_adj); cnts2.append(st.cnt); known2 += 1

        means2 = np.asarray(means2, dtype=float)
        meds2 = np.asarray(meds2, dtype=float)
        cnts2 = np.asarray(cnts2, dtype=float)

        feats[i, 8:16] = np.array([
            float(np.sum(w2 * means2)),
            float(np.sum(w2 * meds2)),
            float(np.mean(means2)),
            float(np.max(means2)),
            float(np.min(means2)),
            float(meds2[top2] if meds2.size else gmed),
            float(known2 / max(1, len(clus))),
            float(np.mean(np.log1p(cnts2))) if cnts2.size else 0.0,
        ], dtype=np.float32)

    return feats


def make_strat_labels(train_df: pd.DataFrame) -> np.ndarray:
    chronic = train_df["primary_chronic"].astype(str)
    cost_bin = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    return (chronic + "_" + cost_bin).values


def best_blend_weight(y_true: np.ndarray, p1: np.ndarray, p2: np.ndarray, step: float) -> Tuple[float, float]:
    best_w = 0.5
    best_m = 1e18
    for w in np.arange(0.0, 1.0 + 1e-9, step):
        blend = w * p1 + (1.0 - w) * p2
        m = mae(y_true, blend)
        if m < best_m:
            best_m = m
            best_w = float(w)
    return best_w, best_m


# =========================
# MAIN
# =========================
def main() -> None:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb, '__version__', 'unknown')}")
    logging.info(f"FAST_MODE={FAST_MODE} | repeats={len(REPEAT_SEEDS)} | folds={N_FOLDS}")
    logging.info(f"GPU config: XGB_DEVICE={XGB_DEVICE}, CAT_TASK_TYPE={CAT_TASK_TYPE}")

    # load
    if not TRAIN_CSV.exists():
        raise FileNotFoundError(f"Missing: {TRAIN_CSV}")
    if not TEST_CSV.exists():
        raise FileNotFoundError(f"Missing: {TEST_CSV}")
    if not PDF_DIR.exists():
        raise FileNotFoundError(f"Missing PDF dir: {PDF_DIR}")

    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train_df.shape}, test={test_df.shape}")

    # optional patients.csv
    if PATIENTS_CSV.exists():
        patients = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in patients.columns:
            train_df = train_df.merge(patients, on="patient_id", how="left")
            test_df = test_df.merge(patients, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train_df.shape}, test={test_df.shape}")
        else:
            logging.warning("patients.csv exists but has no patient_id; skipping merge.")
    else:
        logging.info("patients.csv not found; continuing without it.")

    train_df["is_train"] = 1
    test_df["is_train"] = 0
    df_all = pd.concat([train_df, test_df], ignore_index=True)

    # receipts parsing (cached)
    all_pids = df_all["patient_id"].astype(int).unique().tolist()
    all_pids.sort()
    receipt_meta_df, profiles = build_receipt_artifacts(all_pids, PDF_DIR, CACHE_PATH)
    df_all = df_all.merge(receipt_meta_df, on="patient_id", how="left")
    logging.info(f"Receipt meta merged: df_all={df_all.shape}")

    # unsupervised features
    logging.info("=== Unsupervised code representation: SVD + clustering ===")
    unsup_df, code_to_idx, code_to_cluster = build_unsupervised_features(
        patient_ids=all_pids,
        profiles=profiles,
        n_svd_tfidf=N_SVD_TFIDF,
        n_svd_spend=N_SVD_SPEND,
        n_clusters=N_CODE_CLUSTERS,
        random_state=2027,
    )
    df_all = df_all.merge(unsup_df, on="patient_id", how="left")
    logging.info(f"Unsupervised features merged: df_all={df_all.shape} | clusters={len(set(code_to_cluster.values())) if code_to_cluster else 0}")

    # Build XGB base sparse matrix
    logging.info("Building XGB base sparse matrix (OHE cats + numeric)...")
    X_all_base, _ = build_base_sparse_matrix(df_all)
    logging.info(f"X_all_base shape: {X_all_base.shape}")

    is_train_mask = df_all["is_train"].values.astype(int) == 1
    X_train_base = X_all_base[is_train_mask]
    X_test_base = X_all_base[~is_train_mask]

    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)
    y_log = np.log1p(y)

    train_pids = train_df["patient_id"].astype(int).tolist()
    test_pids = test_df["patient_id"].astype(int).tolist()
    pid_to_row = {pid: i for i, pid in enumerate(train_pids)}

    # CatBoost base frames
    logging.info("Preparing CatBoost base frames...")
    df_cb_all = df_all.drop(columns=["ed_cost_next3y_usd"], errors="ignore").copy()
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df_cb_all.columns:
            df_cb_all[c] = df_cb_all[c].astype(str).fillna("missing")

    drop_cols = [c for c in ["patient_id", "is_train"] if c in df_cb_all.columns]
    df_cb_all = df_cb_all.drop(columns=drop_cols, errors="ignore")

    df_train_cb = df_cb_all[is_train_mask].reset_index(drop=True)
    df_test_cb = df_cb_all[~is_train_mask].reset_index(drop=True)

    cb_cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                   if c in df_train_cb.columns]
    cb_cat_idx = [df_train_cb.columns.get_loc(c) for c in cb_cat_cols]
    logging.info(f"CatBoost cat_features={cb_cat_cols}")

    # CV splits
    strat = make_strat_labels(train_df)
    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), strat)):
            splits.append((rep, fold, tr_idx, va_idx))
    logging.info(f"Total CV splits: {len(splits)}")

    # storage
    oof_xgb = np.zeros(len(y), dtype=np.float64)
    oof_cat = np.zeros(len(y), dtype=np.float64)
    cnt_xgb = np.zeros(len(y), dtype=np.float64)
    cnt_cat = np.zeros(len(y), dtype=np.float64)

    test_xgb_sum = np.zeros(len(test_df), dtype=np.float64)
    test_cat_sum = np.zeros(len(test_df), dtype=np.float64)
    n_xgb = 0
    n_cat = 0

    # Model params (GPU + heartbeats)
    xgb_params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=XGB_DEVICE,
        n_estimators=6000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_lambda=1.0,
        min_child_weight=1.0,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    cat_params = dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=8000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=6.0,
        bagging_temperature=0.8,
        random_strength=0.5,
        od_type="Iter",
        od_wait=EARLY_STOPPING,
        task_type=CAT_TASK_TYPE,
        devices="0",
        verbose=CAT_VERBOSE_EVERY,
    )

    te_cols = [f"te_{i:02d}" for i in range(16)]
    fit_sig_xgb = None

    # CV loop
    for rep, fold, tr_idx, va_idx in tqdm(splits, desc="CV folds", ncols=110):
        t0 = time.time()
        fold_tag = f"rep={rep} fold={fold}"
        logging.info(f"{fold_tag} | START | train={len(tr_idx)} val={len(va_idx)}")

        tr_pids = [train_pids[i] for i in tr_idx.tolist()]
        va_pids = [train_pids[i] for i in va_idx.tolist()]

        # TE maps from TRAIN fold only (no leakage)
        logging.info(f"{fold_tag} | TE stats build (codes + learned clusters)...")
        stats_code, stats_clu, gmean, gmed = build_te_stats_for_fold(
            train_pids=tr_pids,
            y_log=y_log,
            pid_to_row=pid_to_row,
            profiles=profiles,
            code_to_cluster=code_to_cluster,
        )

        logging.info(f"{fold_tag} | TE feature build (train/val/test)...")
        te_tr = te_features_for_patients(tr_pids, profiles, stats_code, stats_clu, code_to_cluster, gmean, gmed)
        te_va = te_features_for_patients(va_pids, profiles, stats_code, stats_clu, code_to_cluster, gmean, gmed)
        te_te = te_features_for_patients(test_pids, profiles, stats_code, stats_clu, code_to_cluster, gmean, gmed)

        # XGB matrices
        X_tr = hstack([X_train_base[tr_idx], csr_matrix(te_tr)], format="csr")
        X_va = hstack([X_train_base[va_idx], csr_matrix(te_va)], format="csr")
        X_te = hstack([X_test_base, csr_matrix(te_te)], format="csr")

        y_tr = y[tr_idx]
        y_va = y[va_idx]

        # ----- XGBoost -----
        logging.info(f"{fold_tag} | XGB fit START | device={xgb_params['device']} | verbose_every={XGB_VERBOSE_EVERY}")
        try:
            xgb_model = XGBRegressor(**xgb_params, random_state=1000 * rep + fold)

            if fit_sig_xgb is None:
                fit_sig_xgb = inspect.signature(xgb_model.fit).parameters

            fit_kwargs = {"eval_set": [(X_va, y_va)]}
            if "verbose_eval" in fit_sig_xgb:
                fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
            else:
                fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

            xgb_model.fit(X_tr, y_tr, **fit_kwargs)
            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_te)

            best_it = getattr(xgb_model, "best_iteration", None)
            logging.info(f"{fold_tag} | XGB fit END | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | XGB GPU failed: {e} -> fallback CPU")
            xgb_cpu = dict(xgb_params)
            xgb_cpu["device"] = "cpu"
            xgb_model = XGBRegressor(**xgb_cpu, random_state=1000 * rep + fold)

            fit_sig_xgb = inspect.signature(xgb_model.fit).parameters
            fit_kwargs = {"eval_set": [(X_va, y_va)]}
            if "verbose_eval" in fit_sig_xgb:
                fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
            else:
                fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

            logging.info(f"{fold_tag} | XGB (CPU fallback) fit START")
            xgb_model.fit(X_tr, y_tr, **fit_kwargs)
            logging.info(f"{fold_tag} | XGB (CPU fallback) fit END")
            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_te)

        if CLIP_AT_ZERO:
            p_va_x = np.maximum(p_va_x, 0.0)
            p_te_x = np.maximum(p_te_x, 0.0)

        oof_xgb[va_idx] += p_va_x
        cnt_xgb[va_idx] += 1.0
        test_xgb_sum += p_te_x
        n_xgb += 1

        # ----- CatBoost -----
        logging.info(f"{fold_tag} | CatBoost pools build...")
        te_train_full = np.zeros((len(train_df), 16), dtype=np.float32)
        te_train_full[tr_idx, :] = te_tr
        te_train_full[va_idx, :] = te_va

        df_tr = df_train_cb.copy()
        df_te = df_test_cb.copy()
        for j, col in enumerate(te_cols):
            df_tr[col] = te_train_full[:, j]
            df_te[col] = te_te[:, j]

        tr_pool = Pool(df_tr.iloc[tr_idx], label=y_tr, cat_features=cb_cat_idx)
        va_pool = Pool(df_tr.iloc[va_idx], label=y_va, cat_features=cb_cat_idx)
        te_pool = Pool(df_te, cat_features=cb_cat_idx)

        logging.info(f"{fold_tag} | CatBoost fit START | task_type={cat_params['task_type']} | verbose={cat_params['verbose']}")
        try:
            cat_model = CatBoostRegressor(**cat_params, random_seed=2000 * rep + fold)
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
            best_it = None
            try:
                best_it = cat_model.get_best_iteration()
            except Exception:
                best_it = None
            logging.info(f"{fold_tag} | CatBoost fit END | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | CatBoost GPU failed: {e} -> fallback CPU")
            cat_cpu = dict(cat_params)
            cat_cpu["task_type"] = "CPU"
            cat_cpu.pop("devices", None)
            cat_model = CatBoostRegressor(**cat_cpu, random_seed=2000 * rep + fold)

            logging.info(f"{fold_tag} | CatBoost (CPU fallback) fit START | verbose={cat_cpu['verbose']}")
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
            logging.info(f"{fold_tag} | CatBoost (CPU fallback) fit END")

        p_va_c = cat_model.predict(va_pool)
        p_te_c = cat_model.predict(te_pool)

        if CLIP_AT_ZERO:
            p_va_c = np.maximum(p_va_c, 0.0)
            p_te_c = np.maximum(p_te_c, 0.0)

        oof_cat[va_idx] += p_va_c
        cnt_cat[va_idx] += 1.0
        test_cat_sum += p_te_c
        n_cat += 1

        dt = time.time() - t0
        logging.info(f"{fold_tag} | END | fold_time={dt:.1f}s")

    # finalize
    oof_x = (oof_xgb / np.maximum(cnt_xgb, 1.0)).astype(np.float32)
    oof_c = (oof_cat / np.maximum(cnt_cat, 1.0)).astype(np.float32)
    te_x = (test_xgb_sum / max(1, n_xgb)).astype(np.float32)
    te_c = (test_cat_sum / max(1, n_cat)).astype(np.float32)

    logging.info(f"OOF MAE | XGB: {mae(y, oof_x):.4f}")
    logging.info(f"OOF MAE | CAT: {mae(y, oof_c):.4f}")

    # blend + median bias (MAE-optimal shift)
    w, blend_mae = best_blend_weight(y, oof_x, oof_c, step=BLEND_STEP)
    oof_blend = w * oof_x + (1.0 - w) * oof_c
    bias = float(np.median(y - oof_blend))
    oof_final = oof_blend + bias
    test_final = w * te_x + (1.0 - w) * te_c + bias

    if CLIP_AT_ZERO:
        oof_final = np.maximum(oof_final, 0.0)
        test_final = np.maximum(test_final, 0.0)

    logging.info(f"Blend w(XGB)={w:.2f} | OOF MAE pre-bias={blend_mae:.4f} | bias={bias:.4f} | OOF MAE final={mae(y, oof_final):.4f}")

    # Save submission
    sub = pd.DataFrame({
        "patient_id": test_df["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": test_final.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")


if __name__ == "__main__":
    main()


2026-02-13 12:26:24,621 | INFO | XGBoost version: 3.0.0
2026-02-13 12:26:24,621 | INFO | FAST_MODE=True | repeats=2 | folds=5
2026-02-13 12:26:24,621 | INFO | GPU config: XGB_DEVICE=cuda, CAT_TASK_TYPE=GPU
2026-02-13 12:26:24,627 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 12:26:24,639 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 12:26:24,809 | INFO | Receipt parsing: all PDFs loaded from cache.
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 12:26:24,827 | INFO | Receipt meta merged: df_all=(4000, 25)
2026-02-13 12:26:24,833 | INFO | === Unsupervised code representation: SVD + clustering ===
2026-02-13 12:26:24,833 | INFO | Building code vocabulary from receipts...
2026-02-13 12:26:24,833 | INFO | Vocabulary size: 18 codes | patients: 4,000
2026-02-13 12:26:24,833 | INFO | Building sparse code matrices (binary/spend/qty)...
2026-02-13 12:26:24,848 | INFO | TF-IDF transform on binary matrix...
2026-02-13 12:26:24,854 | INFO | Fitting Truncated

[0]	validation_0-mae:1329.92674
[100]	validation_0-mae:465.32523


CV folds:   0%|                                                                        | 0/10 [00:02<?, ?it/s]


KeyboardInterrupt: 

# Iteration 5
- 488.6998

In [17]:
# -*- coding: utf-8 -*-
r"""
ITERATION 5 — "Scale the Simple" (Sparse TF-IDF + Sparse Spend-Share, No SVD, No Manual Groupings)
=================================================================================================

Why this should beat Iteration 2 (478.4) without falling into the complexity trap:
- Keeps the proven "blind" sparse token signal (TF-IDF on raw billing codes).
- Adds ONE extra sparse view that TF-IDF ignores: per-code spend composition (spend share per code),
  still high-dimensional + sparse (no embeddings, no dimensionality reduction).
- Adds a tiny MAE-aligned calibration: optional per-primary_chronic median bias correction
  (constant shift per group), which often improves MAE for biased residuals.

Hard constraints respected:
- NO SVD/PCA/NMF/embeddings.
- NO manual CPT range buckets or clinical rules.
- GPU enabled for XGBoost; CatBoost GPU attempted but safely falls back if needed.
- Rich, timestamped logs + training heartbeats.

===========================================================
INSTALL (run once)
===========================================================
pip install -U pandas numpy scipy scikit-learn xgboost catboost pymupdf joblib tqdm
"""

from __future__ import annotations

import re
import math
import time
import logging
import warnings
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfTransformer

import xgboost as xgb
from xgboost import XGBRegressor

from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# PATHS
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"          # optional
PDF_DIR = BASE_DIR / "receipts_pdf"

SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

CACHE_PATH = BASE_DIR / "cache_receipts_iter5_sparse.joblib"
CACHE_VERSION = 51


# =========================
# TRAINING CONFIG
# =========================
FAST_MODE = True
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026] if FAST_MODE else [2024, 2025, 2026]
EARLY_STOPPING = 200

# Text / sparse code settings
CODE_MIN_DF = 1         # keep as 1 to match "don't throw info away"; raise to 2 if overfitting suspected

# GPU
XGB_DEVICE = "cuda"
CAT_TASK_TYPE = "GPU"

# Heartbeats
XGB_VERBOSE_EVERY = 100
CAT_VERBOSE_EVERY = 100

# Post-process
CLIP_AT_ZERO = True

# Blend
BLEND_STEP = 0.02

# Calibration
USE_GROUP_BIAS_BY_CHRONIC = True


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


# =========================
# UTIL
# =========================
_money_re = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d{2})$|^\d+(?:\.\d{2})$")
_code_re = re.compile(r"^(?:[A-Z]\d{4}|\d{4,5})$")

LINE_ITEM_RE = re.compile(r"^([A-Z]\d{4}|\d{4,5})\s+(.+?)\s+(\d+)\s+([\d,]+\.\d{2})\s+([\d,]+\.\d{2})$")
TOTAL_RE = re.compile(r"^TOTAL\s+([\d,]+\.\d{2})$")

def is_money(tok: str) -> bool:
    return bool(_money_re.match(tok))

def money_to_float(tok: str) -> float:
    return float(tok.replace(",", ""))

def is_code(tok: str) -> bool:
    return bool(_code_re.match(tok))

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))

def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def ensure_csr_float32_int32(X: csr_matrix) -> csr_matrix:
    X = X.tocsr()
    if X.data.dtype != np.float32:
        X.data = X.data.astype(np.float32, copy=False)
    # CatBoost sometimes dislikes int64 indices; keep int32 to be safe
    if X.indices.dtype != np.int32:
        X.indices = X.indices.astype(np.int32, copy=False)
    if X.indptr.dtype != np.int32:
        X.indptr = X.indptr.astype(np.int32, copy=False)
    return X


# =========================
# PDF PARSING (simple + robust)
# =========================
def parse_header_fields(text: str) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    pid = None
    zip3 = None
    insurance = None

    m = re.search(r"Patient ID:\s*(\d+)", text)
    if m:
        pid = int(m.group(1))

    m = re.search(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)", text)
    if m:
        zip3 = str(m.group(1))
        insurance = str(m.group(2)).lower()

    return pid, zip3, insurance


@dataclass
class ReceiptProfile:
    meta: Dict[str, Any]
    code_cnt: Dict[str, float]
    code_spend: Dict[str, float]
    code_qty: Dict[str, float]


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptProfile:
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"

    empty_meta = dict(
        pdf_ok=0,
        pdf_n_lines=0,
        pdf_n_unique_codes=0,
        pdf_sum_qty=0.0,
        pdf_sum_line_total=0.0,
        pdf_total=0.0,
        pdf_total_diff=0.0,
        pdf_insurance=None,
        pdf_zip3=None,
        pdf_parse_notes="missing_pdf",
    )

    if not pdf_path.exists():
        return ReceiptProfile(meta=empty_meta, code_cnt={}, code_spend={}, code_qty={})

    try:
        with fitz.open(pdf_path) as doc:
            full_text = ""
            for page in doc:
                full_text += (page.get_text("text") or "") + "\n"
    except Exception as e:
        empty_meta["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return ReceiptProfile(meta=empty_meta, code_cnt={}, code_spend={}, code_qty={})

    # header fields
    _, zip3, insurance = parse_header_fields(full_text)

    # parse line items from text lines
    lines = [ln.strip() for ln in full_text.splitlines() if ln.strip()]
    if not lines:
        empty_meta["pdf_parse_notes"] = "empty_text"
        empty_meta["pdf_zip3"] = zip3
        empty_meta["pdf_insurance"] = insurance
        return ReceiptProfile(meta=empty_meta, code_cnt={}, code_spend={}, code_qty={})

    # find table header
    start = None
    for i, ln in enumerate(lines):
        if ln.lower().startswith("code description qty"):
            start = i + 1
            break
    if start is None:
        # fallback: try the exact header fragment
        for i, ln in enumerate(lines):
            if "code description" in ln.lower() and "line total" in ln.lower():
                start = i + 1
                break

    if start is None:
        empty_meta["pdf_parse_notes"] = "no_table_header"
        empty_meta["pdf_zip3"] = zip3
        empty_meta["pdf_insurance"] = insurance
        return ReceiptProfile(meta=empty_meta, code_cnt={}, code_spend={}, code_qty={})

    code_cnt: Dict[str, float] = {}
    code_spend: Dict[str, float] = {}
    code_qty: Dict[str, float] = {}
    total_val: Optional[float] = None
    sum_line_total = 0.0
    sum_qty = 0.0

    for ln in lines[start:]:
        m_tot = TOTAL_RE.match(ln)
        if m_tot:
            total_val = money_to_float(m_tot.group(1))
            break

        m = LINE_ITEM_RE.match(ln)
        if m:
            code = m.group(1)
            qty = float(m.group(3))
            line_total = money_to_float(m.group(5))

            code_cnt[code] = code_cnt.get(code, 0.0) + 1.0
            code_qty[code] = code_qty.get(code, 0.0) + qty
            code_spend[code] = code_spend.get(code, 0.0) + line_total

            sum_line_total += line_total
            sum_qty += qty
        else:
            # very light fallback: split-based heuristic
            parts = ln.split()
            if len(parts) >= 5 and is_code(parts[0]) and parts[-3].isdigit() and is_money(parts[-2]) and is_money(parts[-1]):
                code = parts[0]
                qty = float(parts[-3])
                line_total = money_to_float(parts[-1])

                code_cnt[code] = code_cnt.get(code, 0.0) + 1.0
                code_qty[code] = code_qty.get(code, 0.0) + qty
                code_spend[code] = code_spend.get(code, 0.0) + line_total

                sum_line_total += line_total
                sum_qty += qty

    if not code_cnt:
        empty_meta["pdf_parse_notes"] = "no_items_parsed"
        empty_meta["pdf_zip3"] = zip3
        empty_meta["pdf_insurance"] = insurance
        return ReceiptProfile(meta=empty_meta, code_cnt={}, code_spend={}, code_qty={})

    pdf_total = float(total_val) if total_val is not None else float(sum_line_total)
    meta = dict(
        pdf_ok=1,
        pdf_n_lines=int(sum(code_cnt.values())),
        pdf_n_unique_codes=int(len(code_cnt)),
        pdf_sum_qty=float(sum_qty),
        pdf_sum_line_total=float(sum_line_total),
        pdf_total=float(pdf_total),
        pdf_total_diff=float(abs(pdf_total - sum_line_total)),
        pdf_insurance=(insurance.lower() if isinstance(insurance, str) else None),
        pdf_zip3=(str(zip3) if zip3 is not None else None),
        pdf_parse_notes="ok" if total_val is not None else "ok_no_total",
    )

    return ReceiptProfile(meta=meta, code_cnt=code_cnt, code_spend=code_spend, code_qty=code_qty)


def load_cache(cache_path: Path) -> Dict[str, Any]:
    if cache_path.exists():
        try:
            obj = joblib.load(cache_path)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "data" in obj:
                return obj
        except Exception:
            pass
    return {"version": CACHE_VERSION, "data": {}}


def save_cache(cache_path: Path, cache_obj: Dict[str, Any]) -> None:
    try:
        joblib.dump(cache_obj, cache_path)
    except Exception as e:
        logging.warning(f"Cache save failed: {e}")


def build_receipt_artifacts(patient_ids: List[int], pdf_dir: Path, cache_path: Path):
    cache_obj = load_cache(cache_path)
    data: Dict[str, Any] = cache_obj.get("data", {})

    to_parse = [pid for pid in patient_ids if str(pid) not in data]
    logging.info(f"Receipt cache: {len(patient_ids)-len(to_parse):,} hit | {len(to_parse):,} miss")

    for pid in tqdm(to_parse, desc="Parsing receipts", ncols=100):
        prof = featurize_receipt(pid, pdf_dir)
        data[str(pid)] = {
            "meta": prof.meta,
            "code_cnt": prof.code_cnt,
            "code_spend": prof.code_spend,
            "code_qty": prof.code_qty,
        }

    cache_obj["data"] = data
    if to_parse:
        save_cache(cache_path, cache_obj)

    meta_rows = []
    profiles: Dict[int, Dict[str, Any]] = {}
    for pid in patient_ids:
        obj = data.get(str(pid), None)
        if obj is None:
            prof = featurize_receipt(pid, pdf_dir)
            obj = {"meta": prof.meta, "code_cnt": prof.code_cnt, "code_spend": prof.code_spend, "code_qty": prof.code_qty}

        meta_rows.append({"patient_id": pid, **(obj.get("meta", {}) or {})})
        profiles[pid] = obj

    receipt_meta_df = pd.DataFrame(meta_rows)
    return receipt_meta_df, profiles


# =========================
# SPARSE CODE MATRICES (NO SVD, NO GROUPING)
# =========================
def build_code_vocab_and_df(profiles: Dict[int, Dict[str, Any]], patient_ids: List[int]) -> Tuple[List[str], Dict[str, int]]:
    df = {}
    for pid in patient_ids:
        codes = (profiles.get(pid, {}) or {}).get("code_cnt", {}) or {}
        for c in codes.keys():
            df[c] = df.get(c, 0) + 1
    # filter by min df
    vocab = [c for c, cdf in df.items() if cdf >= CODE_MIN_DF]
    vocab.sort()
    code_to_idx = {c: i for i, c in enumerate(vocab)}
    return vocab, code_to_idx


def build_sparse_code_views(
    patient_ids: List[int],
    profiles: Dict[int, Dict[str, Any]],
    code_to_idx: Dict[str, int],
) -> Tuple[csr_matrix, csr_matrix]:
    """
    Returns:
      X_bin   : 1 if code appears for patient
      X_share : spend share for code (sum(code_spend)/total_spend_patient)
    """
    n_pat = len(patient_ids)
    n_code = len(code_to_idx)

    rb, cb, db = [], [], []
    rs, cs, ds = [], [], []

    for i, pid in enumerate(patient_ids):
        obj = profiles.get(pid, {}) or {}
        cnt = obj.get("code_cnt", {}) or {}
        spend = obj.get("code_spend", {}) or {}
        total_spend = float(sum(float(v) for v in spend.values()))
        total_spend = max(total_spend, 1e-9)

        for c in cnt.keys():
            j = code_to_idx.get(str(c), None)
            if j is None:
                continue
            rb.append(i); cb.append(j); db.append(1.0)

        for c, sp in spend.items():
            j = code_to_idx.get(str(c), None)
            if j is None:
                continue
            rs.append(i); cs.append(j); ds.append(float(sp) / total_spend)

    X_bin = csr_matrix((np.asarray(db, dtype=np.float32), (np.asarray(rb), np.asarray(cb))), shape=(n_pat, n_code))
    X_share = csr_matrix((np.asarray(ds, dtype=np.float32), (np.asarray(rs), np.asarray(cs))), shape=(n_pat, n_code))
    return X_bin, X_share


# =========================
# TABULAR FEATURES
# =========================
def build_tabular_sparse_matrix(df_all: pd.DataFrame) -> Tuple[csr_matrix, Dict[str, Any]]:
    df = df_all.copy()

    # basic derived features (simple + safe)
    df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
    df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
    df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

    # normalize likely categorical columns if present
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("missing")

    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train"}
    cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                if c in df.columns]
    num_cols = [c for c in df.columns
                if c not in exclude and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    X = preprocessor.fit_transform(df)
    X = X.tocsr() if hasattr(X, "tocsr") else csr_matrix(X)
    return X, {"preprocessor": preprocessor, "num_cols": num_cols, "cat_cols": cat_cols}


def make_strat_labels(train_df: pd.DataFrame) -> np.ndarray:
    chronic = train_df["primary_chronic"].astype(str)
    cost_bin = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    return (chronic + "_" + cost_bin).values


def best_blend_weight(y_true: np.ndarray, p1: np.ndarray, p2: np.ndarray, step: float) -> Tuple[float, float]:
    best_w = 0.5
    best_m = 1e18
    for w in np.arange(0.0, 1.0 + 1e-9, step):
        blend = w * p1 + (1.0 - w) * p2
        m = mae(y_true, blend)
        if m < best_m:
            best_m = m
            best_w = float(w)
    return best_w, best_m


# =========================
# MAIN
# =========================
def main() -> None:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb, '__version__', 'unknown')}")
    logging.info(f"FAST_MODE={FAST_MODE} | repeats={len(REPEAT_SEEDS)} | folds={N_FOLDS}")
    logging.info(f"GPU config: XGB_DEVICE={XGB_DEVICE}, CAT_TASK_TYPE={CAT_TASK_TYPE}")

    # load
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train_df.shape}, test={test_df.shape}")

    # optional patients
    if PATIENTS_CSV.exists():
        patients = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in patients.columns:
            train_df = train_df.merge(patients, on="patient_id", how="left")
            test_df = test_df.merge(patients, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train_df.shape}, test={test_df.shape}")
        else:
            logging.warning("patients.csv exists but no patient_id; skipping.")

    train_df["is_train"] = 1
    test_df["is_train"] = 0
    df_all = pd.concat([train_df, test_df], ignore_index=True)

    # parse receipts
    all_pids = df_all["patient_id"].astype(int).unique().tolist()
    all_pids.sort()
    receipt_meta_df, profiles = build_receipt_artifacts(all_pids, PDF_DIR, CACHE_PATH)
    df_all = df_all.merge(receipt_meta_df, on="patient_id", how="left")
    logging.info(f"Merged receipt meta: df_all={df_all.shape}")

    # build code vocab + sparse code matrices
    logging.info("Building sparse code views (TF-IDF codes + spend-share codes)...")
    vocab, code_to_idx = build_code_vocab_and_df(profiles, all_pids)
    logging.info(f"Code vocab size={len(vocab):,} (CODE_MIN_DF={CODE_MIN_DF})")

    if len(vocab) == 0:
        logging.warning("No codes parsed at all. Falling back to tabular-only.")
        X_code_all = None
    else:
        X_bin, X_share = build_sparse_code_views(all_pids, profiles, code_to_idx)

        # TF-IDF on binary presence
        tfidf = TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True, sublinear_tf=False)
        X_tfidf = tfidf.fit_transform(X_bin)

        # Keep spend-share raw (already normalized per patient)
        X_code_all = hstack([X_tfidf, X_share], format="csr")
        X_code_all = ensure_csr_float32_int32(X_code_all)

        logging.info(f"X_code_all shape={X_code_all.shape} | nnz={X_code_all.nnz:,}")

    # tabular sparse
    logging.info("Building tabular sparse matrix (OHE + numeric)...")
    X_tab_all, _ = build_tabular_sparse_matrix(df_all)
    X_tab_all = ensure_csr_float32_int32(X_tab_all)
    logging.info(f"X_tab_all shape={X_tab_all.shape} | nnz={X_tab_all.nnz:,}")

    # combine
    if X_code_all is None:
        X_all = X_tab_all
    else:
        X_all = hstack([X_tab_all, X_code_all], format="csr")
        X_all = ensure_csr_float32_int32(X_all)

    logging.info(f"FINAL X_all shape={X_all.shape} | nnz={X_all.nnz:,}")

    is_train_mask = df_all["is_train"].values.astype(int) == 1
    X_train = X_all[is_train_mask]
    X_test = X_all[~is_train_mask]

    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)

    # CV splits
    strat = make_strat_labels(train_df)
    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), strat)):
            splits.append((rep, fold, tr_idx, va_idx))
    logging.info(f"Total CV splits: {len(splits)}")

    # storage
    oof_xgb = np.zeros(len(y), dtype=np.float64)
    oof_cat = np.zeros(len(y), dtype=np.float64)
    cnt_xgb = np.zeros(len(y), dtype=np.float64)
    cnt_cat = np.zeros(len(y), dtype=np.float64)

    test_xgb_sum = np.zeros(X_test.shape[0], dtype=np.float64)
    test_cat_sum = np.zeros(X_test.shape[0], dtype=np.float64)
    n_xgb = 0
    n_cat = 0

    # model params (kept conservative; aligned to MAE)
    xgb_params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=XGB_DEVICE,
        n_estimators=6000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_lambda=1.0,
        min_child_weight=1.0,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    cat_params = dict(
        loss_function="MAE",
        eval_metric="MAE",
        iterations=8000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=6.0,
        bagging_temperature=0.8,
        random_strength=0.5,
        od_type="Iter",
        od_wait=EARLY_STOPPING,
        task_type=CAT_TASK_TYPE,
        devices="0",
        verbose=CAT_VERBOSE_EVERY,
    )

    fit_sig_xgb = None

    # CV loop
    for rep, fold, tr_idx, va_idx in tqdm(splits, desc="CV folds", ncols=110):
        t0 = time.time()
        fold_tag = f"rep={rep} fold={fold}"
        logging.info(f"{fold_tag} | START | train={len(tr_idx)} val={len(va_idx)}")

        X_tr = X_train[tr_idx]
        X_va = X_train[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        # ----- XGBoost -----
        logging.info(f"{fold_tag} | XGB fit START | device={xgb_params['device']} | verbose_every={XGB_VERBOSE_EVERY}")
        try:
            xgb_model = XGBRegressor(**xgb_params, random_state=1000 * rep + fold)

            if fit_sig_xgb is None:
                fit_sig_xgb = inspect.signature(xgb_model.fit).parameters

            fit_kwargs = {"eval_set": [(X_va, y_va)]}
            # For sklearn API, verbose=int prints metrics every N boosting stages when eval_set is provided
            if "verbose_eval" in fit_sig_xgb:
                fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
            else:
                fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

            xgb_model.fit(X_tr, y_tr, **fit_kwargs)
            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_test)

            best_it = getattr(xgb_model, "best_iteration", None)
            logging.info(f"{fold_tag} | XGB fit END | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | XGB GPU failed: {e} -> fallback CPU")
            xgb_cpu = dict(xgb_params)
            xgb_cpu["device"] = "cpu"
            xgb_model = XGBRegressor(**xgb_cpu, random_state=1000 * rep + fold)

            fit_sig_xgb = inspect.signature(xgb_model.fit).parameters
            fit_kwargs = {"eval_set": [(X_va, y_va)]}
            if "verbose_eval" in fit_sig_xgb:
                fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
            else:
                fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

            logging.info(f"{fold_tag} | XGB (CPU fallback) fit START")
            xgb_model.fit(X_tr, y_tr, **fit_kwargs)
            logging.info(f"{fold_tag} | XGB (CPU fallback) fit END")
            p_va_x = xgb_model.predict(X_va)
            p_te_x = xgb_model.predict(X_test)

        if CLIP_AT_ZERO:
            p_va_x = np.maximum(p_va_x, 0.0)
            p_te_x = np.maximum(p_te_x, 0.0)

        oof_xgb[va_idx] += p_va_x
        cnt_xgb[va_idx] += 1.0
        test_xgb_sum += p_te_x
        n_xgb += 1

        # ----- CatBoost (sparse numeric) -----
        logging.info(f"{fold_tag} | CatBoost fit START | task_type={cat_params['task_type']} | verbose={cat_params['verbose']}")
        try:
            tr_pool = Pool(X_tr, label=y_tr)
            va_pool = Pool(X_va, label=y_va)
            te_pool = Pool(X_test)

            cat_model = CatBoostRegressor(**cat_params, random_seed=2000 * rep + fold)
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)

            p_va_c = cat_model.predict(va_pool)
            p_te_c = cat_model.predict(te_pool)

            best_it = None
            try:
                best_it = cat_model.get_best_iteration()
            except Exception:
                best_it = None
            logging.info(f"{fold_tag} | CatBoost fit END | best_iteration={best_it}")

        except Exception as e:
            logging.warning(f"{fold_tag} | CatBoost GPU failed: {e} -> fallback CPU")
            cat_cpu = dict(cat_params)
            cat_cpu["task_type"] = "CPU"
            cat_cpu.pop("devices", None)
            cat_model = CatBoostRegressor(**cat_cpu, random_seed=2000 * rep + fold)

            tr_pool = Pool(X_tr, label=y_tr)
            va_pool = Pool(X_va, label=y_va)
            te_pool = Pool(X_test)

            logging.info(f"{fold_tag} | CatBoost (CPU fallback) fit START | verbose={cat_cpu['verbose']}")
            cat_model.fit(tr_pool, eval_set=va_pool, use_best_model=True)
            logging.info(f"{fold_tag} | CatBoost (CPU fallback) fit END")

            p_va_c = cat_model.predict(va_pool)
            p_te_c = cat_model.predict(te_pool)

        if CLIP_AT_ZERO:
            p_va_c = np.maximum(p_va_c, 0.0)
            p_te_c = np.maximum(p_te_c, 0.0)

        oof_cat[va_idx] += p_va_c
        cnt_cat[va_idx] += 1.0
        test_cat_sum += p_te_c
        n_cat += 1

        dt = time.time() - t0
        # quick fold-level MAE snapshots
        fold_mae_x = mae(y_va, p_va_x)
        fold_mae_c = mae(y_va, p_va_c)
        logging.info(f"{fold_tag} | END | fold_time={dt:.1f}s | fold_MAE(XGB)={fold_mae_x:.3f} | fold_MAE(CAT)={fold_mae_c:.3f}")

    # finalize OOF/test
    oof_x = (oof_xgb / np.maximum(cnt_xgb, 1.0)).astype(np.float32)
    oof_c = (oof_cat / np.maximum(cnt_cat, 1.0)).astype(np.float32)
    te_x = (test_xgb_sum / max(1, n_xgb)).astype(np.float32)
    te_c = (test_cat_sum / max(1, n_cat)).astype(np.float32)

    mae_x = mae(y, oof_x)
    mae_c = mae(y, oof_c)
    logging.info(f"OOF MAE | XGB: {mae_x:.4f}")
    logging.info(f"OOF MAE | CAT: {mae_c:.4f}")

    # blend weight on OOF
    w, blend_mae = best_blend_weight(y, oof_x, oof_c, step=BLEND_STEP)
    oof_blend = w * oof_x + (1.0 - w) * oof_c
    test_blend = w * te_x + (1.0 - w) * te_c
    logging.info(f"Blend search | best w(XGB)={w:.2f} | OOF MAE={blend_mae:.4f}")

    # MAE-aligned bias correction (global and optional per-chronic)
    train_chronic = train_df["primary_chronic"].astype(str).values
    test_chronic = test_df["primary_chronic"].astype(str).values

    # global bias
    bias_global = float(np.median(y - oof_blend))
    oof_global = oof_blend + bias_global
    test_global = test_blend + bias_global
    mae_global = mae(y, oof_global)
    logging.info(f"Calibration(global) | bias={bias_global:.4f} | OOF MAE={mae_global:.4f}")

    # per-chronic bias (very low complexity; 3 groups)
    if USE_GROUP_BIAS_BY_CHRONIC:
        biases = {}
        for g in np.unique(train_chronic):
            mask = (train_chronic == g)
            if mask.sum() == 0:
                continue
            biases[g] = float(np.median(y[mask] - oof_blend[mask]))

        oof_group = oof_blend.copy()
        for g, b in biases.items():
            oof_group[train_chronic == g] += b

        test_group = test_blend.copy()
        for i in range(len(test_group)):
            b = biases.get(test_chronic[i], 0.0)
            test_group[i] += b

        mae_group = mae(y, oof_group)
        logging.info(f"Calibration(by_chronic) | biases={biases} | OOF MAE={mae_group:.4f}")

        if mae_group <= mae_global:
            oof_final = oof_group
            test_final = test_group
            logging.info("Using per-chronic bias correction (better or equal OOF MAE).")
        else:
            oof_final = oof_global
            test_final = test_global
            logging.info("Using global bias correction (better OOF MAE).")
    else:
        oof_final = oof_global
        test_final = test_global

    if CLIP_AT_ZERO:
        oof_final = np.maximum(oof_final, 0.0)
        test_final = np.maximum(test_final, 0.0)

    logging.info(f"FINAL OOF MAE (post-calibration): {mae(y, oof_final):.4f}")

    # save submission
    sub = pd.DataFrame({
        "patient_id": test_df["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": test_final.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")


if __name__ == "__main__":
    main()


2026-02-13 14:13:18,294 | INFO | XGBoost version: 3.0.0
2026-02-13 14:13:18,294 | INFO | FAST_MODE=True | repeats=2 | folds=5
2026-02-13 14:13:18,294 | INFO | GPU config: XGB_DEVICE=cuda, CAT_TASK_TYPE=GPU
2026-02-13 14:13:18,309 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 14:13:18,318 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 14:13:18,321 | INFO | Receipt cache: 0 hit | 4,000 miss
Parsing receipts: 100%|████████████████████████████████████████| 4000/4000 [00:06<00:00, 610.58it/s]
2026-02-13 14:13:25,013 | INFO | Merged receipt meta: df_all=(4000, 20)
2026-02-13 14:13:25,013 | INFO | Building sparse code views (TF-IDF codes + spend-share codes)...
2026-02-13 14:13:25,017 | INFO | Code vocab size=0 (CODE_MIN_DF=1)
2026-02-13 14:13:25,018 | WARNING | No codes parsed at all. Falling back to tabular-only.
2026-02-13 14:13:25,018 | INFO | Building tabular sparse matrix (OHE + numeric)...
2026-02-13 14:13:25,122 | INFO | X_tab_all shape=(4000, 45

[0]	validation_0-mae:1328.22824
[100]	validation_0-mae:482.57967
[200]	validation_0-mae:481.00177
[300]	validation_0-mae:483.76434
[375]	validation_0-mae:486.26440


2026-02-13 14:13:27,004 | INFO | rep=0 fold=0 | XGB fit END | best_iteration=176
2026-02-13 14:13:27,010 | INFO | rep=0 fold=0 | CatBoost fit START | task_type=GPU | verbose=100
Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1444.3681250	test: 1388.4382813	best: 1388.4382813 (0)	total: 13.9ms	remaining: 1m 51s
100:	learn: 1442.7225000	test: 1386.8281250	best: 1386.8281250 (100)	total: 1.14s	remaining: 1m 28s
200:	learn: 1441.0703125	test: 1385.2179687	best: 1385.2179687 (200)	total: 2.16s	remaining: 1m 23s
300:	learn: 1439.4123437	test: 1383.6134375	best: 1383.6134375 (300)	total: 3.35s	remaining: 1m 25s
400:	learn: 1437.7543750	test: 1382.0068750	best: 1382.0068750 (400)	total: 4.48s	remaining: 1m 24s
500:	learn: 1436.0965625	test: 1380.4009375	best: 1380.4009375 (500)	total: 5.56s	remaining: 1m 23s
600:	learn: 1434.4415625	test: 1378.7984375	best: 1378.7984375 (600)	total: 6.65s	remaining: 1m 21s
700:	learn: 1432.7901563	test: 1377.1946875	best: 1377.1946875 (700)	total: 8.22s	remaining: 1m 25s
800:	learn: 1431.1456250	test: 1375.5937500	best: 1375.5937500 (800)	total: 9.75s	remaining: 1m 27s
900:	learn: 1429.5034375	test: 1373.9931250	best: 1373.9931250 (900)	total: 11.1s	remaining: 1m 27s
100

2026-02-13 14:15:12,533 | INFO | rep=0 fold=0 | CatBoost fit END | best_iteration=7999
2026-02-13 14:15:12,533 | INFO | rep=0 fold=0 | END | fold_time=107.4s | fold_MAE(XGB)=479.541 | fold_MAE(CAT)=1263.119
CV folds:  10%|██████▎                                                        | 1/10 [01:47<16:06, 107.39s/it]2026-02-13 14:15:12,533 | INFO | rep=0 fold=1 | START | train=1600 val=400
2026-02-13 14:15:12,539 | INFO | rep=0 fold=1 | XGB fit START | device=cuda | verbose_every=100


...

2026-02-13 14:29:52,229 | INFO | rep=1 fold=3 | CatBoost fit END | best_iteration=7999
2026-02-13 14:29:52,230 | INFO | rep=1 fold=3 | END | fold_time=106.8s | fold_MAE(XGB)=498.908 | fold_MAE(CAT)=1308.003
CV folds:  90%|████████████████████████████████████████████████████████▋      | 9/10 [16:27<01:49, 109.33s/it]2026-02-13 14:29:52,231 | INFO | rep=1 fold=4 | START | train=1600 val=400
2026-02-13 14:29:52,233 | INFO | rep=1 fold=4 | XGB fit START | device=cuda | verbose_every=100


[0]	validation_0-mae:1347.98318
[100]	validation_0-mae:476.05595
[200]	validation_0-mae:477.95101
[300]	validation_0-mae:480.35630
[321]	validation_0-mae:480.14503


2026-02-13 14:29:53,355 | INFO | rep=1 fold=4 | XGB fit END | best_iteration=122
2026-02-13 14:29:53,356 | INFO | rep=1 fold=4 | CatBoost fit START | task_type=GPU | verbose=100
Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 1440.0225000	test: 1406.4240625	best: 1406.4240625 (0)	total: 12.3ms	remaining: 1m 38s
100:	learn: 1438.3656250	test: 1404.7771875	best: 1404.7771875 (100)	total: 1.28s	remaining: 1m 40s
200:	learn: 1436.6984375	test: 1403.1231250	best: 1403.1231250 (200)	total: 2.61s	remaining: 1m 41s
300:	learn: 1435.0240625	test: 1401.4675000	best: 1401.4675000 (300)	total: 4.02s	remaining: 1m 42s
400:	learn: 1433.3518750	test: 1399.8168750	best: 1399.8168750 (400)	total: 5.5s	remaining: 1m 44s
500:	learn: 1431.6809375	test: 1398.1670313	best: 1398.1670313 (500)	total: 6.96s	remaining: 1m 44s
600:	learn: 1430.0098438	test: 1396.5182812	best: 1396.5182812 (600)	total: 8.41s	remaining: 1m 43s
700:	learn: 1428.3384375	test: 1394.8690625	best: 1394.8690625 (700)	total: 9.84s	remaining: 1m 42s
800:	learn: 1426.6675000	test: 1393.2253125	best: 1393.2253125 (800)	total: 11.2s	remaining: 1m 41s
900:	learn: 1424.9967187	test: 1391.5960937	best: 1391.5960937 (900)	total: 12.7s	remaining: 1m 40s
1000

2026-02-13 14:31:52,007 | INFO | rep=1 fold=4 | CatBoost fit END | best_iteration=7999
2026-02-13 14:31:52,008 | INFO | rep=1 fold=4 | END | fold_time=119.8s | fold_MAE(XGB)=474.971 | fold_MAE(CAT)=1281.080
CV folds: 100%|██████████████████████████████████████████████████████████████| 10/10 [18:26<00:00, 110.69s/it]
2026-02-13 14:31:52,012 | INFO | OOF MAE | XGB: 477.7906
2026-02-13 14:31:52,012 | INFO | OOF MAE | CAT: 1307.7126
2026-02-13 14:31:52,013 | INFO | Blend search | best w(XGB)=1.00 | OOF MAE=477.7906
2026-02-13 14:31:52,014 | INFO | Calibration(global) | bias=-4.3629 | OOF MAE=477.7767
2026-02-13 14:31:52,016 | INFO | Calibration(by_chronic) | biases={'DiabetesComp': -15.25439453125, 'HF': 59.30517578125, 'Pneumonia': -84.23126220703125} | OOF MAE=475.7624
2026-02-13 14:31:52,016 | INFO | Using per-chronic bias correction (better or equal OOF MAE).
2026-02-13 14:31:52,016 | INFO | FINAL OOF MAE (post-calibration): 475.7624
2026-02-13 14:31:52,024 | INFO | Saved submission: D

# Iteration 6
- 486.4260

In [20]:
# -*- coding: utf-8 -*-
r"""
ITERATION 6 (FIXED) — Monolithic XGBoost (GPU) + TF-IDF (1–3 grams, 1000 features) + Chronic Bias Correction
===========================================================================================================

Fixes vs previous Iter 6:
- Robust code extraction: regex-find all CPT/HCPCS-like codes from full PDF text (handles PyMuPDF line breaks).
- TF-IDF guardrails: if vocabulary is empty, logs + falls back to tabular-only instead of crashing.
- Cache version bumped so a previously-bad cache won't poison current run.

===========================================================
INSTALL (run once)
===========================================================
pip install -U pandas numpy scipy scikit-learn xgboost pymupdf joblib tqdm
"""

from __future__ import annotations

import re
import time
import logging
import warnings
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer

import xgboost as xgb
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# CONFIG
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"  # optional
PDF_DIR = BASE_DIR / "receipts_pdf"

SUBMISSION_PATH = BASE_DIR / "submission.csv"

# Cache (bumped to invalidate any previous "empty codes_doc" cache)
CACHE_PATH = BASE_DIR / "cache_receipts_iter6_codes.joblib"
CACHE_VERSION = 62

# CV
FAST_MODE = True
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026] if FAST_MODE else [2024, 2025, 2026]
EARLY_STOPPING = 200

# TF-IDF
TFIDF_MAX_FEATURES = 1000
TFIDF_NGRAM_RANGE = (1, 3)

# XGB
XGB_DEVICE = "cuda"     # "cuda" or "cuda:0"
N_ESTIMATORS = 8000
LEARNING_RATE = 0.02

# Heartbeats
XGB_VERBOSE_EVERY = 100

# Post-process
CLIP_AT_ZERO = True


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def ensure_csr_float32_int32(X: csr_matrix) -> csr_matrix:
    X = X.tocsr()
    if X.data.dtype != np.float32:
        X.data = X.data.astype(np.float32, copy=False)
    if X.indices.dtype != np.int32:
        X.indices = X.indices.astype(np.int32, copy=False)
    if X.indptr.dtype != np.int32:
        X.indptr = X.indptr.astype(np.int32, copy=False)
    return X


# =========================
# REGEX (RAW STRINGS)
# =========================
PID_RE = re.compile(r"Patient ID:\s*(\d+)")
ZIP_INS_RE = re.compile(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)")
TOTAL_ANY_RE = re.compile(r"TOTAL\s+([\d,]+\.\d{2})")

# IMPORTANT:
# Use 5-digit numeric CPTs + HCPCS like "G0378" (letter + 4 digits).
# This avoids accidentally capturing 3-digit ZIPs or small patient IDs.
CODE_TOKEN_RE = re.compile(r"\b(?:[A-Z]\d{4}|\d{5})\b")


def parse_header_fields(text: str) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    pid = None
    zip3 = None
    insurance = None

    m = PID_RE.search(text)
    if m:
        pid = int(m.group(1))

    m = ZIP_INS_RE.search(text)
    if m:
        zip3 = str(m.group(1))
        insurance = str(m.group(2)).lower()

    return pid, zip3, insurance


def money_to_float(tok: str) -> float:
    return float(tok.replace(",", ""))


@dataclass
class ReceiptProfile:
    meta: Dict[str, Any]
    codes_doc: str


def extract_codes_doc_from_pdf_text(text: str, patient_id: int) -> Tuple[str, Dict[str, Any]]:
    """
    Robustly build a whitespace-separated "document" consisting of CPT/HCPCS-like code tokens.

    Key design:
    - Do NOT rely on line structure (PyMuPDF can break tables into column-cells).
    - Just regex-extract all codes from the full text.
    """
    _, zip3, insurance = parse_header_fields(text)

    codes = CODE_TOKEN_RE.findall(text or "")
    # Defensive: if patient_id happens to be 5 digits and appears in text, drop it
    pid_str = str(patient_id)
    if len(pid_str) == 5:
        codes = [c for c in codes if c != pid_str]

    codes_doc = " ".join(codes)

    # total (optional meta feature)
    total_val = None
    m = TOTAL_ANY_RE.search(text or "")
    if m:
        try:
            total_val = money_to_float(m.group(1))
        except Exception:
            total_val = None

    meta = dict(
        pdf_ok=1 if len(codes) > 0 else 0,
        pdf_parse_notes="ok" if len(codes) > 0 else "no_codes_found",
        pdf_zip3=zip3,
        pdf_insurance=insurance,
        pdf_n_lines=int(len(codes)),                 # "lines" here = number of code tokens found
        pdf_n_unique_codes=int(len(set(codes))),
        pdf_total=float(total_val) if total_val is not None else 0.0,
    )
    return codes_doc, meta


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptProfile:
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"

    empty = ReceiptProfile(
        meta=dict(pdf_ok=0, pdf_parse_notes="missing_pdf", pdf_zip3=None, pdf_insurance=None,
                  pdf_n_lines=0, pdf_n_unique_codes=0, pdf_total=0.0),
        codes_doc=""
    )

    if not pdf_path.exists():
        return empty

    try:
        with fitz.open(pdf_path) as doc:
            text = ""
            for page in doc:
                text += (page.get_text("text") or "") + "\n"
    except Exception as e:
        empty.meta["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return empty

    codes_doc, meta = extract_codes_doc_from_pdf_text(text, patient_id)
    return ReceiptProfile(meta=meta, codes_doc=codes_doc)


def load_cache(cache_path: Path) -> Dict[str, Any]:
    if cache_path.exists():
        try:
            obj = joblib.load(cache_path)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "data" in obj:
                return obj
        except Exception:
            pass
    return {"version": CACHE_VERSION, "data": {}}


def save_cache(cache_path: Path, cache_obj: Dict[str, Any]) -> None:
    try:
        joblib.dump(cache_obj, cache_path)
    except Exception as e:
        logging.warning(f"Cache save failed: {e}")


def build_receipt_tables(patient_ids: List[int], pdf_dir: Path, cache_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      receipt_meta_df: patient_id + pdf_* meta columns
      receipt_doc_df : patient_id + codes_doc
    """
    cache_obj = load_cache(cache_path)
    data: Dict[str, Any] = cache_obj.get("data", {})

    to_parse = [pid for pid in patient_ids if str(pid) not in data]
    logging.info(f"Receipt cache: {len(patient_ids) - len(to_parse):,} hit | {len(to_parse):,} miss | cache_version={CACHE_VERSION}")

    for pid in tqdm(to_parse, desc="Parsing receipts", ncols=100):
        prof = featurize_receipt(pid, pdf_dir)
        data[str(pid)] = {"meta": prof.meta, "codes_doc": prof.codes_doc}

    cache_obj["data"] = data
    if to_parse:
        save_cache(cache_path, cache_obj)

    meta_rows = []
    doc_rows = []
    for pid in patient_ids:
        obj = data.get(str(pid))
        if obj is None:
            prof = featurize_receipt(pid, pdf_dir)
            obj = {"meta": prof.meta, "codes_doc": prof.codes_doc}

        meta_rows.append({"patient_id": pid, **(obj.get("meta", {}) or {})})
        doc_rows.append({"patient_id": pid, "codes_doc": obj.get("codes_doc", "") or ""})

    return pd.DataFrame(meta_rows), pd.DataFrame(doc_rows)


# =========================
# TABULAR SPARSE MATRIX
# =========================
def build_tabular_sparse_matrix(df_all: pd.DataFrame) -> csr_matrix:
    df = df_all.copy()

    # simple derived features
    df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
    df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
    df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

    # categorical normalization
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("missing")

    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train", "codes_doc"}
    cat_cols = [c for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
                if c in df.columns]
    num_cols = [c for c in df.columns
                if c not in exclude and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    X = preprocessor.fit_transform(df)
    X = X.tocsr() if hasattr(X, "tocsr") else csr_matrix(X)
    return ensure_csr_float32_int32(X)


def make_strat_labels(train_df: pd.DataFrame) -> np.ndarray:
    chronic = train_df["primary_chronic"].astype(str)
    cost_bin = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    return (chronic + "_" + cost_bin).values


def chronic_bias_correction_mean(
    y_true: np.ndarray,
    oof_pred: np.ndarray,
    train_group: np.ndarray,
    test_pred: np.ndarray,
    test_group: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float], float]:
    """
    Mean residual correction per group:
      bias[g] = mean(y - pred) over OOF
      pred_adj = pred + bias[group]
    """
    biases: Dict[str, float] = {}
    global_bias = float(np.mean(y_true - oof_pred))

    for g in np.unique(train_group):
        m = (train_group == g)
        if m.sum() == 0:
            continue
        biases[str(g)] = float(np.mean(y_true[m] - oof_pred[m]))

    oof_adj = oof_pred.copy()
    for g, b in biases.items():
        oof_adj[train_group == g] += b

    test_adj = test_pred.copy()
    for i in range(len(test_adj)):
        g = str(test_group[i])
        test_adj[i] += biases.get(g, global_bias)

    return oof_adj, test_adj, biases, global_bias


# =========================
# MAIN
# =========================
def main() -> None:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb, '__version__', 'unknown')}")
    logging.info("=== Iteration 6 (Fixed): Monolithic XGBoost (GPU) + TF-IDF codes ===")
    logging.info(f"CV: FAST_MODE={FAST_MODE} | repeats={len(REPEAT_SEEDS)} | folds={N_FOLDS}")
    logging.info(f"XGB: n_estimators={N_ESTIMATORS} | lr={LEARNING_RATE} | device={XGB_DEVICE}")

    # Load
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train_df.shape}, test={test_df.shape}")

    # Optional patients.csv
    if PATIENTS_CSV.exists():
        patients = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in patients.columns:
            train_df = train_df.merge(patients, on="patient_id", how="left")
            test_df = test_df.merge(patients, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train_df.shape}, test={test_df.shape}")
        else:
            logging.warning("patients.csv exists but has no patient_id; skipping merge.")
    else:
        logging.info("patients.csv not found; continuing without it.")

    train_df["is_train"] = 1
    test_df["is_train"] = 0
    df_all = pd.concat([train_df, test_df], ignore_index=True)

    # Parse receipts -> codes_doc + meta
    all_pids = df_all["patient_id"].astype(int).unique().tolist()
    all_pids.sort()
    meta_df, doc_df = build_receipt_tables(all_pids, PDF_DIR, CACHE_PATH)

    df_all = df_all.merge(meta_df, on="patient_id", how="left").merge(doc_df, on="patient_id", how="left")
    df_all["codes_doc"] = df_all["codes_doc"].fillna("").astype(str)

    nonempty_docs = int((df_all["codes_doc"].str.len() > 0).sum())
    logging.info(f"codes_doc non-empty: {nonempty_docs}/{len(df_all)}")

    # Build TF-IDF
    logging.info("Building TF-IDF on code documents...")
    if nonempty_docs == 0:
        logging.warning("No code tokens found in any PDF -> TF-IDF disabled (tabular-only run).")
        X_tfidf_all = csr_matrix((len(df_all), 0), dtype=np.float32)
    else:
        vectorizer = TfidfVectorizer(
            analyzer="word",
            token_pattern=r"(?u)\b[A-Za-z0-9]+\b",
            lowercase=False,
            ngram_range=TFIDF_NGRAM_RANGE,
            max_features=TFIDF_MAX_FEATURES,
            min_df=1,
            max_df=1.0,
            sublinear_tf=True,
            norm="l2",
            use_idf=True,
            smooth_idf=True,
            dtype=np.float32,
        )
        try:
            X_tfidf_all = vectorizer.fit_transform(df_all["codes_doc"].values)
            X_tfidf_all = ensure_csr_float32_int32(X_tfidf_all)
            logging.info(f"TF-IDF OK: shape={X_tfidf_all.shape} | nnz={X_tfidf_all.nnz:,} | vocab={len(vectorizer.vocabulary_):,}")
        except ValueError as e:
            # Robust fallback: restrict token_pattern to code-like tokens
            logging.warning(f"TF-IDF failed ({e}). Retrying with stricter code token_pattern...")
            vectorizer2 = TfidfVectorizer(
                analyzer="word",
                token_pattern=r"(?u)\b(?:[A-Z]\d{4}|\d{5})\b",
                lowercase=False,
                ngram_range=TFIDF_NGRAM_RANGE,
                max_features=TFIDF_MAX_FEATURES,
                min_df=1,
                max_df=1.0,
                sublinear_tf=True,
                norm="l2",
                use_idf=True,
                smooth_idf=True,
                dtype=np.float32,
            )
            try:
                X_tfidf_all = vectorizer2.fit_transform(df_all["codes_doc"].values)
                X_tfidf_all = ensure_csr_float32_int32(X_tfidf_all)
                logging.info(f"TF-IDF fallback OK: shape={X_tfidf_all.shape} | nnz={X_tfidf_all.nnz:,} | vocab={len(vectorizer2.vocabulary_):,}")
            except ValueError as e2:
                logging.warning(f"TF-IDF fallback still failed ({e2}). Disabling TF-IDF (tabular-only run).")
                X_tfidf_all = csr_matrix((len(df_all), 0), dtype=np.float32)

    # Build tabular sparse
    logging.info("Building tabular sparse matrix (OHE + numeric)...")
    X_tab_all = build_tabular_sparse_matrix(df_all)
    logging.info(f"Tabular OK: shape={X_tab_all.shape} | nnz={X_tab_all.nnz:,}")

    # Combine
    X_all = hstack([X_tab_all, X_tfidf_all], format="csr")
    X_all = ensure_csr_float32_int32(X_all)
    logging.info(f"FINAL X_all: shape={X_all.shape} | nnz={X_all.nnz:,}")

    # Split
    is_train_mask = df_all["is_train"].values.astype(int) == 1
    X_train = X_all[is_train_mask]
    X_test = X_all[~is_train_mask]

    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)
    train_chronic = train_df["primary_chronic"].astype(str).values
    test_chronic = test_df["primary_chronic"].astype(str).values

    # CV splits (stratified; fallback to KFold if needed)
    strat = make_strat_labels(train_df)
    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        try:
            skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
            for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), strat)):
                splits.append((rep, fold, tr_idx, va_idx))
        except ValueError as e:
            logging.warning(f"StratifiedKFold failed ({e}); falling back to KFold for seed={seed}.")
            kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
            for fold, (tr_idx, va_idx) in enumerate(kf.split(np.zeros(len(y)))):
                splits.append((rep, fold, tr_idx, va_idx))

    logging.info(f"Total CV splits: {len(splits)}")

    # Storage
    oof_sum = np.zeros(len(y), dtype=np.float64)
    oof_cnt = np.zeros(len(y), dtype=np.float64)
    test_sum = np.zeros(X_test.shape[0], dtype=np.float64)
    n_models = 0

    # XGB params
    xgb_params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=XGB_DEVICE,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.75,
        reg_lambda=1.0,
        min_child_weight=1.0,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,  # required in __init__ for newer xgboost sklearn API
    )

    fit_sig = None

    # Train
    for rep, fold, tr_idx, va_idx in tqdm(splits, desc="CV folds", ncols=110):
        t0 = time.time()
        tag = f"rep={rep} fold={fold}"
        logging.info(f"{tag} | START | train={len(tr_idx)} val={len(va_idx)}")

        X_tr = X_train[tr_idx]
        X_va = X_train[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        logging.info(f"{tag} | XGB fit START | verbose_every={XGB_VERBOSE_EVERY}")
        model = XGBRegressor(**xgb_params, random_state=1000 * rep + fold)

        if fit_sig is None:
            fit_sig = inspect.signature(model.fit).parameters

        fit_kwargs = {"eval_set": [(X_va, y_va)]}
        if "verbose_eval" in fit_sig:
            fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
        else:
            fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

        model.fit(X_tr, y_tr, **fit_kwargs)

        best_it = getattr(model, "best_iteration", None)
        logging.info(f"{tag} | XGB fit END | best_iteration={best_it}")

        p_va = model.predict(X_va)
        p_te = model.predict(X_test)

        if CLIP_AT_ZERO:
            p_va = np.maximum(p_va, 0.0)
            p_te = np.maximum(p_te, 0.0)

        oof_sum[va_idx] += p_va
        oof_cnt[va_idx] += 1.0
        test_sum += p_te
        n_models += 1

        fold_mae = mae(y_va, p_va)
        dt = time.time() - t0
        logging.info(f"{tag} | END | fold_time={dt:.1f}s | fold_MAE={fold_mae:.3f}")

    # Aggregate predictions
    oof = (oof_sum / np.maximum(oof_cnt, 1.0)).astype(np.float32)
    test_pred = (test_sum / max(1, n_models)).astype(np.float32)

    base_oof_mae = mae(y, oof)
    logging.info(f"OOF MAE (raw): {base_oof_mae:.4f}")

    # Chronic condition bias correction (mean residual per chronic group)
    logging.info("Applying chronic condition mean-residual bias correction (OOF-based)...")
    oof_cal, test_cal, biases, global_bias = chronic_bias_correction_mean(
        y_true=y,
        oof_pred=oof,
        train_group=train_chronic,
        test_pred=test_pred,
        test_group=test_chronic,
    )

    if CLIP_AT_ZERO:
        oof_cal = np.maximum(oof_cal, 0.0)
        test_cal = np.maximum(test_cal, 0.0)

    cal_mae = mae(y, oof_cal)
    logging.info(f"Biases by chronic (mean residual): {biases} | global_bias={global_bias:.4f}")
    logging.info(f"OOF MAE (after chronic bias correction): {cal_mae:.4f}")

    # Safety: if correction unexpectedly worsens OOF, revert
    if cal_mae > base_oof_mae + 1e-6:
        logging.warning("Chronic bias correction worsened OOF MAE. Reverting to raw predictions.")
        test_out = test_pred
    else:
        test_out = test_cal

    if CLIP_AT_ZERO:
        test_out = np.maximum(test_out, 0.0)

    # Save submission
    sub = pd.DataFrame({
        "patient_id": test_df["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": test_out.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")


if __name__ == "__main__":
    main()


2026-02-13 15:45:38,019 | INFO | XGBoost version: 3.0.0
2026-02-13 15:45:38,021 | INFO | === Iteration 6 (Fixed): Monolithic XGBoost (GPU) + TF-IDF codes ===
2026-02-13 15:45:38,022 | INFO | CV: FAST_MODE=True | repeats=2 | folds=5
2026-02-13 15:45:38,023 | INFO | XGB: n_estimators=8000 | lr=0.02 | device=cuda
2026-02-13 15:45:38,032 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 15:45:38,041 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 15:45:38,120 | INFO | Receipt cache: 0 hit | 4,000 miss | cache_version=62
Parsing receipts: 100%|████████████████████████████████████████| 4000/4000 [00:05<00:00, 685.58it/s]
2026-02-13 15:45:44,073 | INFO | codes_doc non-empty: 2319/4000
2026-02-13 15:45:44,073 | INFO | Building TF-IDF on code documents...
2026-02-13 15:45:44,119 | INFO | TF-IDF OK: shape=(4000, 1000) | nnz=31,270 | vocab=1,000
2026-02-13 15:45:44,119 | INFO | Building tabular sparse matrix (OHE + numeric)...
2026-02-13 15:45:44,181 | INFO | Tab

[0]	validation_0-mae:1365.80872
[100]	validation_0-mae:535.22839
[200]	validation_0-mae:479.15629
[300]	validation_0-mae:473.82716
[400]	validation_0-mae:473.28990
[500]	validation_0-mae:473.47484
[600]	validation_0-mae:473.64385
[649]	validation_0-mae:473.86880


2026-02-13 15:45:47,283 | INFO | rep=0 fold=0 | XGB fit END | best_iteration=450
2026-02-13 15:45:47,318 | INFO | rep=0 fold=0 | END | fold_time=3.1s | fold_MAE=459.269
CV folds:  10%|██████▍                                                         | 1/10 [00:03<00:28,  3.12s/it]2026-02-13 15:45:47,320 | INFO | rep=0 fold=1 | START | train=1600 val=400
2026-02-13 15:45:47,322 | INFO | rep=0 fold=1 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1407.53680
[100]	validation_0-mae:570.32653
[200]	validation_0-mae:506.02738
[300]	validation_0-mae:504.23906
[400]	validation_0-mae:504.87391
[469]	validation_0-mae:504.93729


2026-02-13 15:45:49,537 | INFO | rep=0 fold=1 | XGB fit END | best_iteration=269
2026-02-13 15:45:49,566 | INFO | rep=0 fold=1 | END | fold_time=2.2s | fold_MAE=500.050
CV folds:  20%|████████████▊                                                   | 2/10 [00:05<00:20,  2.61s/it]2026-02-13 15:45:49,568 | INFO | rep=0 fold=2 | START | train=1600 val=400
2026-02-13 15:45:49,570 | INFO | rep=0 fold=2 | XGB fit START | verbose_every=100


...

2026-02-13 15:46:24,285 | INFO | rep=1 fold=2 | XGB fit END | best_iteration=1465
2026-02-13 15:46:24,376 | INFO | rep=1 fold=2 | END | fold_time=7.1s | fold_MAE=465.229
CV folds:  80%|███████████████████████████████████████████████████▏            | 8/10 [00:40<00:11,  5.51s/it]2026-02-13 15:46:24,378 | INFO | rep=1 fold=3 | START | train=1600 val=400
2026-02-13 15:46:24,380 | INFO | rep=1 fold=3 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1415.27399
[100]	validation_0-mae:573.70504
[200]	validation_0-mae:511.26305
[300]	validation_0-mae:505.79674
[400]	validation_0-mae:503.30366
[500]	validation_0-mae:501.94144
[600]	validation_0-mae:500.62385
[700]	validation_0-mae:500.54632
[800]	validation_0-mae:499.80315
[900]	validation_0-mae:499.80146
[1000]	validation_0-mae:499.28395
[1100]	validation_0-mae:499.02064
[1200]	validation_0-mae:498.99499
[1300]	validation_0-mae:498.82141
[1400]	validation_0-mae:498.70902
[1500]	validation_0-mae:498.32258
[1600]	validation_0-mae:498.20910
[1700]	validation_0-mae:498.38402
[1794]	validation_0-mae:498.65357


2026-02-13 15:46:31,372 | INFO | rep=1 fold=3 | XGB fit END | best_iteration=1595
2026-02-13 15:46:31,465 | INFO | rep=1 fold=3 | END | fold_time=7.1s | fold_MAE=505.942
CV folds:  90%|█████████████████████████████████████████████████████████▌      | 9/10 [00:47<00:06,  6.01s/it]2026-02-13 15:46:31,467 | INFO | rep=1 fold=4 | START | train=1600 val=400
2026-02-13 15:46:31,468 | INFO | rep=1 fold=4 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1383.37445
[100]	validation_0-mae:534.30220
[200]	validation_0-mae:469.90922
[300]	validation_0-mae:464.93745
[400]	validation_0-mae:463.75611
[500]	validation_0-mae:464.18507
[600]	validation_0-mae:463.80314
[700]	validation_0-mae:463.79656
[780]	validation_0-mae:464.15700


2026-02-13 15:46:34,665 | INFO | rep=1 fold=4 | XGB fit END | best_iteration=581
2026-02-13 15:46:34,710 | INFO | rep=1 fold=4 | END | fold_time=3.2s | fold_MAE=467.958
CV folds: 100%|███████████████████████████████████████████████████████████████| 10/10 [00:50<00:00,  5.05s/it]
2026-02-13 15:46:34,713 | INFO | OOF MAE (raw): 469.5084
2026-02-13 15:46:34,714 | INFO | Applying chronic condition mean-residual bias correction (OOF-based)...
2026-02-13 15:46:34,716 | INFO | Biases by chronic (mean residual): {'DiabetesComp': -35.36746597290039, 'HF': 55.52805709838867, 'Pneumonia': -151.41448974609375} | global_bias=-19.0170
2026-02-13 15:46:34,717 | INFO | OOF MAE (after chronic bias correction): 465.6756
2026-02-13 15:46:34,722 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission.csv


# Iteration 7
- 486.2904

In [ ]:
# -*- coding: utf-8 -*-
r"""
ITERATION 7 — Drift-Aware XGBoost (GPU) + TF-IDF Codes + Importance Weighting + MAE-Calibration
============================================================================================

Notebook-safe update:
- No argparse crash in Jupyter (ipykernel injects --f=... which breaks parse_args()).
- In notebooks, this file does NOT auto-run training. You call run_iteration7(...) manually.
- If run as a .py script, it supports CLI flags via parse_known_args().

Core idea:
1) Adversarial validation: classify train vs test to estimate test-likeness p(test|x).
2) Drift-aware folds: build validation folds by sorting train rows by p(test|x).
3) Importance weighting: weight regression training by odds p/(1-p) (stabilized).
4) MAE-aligned calibration: shrunk median residual per chronic group, selected by a test-like proxy slice.

INSTALL:
pip install -U pandas numpy scipy scikit-learn xgboost pymupdf joblib tqdm
"""

from __future__ import annotations

import re
import time
import math
import sys
import argparse
import logging
import warnings
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

import xgboost as xgb
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# PATHS / CONFIG
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"

ADMISSIONS_TRAIN = BASE_DIR / "admissions_train.csv"
ADMISSIONS_TEST = BASE_DIR / "admissions_test.csv"
STAYS_TRAIN = BASE_DIR / "stays_train.csv"
STAYS_TEST = BASE_DIR / "stays_test.csv"

PDF_DIR = BASE_DIR / "receipts_pdf"
SUBMISSION_PATH = BASE_DIR / "submission.csv"

# Receipt cache
CACHE_PATH = BASE_DIR / "cache_receipts_iter7_codes.joblib"
CACHE_VERSION = 72  # bump if you change parsing logic

# TF-IDF (NO SVD/PCA)
TFIDF_MAX_FEATURES = 1000
TFIDF_NGRAM_RANGE = (1, 3)

# Adversarial validation
ADV_N_SPLITS = 5
ADV_RANDOM_STATE = 2027

# Drift-aware folds for regression
REG_N_FOLDS = 5
REG_REPEAT_SEEDS = [2025, 2026]
EARLY_STOPPING = 200

# Importance weighting
WEIGHT_EPS = 1e-4
WEIGHT_BETA = 0.50          # sqrt-odds to reduce variance
WEIGHT_CLIP_MIN = 0.25
WEIGHT_CLIP_MAX = 4.0

# Calibration (MAE-aligned)
CALIB_TOP_QUANTILE = 0.80   # use most test-like 20% as proxy
CALIB_ALPHA = 50.0          # shrinkage strength

# XGBoost
N_ESTIMATORS = 8000
LEARNING_RATE = 0.02
XGB_MAX_DEPTH = 6
XGB_SUBSAMPLE = 0.85
XGB_COLSAMPLE = 0.75
XGB_LAMBDA = 1.0
XGB_MIN_CHILD_WEIGHT = 1.0

# Heartbeats
XGB_VERBOSE_EVERY = 100

# Post
CLIP_AT_ZERO = True


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    root = logging.getLogger()
    if not root.handlers:
        logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    else:
        root.setLevel(logging.INFO)


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def ensure_csr_float32_int32(X: Any) -> csr_matrix:
    # If it's not already a CSR matrix, convert it safely
    if not isinstance(X, csr_matrix):
        if hasattr(X, "tocsr"):
            X = X.tocsr()
        else:
            # This handles dense numpy arrays returned by ColumnTransformer
            X = csr_matrix(X)
            
    # Cast to requested dtypes for XGBoost GPU memory efficiency
    if X.data.dtype != np.float32:
        X.data = X.data.astype(np.float32, copy=False)
    if X.indices.dtype != np.int32:
        X.indices = X.indices.astype(np.int32, copy=False)
    if X.indptr.dtype != np.int32:
        X.indptr = X.indptr.astype(np.int32, copy=False)
    return X

# =========================
# REGEX (RAW STRINGS)
# =========================
PID_RE = re.compile(r"Patient ID:\s*(\d+)")
ZIP_INS_RE = re.compile(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)")
TOTAL_ANY_RE = re.compile(r"TOTAL\s+([\d,]+\.\d{2})")
CODE_TOKEN_RE = re.compile(r"\b(?:[A-Z]\d{4}|\d{5})\b")  # HCPCS or 5-digit CPT


def money_to_float(tok: str) -> float:
    return float(tok.replace(",", ""))


def parse_header_fields(text: str) -> Tuple[Optional[int], Optional[str], Optional[str]]:
    pid = None
    zip3 = None
    insurance = None

    m = PID_RE.search(text or "")
    if m:
        pid = int(m.group(1))

    m = ZIP_INS_RE.search(text or "")
    if m:
        zip3 = str(m.group(1))
        insurance = str(m.group(2)).lower()

    return pid, zip3, insurance


@dataclass
class ReceiptProfile:
    meta: Dict[str, Any]
    codes_doc: str


def extract_codes_doc_from_pdf_text(text: str, patient_id: int) -> ReceiptProfile:
    _, zip3, insurance = parse_header_fields(text or "")

    codes = CODE_TOKEN_RE.findall(text or "")
    pid_str = str(patient_id)
    if len(pid_str) == 5:
        codes = [c for c in codes if c != pid_str]

    codes_doc = " ".join(codes)

    total_val = None
    m = TOTAL_ANY_RE.search(text or "")
    if m:
        try:
            total_val = money_to_float(m.group(1))
        except Exception:
            total_val = None

    meta = dict(
        pdf_ok=1 if len(codes) > 0 else 0,
        pdf_parse_notes="ok" if len(codes) > 0 else "no_codes_found",
        pdf_zip3=zip3,
        pdf_insurance=insurance,
        pdf_n_codes=int(len(codes)),
        pdf_n_unique_codes=int(len(set(codes))),
        pdf_total=float(total_val) if total_val is not None else 0.0,
    )
    return ReceiptProfile(meta=meta, codes_doc=codes_doc)


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptProfile:
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"

    empty = ReceiptProfile(
        meta=dict(
            pdf_ok=0,
            pdf_parse_notes="missing_pdf",
            pdf_zip3=None,
            pdf_insurance=None,
            pdf_n_codes=0,
            pdf_n_unique_codes=0,
            pdf_total=0.0,
        ),
        codes_doc="",
    )

    if not pdf_path.exists():
        return empty

    try:
        with fitz.open(pdf_path) as doc:
            text = ""
            for page in doc:
                text += (page.get_text("text") or "") + "\n"
    except Exception as e:
        empty.meta["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return empty

    return extract_codes_doc_from_pdf_text(text, patient_id)


def load_cache(cache_path: Path) -> Dict[str, Any]:
    if cache_path.exists():
        try:
            obj = joblib.load(cache_path)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "data" in obj:
                return obj
        except Exception:
            pass
    return {"version": CACHE_VERSION, "data": {}}


def save_cache(cache_path: Path, cache_obj: Dict[str, Any]) -> None:
    try:
        joblib.dump(cache_obj, cache_path)
    except Exception as e:
        logging.warning(f"Cache save failed: {e}")


def build_receipt_tables(patient_ids: List[int], pdf_dir: Path, cache_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    cache_obj = load_cache(cache_path)
    data: Dict[str, Any] = cache_obj.get("data", {})

    to_parse = [pid for pid in patient_ids if str(pid) not in data]
    logging.info(f"Receipt cache: {len(patient_ids) - len(to_parse):,} hit | {len(to_parse):,} miss | cache_version={CACHE_VERSION}")

    for pid in tqdm(to_parse, desc="Parsing receipts", ncols=100):
        prof = featurize_receipt(pid, pdf_dir)
        data[str(pid)] = {"meta": prof.meta, "codes_doc": prof.codes_doc}

    cache_obj["data"] = data
    if to_parse:
        save_cache(cache_path, cache_obj)

    meta_rows = []
    doc_rows = []
    for pid in patient_ids:
        obj = data.get(str(pid))
        if obj is None:
            prof = featurize_receipt(pid, pdf_dir)
            obj = {"meta": prof.meta, "codes_doc": prof.codes_doc}
        meta_rows.append({"patient_id": pid, **(obj.get("meta", {}) or {})})
        doc_rows.append({"patient_id": pid, "codes_doc": obj.get("codes_doc", "") or ""})

    return pd.DataFrame(meta_rows), pd.DataFrame(doc_rows)


# =========================
# OPTIONAL RELATIONAL AGGS
# =========================
def safe_read_csv(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists():
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        logging.warning(f"Failed reading {path}: {e}")
        return None


def add_admissions_aggs(df_all: pd.DataFrame) -> pd.DataFrame:
    tr = safe_read_csv(ADMISSIONS_TRAIN)
    te = safe_read_csv(ADMISSIONS_TEST)
    if tr is None or te is None:
        logging.info("Admissions files not found. Skipping admissions aggregates.")
        return df_all

    adm = pd.concat([tr, te], ignore_index=True)
    if "readmit_30d" in adm.columns:
        adm = adm.drop(columns=["readmit_30d"])

    if "patient_id" not in adm.columns:
        logging.info("Admissions table missing patient_id. Skipping.")
        return df_all

    logging.info(f"Admissions rows: {adm.shape}")

    num_cols = [c for c in ["los_days", "acuity_emergent", "charlson_band", "ed_visits_6m"] if c in adm.columns]
    agg_dict = {}
    agg_dict["adm_n"] = ("admission_id", "count") if "admission_id" in adm.columns else ("patient_id", "size")

    for c in num_cols:
        agg_dict[f"adm_{c}_mean"] = (c, "mean")
        agg_dict[f"adm_{c}_max"] = (c, "max")

    agg = adm.groupby("patient_id").agg(**agg_dict).reset_index()

    if "primary_dx" in adm.columns:
        dx_ct = pd.crosstab(adm["patient_id"], adm["primary_dx"]).reset_index()
        dx_ct.columns = ["patient_id"] + [f"adm_primary_dx_{str(c)}_count" for c in dx_ct.columns[1:]]
        agg = agg.merge(dx_ct, on="patient_id", how="left")

    if "discharge_weekday" in adm.columns:
        wd_ct = pd.crosstab(adm["patient_id"], adm["discharge_weekday"]).reset_index()
        wd_ct.columns = ["patient_id"] + [f"adm_discharge_wd_{str(c)}_count" for c in wd_ct.columns[1:]]
        agg = agg.merge(wd_ct, on="patient_id", how="left")

    logging.info(f"Admissions aggregates shape: {agg.shape}")
    return df_all.merge(agg, on="patient_id", how="left")


def add_stays_aggs(df_all: pd.DataFrame) -> pd.DataFrame:
    tr = safe_read_csv(STAYS_TRAIN)
    te = safe_read_csv(STAYS_TEST)
    if tr is None or te is None:
        logging.info("Stays files not found. Skipping stays aggregates.")
        return df_all

    st = pd.concat([tr, te], ignore_index=True)
    if "discharge_ready_day11" in st.columns:
        st = st.drop(columns=["discharge_ready_day11"])

    if "patient_id" not in st.columns:
        logging.info("Stays table missing patient_id. Skipping.")
        return df_all

    logging.info(f"Stays rows: {st.shape}")

    agg = st.groupby("patient_id").agg(stay_n=("stay_id", "count") if "stay_id" in st.columns else ("patient_id", "size")).reset_index()

    if "unit_type" in st.columns:
        ut = pd.crosstab(st["patient_id"], st["unit_type"]).reset_index()
        ut.columns = ["patient_id"] + [f"stay_unit_{str(c)}_count" for c in ut.columns[1:]]
        agg = agg.merge(ut, on="patient_id", how="left")

    if "admission_reason" in st.columns:
        ar = pd.crosstab(st["patient_id"], st["admission_reason"]).reset_index()
        ar.columns = ["patient_id"] + [f"stay_reason_{str(c)}_count" for c in ar.columns[1:]]
        agg = agg.merge(ar, on="patient_id", how="left")

    logging.info(f"Stays aggregates shape: {agg.shape}")
    return df_all.merge(agg, on="patient_id", how="left")


# =========================
# FEATURE BUILDING
# =========================
def build_tabular_sparse(df_all: pd.DataFrame) -> Tuple[csr_matrix, Optional[np.ndarray]]:
    df = df_all.copy()

    if "prior_ed_cost_5y_usd" in df.columns and "prior_ed_visits_5y" in df.columns:
        df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
        df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
        df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

    if "pdf_total" in df.columns and "prior_ed_cost_5y_usd" in df.columns:
        df["pdf_total_minus_prior_cost"] = df["pdf_total"].astype(float) - df["prior_ed_cost_5y_usd"].astype(float)
        df["pdf_total_absdiff_prior_cost"] = np.abs(df["pdf_total_minus_prior_cost"])

    cat_candidates = ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]
    for c in cat_candidates:
        if c in df.columns:
            df[c] = df[c].astype("string").fillna("missing")

    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train", "codes_doc"}
    cat_cols = [c for c in cat_candidates if c in df.columns]
    num_cols = [c for c in df.columns if c not in exclude and c not in cat_cols and pd.api.types.is_numeric_dtype(df[c])]

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                              ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    X = pre.fit_transform(df)
    X = X.tocsr() if hasattr(X, "tocsr") else csr_matrix(X)
    X = ensure_csr_float32_int32(X)

    feat_names = None
    try:
        feat_names = pre.get_feature_names_out()
    except Exception:
        pass

    return X, feat_names


def build_tfidf_sparse(codes_doc: np.ndarray) -> Tuple[csr_matrix, Optional[np.ndarray]]:
    nonempty = int(np.sum(pd.Series(codes_doc).astype(str).str.len().values > 0))
    if nonempty == 0:
        logging.warning("No non-empty codes_doc found. TF-IDF disabled.")
        X = csr_matrix((len(codes_doc), 0), dtype=np.float32)
        return X, None

    vec = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b(?:[A-Z]\d{4}|\d{5})\b",
        lowercase=False,
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=1,
        max_df=1.0,
        sublinear_tf=True,
        norm="l2",
        use_idf=True,
        smooth_idf=True,
        dtype=np.float32,
    )
    X = vec.fit_transform(codes_doc.astype(str))
    X = ensure_csr_float32_int32(X)
    names = vec.get_feature_names_out()
    return X, names


def build_full_features(df_all: pd.DataFrame) -> Tuple[csr_matrix, Optional[np.ndarray]]:
    X_tab, n_tab = build_tabular_sparse(df_all)
    X_txt, n_txt = build_tfidf_sparse(df_all["codes_doc"].values.astype(str))
    X_all = hstack([X_tab, X_txt], format="csr")
    X_all = ensure_csr_float32_int32(X_all)

    if (n_tab is not None) and (n_txt is not None):
        feat_names = np.concatenate([n_tab, n_txt])
    else:
        feat_names = None
    return X_all, feat_names


# =========================
# ADVERSARIAL VALIDATION
# =========================
def adversarial_oof_proba(X_all: csr_matrix, is_test: np.ndarray, diagnostic: bool = False) -> Tuple[np.ndarray, float]:
    y_adv = is_test.astype(int)
    skf = StratifiedKFold(n_splits=ADV_N_SPLITS, shuffle=True, random_state=ADV_RANDOM_STATE)

    p = np.zeros(X_all.shape[0], dtype=np.float32)
    fold_aucs = []

    logging.info("Adversarial validation: LogisticRegression(train=0 vs test=1)...")
    for k, (tr, va) in enumerate(skf.split(np.zeros(len(y_adv)), y_adv)):
        t0 = time.time()
        clf = LogisticRegression(
            solver="liblinear",
            C=1.0,
            class_weight="balanced",
            max_iter=200,
        )
        clf.fit(X_all[tr], y_adv[tr])
        p_va = clf.predict_proba(X_all[va])[:, 1].astype(np.float32)
        p[va] = p_va
        auc = roc_auc_score(y_adv[va], p_va)
        fold_aucs.append(float(auc))
        logging.info(f"ADV fold {k} | AUC={auc:.4f} | time={time.time()-t0:.1f}s")

    auc_all = float(roc_auc_score(y_adv, p))
    logging.info(f"Adversarial OOF AUC: {auc_all:.4f} | fold_AUCs={fold_aucs}")
    return p, auc_all


def make_drift_folds(p_train: np.ndarray, n_folds: int) -> np.ndarray:
    n = len(p_train)
    order = np.argsort(p_train)  # low -> high test-likeness
    fold_id = np.empty(n, dtype=np.int32)
    bins = (np.arange(n) * n_folds) // n
    fold_id[order] = np.clip(bins, 0, n_folds - 1).astype(np.int32)
    return fold_id


def compute_importance_weights(p_train: np.ndarray) -> np.ndarray:
    p = np.clip(p_train.astype(np.float64), WEIGHT_EPS, 1.0 - WEIGHT_EPS)
    odds = p / (1.0 - p)
    w = np.power(odds, WEIGHT_BETA)
    w = np.clip(w, WEIGHT_CLIP_MIN, WEIGHT_CLIP_MAX)
    w = w / np.mean(w)
    return w.astype(np.float32)


# =========================
# MAE CALIBRATION (SHRUNK MEDIAN RESIDUAL)
# =========================
def shrunk_median_bias(residuals: np.ndarray, groups: np.ndarray, alpha: float) -> Tuple[Dict[str, float], float]:
    g = groups.astype(str)
    global_med = float(np.median(residuals))
    biases: Dict[str, float] = {}
    for key in np.unique(g):
        m = (g == key)
        rg = residuals[m]
        if rg.size == 0:
            continue
        med_g = float(np.median(rg))
        n = float(rg.size)
        b = (n * med_g + alpha * global_med) / (n + alpha)
        biases[str(key)] = float(b)
    return biases, global_med


def apply_group_bias(pred: np.ndarray, groups: np.ndarray, biases: Dict[str, float], default_bias: float) -> np.ndarray:
    out = pred.astype(np.float64).copy()
    g = groups.astype(str)
    for i in range(len(out)):
        out[i] += biases.get(str(g[i]), default_bias)
    return out.astype(np.float32)


# =========================
# DRIVER (NOTEBOOK FRIENDLY)
# =========================
def run_iteration7(
    diagnostic: bool = False,
    use_relational: bool = True,
    use_weights: bool = False,
    force_cpu: bool = False,
) -> Dict[str, Any]:
    """
    Notebook-friendly entrypoint.

    Examples:
      run_iteration7(diagnostic=True)
      run_iteration7()
      run_iteration7(use_weights=False)
      run_iteration7(use_relational=False)
    """
    setup_logging()

    logging.info(f"XGBoost version: {getattr(xgb, '__version__', 'unknown')}")
    logging.info("=== Iteration 7 (Notebook-safe) ===")
    logging.info(f"diagnostic={diagnostic} | use_relational={use_relational} | use_weights={use_weights} | force_cpu={force_cpu}")

    # Load base
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train_df.shape}, test={test_df.shape}")

    # patients.csv
    if PATIENTS_CSV.exists():
        pat = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in pat.columns:
            train_df = train_df.merge(pat, on="patient_id", how="left")
            test_df = test_df.merge(pat, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train_df.shape}, test={test_df.shape}")

    train_df["is_train"] = 1
    test_df["is_train"] = 0
    df_all = pd.concat([train_df, test_df], ignore_index=True)

    # Optional relational aggregates
    if use_relational:
        df_all = add_admissions_aggs(df_all)
        df_all = add_stays_aggs(df_all)
        logging.info(f"After relational merges: df_all={df_all.shape}")

    # Receipts parsing
    all_pids = df_all["patient_id"].astype(int).unique().tolist()
    all_pids.sort()
    meta_df, doc_df = build_receipt_tables(all_pids, PDF_DIR, CACHE_PATH)
    df_all = df_all.merge(meta_df, on="patient_id", how="left").merge(doc_df, on="patient_id", how="left")
    df_all["codes_doc"] = df_all["codes_doc"].fillna("").astype(str)

    nonempty_docs = int((df_all["codes_doc"].str.len() > 0).sum())
    logging.info(f"codes_doc non-empty: {nonempty_docs}/{len(df_all)}")

    # Feature build
    logging.info("Building full sparse feature matrix: tabular(OHE) + TF-IDF(codes)...")
    X_all, feat_names = build_full_features(df_all)
    logging.info(f"X_all: shape={X_all.shape} | nnz={X_all.nnz:,}")

    is_train_mask = (df_all["is_train"].values.astype(int) == 1)
    is_test_mask = ~is_train_mask

    # Adversarial validation scores
    p_adv, adv_auc = adversarial_oof_proba(X_all, is_test_mask.astype(int), diagnostic=False)
    p_train = p_adv[is_train_mask]
    p_test = p_adv[is_test_mask]

    def q(x, qs=(0.01, 0.05, 0.50, 0.95, 0.99)):
        return {str(qq): float(np.quantile(x, qq)) for qq in qs}

    logging.info(f"Adv proba TRAIN | mean={p_train.mean():.4f} qs={q(p_train)}")
    logging.info(f"Adv proba TEST  | mean={p_test.mean():.4f} qs={q(p_test)}")

    out = {
        "adv_auc": adv_auc,
        "p_train_mean": float(p_train.mean()),
        "p_test_mean": float(p_test.mean()),
    }

    if diagnostic:
        logging.info("Diagnostic mode: stopping after adversarial validation.")
        return out

    # Regression data
    X_train = X_all[is_train_mask]
    X_test = X_all[is_test_mask]
    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)

    train_chronic = train_df["primary_chronic"].astype(str).values
    test_chronic = test_df["primary_chronic"].astype(str).values

    # Drift-aware folds
    fold_id = make_drift_folds(p_train, REG_N_FOLDS)
    for k in range(REG_N_FOLDS):
        m = (fold_id == k)
        logging.info(f"Fold {k}: n={int(m.sum())} | p_mean={p_train[m].mean():.4f} | p_range=[{p_train[m].min():.4f},{p_train[m].max():.4f}]")

    # Weights
    if use_weights:
        w_train = compute_importance_weights(p_train)
        logging.info(f"Weights: mean={w_train.mean():.3f} min={w_train.min():.3f} p50={np.quantile(w_train,0.5):.3f} max={w_train.max():.3f}")
    else:
        w_train = np.ones_like(p_train, dtype=np.float32)
        logging.info("Weights disabled.")

    device = "cpu" if force_cpu else "cuda"

    xgb_params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=device,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=XGB_MAX_DEPTH,
        subsample=XGB_SUBSAMPLE,
        colsample_bytree=XGB_COLSAMPLE,
        reg_lambda=XGB_LAMBDA,
        min_child_weight=XGB_MIN_CHILD_WEIGHT,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    fit_sig = None

    # Bagging over drift folds and repeats
    oof_sum = np.zeros(len(y), dtype=np.float64)
    oof_cnt = np.zeros(len(y), dtype=np.float64)
    test_sum = np.zeros(X_test.shape[0], dtype=np.float64)
    n_models = 0

    splits = []
    for rep, seed in enumerate(REG_REPEAT_SEEDS):
        for k in range(REG_N_FOLDS):
            va_idx = np.where(fold_id == k)[0]
            tr_idx = np.where(fold_id != k)[0]
            splits.append((rep, k, seed, tr_idx, va_idx))
    logging.info(f"Total regression fits: {len(splits)}")

    for rep, k, seed, tr_idx, va_idx in tqdm(splits, desc="REG folds", ncols=110):
        tag = f"rep={rep} fold={k}"
        t0 = time.time()

        X_tr = X_train[tr_idx]
        y_tr = y[tr_idx]
        X_va = X_train[va_idx]
        y_va = y[va_idx]
        sw_tr = w_train[tr_idx]

        logging.info(f"{tag} | START | train={len(tr_idx)} val={len(va_idx)} | device={device}")

        model = XGBRegressor(**xgb_params, random_state=1000 * rep + seed + k)

        if fit_sig is None:
            fit_sig = inspect.signature(model.fit).parameters

        fit_kwargs = {
            "eval_set": [(X_va, y_va)],
            "sample_weight": sw_tr,
        }
        if "verbose_eval" in fit_sig:
            fit_kwargs["verbose_eval"] = XGB_VERBOSE_EVERY
        else:
            fit_kwargs["verbose"] = XGB_VERBOSE_EVERY

        logging.info(f"{tag} | XGB fit START | verbose_every={XGB_VERBOSE_EVERY}")
        model.fit(X_tr, y_tr, **fit_kwargs)
        best_it = getattr(model, "best_iteration", None)
        logging.info(f"{tag} | XGB fit END | best_iteration={best_it}")

        p_va = model.predict(X_va)
        p_te = model.predict(X_test)

        if CLIP_AT_ZERO:
            p_va = np.maximum(p_va, 0.0)
            p_te = np.maximum(p_te, 0.0)

        oof_sum[va_idx] += p_va
        oof_cnt[va_idx] += 1.0
        test_sum += p_te
        n_models += 1

        fold_mae = mae(y_va, p_va)
        logging.info(f"{tag} | END | fold_MAE={fold_mae:.3f} | time={time.time()-t0:.1f}s")

    oof = (oof_sum / np.maximum(oof_cnt, 1.0)).astype(np.float32)
    test_pred = (test_sum / max(1, n_models)).astype(np.float32)

    raw_mae_all = mae(y, oof)
    logging.info(f"OOF MAE (raw, drift-fold): {raw_mae_all:.4f}")

    thr = float(np.quantile(p_train, CALIB_TOP_QUANTILE))
    idx_top = (p_train >= thr)
    raw_mae_top = mae(y[idx_top], oof[idx_top]) if idx_top.any() else raw_mae_all
    logging.info(f"OOF MAE on top-testlike (p>={thr:.4f}): {raw_mae_top:.4f}")

    # Calibration (shrunk median residual)
    residuals = (y.astype(np.float64) - oof.astype(np.float64)).astype(np.float64)
    biases, global_med = shrunk_median_bias(residuals, train_chronic, alpha=CALIB_ALPHA)

    oof_cal = apply_group_bias(oof, train_chronic, biases, default_bias=global_med)
    test_cal = apply_group_bias(test_pred, test_chronic, biases, default_bias=global_med)

    if CLIP_AT_ZERO:
        oof_cal = np.maximum(oof_cal, 0.0)
        test_cal = np.maximum(test_cal, 0.0)

    cal_mae_all = mae(y, oof_cal)
    cal_mae_top = mae(y[idx_top], oof_cal[idx_top]) if idx_top.any() else cal_mae_all

    logging.info(f"Calibration biases (shrunk median) by chronic: {biases} | global_med={global_med:.4f}")
    logging.info(f"OOF MAE (calibrated): all={cal_mae_all:.4f} | top-testlike={cal_mae_top:.4f}")

    # Choose based on top-testlike proxy
    if cal_mae_top <= raw_mae_top + 1e-9:
        logging.info("Using CALIBRATED predictions (better on test-like subset).")
        final_test = test_cal
        chosen = "calibrated"
    else:
        logging.info("Using RAW predictions (calibration hurt test-like subset).")
        final_test = test_pred
        chosen = "raw"

    if CLIP_AT_ZERO:
        final_test = np.maximum(final_test, 0.0)

    sub = pd.DataFrame({
        "patient_id": test_df["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": final_test.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")

    out.update({
        "oof_mae_raw_all": float(raw_mae_all),
        "oof_mae_raw_top": float(raw_mae_top),
        "oof_mae_cal_all": float(cal_mae_all),
        "oof_mae_cal_top": float(cal_mae_top),
        "chosen": chosen,
        "submission_path": str(SUBMISSION_PATH),
    })
    return out


# =========================
# CLI SUPPORT (SAFE IN NOTEBOOK TOO)
# =========================
def parse_cli_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument("--diagnostic", action="store_true")
    parser.add_argument("--no_relational", action="store_true")
    parser.add_argument("--no_weights", action="store_true")
    parser.add_argument("--cpu", action="store_true")
    # parse_known_args avoids crashing on ipykernel's extra args (e.g., --f=...json)
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        logging.info(f"Ignoring unknown args: {unknown}")
    return args


def _in_notebook() -> bool:
    # robust notebook check
    try:
        import IPython
        ip = IPython.get_ipython()
        if ip is None:
            return False
        return ("IPKernelApp" in ip.config) or ("ipykernel" in sys.modules)
    except Exception:
        return ("ipykernel" in sys.modules)


if __name__ == "__main__":
    setup_logging()
    if _in_notebook():
        logging.info("Notebook detected: not auto-running. Call run_iteration7(diagnostic=True) or run_iteration7().")
    else:
        a = parse_cli_args()
        run_iteration7(
            diagnostic=a.diagnostic,
            use_relational=not a.no_relational,
            use_weights=not a.no_weights,
            force_cpu=a.cpu,
        )


2026-02-13 17:22:39,273 | INFO | Notebook detected: not auto-running. Call run_iteration7(diagnostic=True) or run_iteration7().


## Step 1 — Run diagnostics only (fast)

In [24]:
results = run_iteration7(diagnostic=True)
results

2026-02-13 17:21:23,986 | INFO | XGBoost version: 3.0.0
2026-02-13 17:21:23,993 | INFO | === Iteration 7 (Notebook-safe) ===
2026-02-13 17:21:23,994 | INFO | diagnostic=True | use_relational=True | use_weights=True | force_cpu=False
2026-02-13 17:21:24,004 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 17:21:24,014 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 17:21:24,033 | INFO | Admissions rows: (10000, 8)
2026-02-13 17:21:24,203 | INFO | Admissions aggregates shape: (4000, 20)
2026-02-13 17:21:24,216 | INFO | Stays rows: (2000, 4)
2026-02-13 17:21:24,273 | INFO | Stays aggregates shape: (2000, 9)
2026-02-13 17:21:24,278 | INFO | After relational merges: df_all=(4000, 37)
2026-02-13 17:21:24,282 | INFO | Receipt cache: 0 hit | 4,000 miss | cache_version=72
Parsing receipts: 100%|████████████████████████████████████████| 4000/4000 [00:06<00:00, 639.10it/s]
2026-02-13 17:21:30,646 | INFO | codes_doc non-empty: 2319/4000
2026-02-13 17:21:30,647 | 

{'adv_auc': 0.508991875,
 'p_train_mean': 0.49753326177597046,
 'p_test_mean': 0.5020621418952942}

## Step 2 — Run full training (creates submission.csv)
- 486.2904


In [26]:
results = run_iteration7(diagnostic=False)
results

2026-02-13 17:22:46,388 | INFO | XGBoost version: 3.0.0
2026-02-13 17:22:46,388 | INFO | === Iteration 7 (Notebook-safe) ===
2026-02-13 17:22:46,389 | INFO | diagnostic=False | use_relational=True | use_weights=False | force_cpu=False
2026-02-13 17:22:46,394 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 17:22:46,401 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 17:22:46,412 | INFO | Admissions rows: (10000, 8)
2026-02-13 17:22:46,548 | INFO | Admissions aggregates shape: (4000, 20)
2026-02-13 17:22:46,554 | INFO | Stays rows: (2000, 4)
2026-02-13 17:22:46,594 | INFO | Stays aggregates shape: (2000, 9)
2026-02-13 17:22:46,597 | INFO | After relational merges: df_all=(4000, 37)
2026-02-13 17:22:46,651 | INFO | Receipt cache: 4,000 hit | 0 miss | cache_version=72
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 17:22:46,669 | INFO | codes_doc non-empty: 2319/4000
2026-02-13 17:22:46,669 | INFO | Building full sparse feature matrix: tabular(OHE) + TF-

[0]	validation_0-mae:1344.07504
[100]	validation_0-mae:542.79021
[200]	validation_0-mae:493.97385
[300]	validation_0-mae:487.81877
[400]	validation_0-mae:486.99917
[500]	validation_0-mae:486.69969
[600]	validation_0-mae:486.48476
[700]	validation_0-mae:486.36111
[800]	validation_0-mae:485.95917
[900]	validation_0-mae:485.50912
[1000]	validation_0-mae:485.19614
[1100]	validation_0-mae:484.60750
[1200]	validation_0-mae:484.33788
[1300]	validation_0-mae:484.51871
[1400]	validation_0-mae:484.10952
[1500]	validation_0-mae:483.73054
[1600]	validation_0-mae:483.55764
[1700]	validation_0-mae:483.78228
[1800]	validation_0-mae:483.49698
[1900]	validation_0-mae:482.62041
[2000]	validation_0-mae:482.05729
[2100]	validation_0-mae:481.50393
[2200]	validation_0-mae:480.91537
[2300]	validation_0-mae:480.54867
[2400]	validation_0-mae:480.61148
[2500]	validation_0-mae:480.47242
[2600]	validation_0-mae:480.23664
[2700]	validation_0-mae:480.08131
[2800]	validation_0-mae:480.28223
[2857]	validation_0-mae:4

2026-02-13 17:22:59,107 | INFO | rep=0 fold=0 | XGB fit END | best_iteration=2658
2026-02-13 17:22:59,283 | INFO | rep=0 fold=0 | END | fold_MAE=478.804 | time=12.0s
REG folds:  10%|██████▎                                                        | 1/10 [00:11<01:47, 11.95s/it]2026-02-13 17:22:59,286 | INFO | rep=0 fold=1 | START | train=1600 val=400 | device=cuda
2026-02-13 17:22:59,300 | INFO | rep=0 fold=1 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1362.21779
[100]	validation_0-mae:546.08425
[200]	validation_0-mae:497.44339
[300]	validation_0-mae:493.98905
[400]	validation_0-mae:491.80423
[500]	validation_0-mae:491.70671
[600]	validation_0-mae:491.48613
[700]	validation_0-mae:491.10423
[800]	validation_0-mae:490.76283
[900]	validation_0-mae:490.43028
[1000]	validation_0-mae:490.70793
[1100]	validation_0-mae:490.53548
[1113]	validation_0-mae:490.43164


2026-02-13 17:23:03,986 | INFO | rep=0 fold=1 | XGB fit END | best_iteration=913
2026-02-13 17:23:04,062 | INFO | rep=0 fold=1 | END | fold_MAE=481.053 | time=4.8s
REG folds:  20%|████████████▌                                                  | 2/10 [00:16<01:01,  7.73s/it]2026-02-13 17:23:04,068 | INFO | rep=0 fold=2 | START | train=1600 val=400 | device=cuda
2026-02-13 17:23:04,071 | INFO | rep=0 fold=2 | XGB fit START | verbose_every=100


...

[0]	validation_0-mae:1502.77882
[100]	validation_0-mae:560.48121
[200]	validation_0-mae:492.49270
[300]	validation_0-mae:481.83068
[400]	validation_0-mae:479.12708
[500]	validation_0-mae:475.80450
[600]	validation_0-mae:474.91166
[700]	validation_0-mae:474.00712
[800]	validation_0-mae:473.53027
[900]	validation_0-mae:472.75207
[1000]	validation_0-mae:472.69227
[1100]	validation_0-mae:472.07521
[1200]	validation_0-mae:472.34839
[1295]	validation_0-mae:472.51830


2026-02-13 17:23:41,656 | INFO | rep=1 fold=3 | XGB fit END | best_iteration=1095
2026-02-13 17:23:41,739 | INFO | rep=1 fold=3 | END | fold_MAE=472.799 | time=5.9s
REG folds:  90%|████████████████████████████████████████████████████████▋      | 9/10 [00:54<00:05,  5.16s/it]2026-02-13 17:23:41,742 | INFO | rep=1 fold=4 | START | train=1600 val=400 | device=cuda
2026-02-13 17:23:41,747 | INFO | rep=1 fold=4 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1498.43210
[100]	validation_0-mae:591.52086
[200]	validation_0-mae:507.37845
[300]	validation_0-mae:503.10314
[400]	validation_0-mae:503.02938
[500]	validation_0-mae:501.49586
[600]	validation_0-mae:500.28500
[700]	validation_0-mae:499.12805
[800]	validation_0-mae:496.95452
[900]	validation_0-mae:496.74752
[1000]	validation_0-mae:496.51447
[1100]	validation_0-mae:496.27364
[1200]	validation_0-mae:495.93013
[1300]	validation_0-mae:495.54283
[1400]	validation_0-mae:494.52417
[1500]	validation_0-mae:494.20450
[1600]	validation_0-mae:493.82038
[1700]	validation_0-mae:493.78880
[1800]	validation_0-mae:493.52834
[1900]	validation_0-mae:493.49141
[2000]	validation_0-mae:493.22473
[2100]	validation_0-mae:492.89073
[2200]	validation_0-mae:492.90315
[2300]	validation_0-mae:492.89424
[2341]	validation_0-mae:492.79596


2026-02-13 17:23:51,477 | INFO | rep=1 fold=4 | XGB fit END | best_iteration=2141
2026-02-13 17:23:51,625 | INFO | rep=1 fold=4 | END | fold_MAE=502.055 | time=9.9s
REG folds: 100%|██████████████████████████████████████████████████████████████| 10/10 [01:04<00:00,  6.43s/it]
2026-02-13 17:23:51,627 | INFO | OOF MAE (raw, drift-fold): 483.7962
2026-02-13 17:23:51,628 | INFO | OOF MAE on top-testlike (p>=0.5756): 496.3192
2026-02-13 17:23:51,636 | INFO | Calibration biases (shrunk median) by chronic: {'DiabetesComp': -23.19183152721774, 'HF': 85.8454507846357, 'Pneumonia': -131.74881899673326} | global_med=-1.9790
2026-02-13 17:23:51,637 | INFO | OOF MAE (calibrated): all=479.1407 | top-testlike=491.1682
2026-02-13 17:23:51,637 | INFO | Using CALIBRATED predictions (better on test-like subset).
2026-02-13 17:23:51,644 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission.csv


{'adv_auc': 0.508991875,
 'p_train_mean': 0.49753326177597046,
 'p_test_mean': 0.5020621418952942,
 'oof_mae_raw_all': 483.7961730957031,
 'oof_mae_raw_top': 496.3191833496094,
 'oof_mae_cal_all': 479.1407165527344,
 'oof_mae_cal_top': 491.1682434082031,
 'chosen': 'calibrated',
 'submission_path': 'D:\\AgentDs\\agent_ds_healthcare\\submission.csv'}

## Step 3 — Quick ablations in notebook
- 485.9746

In [28]:
# No relational tables
run_iteration7(use_relational=False)

2026-02-13 17:27:06,629 | INFO | XGBoost version: 3.0.0
2026-02-13 17:27:06,630 | INFO | === Iteration 7 (Notebook-safe) ===
2026-02-13 17:27:06,630 | INFO | diagnostic=False | use_relational=False | use_weights=False | force_cpu=False
2026-02-13 17:27:06,634 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 17:27:06,642 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 17:27:06,694 | INFO | Receipt cache: 4,000 hit | 0 miss | cache_version=72
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 17:27:06,716 | INFO | codes_doc non-empty: 2319/4000
2026-02-13 17:27:06,717 | INFO | Building full sparse feature matrix: tabular(OHE) + TF-IDF(codes)...
2026-02-13 17:27:06,801 | INFO | X_all: shape=(4000, 1044) | nnz=95,802
2026-02-13 17:27:06,802 | INFO | Adversarial validation: LogisticRegression(train=0 vs test=1)...
2026-02-13 17:27:06,863 | INFO | ADV fold 0 | AUC=0.4949 | time=0.1s
2026-02-13 17:27:06,936 | INFO | ADV fold 1 | AUC=0.5068 | time=0.1s
2026-02-1

[0]	validation_0-mae:1353.50959
[100]	validation_0-mae:542.67809
[200]	validation_0-mae:485.17533
[300]	validation_0-mae:479.12043
[400]	validation_0-mae:476.39842
[500]	validation_0-mae:474.55326
[600]	validation_0-mae:473.01165
[700]	validation_0-mae:472.15809
[800]	validation_0-mae:471.70635
[900]	validation_0-mae:470.94604
[1000]	validation_0-mae:470.19455
[1100]	validation_0-mae:470.30285
[1200]	validation_0-mae:469.44334
[1300]	validation_0-mae:469.14878
[1400]	validation_0-mae:468.69987
[1500]	validation_0-mae:468.67272
[1600]	validation_0-mae:467.97312
[1700]	validation_0-mae:467.87815
[1800]	validation_0-mae:467.90178
[1900]	validation_0-mae:467.72501
[2000]	validation_0-mae:467.35625
[2100]	validation_0-mae:467.36599
[2200]	validation_0-mae:467.47641
[2239]	validation_0-mae:467.45633


2026-02-13 17:27:16,781 | INFO | rep=0 fold=0 | XGB fit END | best_iteration=2039
2026-02-13 17:27:16,909 | INFO | rep=0 fold=0 | END | fold_MAE=488.057 | time=9.8s
REG folds:  10%|██████▎                                                        | 1/10 [00:09<01:28,  9.80s/it]2026-02-13 17:27:16,912 | INFO | rep=0 fold=1 | START | train=1600 val=400 | device=cuda
2026-02-13 17:27:16,919 | INFO | rep=0 fold=1 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1350.94041
[100]	validation_0-mae:551.98669
[200]	validation_0-mae:493.68845
[300]	validation_0-mae:486.17338
[400]	validation_0-mae:483.59544
[500]	validation_0-mae:481.83899
[600]	validation_0-mae:481.07270
[700]	validation_0-mae:480.70523
[800]	validation_0-mae:480.47995
[900]	validation_0-mae:480.79708
[982]	validation_0-mae:480.95332


2026-02-13 17:27:21,510 | INFO | rep=0 fold=1 | XGB fit END | best_iteration=782
2026-02-13 17:27:21,557 | INFO | rep=0 fold=1 | END | fold_MAE=485.314 | time=4.6s
REG folds:  20%|████████████▌                                                  | 2/10 [00:14<00:54,  6.77s/it]2026-02-13 17:27:21,560 | INFO | rep=0 fold=2 | START | train=1600 val=400 | device=cuda
2026-02-13 17:27:21,562 | INFO | rep=0 fold=2 | XGB fit START | verbose_every=100


...

[0]	validation_0-mae:1383.52110
[100]	validation_0-mae:531.29074
[200]	validation_0-mae:487.67443
[300]	validation_0-mae:487.82728
[400]	validation_0-mae:487.85180
[500]	validation_0-mae:486.42163
[600]	validation_0-mae:484.83856
[700]	validation_0-mae:484.03315
[800]	validation_0-mae:482.99726
[900]	validation_0-mae:482.01295
[1000]	validation_0-mae:481.19607
[1100]	validation_0-mae:480.95050
[1200]	validation_0-mae:480.63520
[1300]	validation_0-mae:480.56819
[1400]	validation_0-mae:480.42110
[1500]	validation_0-mae:480.49245
[1600]	validation_0-mae:480.48257
[1659]	validation_0-mae:480.48301


2026-02-13 17:28:30,076 | INFO | rep=1 fold=3 | XGB fit END | best_iteration=1459
2026-02-13 17:28:30,168 | INFO | rep=1 fold=3 | END | fold_MAE=482.477 | time=7.3s
REG folds:  90%|████████████████████████████████████████████████████████▋      | 9/10 [01:23<00:08,  8.36s/it]2026-02-13 17:28:30,171 | INFO | rep=1 fold=4 | START | train=1600 val=400 | device=cuda
2026-02-13 17:28:30,174 | INFO | rep=1 fold=4 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1626.94941
[100]	validation_0-mae:626.69410
[200]	validation_0-mae:518.86174
[300]	validation_0-mae:510.07816
[400]	validation_0-mae:504.72644
[500]	validation_0-mae:500.34031
[600]	validation_0-mae:498.37931
[700]	validation_0-mae:497.93489
[800]	validation_0-mae:497.27049
[900]	validation_0-mae:496.33668
[1000]	validation_0-mae:496.01524
[1100]	validation_0-mae:495.30476
[1200]	validation_0-mae:494.77665
[1300]	validation_0-mae:494.48482
[1400]	validation_0-mae:493.35418
[1500]	validation_0-mae:493.62487
[1600]	validation_0-mae:493.99871
[1641]	validation_0-mae:493.97919


2026-02-13 17:28:37,159 | INFO | rep=1 fold=4 | XGB fit END | best_iteration=1442
2026-02-13 17:28:37,247 | INFO | rep=1 fold=4 | END | fold_MAE=497.832 | time=7.1s
REG folds: 100%|██████████████████████████████████████████████████████████████| 10/10 [01:30<00:00,  9.01s/it]
2026-02-13 17:28:37,250 | INFO | OOF MAE (raw, drift-fold): 477.7679
2026-02-13 17:28:37,252 | INFO | OOF MAE on top-testlike (p>=0.5581): 495.0306
2026-02-13 17:28:37,258 | INFO | Calibration biases (shrunk median) by chronic: {'DiabetesComp': -29.063240297379032, 'HF': 78.07981308826517, 'Pneumonia': -126.64996094830268} | global_med=-11.1260
2026-02-13 17:28:37,259 | INFO | OOF MAE (calibrated): all=474.0898 | top-testlike=491.3989
2026-02-13 17:28:37,260 | INFO | Using CALIBRATED predictions (better on test-like subset).
2026-02-13 17:28:37,266 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission.csv


{'adv_auc': 0.5074105,
 'p_train_mean': 0.49792784452438354,
 'p_test_mean': 0.5021163821220398,
 'oof_mae_raw_all': 477.7678527832031,
 'oof_mae_raw_top': 495.0306396484375,
 'oof_mae_cal_all': 474.0897521972656,
 'oof_mae_cal_top': 491.3988952636719,
 'chosen': 'calibrated',
 'submission_path': 'D:\\AgentDs\\agent_ds_healthcare\\submission.csv'}

- 484.4195

In [30]:
# importance weighting
run_iteration7(use_weights=True)

2026-02-13 17:29:20,163 | INFO | XGBoost version: 3.0.0
2026-02-13 17:29:20,163 | INFO | === Iteration 7 (Notebook-safe) ===
2026-02-13 17:29:20,163 | INFO | diagnostic=False | use_relational=True | use_weights=True | force_cpu=False
2026-02-13 17:29:20,169 | INFO | Loaded train=(2000, 5), test=(2000, 4)
2026-02-13 17:29:20,175 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 17:29:20,185 | INFO | Admissions rows: (10000, 8)
2026-02-13 17:29:20,352 | INFO | Admissions aggregates shape: (4000, 20)
2026-02-13 17:29:20,359 | INFO | Stays rows: (2000, 4)
2026-02-13 17:29:20,403 | INFO | Stays aggregates shape: (2000, 9)
2026-02-13 17:29:20,406 | INFO | After relational merges: df_all=(4000, 37)
2026-02-13 17:29:20,452 | INFO | Receipt cache: 4,000 hit | 0 miss | cache_version=72
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 17:29:20,471 | INFO | codes_doc non-empty: 2319/4000
2026-02-13 17:29:20,472 | INFO | Building full sparse feature matrix: tabular(OHE) + TF-I

[0]	validation_0-mae:1346.80181
[100]	validation_0-mae:545.11726
[200]	validation_0-mae:496.80871
[300]	validation_0-mae:489.84654
[400]	validation_0-mae:487.25472
[500]	validation_0-mae:485.93842
[600]	validation_0-mae:484.98538
[700]	validation_0-mae:484.38381
[800]	validation_0-mae:483.66814
[900]	validation_0-mae:483.36932
[1000]	validation_0-mae:482.27762
[1100]	validation_0-mae:480.80564
[1200]	validation_0-mae:479.84224
[1300]	validation_0-mae:479.18210
[1400]	validation_0-mae:479.05946
[1500]	validation_0-mae:478.70432
[1600]	validation_0-mae:478.72861
[1700]	validation_0-mae:478.53067
[1800]	validation_0-mae:479.01898
[1900]	validation_0-mae:479.38262
[1924]	validation_0-mae:479.33507


2026-02-13 17:29:29,384 | INFO | rep=0 fold=0 | XGB fit END | best_iteration=1725
2026-02-13 17:29:29,478 | INFO | rep=0 fold=0 | END | fold_MAE=481.474 | time=8.3s
REG folds:  10%|██████▎                                                        | 1/10 [00:08<01:15,  8.34s/it]2026-02-13 17:29:29,481 | INFO | rep=0 fold=1 | START | train=1600 val=400 | device=cuda
2026-02-13 17:29:29,484 | INFO | rep=0 fold=1 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1362.90054
[100]	validation_0-mae:544.42089
[200]	validation_0-mae:497.85606
[300]	validation_0-mae:492.39358
[400]	validation_0-mae:490.06074
[500]	validation_0-mae:489.79451
[600]	validation_0-mae:489.76536
[700]	validation_0-mae:489.72961
[800]	validation_0-mae:490.02862
[839]	validation_0-mae:489.85224


2026-02-13 17:29:33,894 | INFO | rep=0 fold=1 | XGB fit END | best_iteration=639
2026-02-13 17:29:33,941 | INFO | rep=0 fold=1 | END | fold_MAE=484.996 | time=4.5s
REG folds:  20%|████████████▌                                                  | 2/10 [00:12<00:48,  6.06s/it]2026-02-13 17:29:33,944 | INFO | rep=0 fold=2 | START | train=1600 val=400 | device=cuda
2026-02-13 17:29:33,946 | INFO | rep=0 fold=2 | XGB fit START | verbose_every=100


...

[0]	validation_0-mae:1501.92994
[100]	validation_0-mae:566.38657
[200]	validation_0-mae:491.20482
[300]	validation_0-mae:479.74880
[400]	validation_0-mae:477.63674
[500]	validation_0-mae:474.93679
[600]	validation_0-mae:473.92005
[700]	validation_0-mae:472.65344
[800]	validation_0-mae:471.65313
[900]	validation_0-mae:471.39151
[1000]	validation_0-mae:470.82461
[1100]	validation_0-mae:471.27003
[1181]	validation_0-mae:471.27878


2026-02-13 17:30:06,093 | INFO | rep=1 fold=3 | XGB fit END | best_iteration=982
2026-02-13 17:30:06,166 | INFO | rep=1 fold=3 | END | fold_MAE=468.658 | time=5.1s
REG folds:  90%|████████████████████████████████████████████████████████▋      | 9/10 [00:45<00:04,  5.00s/it]2026-02-13 17:30:06,169 | INFO | rep=1 fold=4 | START | train=1600 val=400 | device=cuda
2026-02-13 17:30:06,171 | INFO | rep=1 fold=4 | XGB fit START | verbose_every=100


[0]	validation_0-mae:1498.41491
[100]	validation_0-mae:588.01257
[200]	validation_0-mae:503.58369
[300]	validation_0-mae:499.09620
[400]	validation_0-mae:497.54444
[500]	validation_0-mae:495.53322
[600]	validation_0-mae:494.10317
[700]	validation_0-mae:493.82380
[800]	validation_0-mae:493.44889
[900]	validation_0-mae:492.20699
[1000]	validation_0-mae:491.80975
[1100]	validation_0-mae:491.32414
[1200]	validation_0-mae:491.20391
[1300]	validation_0-mae:491.20337
[1400]	validation_0-mae:490.55840
[1500]	validation_0-mae:489.88670
[1600]	validation_0-mae:490.03419
[1700]	validation_0-mae:490.23621
[1745]	validation_0-mae:490.17889


2026-02-13 17:30:12,877 | INFO | rep=1 fold=4 | XGB fit END | best_iteration=1545
2026-02-13 17:30:12,981 | INFO | rep=1 fold=4 | END | fold_MAE=502.207 | time=6.8s
REG folds: 100%|██████████████████████████████████████████████████████████████| 10/10 [00:51<00:00,  5.18s/it]
2026-02-13 17:30:12,984 | INFO | OOF MAE (raw, drift-fold): 483.4443
2026-02-13 17:30:12,985 | INFO | OOF MAE on top-testlike (p>=0.5756): 500.4191
2026-02-13 17:30:12,991 | INFO | Calibration biases (shrunk median) by chronic: {'DiabetesComp': -31.586575415826612, 'HF': 53.004570957898835, 'Pneumonia': -149.17360829041067} | global_med=-18.3030
2026-02-13 17:30:12,992 | INFO | OOF MAE (calibrated): all=479.1780 | top-testlike=495.2959
2026-02-13 17:30:12,993 | INFO | Using CALIBRATED predictions (better on test-like subset).
2026-02-13 17:30:13,001 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission.csv


{'adv_auc': 0.508991875,
 'p_train_mean': 0.49753326177597046,
 'p_test_mean': 0.5020621418952942,
 'oof_mae_raw_all': 483.4443054199219,
 'oof_mae_raw_top': 500.4190673828125,
 'oof_mae_cal_all': 479.17803955078125,
 'oof_mae_cal_top': 495.2959289550781,
 'chosen': 'calibrated',
 'submission_path': 'D:\\AgentDs\\agent_ds_healthcare\\submission.csv'}

# Iteration 8
- 496.6276

In [ ]:
# -*- coding: utf-8 -*-
r"""
ITERATION 8 — Multi-Source Patient Features (Receipts + Notes + Vitals) + XGBoost GPU
===================================================================================

:contentReference[oaicite:7]{index=7}hould beat the stuck 478~480 plateau):
1) Your adversarial AUC ~0.51 => train/test covariate shift is basically absent.
   So "drift folds" and importance weighting are the wrong framework.
2) CV looks great but leaderboard/test gets worse — classic symptom of preprocessing leakage
   (vectorizers/encoders fitted globally) or overly-rich n-grams overfitting on small data.
   This script makes CV leakage-safe by fitting ALL encoders/vectorizers strictly inside each fold.
3) Add new modalities that exist in this dataset package (joinable via patient_id):
   admissions + discharge notes + stays + vitals time series notes (auto-skip if missing).
4) Receipts: not "codes only" — parse the table structure and inject Description words into receipt_doc.
   This learns code similarity statistically without manual clinical grouping.

MODEL:
- Two XGBoost models (GPU):
  (A) reg:squarederror (often generalizes better even when metric is MAE)
  (B) reg:quantileerror with quantile_alpha=0.5 (median reg; MAE-aligned)
- Blend weight optimized on leakage-safe OOF MAE.
- Optional MAE-aligned calibration (global median residual / shrunk median residual by chronic).

USAGE (Jupyter):
    res = run_iteration8(diagnostic=True)      # check coverage of extra modalities
    res = run_iteration8(diagnostic=False)     # train + write submission.csv

INSTALL:
pip install -U pandas numpy scipy scikit-learn xgboost pymupdf joblib tqdm
"""

from __future__ import annotations

import re
import sys
import json
import time
import argparse
import logging
import warnings
import inspect
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib

import fitz  # PyMuPDF
from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_error

import xgboost as xgb
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# PATHS (edit if needed)
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")

TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"

ADMISSIONS_TRAIN = BASE_DIR / "admissions_train.csv"
ADMISSIONS_TEST = BASE_DIR / "admissions_test.csv"
DISCHARGE_NOTES_JSON = BASE_DIR / "discharge_notes.json"

STAYS_TRAIN = BASE_DIR / "stays_train.csv"
STAYS_TEST = BASE_DIR / "stays_test.csv"
VITALS_JSON = BASE_DIR / "vitals_timeseries.json"

PDF_DIR = BASE_DIR / "receipts_pdf"
SUBMISSION_PATH = BASE_DIR / "submission.csv"

CACHE_DIR = BASE_DIR / "cache_iter8"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RECEIPT_CACHE = CACHE_DIR / "receipts.joblib"
DISCHARGE_CACHE = CACHE_DIR / "discharge_docs.joblib"
VITALS_CACHE = CACHE_DIR / "vitals_features.joblib"
RELATIONAL_CACHE = CACHE_DIR / "relational_aggs.joblib"

CACHE_VERSION = 82  # bump if you change parsing logic


# =========================
# GLOBAL CONFIG
# =========================
# Text feature caps (keep bounded to avoid overfitting)
MAXF_RECEIPT = 4000
MAXF_DISCHARGE = 2500
MAXF_VITALS_NOTE = 2500

# For small datasets: filter 1-off tokens to reduce noise
MIN_DF_RECEIPT = 2
MIN_DF_NOTES = 2

# CV
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026]   # 10 fits total
EARLY_STOPPING = 200

# XGB
N_ESTIMATORS = 8000
LR = 0.02
MAX_DEPTH = 6
SUBSAMPLE = 0.85
COLSAMPLE = 0.75
REG_LAMBDA = 1.0
MIN_CHILD_WEIGHT = 1.0

# Objectives to try
USE_L2 = True
USE_Q50 = True

# Blend grid
BLEND_GRID_STEP = 0.02

# Calibration
USE_CALIBRATION = True
CALIB_ALPHA = 50.0  # shrinkage for group median residual
CLIP_AT_ZERO = True

# Heartbeats
VERBOSE_EVERY = 100


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    root = logging.getLogger()
    if not root.handlers:
        logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    else:
        root.setLevel(logging.INFO)


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(mean_absolute_error(y_true, y_pred))


def ensure_csr_float32_int32(X: Any) -> csr_matrix:
    # If it's not already a CSR matrix, convert it safely
    if not isinstance(X, csr_matrix):
        if hasattr(X, "tocsr"):
            X = X.tocsr()
        else:
            # This handles dense numpy arrays returned by ColumnTransformer
            X = csr_matrix(X)
            
    # Cast to requested dtypes for XGBoost GPU memory efficiency
    if X.data.dtype != np.float32:
        X.data = X.data.astype(np.float32, copy=False)
    if X.indices.dtype != np.int32:
        X.indices = X.indices.astype(np.int32, copy=False)
    if X.indptr.dtype != np.int32:
        X.indptr = X.indptr.astype(np.int32, copy=False)
    return X


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


# =========================
# RECEIPT PARSING (table-structure aware)
# =========================
ZIP_INS_RE = re.compile(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)")
TOTAL_RE = re.compile(r"TOTAL\s+([\d,]+\.\d{2})")

CODE_FULL_RE = re.compile(r"^(?:[A-Z]\d{4}|\d{5})$")
MONEY_RE = re.compile(r"^[\d,]+\.\d{2}$")


def money_to_float(s: str) -> float:
    return float(s.replace(",", ""))


@dataclass
class ReceiptProfile:
    meta: Dict[str, Any]
    receipt_doc: str


def parse_receipt_table(lines: List[str]) -> List[Tuple[str, int, float, float, str]]:
    """
    Parse PyMuPDF-extracted receipt table lines.

    Many PDFs come out "columnar": code, description, qty, unit, line_total each on its own line.
    This parser reconstructs line items by:
      code_line -> (one or more desc lines) -> qty_line -> unit_line -> line_total_line
    """
    items: List[Tuple[str, int, float, float, str]] = []
    i = 0
    n = len(lines)

    while i < n:
        s = lines[i].strip()
        if CODE_FULL_RE.match(s):
            code = s
            j = i + 1
            desc_parts: List[str] = []

            # accumulate description (can be multi-line) until we hit qty
            while j < n:
                sj = lines[j].strip()
                if sj.isdigit():
                    break
                if sj.startswith("TOTAL"):
                    break
                # guard against header lines repeating
                if sj in {"Code", "Description", "Qty", "Unit ($)", "Line Total ($)"}:
                    j += 1
                    continue
                if sj:
                    desc_parts.append(sj)
                j += 1

            # need qty + unit + line
            if j + 2 < n:
                qty_s = lines[j].strip()
                unit_s = lines[j + 1].strip()
                line_s = lines[j + 2].strip()
                if qty_s.isdigit() and MONEY_RE.match(unit_s) and MONEY_RE.match(line_s):
                    qty = int(qty_s)
                    qty = max(1, min(qty, 50))
                    unit = money_to_float(unit_s)
                    line_total = money_to_float(line_s)
                    desc = " ".join(desc_parts)
                    items.append((code, qty, unit, line_total, desc))
                    i = j + 3
                    continue
        i += 1

    return items


def featurize_receipt(patient_id: int, pdf_dir: Path) -> ReceiptProfile:
    pdf_path = pdf_dir / f"receipt_{patient_id}.pdf"

    empty = ReceiptProfile(
        meta=dict(
            pdf_ok=0,
            pdf_zip3=None,
            pdf_insurance=None,
            pdf_n_items=0,
            pdf_n_codes=0,
            pdf_n_unique_codes=0,
            pdf_qty_sum=0,
            pdf_unit_mean=np.nan,
            pdf_unit_max=np.nan,
            pdf_line_mean=np.nan,
            pdf_line_max=np.nan,
            pdf_cost_concentration=np.nan,  # max(line_total)/total
            pdf_total=0.0,
        ),
        receipt_doc="",
    )

    if not pdf_path.exists():
        return empty

    try:
        with fitz.open(pdf_path) as doc:
            text = ""
            for page in doc:
                text += (page.get_text("text") or "") + "\n"
    except Exception:
        return empty

    text = text or ""
    lines = text.splitlines()

    # Header info (zip/insurance)
    zip3 = None
    ins = None
    m = ZIP_INS_RE.search(text)
    if m:
        zip3 = str(m.group(1))
        ins = str(m.group(2)).lower()

    # Total (should match prior cost)
    total_val = 0.0
    m = TOTAL_RE.search(text)
    if m:
        try:
            total_val = money_to_float(m.group(1))
        except Exception:
            total_val = 0.0

    items = parse_receipt_table(lines)

    # Build receipt_doc = code + description words (repeat by qty to encode intensity)
    tokens: List[str] = []
    units = []
    lines_total = []
    qty_sum = 0

    for code, qty, unit, line_total, desc in items:
        qty_sum += qty
        units.append(unit)
        lines_total.append(line_total)
        # repeat by qty, but cap repetition to avoid huge docs
        rep = max(1, min(qty, 10))
        for _ in range(rep):
            if desc:
                tokens.append(f"{code} {desc}")
            else:
                tokens.append(f"{code}")

    receipt_doc = " ".join(tokens)

    # If parsing failed, fallback to codes only (still useful)
    if len(items) == 0:
        code_tokens = re.findall(r"\b(?:[A-Z]\d{4}|\d{5})\b", text)
        receipt_doc = " ".join(code_tokens)

    n_codes = len(re.findall(r"\b(?:[A-Z]\d{4}|\d{5})\b", receipt_doc))
    uniq_codes = len(set(re.findall(r"\b(?:[A-Z]\d{4}|\d{5})\b", receipt_doc)))

    unit_mean = float(np.mean(units)) if units else np.nan
    unit_max = float(np.max(units)) if units else np.nan
    line_mean = float(np.mean(lines_total)) if lines_total else np.nan
    line_max = float(np.max(lines_total)) if lines_total else np.nan
    conc = float(line_max / total_val) if (lines_total and total_val > 0) else np.nan

    meta = dict(
        pdf_ok=1 if len(receipt_doc) > 0 else 0,
        pdf_zip3=zip3,
        pdf_insurance=ins,
        pdf_n_items=int(len(items)),
        pdf_n_codes=int(n_codes),
        pdf_n_unique_codes=int(uniq_codes),
        pdf_qty_sum=int(qty_sum),
        pdf_unit_mean=unit_mean,
        pdf_unit_max=unit_max,
        pdf_line_mean=line_mean,
        pdf_line_max=line_max,
        pdf_cost_concentration=conc,
        pdf_total=float(total_val),
    )
    return ReceiptProfile(meta=meta, receipt_doc=receipt_doc)


def load_or_build_receipts(patient_ids: List[int]) -> pd.DataFrame:
    cache = {"version": CACHE_VERSION, "data": {}}
    if RECEIPT_CACHE.exists():
        try:
            cache = joblib.load(RECEIPT_CACHE)
        except Exception:
            cache = {"version": CACHE_VERSION, "data": {}}

    if not isinstance(cache, dict) or cache.get("version") != CACHE_VERSION:
        cache = {"version": CACHE_VERSION, "data": {}}

    data = cache.get("data", {})
    missing = [pid for pid in patient_ids if str(pid) not in data]

    logging.info(f"Receipts cache: hit={len(patient_ids)-len(missing):,} miss={len(missing):,}")

    for pid in tqdm(missing, desc="Parsing receipts", ncols=100):
        prof = featurize_receipt(pid, PDF_DIR)
        data[str(pid)] = {"meta": prof.meta, "receipt_doc": prof.receipt_doc}

    cache["data"] = data
    if missing:
        try:
            joblib.dump(cache, RECEIPT_CACHE)
        except Exception as e:
            logging.warning(f"Receipt cache save failed: {e}")

    rows = []
    for pid in patient_ids:
        obj = data.get(str(pid), {"meta": {}, "receipt_doc": ""})
        row = {"patient_id": pid, "receipt_doc": obj.get("receipt_doc", "") or ""}
        row.update(obj.get("meta") or {})
        rows.append(row)

    return pd.DataFrame(rows)


# =========================
# DISCHARGE NOTES -> patient docs
# =========================
def load_discharge_docs() -> pd.DataFrame:
    if DISCHARGE_CACHE.exists():
        try:
            obj = joblib.load(DISCHARGE_CACHE)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "df" in obj:
                return obj["df"]
        except Exception:
            pass

    if (not ADMISSIONS_TRAIN.exists()) or (not ADMISSIONS_TEST.exists()) or (not DISCHARGE_NOTES_JSON.exists()):
        logging.info("Discharge notes files missing. Skipping discharge_doc.")
        df = pd.DataFrame({"patient_id": [], "discharge_doc": [], "discharge_n_notes": []})
        joblib.dump({"version": CACHE_VERSION, "df": df}, DISCHARGE_CACHE)
        return df

    adm = pd.concat([pd.read_csv(ADMISSIONS_TRAIN), pd.read_csv(ADMISSIONS_TEST)], ignore_index=True)
    if "readmit_30d" in adm.columns:
        adm = adm.drop(columns=["readmit_30d"])

    if "admission_id" not in adm.columns or "patient_id" not in adm.columns:
        logging.info("Admissions schema unexpected. Skipping discharge_doc.")
        df = pd.DataFrame({"patient_id": [], "discharge_doc": [], "discharge_n_notes": []})
        joblib.dump({"version": CACHE_VERSION, "df": df}, DISCHARGE_CACHE)
        return df

    with open(DISCHARGE_NOTES_JSON, "r", encoding="utf-8") as f:
        notes_list = json.load(f)

    notes = pd.DataFrame(notes_list)
    if "admission_id" not in notes.columns or "note" not in notes.columns:
        logging.info("discharge_notes.json schema unexpected. Skipping discharge_doc.")
        df = pd.DataFrame({"patient_id": [], "discharge_doc": [], "discharge_n_notes": []})
        joblib.dump({"version": CACHE_VERSION, "df": df}, DISCHARGE_CACHE)
        return df

    merged = notes.merge(adm[["admission_id", "patient_id"]], on="admission_id", how="inner")
    merged["note"] = merged["note"].astype(str).fillna("")
    agg = merged.groupby("patient_id").agg(
        discharge_doc=("note", lambda s: " ".join(s.tolist())),
        discharge_n_notes=("note", "size"),
    ).reset_index()

    joblib.dump({"version": CACHE_VERSION, "df": agg}, DISCHARGE_CACHE)
    logging.info(f"Built discharge docs: {agg.shape}")
    return agg


# =========================
# VITALS -> numeric stats + patient doc
# =========================
def load_vitals_features() -> pd.DataFrame:
    if VITALS_CACHE.exists():
        try:
            obj = joblib.load(VITALS_CACHE)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "df" in obj:
                return obj["df"]
        except Exception:
            pass

    if (not STAYS_TRAIN.exists()) or (not STAYS_TEST.exists()) or (not VITALS_JSON.exists()):
        logging.info("Vitals files missing. Skipping vitals features.")
        df = pd.DataFrame({"patient_id": [], "vitals_doc": [], "vitals_n_stays": []})
        joblib.dump({"version": CACHE_VERSION, "df": df}, VITALS_CACHE)
        return df

    st = pd.concat([pd.read_csv(STAYS_TRAIN), pd.read_csv(STAYS_TEST)], ignore_index=True)
    if "discharge_ready_day11" in st.columns:
        st = st.drop(columns=["discharge_ready_day11"])

    if "stay_id" not in st.columns or "patient_id" not in st.columns:
        logging.info("Stays schema unexpected. Skipping vitals.")
        df = pd.DataFrame({"patient_id": [], "vitals_doc": [], "vitals_n_stays": []})
        joblib.dump({"version": CACHE_VERSION, "df": df}, VITALS_CACHE)
        return df

    stay2pid = dict(zip(st["stay_id"].astype(int).values, st["patient_id"].astype(int).values))

    with open(VITALS_JSON, "r", encoding="utf-8") as f:
        vit_list = json.load(f)

    rows = []
    for item in vit_list:
        stay_id = int(item.get("stay_id"))
        pid = stay2pid.get(stay_id)
        if pid is None:
            continue
        days = item.get("days", []) or []
        if len(days) == 0:
            continue

        def arr(key: str) -> np.ndarray:
            vals = []
            for d in days:
                v = d.get(key)
                if v is None:
                    continue
                try:
                    vals.append(float(v))
                except Exception:
                    pass
            return np.array(vals, dtype=np.float32)

        notes = []
        for d in days:
            n = d.get("note")
            if n:
                notes.append(str(n))

        feat = {"patient_id": pid, "stay_id": stay_id}
        for key in ["hr", "sbp", "dbp", "temp_c", "rr"]:
            a = arr(key)
            if a.size > 0:
                feat[f"v_{key}_mean"] = float(np.mean(a))
                feat[f"v_{key}_std"] = float(np.std(a))
                feat[f"v_{key}_min"] = float(np.min(a))
                feat[f"v_{key}_max"] = float(np.max(a))
                feat[f"v_{key}_delta"] = float(a[-1] - a[0])
            else:
                feat[f"v_{key}_mean"] = np.nan
                feat[f"v_{key}_std"] = np.nan
                feat[f"v_{key}_min"] = np.nan
                feat[f"v_{key}_max"] = np.nan
                feat[f"v_{key}_delta"] = np.nan

        feat["vitals_doc"] = " ".join(notes)
        feat["v_n_days"] = int(len(days))
        rows.append(feat)

    if len(rows) == 0:
        logging.info("Vitals JSON parsed but no matching stay_id→patient_id. Skipping vitals.")
        df = pd.DataFrame({"patient_id": [], "vitals_doc": [], "vitals_n_stays": []})
        joblib.dump({"version": CACHE_VERSION, "df": df}, VITALS_CACHE)
        return df

    stay_df = pd.DataFrame(rows)

    num_cols = [c for c in stay_df.columns if c not in ["patient_id", "stay_id", "vitals_doc"] and pd.api.types.is_numeric_dtype(stay_df[c])]
    agg_num = stay_df.groupby("patient_id")[num_cols].agg(["mean", "max"])
    agg_num.columns = [f"{c}_{stat}" for c, stat in agg_num.columns]
    agg_num = agg_num.reset_index()

    agg_txt = stay_df.groupby("patient_id").agg(
        vitals_doc=("vitals_doc", lambda s: " ".join([x for x in s.tolist() if isinstance(x, str) and len(x) > 0])),
        vitals_n_stays=("stay_id", "nunique"),
    ).reset_index()

    agg = agg_num.merge(agg_txt, on="patient_id", how="left")
    joblib.dump({"version": CACHE_VERSION, "df": agg}, VITALS_CACHE)
    logging.info(f"Built vitals features: {agg.shape}")
    return agg


# =========================
# RELATIONAL AGGS (admissions + stays meta counts)
# =========================
def load_relational_aggs() -> pd.DataFrame:
    if RELATIONAL_CACHE.exists():
        try:
            obj = joblib.load(RELATIONAL_CACHE)
            if isinstance(obj, dict) and obj.get("version") == CACHE_VERSION and "df" in obj:
                return obj["df"]
        except Exception:
            pass

    parts = []

    if ADMISSIONS_TRAIN.exists() and ADMISSIONS_TEST.exists():
        adm = pd.concat([pd.read_csv(ADMISSIONS_TRAIN), pd.read_csv(ADMISSIONS_TEST)], ignore_index=True)
        if "readmit_30d" in adm.columns:
            adm = adm.drop(columns=["readmit_30d"])
        if "patient_id" in adm.columns:
            agg = adm.groupby("patient_id").agg(adm_n=("admission_id", "count") if "admission_id" in adm.columns else ("patient_id", "size")).reset_index()
            for c in ["los_days", "acuity_emergent", "charlson_band", "ed_visits_6m"]:
                if c in adm.columns:
                    g = adm.groupby("patient_id")[c]
                    agg[f"adm_{c}_mean"] = g.mean().values
                    agg[f"adm_{c}_max"] = g.max().values
            parts.append(agg)

    if STAYS_TRAIN.exists() and STAYS_TEST.exists():
        st = pd.concat([pd.read_csv(STAYS_TRAIN), pd.read_csv(STAYS_TEST)], ignore_index=True)
        if "discharge_ready_day11" in st.columns:
            st = st.drop(columns=["discharge_ready_day11"])
        if "patient_id" in st.columns:
            agg = st.groupby("patient_id").agg(stay_n=("stay_id", "count") if "stay_id" in st.columns else ("patient_id", "size")).reset_index()
            parts.append(agg)

    if len(parts) == 0:
        df = pd.DataFrame({"patient_id": []})
        joblib.dump({"version": CACHE_VERSION, "df": df}, RELATIONAL_CACHE)
        return df

    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on="patient_id", how="outer")

    joblib.dump({"version": CACHE_VERSION, "df": out}, RELATIONAL_CACHE)
    logging.info(f"Built relational aggs: {out.shape}")
    return out


# =========================
# DATA ASSEMBLY
# =========================
def assemble_dataset() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(TRAIN_CSV)
    test = pd.read_csv(TEST_CSV)

    if PATIENTS_CSV.exists():
        pat = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in pat.columns:
            train = train.merge(pat, on="patient_id", how="left")
            test = test.merge(pat, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train.shape}, test={test.shape}")

    train["is_train"] = 1
    test["is_train"] = 0
    all_df = pd.concat([train, test], ignore_index=True)

    pids = all_df["patient_id"].astype(int).tolist()
    rec = load_or_build_receipts(pids)
    all_df = all_df.merge(rec, on="patient_id", how="left")

    rel = load_relational_aggs()
    if "patient_id" in rel.columns and len(rel) > 0:
        all_df = all_df.merge(rel, on="patient_id", how="left")

    dis = load_discharge_docs()
    if "patient_id" in dis.columns and len(dis) > 0:
        all_df = all_df.merge(dis, on="patient_id", how="left")
    else:
        all_df["discharge_doc"] = ""
        all_df["discharge_n_notes"] = 0

    vit = load_vitals_features()
    if "patient_id" in vit.columns and len(vit) > 0:
        all_df = all_df.merge(vit, on="patient_id", how="left")
    else:
        all_df["vitals_doc"] = ""
        all_df["vitals_n_stays"] = 0

    for c in ["receipt_doc", "discharge_doc", "vitals_doc"]:
        if c in all_df.columns:
            all_df[c] = all_df[c].fillna("").astype(str)

    if "prior_ed_cost_5y_usd" in all_df.columns and "prior_ed_visits_5y" in all_df.columns:
        all_df["prior_cost_per_visit"] = all_df["prior_ed_cost_5y_usd"] / (all_df["prior_ed_visits_5y"] + 1.0)
        all_df["log_prior_cost"] = np.log1p(all_df["prior_ed_cost_5y_usd"].astype(float))
        all_df["log_prior_visits"] = np.log1p(all_df["prior_ed_visits_5y"].astype(float))

    train_out = all_df[all_df["is_train"] == 1].reset_index(drop=True)
    test_out = all_df[all_df["is_train"] == 0].reset_index(drop=True)
    return train_out, test_out


# =========================
# FEATURE BUILDERS (fold-safe)
# =========================
def build_tab_preprocessor(num_cols: List[str], cat_cols: List[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def safe_tfidf_fit_transform(vec: TfidfVectorizer, tr_text: np.ndarray, va_text: np.ndarray, te_text: np.ndarray) -> Tuple[csr_matrix, csr_matrix, csr_matrix]:
    try:
        Xtr = vec.fit_transform(tr_text)
        Xva = vec.transform(va_text)
        Xte = vec.transform(te_text)
        return ensure_csr_float32_int32(Xtr), ensure_csr_float32_int32(Xva), ensure_csr_float32_int32(Xte)
    except ValueError:
        n_tr = len(tr_text); n_va = len(va_text); n_te = len(te_text)
        Ztr = csr_matrix((n_tr, 0), dtype=np.float32)
        Zva = csr_matrix((n_va, 0), dtype=np.float32)
        Zte = csr_matrix((n_te, 0), dtype=np.float32)
        return Ztr, Zva, Zte


def build_fold_matrices(
    df_tr: pd.DataFrame,
    df_va: pd.DataFrame,
    df_te: pd.DataFrame,
    num_cols: List[str],
    cat_cols: List[str],
) -> Tuple[csr_matrix, csr_matrix, csr_matrix]:
    pre = build_tab_preprocessor(num_cols, cat_cols)
    Xtr_tab = ensure_csr_float32_int32(pre.fit_transform(df_tr))
    Xva_tab = ensure_csr_float32_int32(pre.transform(df_va))
    Xte_tab = ensure_csr_float32_int32(pre.transform(df_te))

    vec_rec = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b[A-Za-z0-9]+\b",
        lowercase=True,
        ngram_range=(1, 2),
        max_features=MAXF_RECEIPT,
        min_df=MIN_DF_RECEIPT,
        max_df=1.0,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    Xtr_r, Xva_r, Xte_r = safe_tfidf_fit_transform(
        vec_rec,
        df_tr["receipt_doc"].values.astype(str),
        df_va["receipt_doc"].values.astype(str),
        df_te["receipt_doc"].values.astype(str),
    )

    vec_dis = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
        lowercase=True,
        ngram_range=(1, 2),
        max_features=MAXF_DISCHARGE,
        min_df=MIN_DF_NOTES,
        max_df=1.0,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    Xtr_d, Xva_d, Xte_d = safe_tfidf_fit_transform(
        vec_dis,
        df_tr["discharge_doc"].values.astype(str),
        df_va["discharge_doc"].values.astype(str),
        df_te["discharge_doc"].values.astype(str),
    )

    vec_vn = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
        lowercase=True,
        ngram_range=(1, 2),
        max_features=MAXF_VITALS_NOTE,
        min_df=MIN_DF_NOTES,
        max_df=1.0,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    Xtr_vt, Xva_vt, Xte_vt = safe_tfidf_fit_transform(
        vec_vn,
        df_tr["vitals_doc"].values.astype(str),
        df_va["vitals_doc"].values.astype(str),
        df_te["vitals_doc"].values.astype(str),
    )

    Xtr = ensure_csr_float32_int32(hstack([Xtr_tab, Xtr_r, Xtr_d, Xtr_vt], format="csr"))
    Xva = ensure_csr_float32_int32(hstack([Xva_tab, Xva_r, Xva_d, Xva_vt], format="csr"))
    Xte = ensure_csr_float32_int32(hstack([Xte_tab, Xte_r, Xte_d, Xte_vt], format="csr"))
    return Xtr, Xva, Xte


# =========================
# TRAINING
# =========================
def fit_xgb(
    X_tr: csr_matrix, y_tr: np.ndarray,
    X_va: csr_matrix, y_va: np.ndarray,
    params: Dict[str, Any],
    seed: int,
    tag: str,
) -> XGBRegressor:
    model = XGBRegressor(**params, random_state=seed)
    fit_sig = inspect.signature(model.fit).parameters
    fit_kwargs = {"eval_set": [(X_va, y_va)]}
    if "verbose_eval" in fit_sig:
        fit_kwargs["verbose_eval"] = VERBOSE_EVERY
    else:
        fit_kwargs["verbose"] = VERBOSE_EVERY
    logging.info(f"{tag} | fit START | obj={params.get('objective')} | device={params.get('device')}")
    model.fit(X_tr, y_tr, **fit_kwargs)
    logging.info(f"{tag} | fit END | best_iteration={getattr(model,'best_iteration',None)}")
    return model


def shrunk_group_median_bias(residuals: np.ndarray, groups: np.ndarray, alpha: float) -> Tuple[Dict[str, float], float]:
    g = groups.astype(str)
    global_med = float(np.median(residuals))
    biases: Dict[str, float] = {}
    for key in np.unique(g):
        m = (g == key)
        rg = residuals[m]
        if rg.size == 0:
            continue
        med_g = float(np.median(rg))
        n = float(rg.size)
        b = (n * med_g + alpha * global_med) / (n + alpha)
        biases[str(key)] = float(b)
    return biases, global_med


def apply_bias(pred: np.ndarray, groups: np.ndarray, biases: Dict[str, float], default_bias: float) -> np.ndarray:
    out = pred.astype(np.float64).copy()
    g = groups.astype(str)
    for i in range(len(out)):
        out[i] += biases.get(str(g[i]), default_bias)
    return out.astype(np.float32)


def run_iteration8(diagnostic: bool = False, force_cpu: bool = False) -> Dict[str, Any]:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb,'__version__','unknown')}")
    logging.info("=== Iteration 8: multi-source + leakage-safe CV + receipt semantic tokens ===")

    train_df, test_df = assemble_dataset()
    logging.info(f"train_df={train_df.shape} test_df={test_df.shape}")

    coverage = {
        "receipt_nonempty": int((train_df["receipt_doc"].str.len() > 0).sum()),
        "discharge_nonempty": int((train_df["discharge_doc"].str.len() > 0).sum()) if "discharge_doc" in train_df else 0,
        "vitals_nonempty": int((train_df["vitals_doc"].str.len() > 0).sum()) if "vitals_doc" in train_df else 0,
    }
    logging.info(f"Coverage (train): {coverage}")
    if diagnostic:
        return {"coverage": coverage}

    y = train_df["ed_cost_next3y_usd"].values.astype(np.float32)

    doc_cols = {"receipt_doc", "discharge_doc", "vitals_doc"}
    exclude = {"patient_id", "ed_cost_next3y_usd", "is_train"} | doc_cols

    preferred_cats = ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3"]
    cat_cols = [c for c in preferred_cats if c in train_df.columns]

    for c in train_df.columns:
        if c in exclude or c in cat_cols:
            continue
        if train_df[c].dtype == "object" or str(train_df[c].dtype).startswith("string"):
            cat_cols.append(c)

    num_cols = []
    for c in train_df.columns:
        if c in exclude or c in cat_cols:
            continue
        if pd.api.types.is_numeric_dtype(train_df[c]):
            num_cols.append(c)

    for c in cat_cols:
        train_df[c] = train_df[c].astype("string").fillna("missing")
        test_df[c] = test_df[c].astype("string").fillna("missing")

    logging.info(f"Num cols={len(num_cols)} | Cat cols={len(cat_cols)} | Text cols=3")

    device = "cpu" if force_cpu else "cuda"
    common_params = dict(
        eval_metric="mae",
        tree_method="hist",
        device=device,
        n_estimators=N_ESTIMATORS,
        learning_rate=LR,
        max_depth=MAX_DEPTH,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        min_child_weight=MIN_CHILD_WEIGHT,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )
    params_l2 = dict(**common_params, objective="reg:squarederror")
    params_q50 = dict(**common_params, objective="reg:quantileerror", quantile_alpha=0.5)

    if "prior_ed_cost_5y_usd" in train_df.columns:
        bins = pd.qcut(train_df["prior_ed_cost_5y_usd"], q=10, duplicates="drop")
        strat = train_df["primary_chronic"].astype(str) + "_" + bins.astype(str)
    else:
        strat = train_df["primary_chronic"].astype(str)

    oof_l2 = np.zeros(len(train_df), dtype=np.float64)
    oof_q = np.zeros(len(train_df), dtype=np.float64)
    cnt = np.zeros(len(train_df), dtype=np.float64)

    te_l2 = np.zeros(len(test_df), dtype=np.float64)
    te_q = np.zeros(len(test_df), dtype=np.float64)
    n_fit = 0

    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train_df)), strat)):
            splits.append((rep, fold, seed, tr_idx, va_idx))
    logging.info(f"Total CV folds: {len(splits)} | device={device}")

    for rep, fold, seed, tr_idx, va_idx in tqdm(splits, desc="CV", ncols=110):
        tag = f"rep={rep} fold={fold}"
        t0 = time.time()

        df_tr = train_df.iloc[tr_idx].copy()
        df_va = train_df.iloc[va_idx].copy()
        df_te = test_df.copy()

        logging.info(f"{tag} | FE START")
        X_tr, X_va, X_te = build_fold_matrices(df_tr, df_va, df_te, num_cols, cat_cols)
        logging.info(f"{tag} | FE END | X_tr={X_tr.shape} nnz={X_tr.nnz:,}")

        if USE_L2:
            mdl_l2 = fit_xgb(X_tr, y[tr_idx], X_va, y[va_idx], params_l2, seed=10000 + seed + fold, tag=f"{tag} | L2")
            p_va = mdl_l2.predict(X_va)
            p_te = mdl_l2.predict(X_te)
            oof_l2[va_idx] += p_va
            te_l2 += p_te
            logging.info(f"{tag} | L2 fold MAE={mae(y[va_idx], p_va):.3f}")

        if USE_Q50:
            mdl_q = fit_xgb(X_tr, y[tr_idx], X_va, y[va_idx], params_q50, seed=20000 + seed + fold, tag=f"{tag} | Q50")
            p_va = mdl_q.predict(X_va)
            p_te = mdl_q.predict(X_te)
            oof_q[va_idx] += p_va
            te_q += p_te
            logging.info(f"{tag} | Q50 fold MAE={mae(y[va_idx], p_va):.3f}")

        cnt[va_idx] += 1.0
        n_fit += 1
        logging.info(f"{tag} | DONE | fold_time={time.time()-t0:.1f}s")

    oof_l2 = (oof_l2 / np.maximum(cnt, 1.0)).astype(np.float32)
    oof_q = (oof_q / np.maximum(cnt, 1.0)).astype(np.float32)

    te_l2 = (te_l2 / max(1, n_fit)).astype(np.float32)
    te_q = (te_q / max(1, n_fit)).astype(np.float32)

    if USE_L2 and not USE_Q50:
        w_best = 1.0
    elif USE_Q50 and not USE_L2:
        w_best = 0.0
    else:
        grid = np.arange(0.0, 1.0 + 1e-9, BLEND_GRID_STEP)
        best_mae = 1e18
        w_best = 0.5
        for w in grid:
            pred = w * oof_l2 + (1.0 - w) * oof_q
            s = mae(y, pred)
            if s < best_mae:
                best_mae = s
                w_best = float(w)
        logging.info(f"Blend best: w(L2)={w_best:.2f} | OOF MAE={best_mae:.4f}")

    oof = (w_best * oof_l2 + (1.0 - w_best) * oof_q).astype(np.float32)
    te = (w_best * te_l2 + (1.0 - w_best) * te_q).astype(np.float32)

    if CLIP_AT_ZERO:
        oof = np.maximum(oof, 0.0)
        te = np.maximum(te, 0.0)

    mae_raw = mae(y, oof)
    logging.info(f"OOF MAE (blend raw): {mae_raw:.4f}")

    chosen = "raw"
    final_te = te

    if USE_CALIBRATION:
        resid = (y.astype(np.float64) - oof.astype(np.float64))

        b0 = float(np.median(resid))
        oof_g = oof + b0
        te_g = te + b0
        if CLIP_AT_ZERO:
            oof_g = np.maximum(oof_g, 0.0)
            te_g = np.maximum(te_g, 0.0)
        mae_g = mae(y, oof_g)

        grp = train_df["primary_chronic"].astype(str).values
        biases, default_bias = shrunk_group_median_bias(resid, grp, alpha=CALIB_ALPHA)
        oof_c = apply_bias(oof, grp, biases, default_bias)
        te_c = apply_bias(te, test_df["primary_chronic"].astype(str).values, biases, default_bias)
        if CLIP_AT_ZERO:
            oof_c = np.maximum(oof_c, 0.0)
            te_c = np.maximum(te_c, 0.0)
        mae_c = mae(y, oof_c)

        logging.info(f"Calibration OOF MAE: raw={mae_raw:.4f} | global_median={mae_g:.4f} | chronic_shrunk={mae_c:.4f}")

        best = (mae_raw, "raw", te)
        if mae_g < best[0] - 1e-9:
            best = (mae_g, "global_median", te_g)
        if mae_c < best[0] - 1e-9:
            best = (mae_c, "chronic_shrunk_median", te_c)

        chosen = best[1]
        final_te = best[2].astype(np.float32)
        logging.info(f"Chosen postprocess: {chosen} | OOF MAE={best[0]:.4f}")

    sub = pd.DataFrame({
        "patient_id": test_df["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": final_te.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")

    return {
        "oof_mae_raw": float(mae_raw),
        "blend_w_l2": float(w_best),
        "postprocess": chosen,
        "submission_path": str(SUBMISSION_PATH),
        "coverage": coverage,
    }


# =========================
# Notebook/CLI glue
# =========================
def _in_notebook() -> bool:
    try:
        import IPython  # noqa
        ip = IPython.get_ipython()
        if ip is None:
            return False
        return ("IPKernelApp" in ip.config) or ("ipykernel" in sys.modules)
    except Exception:
        return ("ipykernel" in sys.modules)


def _parse_cli(argv: Optional[List[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument("--diagnostic", action="store_true")
    parser.add_argument("--cpu", action="store_true")
    args, unknown = parser.parse_known_args(argv)
    if unknown:
        logging.info(f"Ignoring unknown args: {unknown}")
    return args


if __name__ == "__main__":
    setup_logging()
    if _in_notebook():
        logging.info("Notebook detected: call run_iteration8(diagnostic=True) or run_iteration8() manually.")
    else:
        a = _parse_cli()
        run_iteration8(diagnostic=a.diagnostic, force_cpu=a.cpu)


2026-02-13 18:22:29,461 | INFO | Notebook detected: call run_iteration8(diagnostic=True) or run_iteration8() manually.


## 第一次先跑覆盖率

In [33]:
res = run_iteration8(diagnostic=True)
res

2026-02-13 18:14:30,340 | INFO | XGBoost version: 3.0.0
2026-02-13 18:14:30,341 | INFO | === Iteration 8: multi-source + leakage-safe CV + receipt semantic tokens ===
2026-02-13 18:14:30,375 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 18:14:30,379 | INFO | Receipts cache: hit=0 miss=4,000
Parsing receipts: 100%|████████████████████████████████████████| 4000/4000 [00:06<00:00, 619.08it/s]
2026-02-13 18:14:37,046 | INFO | Built relational aggs: (4000, 11)
2026-02-13 18:14:37,134 | INFO | Built discharge docs: (4000, 3)
2026-02-13 18:14:37,635 | INFO | Built vitals features: (2000, 55)
2026-02-13 18:14:37,660 | INFO | train_df=(2000, 93) test_df=(2000, 93)
2026-02-13 18:14:37,664 | INFO | Coverage (train): {'receipt_nonempty': 1154, 'discharge_nonempty': 2000, 'vitals_nonempty': 972}


{'coverage': {'receipt_nonempty': 1154,
  'discharge_nonempty': 2000,
  'vitals_nonempty': 972}}

## training
- 496.6276

In [36]:
res = run_iteration8(diagnostic=False)
res


2026-02-13 18:22:35,922 | INFO | XGBoost version: 3.0.0
2026-02-13 18:22:35,922 | INFO | === Iteration 8: multi-source + leakage-safe CV + receipt semantic tokens ===
2026-02-13 18:22:35,933 | INFO | Merged patients.csv: train=(2000, 9), test=(2000, 8)
2026-02-13 18:22:36,008 | INFO | Receipts cache: hit=4,000 miss=0
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 18:22:36,055 | INFO | train_df=(2000, 93) test_df=(2000, 93)
2026-02-13 18:22:36,058 | INFO | Coverage (train): {'receipt_nonempty': 1154, 'discharge_nonempty': 2000, 'vitals_nonempty': 972}
2026-02-13 18:22:36,065 | INFO | Num cols=81 | Cat cols=6 | Text cols=3
2026-02-13 18:22:36,074 | INFO | Total CV folds: 10 | device=cuda
CV:   0%|                                                                              | 0/10 [00:00<?, ?it/s]2026-02-13 18:22:36,080 | INFO | rep=0 fold=0 | FE START
2026-02-13 18:22:36,480 | INFO | rep=0 fold=0 | FE END | X_tr=(1600, 4847) nnz=337,520
2026-02-13 18:22:36,481 | INFO | rep=0 fold=0 | L2

[0]	validation_0-mae:1386.63916
[100]	validation_0-mae:547.69932
[200]	validation_0-mae:602.87091
[300]	validation_0-mae:667.59532
[308]	validation_0-mae:671.53743


2026-02-13 18:22:41,651 | INFO | rep=0 fold=0 | L2 | fit END | best_iteration=109
2026-02-13 18:22:41,685 | INFO | rep=0 fold=0 | L2 fold MAE=538.174
2026-02-13 18:22:41,688 | INFO | rep=0 fold=0 | Q50 | fit START | obj=reg:quantileerror | device=cuda


[0]	validation_0-mae:1366.16789
[100]	validation_0-mae:556.37253
[200]	validation_0-mae:495.02189
[300]	validation_0-mae:489.38689
[400]	validation_0-mae:489.71920
[500]	validation_0-mae:491.54499
[518]	validation_0-mae:491.71394


2026-02-13 18:22:48,274 | INFO | rep=0 fold=0 | Q50 | fit END | best_iteration=319
2026-02-13 18:22:48,312 | INFO | rep=0 fold=0 | Q50 fold MAE=488.233
2026-02-13 18:22:48,313 | INFO | rep=0 fold=0 | DONE | fold_time=12.2s
CV:  10%|███████                                                               | 1/10 [00:12<01:50, 12.24s/it]2026-02-13 18:22:48,324 | INFO | rep=0 fold=1 | FE START
2026-02-13 18:22:48,887 | INFO | rep=0 fold=1 | FE END | X_tr=(1600, 4907) nnz=337,374
2026-02-13 18:22:48,889 | INFO | rep=0 fold=1 | L2 | fit START | obj=reg:squarederror | device=cuda


...

[0]	validation_0-mae:1414.91766
[100]	validation_0-mae:558.82668
[200]	validation_0-mae:639.84448
[300]	validation_0-mae:683.63734
[324]	validation_0-mae:728.31564


2026-02-13 18:24:41,843 | INFO | rep=1 fold=4 | L2 | fit END | best_iteration=124
2026-02-13 18:24:41,868 | INFO | rep=1 fold=4 | L2 fold MAE=538.084
2026-02-13 18:24:41,869 | INFO | rep=1 fold=4 | Q50 | fit START | obj=reg:quantileerror | device=cuda


[0]	validation_0-mae:1385.42276
[100]	validation_0-mae:555.30298
[200]	validation_0-mae:481.98214
[300]	validation_0-mae:479.29411
[400]	validation_0-mae:481.89024
[459]	validation_0-mae:481.28887


2026-02-13 18:24:48,452 | INFO | rep=1 fold=4 | Q50 | fit END | best_iteration=260
2026-02-13 18:24:48,481 | INFO | rep=1 fold=4 | Q50 fold MAE=477.651
2026-02-13 18:24:48,482 | INFO | rep=1 fold=4 | DONE | fold_time=12.9s
CV: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [02:12<00:00, 13.24s/it]
2026-02-13 18:24:48,496 | INFO | Blend best: w(L2)=0.04 | OOF MAE=488.2213
2026-02-13 18:24:48,498 | INFO | OOF MAE (blend raw): 488.2213
2026-02-13 18:24:48,505 | INFO | Calibration OOF MAE: raw=488.2213 | global_median=488.1370 | chronic_shrunk=478.1611
2026-02-13 18:24:48,507 | INFO | Chosen postprocess: chronic_shrunk_median | OOF MAE=478.1611
2026-02-13 18:24:48,532 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission.csv


{'oof_mae_raw': 488.22125244140625,
 'blend_w_l2': 0.04,
 'postprocess': 'chronic_shrunk_median',
 'submission_path': 'D:\\AgentDs\\agent_ds_healthcare\\submission.csv',
 'coverage': {'receipt_nonempty': 1154,
  'discharge_nonempty': 2000,
  'vitals_nonempty': 972}}

# Iteration 9
- 490.2396

In [44]:

# -*- coding: utf-8 -*-
r"""
ITERATION 9 — Outlier-Aware Segmented Regression + Extreme Calibration (XGBoost GPU)
===================================================================================

Why this is a "different mathematical path":
- Your adversarial AUC ~ 0.51 means train/test feature distributions are similar; drift weighting is noise.  (AUC~0.5 => similar)  :contentReference[oaicite:7]{index=7}
- We're focusing on heavy-tail/outliers: a hurdle/segmented model:
    y = body (<= cap) + tail_excess (only if y > cap)
- Body model: quantile regression alpha=0.5 (MAE-aligned).  :contentReference[oaicite:8]{index=8}
- Tail model: (1) classifier for tail event (2) regressor for excess magnitude (log1p).
- Then do "extreme calibration": tune (k, gamma) on OOF without retraining, and do bin-wise median residual correction.

Constraints honored:
- NO SVD/PCA/NMF/embeddings
- NO manual clinical CPT grouping
- GPU-friendly XGBoost only
- Rich logs, notebook-safe (no argparse crash)

INSTALL:
pip install -U pandas numpy scipy scikit-learn xgboost pymupdf joblib tqdm
"""

from __future__ import annotations

import re
import time
import sys
import logging
import warnings
import inspect
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib

import fitz  # PyMuPDF
from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import xgboost as xgb
from xgboost import XGBRegressor, XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# PATHS
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")

TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
PDF_DIR = BASE_DIR / "receipts_pdf"
SUBMISSION_PATH = BASE_DIR / "submission.csv"

CACHE_DIR = BASE_DIR / "cache_iter9"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RECEIPT_CACHE = CACHE_DIR / "receipts_codes_meta.joblib"
CACHE_VERSION = 91


# =========================
# CONFIG
# =========================
# TF-IDF (simple, sparse, no DR)
TFIDF_MAX_FEATURES = 1500      # you can try 1000/1500/2000; 1500 is a safe middle
TFIDF_NGRAM_RANGE = (1, 3)
TFIDF_MIN_DF = 1
TFIDF_MAX_DF = 1.0

# CV
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026]     # total 10 folds
EARLY_STOPPING = 200

# Outlier segmentation
TAIL_Q = 0.90                  # 0.90 => ~200 tail samples if n=2000
CLIP_AT_ZERO = True

# Tail mixing calibration search
GAMMA_GRID = [0.6, 0.8, 1.0, 1.2, 1.5]
K_GRID = [0.6, 0.8, 1.0, 1.2, 1.5]

# Piecewise median residual calibration
CALIB_BINS = 10
CALIB_SHRINK_ALPHA = 80.0      # larger => less overfit
CALIB_CORR_CLIP = 500.0        # clip residual correction per bin for safety

# XGB common
N_ESTIMATORS = 8000
LR = 0.02
SUBSAMPLE = 0.85
COLSAMPLE = 0.75
REG_LAMBDA = 1.0
MIN_CHILD_WEIGHT = 1.0

# Separate depths for stability
DEPTH_BODY = 6
DEPTH_TAIL_CLS = 4
DEPTH_TAIL_REG = 4

# Heartbeats
VERBOSE_EVERY = 100


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    root = logging.getLogger()
    if not root.handlers:
        logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    else:
        root.setLevel(logging.INFO)


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def ensure_csr_float32_int32(X: Any) -> csr_matrix:
    # If it's not already a CSR matrix, convert it safely
    if not isinstance(X, csr_matrix):
        if hasattr(X, "tocsr"):
            X = X.tocsr()
        else:
            # This handles dense numpy arrays returned by ColumnTransformer
            X = csr_matrix(X)
            
    # Cast to requested dtypes for XGBoost GPU memory efficiency
    if X.data.dtype != np.float32:
        X.data = X.data.astype(np.float32, copy=False)
    if X.indices.dtype != np.int32:
        X.indices = X.indices.astype(np.int32, copy=False)
    if X.indptr.dtype != np.int32:
        X.indptr = X.indptr.astype(np.int32, copy=False)
    return X


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


# =========================
# RECEIPT PARSING (FAST, ROBUST)
# =========================
ZIP_INS_RE = re.compile(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)")
TOTAL_RE = re.compile(r"TOTAL\s+([\d,]+\.\d{2})")

LINE_ITEM_RE = re.compile(
    r"^([A-Z]\d{4}|\d{5})\s+(.+?)\s+(\d+)\s+([\d,]+\.\d{2})\s+([\d,]+\.\d{2})\s*$"
)

CODE_TOKEN_RE = re.compile(r"\b(?:[A-Z]\d{4}|\d{5})\b")


def _money_to_float(s: str) -> float:
    return float(s.replace(",", ""))


def featurize_receipt(patient_id: int) -> Dict[str, Any]:
    """
    Output dict:
      - codes_doc: string of repeated CPT/HCPCS tokens (repeated by qty capped)
      - meta numeric/cat features
    """
    pdf_path = PDF_DIR / f"receipt_{patient_id}.pdf"

    out = dict(
        patient_id=int(patient_id),
        codes_doc="",
        pdf_ok=0,
        pdf_zip3="missing",
        pdf_insurance="missing",
        pdf_n_items=0,
        pdf_qty_sum=0,
        pdf_n_unique_codes=0,
        pdf_line_max=0.0,
        pdf_line_mean=0.0,
        pdf_total=0.0,
        pdf_cost_concentration=0.0,
        pdf_parse_notes="missing_pdf",
    )

    if not pdf_path.exists():
        return out

    try:
        with fitz.open(pdf_path) as doc:
            text = ""
            for page in doc:
                text += (page.get_text("text") or "") + "\n"
    except Exception as e:
        out["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return out

    text = text or ""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    # header
    m = ZIP_INS_RE.search(text)
    if m:
        out["pdf_zip3"] = str(m.group(1))
        out["pdf_insurance"] = str(m.group(2)).lower()

    m = TOTAL_RE.search(text)
    if m:
        try:
            out["pdf_total"] = float(_money_to_float(m.group(1)))
        except Exception:
            out["pdf_total"] = 0.0

    # line items
    tokens: List[str] = []
    line_totals: List[float] = []
    qty_sum = 0
    n_items = 0

    for ln in lines:
        mm = LINE_ITEM_RE.match(ln)
        if not mm:
            continue
        code, desc, qty_s, unit_s, line_s = mm.groups()
        try:
            qty = int(qty_s)
        except Exception:
            qty = 1
        qty = max(1, min(qty, 20))
        qty_sum += qty
        n_items += 1

        try:
            line_total = float(_money_to_float(line_s))
            line_totals.append(line_total)
        except Exception:
            pass

        # repeat by qty (capped) to preserve intensity without manual grouping
        rep = max(1, min(qty, 10))
        tokens.extend([code] * rep)

    if len(tokens) == 0:
        # fallback: just extract any code tokens from whole text
        tokens = CODE_TOKEN_RE.findall(text)

    out["codes_doc"] = " ".join(tokens)
    out["pdf_ok"] = 1 if len(out["codes_doc"]) > 0 else 0
    out["pdf_n_items"] = int(n_items)
    out["pdf_qty_sum"] = int(qty_sum)
    out["pdf_n_unique_codes"] = int(len(set(tokens)))

    if len(line_totals) > 0:
        out["pdf_line_max"] = float(np.max(line_totals))
        out["pdf_line_mean"] = float(np.mean(line_totals))
        if out["pdf_total"] > 0:
            out["pdf_cost_concentration"] = float(out["pdf_line_max"] / out["pdf_total"])
    else:
        out["pdf_line_max"] = 0.0
        out["pdf_line_mean"] = 0.0
        out["pdf_cost_concentration"] = 0.0

    out["pdf_parse_notes"] = "ok"
    return out


def load_receipts(patient_ids: List[int]) -> pd.DataFrame:
    cache = {"version": CACHE_VERSION, "data": {}}
    if RECEIPT_CACHE.exists():
        try:
            cache = joblib.load(RECEIPT_CACHE)
        except Exception:
            cache = {"version": CACHE_VERSION, "data": {}}
    if not isinstance(cache, dict) or cache.get("version") != CACHE_VERSION:
        cache = {"version": CACHE_VERSION, "data": {}}

    data = cache.get("data", {})
    missing = [pid for pid in patient_ids if str(pid) not in data]

    logging.info(f"Receipt cache | hit={len(patient_ids)-len(missing):,} miss={len(missing):,} version={CACHE_VERSION}")

    for pid in tqdm(missing, desc="Parsing receipts", ncols=100):
        data[str(pid)] = featurize_receipt(int(pid))

    cache["data"] = data
    if missing:
        try:
            joblib.dump(cache, RECEIPT_CACHE)
        except Exception as e:
            logging.warning(f"Receipt cache save failed: {e}")

    rows = []
    for pid in patient_ids:
        obj = data.get(str(pid))
        if obj is None:
            obj = featurize_receipt(int(pid))
        rows.append(obj)
    return pd.DataFrame(rows)


# =========================
# FEATURE BUILDING (fold-safe)
# =========================
def build_preprocessor(num_cols: List[str], cat_cols: List[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def safe_tfidf_fit_transform(tr_text: np.ndarray, va_text: np.ndarray, te_text: np.ndarray) -> Tuple[csr_matrix, csr_matrix, csr_matrix]:
    vec = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b(?:[A-Z]\d{4}|\d{5})\b",
        lowercase=False,
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    try:
        Xtr = vec.fit_transform(tr_text.astype(str))
        Xva = vec.transform(va_text.astype(str))
        Xte = vec.transform(te_text.astype(str))
        return ensure_csr_float32_int32(Xtr), ensure_csr_float32_int32(Xva), ensure_csr_float32_int32(Xte)
    except ValueError:
        # empty vocabulary etc.
        n_tr, n_va, n_te = len(tr_text), len(va_text), len(te_text)
        Ztr = csr_matrix((n_tr, 0), dtype=np.float32)
        Zva = csr_matrix((n_va, 0), dtype=np.float32)
        Zte = csr_matrix((n_te, 0), dtype=np.float32)
        return Ztr, Zva, Zte


def build_fold_matrices(
    df_tr: pd.DataFrame, df_va: pd.DataFrame, df_te: pd.DataFrame,
    num_cols: List[str], cat_cols: List[str],
) -> Tuple[csr_matrix, csr_matrix, csr_matrix]:
    pre = build_preprocessor(num_cols, cat_cols)
    Xtr_tab = ensure_csr_float32_int32(pre.fit_transform(df_tr))
    Xva_tab = ensure_csr_float32_int32(pre.transform(df_va))
    Xte_tab = ensure_csr_float32_int32(pre.transform(df_te))

    Xtr_txt, Xva_txt, Xte_txt = safe_tfidf_fit_transform(
        df_tr["codes_doc"].values,
        df_va["codes_doc"].values,
        df_te["codes_doc"].values,
    )

    Xtr = ensure_csr_float32_int32(hstack([Xtr_tab, Xtr_txt], format="csr"))
    Xva = ensure_csr_float32_int32(hstack([Xva_tab, Xva_txt], format="csr"))
    Xte = ensure_csr_float32_int32(hstack([Xte_tab, Xte_txt], format="csr"))
    return Xtr, Xva, Xte


# =========================
# MODEL HELPERS
# =========================
def _fit_with_verbose(model, X_tr, y_tr, X_va, y_va, tag: str):
    fit_sig = inspect.signature(model.fit).parameters
    fit_kwargs = {"eval_set": [(X_va, y_va)]}
    if "verbose_eval" in fit_sig:
        fit_kwargs["verbose_eval"] = VERBOSE_EVERY
    else:
        fit_kwargs["verbose"] = VERBOSE_EVERY

    logging.info(f"{tag} | fit START")
    model.fit(X_tr, y_tr, **fit_kwargs)
    logging.info(f"{tag} | fit END | best_iteration={getattr(model,'best_iteration',None)}")
    return model


# =========================
# CALIBRATION: bin-wise median residual (shrunk)
# =========================
def build_bin_median_correction(oof_pred: np.ndarray, y: np.ndarray, n_bins: int, alpha: float) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Returns:
      edges: quantile bin edges (len n_bins+1)
      corr: per-bin correction (len n_bins)
      global_med: global median residual
    """
    oof = oof_pred.astype(np.float64)
    resid = (y.astype(np.float64) - oof)

    # quantile edges; ensure strictly increasing
    qs = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(oof, qs)
    # fix duplicates
    for i in range(1, len(edges)):
        if edges[i] <= edges[i-1]:
            edges[i] = edges[i-1] + 1e-6

    bin_id = np.clip(np.digitize(oof, edges[1:-1], right=False), 0, n_bins - 1)

    global_med = float(np.median(resid))
    corr = np.zeros(n_bins, dtype=np.float64)

    for b in range(n_bins):
        m = (bin_id == b)
        rb = resid[m]
        if rb.size == 0:
            corr[b] = global_med
            continue
        med_b = float(np.median(rb))
        n = float(rb.size)
        c = (n * med_b + alpha * global_med) / (n + alpha)
        c = float(np.clip(c, -CALIB_CORR_CLIP, CALIB_CORR_CLIP))
        corr[b] = c

    return edges.astype(np.float64), corr.astype(np.float64), global_med


def apply_bin_correction(pred: np.ndarray, edges: np.ndarray, corr: np.ndarray) -> np.ndarray:
    p = pred.astype(np.float64)
    n_bins = len(corr)
    bin_id = np.clip(np.digitize(p, edges[1:-1], right=False), 0, n_bins - 1)
    p2 = p + corr[bin_id]
    return p2.astype(np.float32)


# =========================
# MAIN RUNNER
# =========================
def run_iteration9(diagnostic: bool = False, force_cpu: bool = False) -> Dict[str, Any]:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb,'__version__','unknown')}")
    logging.info("=== Iteration 9: Outlier-aware segmented regression + extreme calibration ===")
    logging.info(f"diagnostic={diagnostic} | force_cpu={force_cpu}")

    train = pd.read_csv(TRAIN_CSV)
    test = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train.shape} test={test.shape}")

    if PATIENTS_CSV.exists():
        pat = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in pat.columns:
            train = train.merge(pat, on="patient_id", how="left")
            test = test.merge(pat, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train.shape} test={test.shape}")

    # receipts
    all_ids = pd.concat([train["patient_id"], test["patient_id"]], ignore_index=True).astype(int).tolist()
    rec = load_receipts(all_ids)
    train = train.merge(rec, on="patient_id", how="left")
    test = test.merge(rec, on="patient_id", how="left")
    for df in [train, test]:
        df["codes_doc"] = df["codes_doc"].fillna("").astype(str)

    # basic feature derivations (simple, numeric)
    for df in [train, test]:
        if "prior_ed_cost_5y_usd" in df.columns and "prior_ed_visits_5y" in df.columns:
            df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
            df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
            df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

        # sanity: pdf_total should match prior_ed_cost_5y_usd per dataset description
        if "pdf_total" in df.columns and "prior_ed_cost_5y_usd" in df.columns:
            df["pdf_total_minus_prior"] = df["pdf_total"].astype(float) - df["prior_ed_cost_5y_usd"].astype(float)
            df["pdf_total_absdiff_prior"] = np.abs(df["pdf_total_minus_prior"])

    y = train["ed_cost_next3y_usd"].values.astype(np.float32)

    cap = float(np.quantile(y, TAIL_Q))
    y_tail = (y > cap).astype(np.int32)
    y_body = np.minimum(y, cap).astype(np.float32)
    y_excess = np.maximum(0.0, y - cap).astype(np.float32)

    logging.info(f"Tail threshold: cap=q{TAIL_Q:.2f}={cap:.2f} | tail_rate={y_tail.mean():.3f} (n_tail={y_tail.sum()})")

    if diagnostic:
        return {"cap": cap, "tail_rate": float(y_tail.mean()), "n_tail": int(y_tail.sum())}

    # columns
    doc_cols = {"codes_doc"}
    exclude = {"patient_id", "ed_cost_next3y_usd"} | doc_cols

    # categorical columns
    base_cat = []
    for c in ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3", "pdf_parse_notes"]:
        if c in train.columns:
            base_cat.append(c)

    # any other object/string => cat
    cat_cols = list(dict.fromkeys(base_cat))
    for c in train.columns:
        if c in exclude or c in cat_cols:
            continue
        if train[c].dtype == "object" or str(train[c].dtype).startswith("string"):
            cat_cols.append(c)

    num_cols = []
    for c in train.columns:
        if c in exclude or c in cat_cols:
            continue
        if pd.api.types.is_numeric_dtype(train[c]):
            num_cols.append(c)

    # fill cat
    for c in cat_cols:
        train[c] = train[c].astype("string").fillna("missing")
        test[c] = test[c].astype("string").fillna("missing")

    logging.info(f"Features: num={len(num_cols)} cat={len(cat_cols)} tfidf_max={TFIDF_MAX_FEATURES}")

    # stratification: ensure tail appears in each fold + chronic + prior cost decile
    if "prior_ed_cost_5y_usd" in train.columns:
        cost_bin = pd.qcut(train["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    else:
        cost_bin = pd.Series(["bin"] * len(train))
    strat = train["primary_chronic"].astype(str) + "_t" + pd.Series(y_tail).astype(str) + "_" + cost_bin

    # storage for components
    oof_body = np.zeros(len(train), dtype=np.float64)
    oof_p = np.zeros(len(train), dtype=np.float64)
    oof_ex = np.zeros(len(train), dtype=np.float64)
    oof_cnt = np.zeros(len(train), dtype=np.float64)

    te_body = np.zeros(len(test), dtype=np.float64)
    te_p = np.zeros(len(test), dtype=np.float64)
    te_ex = np.zeros(len(test), dtype=np.float64)
    n_models = 0

    device = "cpu" if force_cpu else "cuda"

    params_body = dict(
        objective="reg:quantileerror",
        quantile_alpha=0.5,
        eval_metric="mae",
        tree_method="hist",
        device=device,
        n_estimators=N_ESTIMATORS,
        learning_rate=LR,
        max_depth=DEPTH_BODY,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        min_child_weight=MIN_CHILD_WEIGHT,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    params_tail_cls = dict(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        device=device,
        n_estimators=6000,
        learning_rate=LR,
        max_depth=DEPTH_TAIL_CLS,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=REG_LAMBDA,
        min_child_weight=5.0,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    params_tail_reg = dict(
        objective="reg:squarederror",
        eval_metric="mae",
        tree_method="hist",
        device=device,
        n_estimators=6000,
        learning_rate=LR,
        max_depth=DEPTH_TAIL_REG,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=REG_LAMBDA,
        min_child_weight=5.0,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train)), strat)):
            splits.append((rep, fold, seed, tr_idx, va_idx))

    logging.info(f"Total folds: {len(splits)} | models per fold=3 | device={device}")

    for rep, fold, seed, tr_idx, va_idx in tqdm(splits, desc="CV", ncols=110):
        tag = f"rep={rep} fold={fold}"
        t0 = time.time()

        df_tr = train.iloc[tr_idx].copy()
        df_va = train.iloc[va_idx].copy()

        logging.info(f"{tag} | FE START")
        X_tr, X_va, X_te = build_fold_matrices(df_tr, df_va, test, num_cols, cat_cols)
        logging.info(f"{tag} | FE END | X_tr={X_tr.shape} nnz={X_tr.nnz:,}")

        # -------- Body model (capped target) --------
        body = XGBRegressor(**params_body, random_state=1000 * rep + seed + fold)
        _fit_with_verbose(body, X_tr, y_body[tr_idx], X_va, y_body[va_idx], f"{tag} | BODY")
        p_body_va = body.predict(X_va).astype(np.float32)
        p_body_te = body.predict(X_te).astype(np.float32)

        # keep body within [0, cap] to avoid double-counting tail
        p_body_va = np.clip(p_body_va, 0.0, cap)
        p_body_te = np.clip(p_body_te, 0.0, cap)

        # -------- Tail classifier --------
        # class imbalance weight
        pos = float(np.sum(y_tail[tr_idx] == 1))
        neg = float(np.sum(y_tail[tr_idx] == 0))
        spw = (neg / max(pos, 1.0))
        cls = XGBClassifier(**params_tail_cls, scale_pos_weight=spw, random_state=2000 * rep + seed + fold)
        _fit_with_verbose(cls, X_tr, y_tail[tr_idx], X_va, y_tail[va_idx], f"{tag} | TAIL_CLS")
        p_tail_va = cls.predict_proba(X_va)[:, 1].astype(np.float32)
        p_tail_te = cls.predict_proba(X_te)[:, 1].astype(np.float32)

        try:
            auc_va = roc_auc_score(y_tail[va_idx], p_tail_va)
            logging.info(f"{tag} | Tail AUC={auc_va:.4f}")
        except Exception:
            pass

        # -------- Tail excess regressor (only tail rows) --------
        tail_tr = np.where(y_tail[tr_idx] == 1)[0]
        tail_va = np.where(y_tail[va_idx] == 1)[0]  # for eval
        # handle rare case: no tail rows in train fold (shouldn't happen due strat, but safe)
        if tail_tr.size >= 20:
            X_tr_tail = X_tr[tail_tr]
            y_tr_tail = np.log1p(y_excess[tr_idx][tail_tr]).astype(np.float32)

            if tail_va.size >= 5:
                X_va_tail = X_va[tail_va]
                y_va_tail = np.log1p(y_excess[va_idx][tail_va]).astype(np.float32)
            else:
                # fallback eval
                X_va_tail = X_va
                y_va_tail = np.log1p(y_excess[va_idx]).astype(np.float32)

            ex = XGBRegressor(**params_tail_reg, random_state=3000 * rep + seed + fold)
            _fit_with_verbose(ex, X_tr_tail, y_tr_tail, X_va_tail, y_va_tail, f"{tag} | TAIL_REG")

            p_ex_va_log = ex.predict(X_va).astype(np.float32)
            p_ex_te_log = ex.predict(X_te).astype(np.float32)

            p_ex_va = np.expm1(np.maximum(p_ex_va_log, 0.0)).astype(np.float32)
            p_ex_te = np.expm1(np.maximum(p_ex_te_log, 0.0)).astype(np.float32)
        else:
            logging.info(f"{tag} | TAIL_REG skipped (tail_tr={tail_tr.size}) -> excess=0")
            p_ex_va = np.zeros(len(va_idx), dtype=np.float32)
            p_ex_te = np.zeros(len(test), dtype=np.float32)

        # store OOF components
        oof_body[va_idx] += p_body_va
        oof_p[va_idx] += p_tail_va
        oof_ex[va_idx] += p_ex_va
        oof_cnt[va_idx] += 1.0

        # store TEST components
        te_body += p_body_te
        te_p += p_tail_te
        te_ex += p_ex_te
        n_models += 1

        logging.info(f"{tag} | fold_time={time.time()-t0:.1f}s")

    # average components
    oof_body = (oof_body / np.maximum(oof_cnt, 1.0)).astype(np.float32)
    oof_p = (oof_p / np.maximum(oof_cnt, 1.0)).astype(np.float32)
    oof_ex = (oof_ex / np.maximum(oof_cnt, 1.0)).astype(np.float32)

    te_body = (te_body / max(1, n_models)).astype(np.float32)
    te_p = (te_p / max(1, n_models)).astype(np.float32)
    te_ex = (te_ex / max(1, n_models)).astype(np.float32)

    # search best (k, gamma) on OOF to minimize MAE
    best = (1e18, 1.0, 1.0)  # (mae, k, gamma)
    for gamma in GAMMA_GRID:
        p_g = np.power(np.clip(oof_p, 0.0, 1.0), gamma).astype(np.float32)
        for k in K_GRID:
            pred = oof_body + float(k) * p_g * oof_ex
            if CLIP_AT_ZERO:
                pred = np.maximum(pred, 0.0)
            s = mae(y, pred)
            if s < best[0]:
                best = (s, float(k), float(gamma))

    best_mae, best_k, best_gamma = best
    logging.info(f"Best tail mix on OOF: MAE={best_mae:.4f} | k={best_k:.2f} gamma={best_gamma:.2f}")

    # build final OOF / TEST pred
    oof_pred = oof_body + best_k * np.power(np.clip(oof_p, 0.0, 1.0), best_gamma) * oof_ex
    te_pred = te_body + best_k * np.power(np.clip(te_p, 0.0, 1.0), best_gamma) * te_ex

    if CLIP_AT_ZERO:
        oof_pred = np.maximum(oof_pred, 0.0)
        te_pred = np.maximum(te_pred, 0.0)

    mae_raw = mae(y, oof_pred)
    logging.info(f"OOF MAE (segmented raw): {mae_raw:.4f}")

    # piecewise median residual calibration
    edges, corr, global_med = build_bin_median_correction(oof_pred, y, n_bins=CALIB_BINS, alpha=CALIB_SHRINK_ALPHA)
    oof_cal = apply_bin_correction(oof_pred, edges, corr)
    te_cal = apply_bin_correction(te_pred, edges, corr)

    if CLIP_AT_ZERO:
        oof_cal = np.maximum(oof_cal, 0.0)
        te_cal = np.maximum(te_cal, 0.0)

    mae_cal = mae(y, oof_cal)
    logging.info(f"OOF MAE (after bin-median calibration): {mae_cal:.4f} | global_med_resid={global_med:.2f}")

    # choose by OOF
    if mae_cal <= mae_raw + 1e-9:
        final = te_cal
        chosen = "bin_median_calibrated"
    else:
        final = te_pred
        chosen = "raw"

    # submission
    sub = pd.DataFrame({
        "patient_id": test["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": final.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH} | postprocess={chosen}")

    return {
        "cap": cap,
        "tail_rate": float(y_tail.mean()),
        "best_k": best_k,
        "best_gamma": best_gamma,
        "oof_mae_raw": float(mae_raw),
        "oof_mae_cal": float(mae_cal),
        "chosen": chosen,
        "submission_path": str(SUBMISSION_PATH),
    }


## diagnose

In [45]:
res = run_iteration9(diagnostic=True)
res

2026-02-13 19:22:34,443 | INFO | XGBoost version: 3.0.0
2026-02-13 19:22:34,444 | INFO | === Iteration 9: Outlier-aware segmented regression + extreme calibration ===
2026-02-13 19:22:34,444 | INFO | diagnostic=True | force_cpu=False
2026-02-13 19:22:34,449 | INFO | Loaded train=(2000, 5) test=(2000, 4)
2026-02-13 19:22:34,457 | INFO | Merged patients.csv: train=(2000, 9) test=(2000, 8)
2026-02-13 19:22:34,510 | INFO | Receipt cache | hit=4,000 miss=0 version=91
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 19:22:34,526 | INFO | Tail threshold: cap=q0.90=6636.48 | tail_rate=0.100 (n_tail=200)


{'cap': 6636.47509765625, 'tail_rate': 0.1, 'n_tail': 200}

## train
- 

In [46]:
res = run_iteration9(diagnostic=False)
res

2026-02-13 19:22:37,280 | INFO | XGBoost version: 3.0.0
2026-02-13 19:22:37,280 | INFO | === Iteration 9: Outlier-aware segmented regression + extreme calibration ===
2026-02-13 19:22:37,281 | INFO | diagnostic=False | force_cpu=False
2026-02-13 19:22:37,286 | INFO | Loaded train=(2000, 5) test=(2000, 4)
2026-02-13 19:22:37,290 | INFO | Merged patients.csv: train=(2000, 9) test=(2000, 8)
2026-02-13 19:22:37,350 | INFO | Receipt cache | hit=4,000 miss=0 version=91
Parsing receipts: 0it [00:00, ?it/s]
2026-02-13 19:22:37,366 | INFO | Tail threshold: cap=q0.90=6636.48 | tail_rate=0.100 (n_tail=200)
2026-02-13 19:22:37,374 | INFO | Features: num=16 cat=7 tfidf_max=1500
2026-02-13 19:22:37,385 | INFO | Total folds: 10 | models per fold=3 | device=cuda
CV:   0%|                                                                              | 0/10 [00:00<?, ?it/s]2026-02-13 19:22:37,391 | INFO | rep=0 fold=0 | FE START
2026-02-13 19:22:37,465 | INFO | rep=0 fold=0 | FE END | X_tr=(1600, 1548) n

[0]	validation_0-mae:1313.27470
[100]	validation_0-mae:478.87179
[200]	validation_0-mae:424.67585
[300]	validation_0-mae:420.75360
[400]	validation_0-mae:417.95396
[500]	validation_0-mae:417.24340
[600]	validation_0-mae:417.80968
[675]	validation_0-mae:417.88770


2026-02-13 19:22:41,038 | INFO | rep=0 fold=0 | BODY | fit END | best_iteration=475
2026-02-13 19:22:41,071 | INFO | rep=0 fold=0 | TAIL_CLS | fit START


[0]	validation_0-auc:0.98104
[100]	validation_0-auc:0.99135
[200]	validation_0-auc:0.99090
[246]	validation_0-auc:0.99125


2026-02-13 19:22:41,761 | INFO | rep=0 fold=0 | TAIL_CLS | fit END | best_iteration=47
2026-02-13 19:22:41,771 | INFO | rep=0 fold=0 | Tail AUC=0.9922
2026-02-13 19:22:41,773 | INFO | rep=0 fold=0 | TAIL_REG | fit START


...

[0]	validation_0-auc:0.97208
[100]	validation_0-auc:0.98674
[200]	validation_0-auc:0.98611
[300]	validation_0-auc:0.98813
[400]	validation_0-auc:0.98569
[494]	validation_0-auc:0.98493


2026-02-13 19:23:26,516 | INFO | rep=1 fold=4 | TAIL_CLS | fit END | best_iteration=295
2026-02-13 19:23:26,541 | INFO | rep=1 fold=4 | Tail AUC=0.9882
2026-02-13 19:23:26,544 | INFO | rep=1 fold=4 | TAIL_REG | fit START


[0]	validation_0-mae:0.69134
[100]	validation_0-mae:0.50666
[200]	validation_0-mae:0.54971
[300]	validation_0-mae:0.57295
[314]	validation_0-mae:0.56980


2026-02-13 19:23:27,266 | INFO | rep=1 fold=4 | TAIL_REG | fit END | best_iteration=114
2026-02-13 19:23:27,276 | INFO | rep=1 fold=4 | fold_time=5.1s
CV: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [00:49<00:00,  4.99s/it]
2026-02-13 19:23:27,280 | INFO | Best tail mix on OOF: MAE=474.5739 | k=1.50 gamma=1.50
2026-02-13 19:23:27,281 | INFO | OOF MAE (segmented raw): 474.5739
2026-02-13 19:23:27,283 | INFO | OOF MAE (after bin-median calibration): 471.8822 | global_med_resid=-36.63
2026-02-13 19:23:27,290 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission.csv | postprocess=bin_median_calibrated


{'cap': 6636.47509765625,
 'tail_rate': 0.1,
 'best_k': 1.5,
 'best_gamma': 1.5,
 'oof_mae_raw': 474.5738830566406,
 'oof_mae_cal': 471.8822021484375,
 'chosen': 'bin_median_calibrated',
 'submission_path': 'D:\\AgentDs\\agent_ds_healthcare\\submission.csv'}

# Iteration 10
- 465.1965

## Diagnose

In [49]:
import re
import pandas as pd
from pathlib import Path
import fitz

BASE = Path(r"D:/AgentDs/agent_ds_healthcare")
TRAIN = pd.read_csv(BASE / "ed_cost_train.csv")
TEST  = pd.read_csv(BASE / "ed_cost_test.csv")
PDF_DIR = BASE / "receipts_pdf"

# A bit more tolerant than the original:
# - case-insensitive
# - allows "TOTAL:", "TOTAL $", extra spaces, etc.
total_re = re.compile(r"\bTOTAL\b\s*[:$]?\s*([\d,]+\.\d{2})", re.IGNORECASE)

def pdf_total(pid: int):
    p = PDF_DIR / f"receipt_{pid}.pdf"
    if not p.exists():
        return None
    try:
        with fitz.open(p) as doc:
            text = ""
            for page in doc:
                text += (page.get_text("text") or "") + "\n"

        m = total_re.search(text)
        if not m:
            return "PARSE_FAIL"
        return float(m.group(1).replace(",", ""))
    except Exception as e:
        return f"ERR:{type(e).__name__}"

def coverage_and_match(df: pd.DataFrame, name: str):
    pids = df["patient_id"].astype(int).tolist()
    exists = [(PDF_DIR / f"receipt_{pid}.pdf").exists() for pid in pids]
    print(f"[{name}] pdf coverage: {sum(exists)}/{len(exists)} = {sum(exists)/len(exists):.3f}")

    sample = pids[:200] if len(pids) > 200 else pids

    # Faster than df.loc[...] inside the loop
    df_i = df.set_index("patient_id")

    rows = []
    for pid in sample:
        tot = pdf_total(pid)
        prior = float(df_i.at[pid, "prior_ed_cost_5y_usd"])
        rows.append((pid, tot, prior))

    chk = pd.DataFrame(rows, columns=["patient_id", "pdf_total", "prior_cost"])
    bad = chk[chk["pdf_total"].isna() | chk["pdf_total"].astype(str).str.contains(r"ERR|PARSE_FAIL", regex=True)]
    ok  = chk.drop(bad.index).copy()

    if len(ok) > 0:
        ok["absdiff"] = (ok["pdf_total"] - ok["prior_cost"]).abs()
        print(f"[{name}] total absdiff median={ok['absdiff'].median():.6f}, max={ok['absdiff'].max():.6f}")
        print(f"[{name}] absdiff > 0.01 count = {(ok['absdiff'] > 0.01).sum()}/{len(ok)}")

    print(f"[{name}] parse_fail_or_error_in_sample = {len(bad)}/{len(chk)}")

coverage_and_match(TRAIN, "TRAIN")
coverage_and_match(TEST,  "TEST")


[TRAIN] pdf coverage: 2000/2000 = 1.000
[TRAIN] total absdiff median=0.000000, max=0.000000
[TRAIN] absdiff > 0.01 count = 0/183
[TRAIN] parse_fail_or_error_in_sample = 17/200
[TEST] pdf coverage: 2000/2000 = 1.000
[TEST] total absdiff median=0.000000, max=0.000000
[TEST] absdiff > 0.01 count = 0/173
[TEST] parse_fail_or_error_in_sample = 27/200


## train

In [54]:
# -*- coding: utf-8 -*-
r"""
ITERATION 10 — Claims-aware ED cost forecasting (Domain+Data-Driven)
===================================================================

What changed vs your previous attempts:
1) Treat receipts as claims: build "structure features" from line items (qty, line totals, concentration).
2) Keep sparse TF-IDF on CPT/HCPCS codes (no SVD, no embeddings).
3) Add OOF target-encoding risk scores per code (smoothed median) -> patient aggregates.
4) Minimal, high-signal domain markers (ED level codes 99281-99285, critical care 99291/99292, obs G0378, intubation 31500, central line 36556).
   (NOT CPT range bucketing.)
5) Strong QC: verify pdf coverage and TOTAL==prior_ed_cost_5y_usd.

INSTALL:
pip install -U pandas numpy scipy scikit-learn xgboost pymupdf joblib tqdm
"""

from __future__ import annotations

import re
import sys
import time
import math
import logging
import warnings
import inspect
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold

import xgboost as xgb
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# -----------------------
# PATHS
# -----------------------
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")
TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
PDF_DIR = BASE_DIR / "receipts_pdf"
SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

CACHE_DIR = BASE_DIR / "cache_iter10"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RECEIPT_CACHE = CACHE_DIR / "receipts_parsed.joblib"
CACHE_VERSION = 101  # bump when you change parsing logic


# -----------------------
# CONFIG
# -----------------------
# TF-IDF
TFIDF_MAX_CODE = 2500
TFIDF_MIN_DF_CODE = 2
TFIDF_NGRAM = (1, 3)

TFIDF_MAX_DESC = 800
TFIDF_MIN_DF_DESC = 2
DESC_NGRAM = (1, 2)

# OOF Target Encoding (code risk)
TE_ALPHA = 50.0   # shrinkage to global median

# CV + Model
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026]  # 10 fits
EARLY_STOPPING = 200

N_ESTIMATORS = 8000
LR = 0.02
MAX_DEPTH = 6
SUBSAMPLE = 0.85
COLSAMPLE = 0.75
REG_LAMBDA = 1.0
MIN_CHILD_WEIGHT = 1.0

USE_PSEUDOHUBER = False
# HUBER_SLOPE = 50.0  # smoother than pure L1, more stable for dollar-scale residuals

VERBOSE_EVERY = 100
CLIP_AT_ZERO = True


# -----------------------
# Logging
# -----------------------
def setup_logging():
    root = logging.getLogger()
    if not root.handlers:
        logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    else:
        root.setLevel(logging.INFO)


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def ensure_csr_float32_int32(X: csr_matrix) -> csr_matrix:
    X = X.tocsr()
    if X.data.dtype != np.float32:
        X.data = X.data.astype(np.float32, copy=False)
    if X.indices.dtype != np.int32:
        X.indices = X.indices.astype(np.int32, copy=False)
    if X.indptr.dtype != np.int32:
        X.indptr = X.indptr.astype(np.int32, copy=False)
    return X


# -----------------------
# Receipt parsing
# -----------------------
ZIP_INS_RE = re.compile(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)")
TOTAL_RE   = re.compile(r"TOTAL\s+([\d,]+\.\d{2})")

# matches: code + desc + qty + unit + line_total (synthetic PDFs are consistent)
LINE_RE = re.compile(
    r"^([A-Z]\d{4}|\d{5})\s+(.+?)\s+(\d+)\s+([\d,]+\.\d{2})\s+([\d,]+\.\d{2})\s*$"
)

CODE_TOKEN_RE = re.compile(r"\b(?:[A-Z]\d{4}|\d{5})\b")
LEVEL_RE = re.compile(r"\(level\s*([1-5])\)", re.IGNORECASE)

ED_LEVEL_CODES = {"99281":1,"99282":2,"99283":3,"99284":4,"99285":5}
SENTINEL_CODES = {
    "99291":"critical_care_99291",
    "99292":"critical_care_99292",
    "G0378":"obs_G0378",
    "31500":"intub_31500",
    "36556":"central_line_36556",
    "85025":"cbc_85025",
}

def _money(s: str) -> float:
    return float(s.replace(",", ""))

def parse_receipt(patient_id: int) -> Dict[str, Any]:
    pdf_path = PDF_DIR / f"receipt_{patient_id}.pdf"
    out = {
        "patient_id": int(patient_id),
        "pdf_ok": 0,
        "pdf_zip3": "missing",
        "pdf_insurance": "missing",
        "pdf_total": 0.0,
        "pdf_parse_notes": "missing_pdf",
        # numeric structure features
        "n_items": 0,
        "n_unique_codes": 0,
        "qty_sum": 0,
        "line_mean": 0.0,
        "line_max": 0.0,
        "line_std": 0.0,
        "cost_concentration": 0.0,  # max_line / total
        "hhi": 0.0,  # sum((line/total)^2)
        # domain markers
        "ed_em_count": 0,
        "ed_em_level_avg": 0.0,
        "ed_em_high_ratio": 0.0,
    }
    # sentinel counts + cost shares
    for k in SENTINEL_CODES.values():
        out[f"{k}_count"] = 0
        out[f"{k}_cost_share"] = 0.0

    # for downstream
    out["codes_doc_qty"] = ""
    out["codes_doc_cost"] = ""
    out["desc_doc"] = ""
    out["_code_qty_dict"] = {}   # code -> qty_sum
    out["_code_cost_dict"] = {}  # code -> line_total_sum

    if not pdf_path.exists():
        return out

    try:
        with fitz.open(pdf_path) as doc:
            text = ""
            for page in doc:
                text += (page.get_text("text") or "") + "\n"
    except Exception as e:
        out["pdf_parse_notes"] = f"open_error:{type(e).__name__}"
        return out

    text = text or ""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    m = ZIP_INS_RE.search(text)
    if m:
        out["pdf_zip3"] = str(m.group(1))
        out["pdf_insurance"] = str(m.group(2)).lower()

    m = TOTAL_RE.search(text)
    if m:
        try:
            out["pdf_total"] = _money(m.group(1))
        except Exception:
            out["pdf_total"] = 0.0

    code_qty = defaultdict(int)
    code_cost = defaultdict(float)
    items = []
    desc_tokens = []
    line_totals = []

    for ln in lines:
        mm = LINE_RE.match(ln)
        if not mm:
            continue
        code, desc, qty_s, unit_s, line_s = mm.groups()
        qty = int(qty_s)
        qty = max(1, min(qty, 50))
        try:
            line_total = _money(line_s)
        except Exception:
            line_total = 0.0

        code_qty[code] += qty
        code_cost[code] += line_total
        items.append((code, desc, qty, line_total))

        # description doc
        if desc:
            desc_tokens.append(desc)

        if line_total > 0:
            line_totals.append(line_total)

    # fallback if table parse failed: extract code tokens only
    if len(items) == 0:
        codes = CODE_TOKEN_RE.findall(text)
        for c in codes:
            code_qty[c] += 1
        out["pdf_parse_notes"] = "fallback_codes_only"
    else:
        out["pdf_parse_notes"] = "ok"

    # build docs
    tokens_qty = []
    tokens_cost = []
    for code, q in code_qty.items():
        rep_q = max(1, min(q, 10))
        tokens_qty.extend([code] * rep_q)

    # cost-based repetition (log bucket)
    for code, csum in code_cost.items():
        rep_c = int(np.clip(np.round(np.log1p(max(csum, 0.0))), 1, 12))
        tokens_cost.extend([code] * rep_c)

    out["codes_doc_qty"] = " ".join(tokens_qty)
    out["codes_doc_cost"] = " ".join(tokens_cost)
    out["desc_doc"] = " ".join(desc_tokens).lower()

    out["_code_qty_dict"] = dict(code_qty)
    out["_code_cost_dict"] = dict(code_cost)

    out["pdf_ok"] = 1 if (len(out["codes_doc_qty"]) > 0) else 0

    # structure features
    out["n_items"] = int(len(items))
    out["n_unique_codes"] = int(len(code_qty))
    out["qty_sum"] = int(sum(code_qty.values()))

    if len(line_totals) > 0:
        arr = np.array(line_totals, dtype=np.float32)
        out["line_mean"] = float(arr.mean())
        out["line_max"] = float(arr.max())
        out["line_std"] = float(arr.std())

        tot = float(out["pdf_total"])
        if tot > 0:
            out["cost_concentration"] = float(out["line_max"] / tot)
            share = arr / tot
            out["hhi"] = float(np.sum(share * share))

    # ED E/M level features
    em_counts = 0
    level_sum = 0
    high = 0
    for c, q in code_qty.items():
        if c in ED_LEVEL_CODES:
            em_counts += q
            lv = ED_LEVEL_CODES[c]
            level_sum += lv * q
            if lv >= 4:
                high += q
    out["ed_em_count"] = int(em_counts)
    out["ed_em_level_avg"] = float(level_sum / em_counts) if em_counts > 0 else 0.0
    out["ed_em_high_ratio"] = float(high / em_counts) if em_counts > 0 else 0.0

    # sentinel features (count + cost share)
    tot = float(out["pdf_total"])
    for code, name in SENTINEL_CODES.items():
        q = int(code_qty.get(code, 0))
        csum = float(code_cost.get(code, 0.0))
        out[f"{name}_count"] = q
        out[f"{name}_cost_share"] = float(csum / tot) if tot > 0 else 0.0

    return out


def load_receipts(patient_ids: List[int]) -> pd.DataFrame:
    cache = {"version": CACHE_VERSION, "data": {}}
    if RECEIPT_CACHE.exists():
        try:
            cache = joblib.load(RECEIPT_CACHE)
        except Exception:
            cache = {"version": CACHE_VERSION, "data": {}}
    if not isinstance(cache, dict) or cache.get("version") != CACHE_VERSION:
        cache = {"version": CACHE_VERSION, "data": {}}

    data = cache.get("data", {})
    missing = [pid for pid in patient_ids if str(pid) not in data]
    logging.info(f"Receipt cache hit={len(patient_ids)-len(missing):,} miss={len(missing):,} version={CACHE_VERSION}")

    for pid in tqdm(missing, desc="Parsing receipts", ncols=100):
        data[str(pid)] = parse_receipt(int(pid))

    cache["data"] = data
    if missing:
        try:
            joblib.dump(cache, RECEIPT_CACHE)
        except Exception as e:
            logging.warning(f"Receipt cache save failed: {e}")

    rows = []
    for pid in patient_ids:
        rows.append(data.get(str(pid), parse_receipt(int(pid))))
    return pd.DataFrame(rows)


# -----------------------
# OOF Target Encoding per code (smoothed median)
# -----------------------
def build_strat_labels(df_train: pd.DataFrame) -> np.ndarray:
    # chronic + prior_cost decile is usually a decent stratification
    if "prior_ed_cost_5y_usd" in df_train.columns:
        b = pd.qcut(df_train["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    else:
        b = pd.Series(["bin"] * len(df_train))
    return (df_train["primary_chronic"].astype(str) + "_" + b).values

def compute_code_te_mapping(code_sets: List[List[str]], y: np.ndarray, alpha: float) -> Tuple[Dict[str,float], float]:
    global_med = float(np.median(y))
    bucket = defaultdict(list)
    for i, codes in enumerate(code_sets):
        for c in codes:
            bucket[c].append(float(y[i]))
    te = {}
    for c, vals in bucket.items():
        n = len(vals)
        med = float(np.median(vals))
        te[c] = (n * med + alpha * global_med) / (n + alpha)
    return te, global_med

def apply_patient_te_features(
    te: Dict[str,float],
    global_med: float,
    code_qty_dicts: List[Dict[str,int]],
    code_cost_dicts: List[Dict[str,float]],
    totals: np.ndarray,
) -> pd.DataFrame:
    rows = []
    for d_qty, d_cost, tot in zip(code_qty_dicts, code_cost_dicts, totals):
        if not d_qty:
            rows.append({
                "te_sum_qty": 0.0, "te_mean_qty": 0.0, "te_max": global_med,
                "te_top3_mean": global_med, "te_sum_costshare": 0.0
            })
            continue

        vals = []
        sum_qty = 0.0
        sum_w = 0.0
        sum_costshare = 0.0

        for c, q in d_qty.items():
            v = float(te.get(c, global_med))
            vals.append(v)
            sum_qty += v * q
            sum_w += q

        # cost-share weighted sum (uses line totals per code)
        if tot and tot > 0:
            for c, cs in d_cost.items():
                v = float(te.get(c, global_med))
                sum_costshare += v * (cs / tot)

        vals_sorted = sorted(vals, reverse=True)
        top3 = vals_sorted[:3] if len(vals_sorted) >= 3 else vals_sorted
        rows.append({
            "te_sum_qty": float(sum_qty),
            "te_mean_qty": float(sum_qty / max(sum_w, 1.0)),
            "te_max": float(vals_sorted[0]) if vals_sorted else global_med,
            "te_top3_mean": float(np.mean(top3)) if top3 else global_med,
            "te_sum_costshare": float(sum_costshare),
        })
    return pd.DataFrame(rows)

def build_oof_te_features(df_train: pd.DataFrame, df_test: pd.DataFrame, alpha: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    y = df_train["ed_cost_next3y_usd"].values.astype(np.float32)

    # prepare code sets (unique codes per patient) from cached dicts
    train_sets = [list(d.keys()) for d in df_train["_code_qty_dict"].tolist()]
    test_sets  = [list(d.keys()) for d in df_test["_code_qty_dict"].tolist()]

    strat = build_strat_labels(df_train)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=2025)

    oof_feats = pd.DataFrame(index=df_train.index, columns=["te_sum_qty","te_mean_qty","te_max","te_top3_mean","te_sum_costshare"], dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(df_train)), strat)):
        te_map, gmed = compute_code_te_mapping([train_sets[i] for i in tr_idx], y[tr_idx], alpha=alpha)
        f_va = apply_patient_te_features(
            te_map, gmed,
            [df_train["_code_qty_dict"].iloc[i] for i in va_idx],
            [df_train["_code_cost_dict"].iloc[i] for i in va_idx],
            df_train["pdf_total"].iloc[va_idx].values.astype(float),
        )
        oof_feats.iloc[va_idx] = f_va.values
        logging.info(f"TE fold {fold} | built mapping codes={len(te_map):,}")

    # test feats from full train mapping
    te_map_full, gmed_full = compute_code_te_mapping(train_sets, y, alpha=alpha)
    test_feats = apply_patient_te_features(
        te_map_full, gmed_full,
        df_test["_code_qty_dict"].tolist(),
        df_test["_code_cost_dict"].tolist(),
        df_test["pdf_total"].values.astype(float),
    )
    return oof_feats.reset_index(drop=True), test_feats.reset_index(drop=True)


# -----------------------
# Feature matrix build
# -----------------------
def safe_tfidf_fit_transform(vec, texts: np.ndarray):
    try:
        X = vec.fit_transform(texts.astype(str))
        return ensure_csr_float32_int32(X)
    except ValueError:
        return csr_matrix((len(texts), 0), dtype=np.float32)

def safe_tfidf_transform(vec, texts: np.ndarray):
    try:
        X = vec.transform(texts.astype(str))
        return ensure_csr_float32_int32(X)
    except Exception:
        return csr_matrix((len(texts), 0), dtype=np.float32)

def build_feature_matrix(df_all: pd.DataFrame) -> Tuple[csr_matrix, ColumnTransformer, Dict[str, Any]]:
    # numeric derivations
    df = df_all.copy()

    if "prior_ed_cost_5y_usd" in df.columns and "prior_ed_visits_5y" in df.columns:
        df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
        df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
        df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))

    # mismatch checks if patients.csv fields exist
    if "insurance" in df.columns:
        df["ins_mismatch"] = ((df["pdf_insurance"].astype(str) != df["insurance"].astype(str)) & (df["pdf_ok"] == 1)).astype(int)
    else:
        df["ins_mismatch"] = 0

    if "zip3" in df.columns:
        df["zip_mismatch"] = ((df["pdf_zip3"].astype(str) != df["zip3"].astype(str)) & (df["pdf_ok"] == 1)).astype(int)
    else:
        df["zip_mismatch"] = 0

    df["total_absdiff_prior"] = np.abs(df["pdf_total"].astype(float) - df.get("prior_ed_cost_5y_usd", 0.0).astype(float))

    # define cat + num
    doc_cols = {"codes_doc_qty","codes_doc_cost","desc_doc"}
    exclude = {"patient_id","ed_cost_next3y_usd","is_train","_code_qty_dict","_code_cost_dict"} | doc_cols

    cat_cols = []
    for c in ["primary_chronic","sex","insurance","zip3","pdf_insurance","pdf_zip3","pdf_parse_notes"]:
        if c in df.columns:
            cat_cols.append(c)

    for c in df.columns:
        if c in exclude or c in cat_cols:
            continue
        if df[c].dtype == "object" or str(df[c].dtype).startswith("string"):
            cat_cols.append(c)

    num_cols = []
    for c in df.columns:
        if c in exclude or c in cat_cols:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            num_cols.append(c)

    # fill cats
    for c in cat_cols:
        df[c] = df[c].astype("string").fillna("missing")

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    X_tab = pre.fit_transform(df)
    X_tab = ensure_csr_float32_int32(X_tab.tocsr() if hasattr(X_tab, "tocsr") else csr_matrix(X_tab, dtype=np.float32))

    # TF-IDF code docs
    vec_code = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b(?:[A-Z]\d{4}|\d{5})\b",
        lowercase=False,
        ngram_range=TFIDF_NGRAM,
        max_features=TFIDF_MAX_CODE,
        min_df=TFIDF_MIN_DF_CODE,
        max_df=1.0,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    # fit on both qty and cost docs separately but same vectorizer? we use two vectorizers for diversity
    vec_cost = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b(?:[A-Z]\d{4}|\d{5})\b",
        lowercase=False,
        ngram_range=(1,2),
        max_features=int(TFIDF_MAX_CODE * 0.7),
        min_df=TFIDF_MIN_DF_CODE,
        max_df=1.0,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    X_code = safe_tfidf_fit_transform(vec_code, df["codes_doc_qty"].values)
    X_cost = safe_tfidf_fit_transform(vec_cost, df["codes_doc_cost"].values)

    # TF-IDF on descriptions (optional small)
    vec_desc = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
        lowercase=True,
        ngram_range=DESC_NGRAM,
        max_features=TFIDF_MAX_DESC,
        min_df=TFIDF_MIN_DF_DESC,
        max_df=1.0,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    X_desc = safe_tfidf_fit_transform(vec_desc, df["desc_doc"].values)

    X = ensure_csr_float32_int32(hstack([X_tab, X_code, X_cost, X_desc], format="csr"))
    meta = {"num_cols": num_cols, "cat_cols": cat_cols}
    vecs = {"vec_code": vec_code, "vec_cost": vec_cost, "vec_desc": vec_desc}
    return X, pre, vecs, meta


# -----------------------
# Calibration (median residual, shrink)
# -----------------------
def shrunk_group_median_bias(resid: np.ndarray, group: np.ndarray, alpha: float) -> Tuple[Dict[str,float], float]:
    g = group.astype(str)
    global_med = float(np.median(resid))
    out = {}
    for key in np.unique(g):
        m = (g == key)
        r = resid[m]
        if r.size == 0:
            continue
        med = float(np.median(r))
        n = float(r.size)
        out[str(key)] = (n * med + alpha * global_med) / (n + alpha)
    return out, global_med

def apply_bias(pred: np.ndarray, group: np.ndarray, bias: Dict[str,float], default_bias: float) -> np.ndarray:
    g = group.astype(str)
    p = pred.astype(np.float64).copy()
    for i in range(len(p)):
        p[i] += bias.get(str(g[i]), default_bias)
    return p.astype(np.float32)


# -----------------------
# Runner
# -----------------------
def run_iteration10(diagnostic: bool = False, force_cpu: bool = False) -> Dict[str, Any]:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb,'__version__','unknown')}")
    logging.info("=== Iteration 10: claims-aware + OOF code risk + sparse TFIDF + XGB GPU ===")
    logging.info(f"diagnostic={diagnostic} | force_cpu={force_cpu}")

    train = pd.read_csv(TRAIN_CSV)
    test = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train.shape} test={test.shape}")

    if PATIENTS_CSV.exists():
        pat = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in pat.columns:
            train = train.merge(pat, on="patient_id", how="left")
            test  = test.merge(pat, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train.shape} test={test.shape}")

    # PDF coverage QC
    train_ids = train["patient_id"].astype(int).tolist()
    test_ids  = test["patient_id"].astype(int).tolist()

    def pdf_exists(pid): return (PDF_DIR / f"receipt_{pid}.pdf").exists()
    cov_tr = np.mean([pdf_exists(pid) for pid in train_ids])
    cov_te = np.mean([pdf_exists(pid) for pid in test_ids])
    logging.info(f"PDF coverage | train={cov_tr:.3f} test={cov_te:.3f}")
    if cov_te < 0.98:
        logging.warning("TEST PDF coverage is not ~100%. Receipt-based models may underperform. Fix your receipts_pdf folder!")

    # parse receipts (train+test)
    all_ids = list(map(int, pd.concat([train["patient_id"], test["patient_id"]]).values))
    rec = load_receipts(all_ids)

    train = train.merge(rec, on="patient_id", how="left")
    test  = test.merge(rec, on="patient_id", how="left")

    # basic TOTAL sanity (sample)
    sample = train.sample(min(200, len(train)), random_state=7)
    absdiff = np.abs(sample["pdf_total"].astype(float) - sample["prior_ed_cost_5y_usd"].astype(float))
    logging.info(f"TOTAL sanity (sample n={len(sample)}): absdiff median={absdiff.median():.6f}, max={absdiff.max():.6f}")

    # add OOF target encoding features
    train_te, test_te = build_oof_te_features(train, test, alpha=TE_ALPHA)
    for c in train_te.columns:
        train[c] = train_te[c].values
        test[c] = test_te[c].values

    # build docs fill
    for df in [train, test]:
        for c in ["codes_doc_qty","codes_doc_cost","desc_doc"]:
            df[c] = df[c].fillna("").astype(str)

    if diagnostic:
        return {
            "pdf_cov_train": float(cov_tr),
            "pdf_cov_test": float(cov_te),
            "total_absdiff_sample_median": float(absdiff.median()),
            "total_absdiff_sample_max": float(absdiff.max()),
            "n_train": int(len(train)),
            "n_test": int(len(test)),
            "nonempty_codes_doc_qty_train": int((train["codes_doc_qty"].str.len() > 0).sum()),
            "nonempty_codes_doc_qty_test": int((test["codes_doc_qty"].str.len() > 0).sum()),
        }

    # build matrix on all (transductive, unsupervised)
    train["is_train"] = 1
    test["is_train"] = 0
    df_all = pd.concat([train, test], ignore_index=True)

    logging.info("Building sparse feature matrix (tabular OHE + TFIDF codes + TFIDF desc)...")
    X_all, pre, vecs, meta = build_feature_matrix(df_all)
    logging.info(f"X_all shape={X_all.shape} nnz={X_all.nnz:,}")

    is_train = (df_all["is_train"].values.astype(int) == 1)
    X_tr_all = X_all[is_train]
    X_te_all = X_all[~is_train]

    y = train["ed_cost_next3y_usd"].values.astype(np.float32)

    # stratification: chronic + cost decile
    if "prior_ed_cost_5y_usd" in train.columns:
        b = pd.qcut(train["prior_ed_cost_5y_usd"], q=10, duplicates="drop").astype(str)
    else:
        b = pd.Series(["bin"] * len(train))
    strat = (train["primary_chronic"].astype(str) + "_" + b).values

    device = "cpu" if force_cpu else "cuda"

    params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=device,
        n_estimators=N_ESTIMATORS,
        learning_rate=LR,
        max_depth=MAX_DEPTH,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        min_child_weight=MIN_CHILD_WEIGHT,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )
    if USE_PSEUDOHUBER:
        params["objective"] = "reg:pseudohubererror"
        params["huber_slope"] = HUBER_SLOPE

    oof_sum = np.zeros(len(train), dtype=np.float64)
    oof_cnt = np.zeros(len(train), dtype=np.float64)
    te_sum  = np.zeros(len(test), dtype=np.float64)
    n_models = 0

    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train)), strat)):
            splits.append((rep, fold, seed, tr_idx, va_idx))

    logging.info(f"Total fits: {len(splits)} | device={device} | objective={params['objective']}")

    # fit loop
    for rep, fold, seed, tr_idx, va_idx in tqdm(splits, desc="CV", ncols=110):
        tag = f"rep={rep} fold={fold}"
        t0 = time.time()

        X_tr = X_tr_all[tr_idx]
        y_tr = y[tr_idx]
        X_va = X_tr_all[va_idx]
        y_va = y[va_idx]

        model = XGBRegressor(**params, random_state=10000 * rep + seed + fold)

        fit_sig = inspect.signature(model.fit).parameters
        fit_kwargs = {"eval_set": [(X_va, y_va)]}
        if "verbose" in fit_sig:
            fit_kwargs["verbose"] = VERBOSE_EVERY
        elif "verbose_eval" in fit_sig:
            fit_kwargs["verbose_eval"] = VERBOSE_EVERY

        logging.info(f"{tag} | fit START")
        try:
            model.fit(X_tr, y_tr, **fit_kwargs)
        except Exception as e:
            # fallback if pseudohuber not supported
            if params.get("objective") == "reg:pseudohubererror":
                logging.warning(f"{tag} | pseudohuber failed ({type(e).__name__}). Falling back to reg:absoluteerror.")
                params2 = dict(params)
                params2["objective"] = "reg:absoluteerror"
                params2.pop("huber_slope", None)
                model = XGBRegressor(**params2, random_state=10000 * rep + seed + fold)
                model.fit(X_tr, y_tr, **fit_kwargs)
            else:
                raise

        logging.info(f"{tag} | fit END | best_iteration={getattr(model,'best_iteration',None)}")

        p_va = model.predict(X_va).astype(np.float32)
        p_te = model.predict(X_te_all).astype(np.float32)

        if CLIP_AT_ZERO:
            p_va = np.maximum(p_va, 0.0)
            p_te = np.maximum(p_te, 0.0)

        oof_sum[va_idx] += p_va
        oof_cnt[va_idx] += 1.0
        te_sum += p_te
        n_models += 1

        fold_mae = mae(y_va, p_va)
        logging.info(f"{tag} | fold_MAE={fold_mae:.3f} | fold_time={time.time()-t0:.1f}s")

    oof = (oof_sum / np.maximum(oof_cnt, 1.0)).astype(np.float32)
    pred_test = (te_sum / max(1, n_models)).astype(np.float32)

    raw_mae = mae(y, oof)
    logging.info(f"OOF MAE (raw): {raw_mae:.4f}")

    # Calibration candidates (median residual shrink)
    resid = (y.astype(np.float64) - oof.astype(np.float64))

    # (1) none
    best = (raw_mae, "none", pred_test)

    # (2) chronic
    bias_c, def_c = shrunk_group_median_bias(resid, train["primary_chronic"].astype(str).values, alpha=TE_ALPHA)
    oof_c = apply_bias(oof, train["primary_chronic"].astype(str).values, bias_c, def_c)
    if CLIP_AT_ZERO: oof_c = np.maximum(oof_c, 0.0)
    mae_c = mae(y, oof_c)
    logging.info(f"OOF MAE (chronic calib): {mae_c:.4f}")
    if mae_c < best[0]:
        pred_c = apply_bias(pred_test, test["primary_chronic"].astype(str).values, bias_c, def_c)
        if CLIP_AT_ZERO: pred_c = np.maximum(pred_c, 0.0)
        best = (mae_c, "chronic", pred_c)

    # (3) chronic+insurance (if available)
    if "insurance" in train.columns and "insurance" in test.columns:
        key_tr = (train["primary_chronic"].astype(str) + "_" + train["insurance"].astype(str)).values
        key_te = (test["primary_chronic"].astype(str) + "_" + test["insurance"].astype(str)).values
        bias_ci, def_ci = shrunk_group_median_bias(resid, key_tr, alpha=TE_ALPHA)
        oof_ci = apply_bias(oof, key_tr, bias_ci, def_ci)
        if CLIP_AT_ZERO: oof_ci = np.maximum(oof_ci, 0.0)
        mae_ci = mae(y, oof_ci)
        logging.info(f"OOF MAE (chronic+ins calib): {mae_ci:.4f}")
        if mae_ci < best[0]:
            pred_ci = apply_bias(pred_test, key_te, bias_ci, def_ci)
            if CLIP_AT_ZERO: pred_ci = np.maximum(pred_ci, 0.0)
            best = (mae_ci, "chronic+insurance", pred_ci)

    chosen_mae, chosen_name, final_test = best
    logging.info(f"Chosen calibration: {chosen_name} | OOF MAE={chosen_mae:.4f}")

    sub = pd.DataFrame({
        "patient_id": test["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": final_test.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")

    return {
        "oof_mae_raw": float(raw_mae),
        "oof_mae_best": float(chosen_mae),
        "calibration": chosen_name,
        "n_models": int(n_models),
        "submission_path": str(SUBMISSION_PATH),
    }


In [55]:
res = run_iteration10(diagnostic=True)
res

2026-02-14 04:31:53,315 | INFO | XGBoost version: 3.0.0
2026-02-14 04:31:53,315 | INFO | === Iteration 10: claims-aware + OOF code risk + sparse TFIDF + XGB GPU ===
2026-02-14 04:31:53,316 | INFO | diagnostic=True | force_cpu=False
2026-02-14 04:31:53,323 | INFO | Loaded train=(2000, 5) test=(2000, 4)
2026-02-14 04:31:53,329 | INFO | Merged patients.csv: train=(2000, 9) test=(2000, 8)
2026-02-14 04:31:53,383 | INFO | PDF coverage | train=1.000 test=1.000
2026-02-14 04:31:53,535 | INFO | Receipt cache hit=4,000 miss=0 version=101
Parsing receipts: 0it [00:00, ?it/s]
2026-02-14 04:32:01,658 | INFO | TOTAL sanity (sample n=200): absdiff median=0.000000, max=25937.300000
2026-02-14 04:32:01,677 | INFO | TE fold 0 | built mapping codes=18
2026-02-14 04:32:01,688 | INFO | TE fold 1 | built mapping codes=18
2026-02-14 04:32:01,700 | INFO | TE fold 2 | built mapping codes=18
2026-02-14 04:32:01,711 | INFO | TE fold 3 | built mapping codes=18
2026-02-14 04:32:01,721 | INFO | TE fold 4 | built m

{'pdf_cov_train': 1.0,
 'pdf_cov_test': 1.0,
 'total_absdiff_sample_median': 0.0,
 'total_absdiff_sample_max': 25937.3,
 'n_train': 2000,
 'n_test': 2000,
 'nonempty_codes_doc_qty_train': 2000,
 'nonempty_codes_doc_qty_test': 2000}

In [56]:
res = run_iteration10(diagnostic=False)
res


2026-02-14 04:32:05,067 | INFO | XGBoost version: 3.0.0
2026-02-14 04:32:05,067 | INFO | === Iteration 10: claims-aware + OOF code risk + sparse TFIDF + XGB GPU ===
2026-02-14 04:32:05,068 | INFO | diagnostic=False | force_cpu=False
2026-02-14 04:32:05,072 | INFO | Loaded train=(2000, 5) test=(2000, 4)
2026-02-14 04:32:05,082 | INFO | Merged patients.csv: train=(2000, 9) test=(2000, 8)
2026-02-14 04:32:05,159 | INFO | PDF coverage | train=1.000 test=1.000
2026-02-14 04:32:05,359 | INFO | Receipt cache hit=4,000 miss=0 version=101
Parsing receipts: 0it [00:00, ?it/s]
2026-02-14 04:32:13,398 | INFO | TOTAL sanity (sample n=200): absdiff median=0.000000, max=25937.300000
2026-02-14 04:32:13,420 | INFO | TE fold 0 | built mapping codes=18
2026-02-14 04:32:13,433 | INFO | TE fold 1 | built mapping codes=18
2026-02-14 04:32:13,444 | INFO | TE fold 2 | built mapping codes=18
2026-02-14 04:32:13,455 | INFO | TE fold 3 | built mapping codes=18
2026-02-14 04:32:13,466 | INFO | TE fold 4 | built 

[0]	validation_0-mae:1365.67722
[100]	validation_0-mae:517.80547
[200]	validation_0-mae:447.91680
[300]	validation_0-mae:442.34992
[400]	validation_0-mae:440.74771
[500]	validation_0-mae:439.92560
[600]	validation_0-mae:440.01832
[700]	validation_0-mae:440.33304
[749]	validation_0-mae:440.32308


2026-02-14 04:32:18,126 | INFO | rep=0 fold=0 | fit END | best_iteration=550
2026-02-14 04:32:18,167 | INFO | rep=0 fold=0 | fold_MAE=450.937 | fold_time=4.6s
CV:  10%|███████                                                               | 1/10 [00:04<00:41,  4.56s/it]2026-02-14 04:32:18,171 | INFO | rep=0 fold=1 | fit START


[0]	validation_0-mae:1407.89542
[100]	validation_0-mae:558.17635
[200]	validation_0-mae:485.34003
[300]	validation_0-mae:478.89452
[400]	validation_0-mae:477.77574
[500]	validation_0-mae:476.64717
[600]	validation_0-mae:477.07697
[700]	validation_0-mae:477.65437
[737]	validation_0-mae:477.61041


2026-02-14 04:32:22,356 | INFO | rep=0 fold=1 | fit END | best_iteration=537
2026-02-14 04:32:22,398 | INFO | rep=0 fold=1 | fold_MAE=482.231 | fold_time=4.2s
CV:  20%|██████████████                                                        | 2/10 [00:08<00:34,  4.36s/it]2026-02-14 04:32:22,404 | INFO | rep=0 fold=2 | fit START


...

[0]	validation_0-mae:1413.92929
[100]	validation_0-mae:561.54118
[200]	validation_0-mae:480.21344
[300]	validation_0-mae:472.63531
[400]	validation_0-mae:471.49042
[500]	validation_0-mae:471.61347
[578]	validation_0-mae:471.80278


2026-02-14 04:33:22,780 | INFO | rep=1 fold=3 | fit END | best_iteration=379
2026-02-14 04:33:22,824 | INFO | rep=1 fold=3 | fold_MAE=472.652 | fold_time=3.1s
CV:  90%|███████████████████████████████████████████████████████████████       | 9/10 [01:09<00:06,  6.82s/it]2026-02-14 04:33:22,828 | INFO | rep=1 fold=4 | fit START


[0]	validation_0-mae:1384.73327
[100]	validation_0-mae:515.83084
[200]	validation_0-mae:440.69597
[300]	validation_0-mae:433.11840
[400]	validation_0-mae:432.13129
[500]	validation_0-mae:431.88253
[600]	validation_0-mae:432.11278
[700]	validation_0-mae:432.13343
[740]	validation_0-mae:432.16613


2026-02-14 04:33:26,598 | INFO | rep=1 fold=4 | fit END | best_iteration=540
2026-02-14 04:33:26,649 | INFO | rep=1 fold=4 | fold_MAE=439.919 | fold_time=3.8s
CV: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [01:13<00:00,  7.30s/it]
2026-02-14 04:33:26,652 | INFO | OOF MAE (raw): 448.1043
2026-02-14 04:33:26,656 | INFO | OOF MAE (chronic calib): 441.9771
2026-02-14 04:33:26,665 | INFO | OOF MAE (chronic+ins calib): 440.1263
2026-02-14 04:33:26,668 | INFO | Chosen calibration: chronic+insurance | OOF MAE=440.1263
2026-02-14 04:33:26,676 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv


{'oof_mae_raw': 448.10430908203125,
 'oof_mae_best': 440.1262512207031,
 'calibration': 'chronic+insurance',
 'n_models': 10,
 'submission_path': 'D:\\AgentDs\\agent_ds_healthcare\\submission_ICHI_V1.csv'}

# Iteration 11
- 469.7176

In [61]:
# -*- coding: utf-8 -*-
r"""
ITERATION 11 — Leakage-safe code risk + robust PDF parsing + monolithic XGBoost (GPU)
===================================================================================

Why this should beat / stabilize vs your current best:
1) Fold-safe target encoding: any target-derived features must be built INSIDE each CV fold,
   otherwise CV becomes overly optimistic (classic target encoding leakage issue).
2) Robust PDF parsing: PyMuPDF text for these synthetic receipts is column-wise (code/desc/qty/unit/line on separate lines),
   so naive single-line regex loses qty/line totals. We reconstruct rows from extracted text.
3) Add claim-structure features + per-code pivot counts/shares (NO manual CPT range groupings, NO SVD/embeddings).
4) Keep single XGBoost reg:absoluteerror on GPU + chronic(+insurance) median-residual calibration.

Run in notebook:
    res = run_iteration11(diagnostic=True)   # quick checks
    res = run_iteration11(diagnostic=False)  # train + write submission

INSTALL:
pip install -U pandas numpy scipy scikit-learn xgboost pymupdf joblib tqdm
"""

from __future__ import annotations

import time
import logging
import warnings
import inspect
import re
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib
import fitz  # PyMuPDF

from scipy.sparse import csr_matrix, hstack

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold

import xgboost as xgb
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# =========================
# PATHS (edit if needed)
# =========================
BASE_DIR = Path(r"D:\AgentDs\agent_ds_healthcare")

TRAIN_CSV = BASE_DIR / "ed_cost_train.csv"
TEST_CSV  = BASE_DIR / "ed_cost_test.csv"
PATIENTS_CSV = BASE_DIR / "patients.csv"
PDF_DIR = BASE_DIR / "receipts_pdf"

SUBMISSION_PATH = BASE_DIR / "submission_ICHI_V1.csv"

CACHE_DIR = BASE_DIR / "cache_iter11"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RECEIPT_CACHE = CACHE_DIR / "receipts_parsed_iter11.joblib"
CACHE_VERSION = 111


# =========================
# CONFIG
# =========================
# TF-IDF
TFIDF_MAX_QTY = 1600
TFIDF_MAX_COST = 900
TFIDF_NGRAM_QTY = (1, 3)
TFIDF_NGRAM_COST = (1, 2)
TFIDF_MIN_DF = 2
TFIDF_MAX_DF = 1.0

# Per-code pivots
TOP_CODE_PIVOTS = 40  # safe upper bound; actual unique codes ~ small in this synthetic dataset

# Target encoding
TE_ALPHA = 80.0              # smoothing strength (higher => less overfit)
TE_USE_GROUP = True          # group-conditioned TE by chronic+insurance (still data-driven)

# CV
N_FOLDS = 5
REPEAT_SEEDS = [2025, 2026]  # 10 models total
EARLY_STOPPING = 200

# XGB
N_ESTIMATORS = 8000
LR = 0.02
MAX_DEPTH = 6
SUBSAMPLE = 0.85
COLSAMPLE = 0.75
REG_LAMBDA = 1.0
MIN_CHILD_WEIGHT = 1.0
GAMMA = 0.0

# Runtime / safety
DEVICE = "cuda"              # set to "cpu" if needed
CLIP_AT_ZERO = True
VERBOSE_EVERY = 100          # fit heartbeat


# =========================
# LOGGING
# =========================
def setup_logging() -> None:
    root = logging.getLogger()
    if not root.handlers:
        logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    else:
        root.setLevel(logging.INFO)


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def make_ohe() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def ensure_csr_float32_int32(X: csr_matrix) -> csr_matrix:
    X = X.tocsr()
    if X.data.dtype != np.float32:
        X.data = X.data.astype(np.float32, copy=False)
    if X.indices.dtype != np.int32:
        X.indices = X.indices.astype(np.int32, copy=False)
    if X.indptr.dtype != np.int32:
        X.indptr = X.indptr.astype(np.int32, copy=False)
    return X


# =========================
# PDF PARSING (robust for synthetic layout)
# =========================
ZIP_INS_RE = re.compile(r"ZIP3:\s*([0-9]{3})\s+Insurance:\s*([A-Za-z_]+)", re.IGNORECASE)
TOTAL_RE   = re.compile(r"TOTAL\s*([\d,]+\.\d{2})", re.IGNORECASE)

CODE_FULL_RE = re.compile(r"^(?:[A-Z]\d{4}|\d{5})$")
CODE_TOKEN_RE = re.compile(r"\b(?:[A-Z]\d{4}|\d{5})\b")
MONEY_RE = re.compile(r"^[\d,]+\.\d{2}$")
LEVEL_RE = re.compile(r"\(level\s*([1-5])\)", re.IGNORECASE)


def _money_to_float(s: str) -> float:
    return float(s.replace(",", ""))


def parse_receipt_table(lines: List[str]) -> List[Tuple[str, int, float, float, str]]:
    """
    Extract table rows from column-wise PDF text:
      code, description (may span lines), qty, unit, line_total
    Returns list of tuples: (code, qty, unit, line_total, desc)
    """
    items: List[Tuple[str, int, float, float, str]] = []
    i, n = 0, len(lines)

    while i < n:
        s = lines[i].strip()

        if CODE_FULL_RE.match(s):
            code = s
            j = i + 1
            desc_parts: List[str] = []

            # gather description lines until we hit qty or TOTAL
            while j < n:
                sj = lines[j].strip()
                if sj.isdigit() or sj.upper().startswith("TOTAL"):
                    break
                # skip headers
                if sj in {"Code", "Description", "Qty", "Unit ($)", "Line Total ($)"}:
                    j += 1
                    continue
                if sj:
                    desc_parts.append(sj)
                j += 1

            # expect qty/unit/line_total next
            if j + 2 < n:
                qty_s = lines[j].strip()
                unit_s = lines[j + 1].strip()
                line_s = lines[j + 2].strip()

                if qty_s.isdigit() and MONEY_RE.match(unit_s) and MONEY_RE.match(line_s):
                    qty = int(qty_s)
                    qty = max(1, min(qty, 50))
                    unit = _money_to_float(unit_s)
                    line = _money_to_float(line_s)
                    desc = " ".join(desc_parts).strip()
                    items.append((code, qty, unit, line, desc))
                    i = j + 3
                    continue

        i += 1

    return items


def parse_receipt_pdf(patient_id: int) -> Dict[str, Any]:
    pdf_path = PDF_DIR / f"receipt_{patient_id}.pdf"

    out: Dict[str, Any] = {
        "patient_id": int(patient_id),
        "pdf_ok": 0,
        "pdf_zip3": "missing",
        "pdf_insurance": "missing",
        "pdf_total_parsed": np.nan,
        "parse_ok": 0,
        "n_items": 0,
        "n_unique_codes": 0,
        "qty_sum": 0,
        "line_sum": 0.0,
        "line_mean": 0.0,
        "line_max": 0.0,
        "line_std": 0.0,
        "cost_concentration": 0.0,
        "hhi": 0.0,
        "ed_level_avg": 0.0,
        "ed_level_high_ratio": 0.0,
        "codes_doc_qty": "",
        "codes_doc_cost": "",
        "desc_doc": "",
        "code_qty_dict": {},
        "code_cost_dict": {},
    }

    if not pdf_path.exists():
        return out

    try:
        with fitz.open(pdf_path) as doc:
            text = ""
            for page in doc:
                text += (page.get_text("text") or "") + "\n"
    except Exception:
        return out

    text = text or ""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    m = ZIP_INS_RE.search(text)
    if m:
        out["pdf_zip3"] = str(m.group(1))
        out["pdf_insurance"] = str(m.group(2)).lower()

    m = TOTAL_RE.search(text)
    if m:
        try:
            out["pdf_total_parsed"] = _money_to_float(m.group(1))
        except Exception:
            out["pdf_total_parsed"] = np.nan

    items = parse_receipt_table(lines)

    # If parsing failed, fallback to just code tokens (still helpful for TF-IDF)
    code_qty: Dict[str, int] = defaultdict(int)
    code_cost: Dict[str, float] = defaultdict(float)

    if items:
        out["parse_ok"] = 1
        out["n_items"] = int(len(items))

        qty_tokens: List[str] = []
        cost_tokens: List[str] = []
        desc_tokens: List[str] = []
        line_totals: List[float] = []
        lvl_list: List[int] = []

        for code, qty, unit, line, desc in items:
            code_qty[code] += qty
            code_cost[code] += line

            rep_q = max(1, min(qty, 10))
            qty_tokens.extend([code] * rep_q)

            rep_c = int(np.clip(round(np.log1p(max(line, 0.0))), 1, 12))
            cost_tokens.extend([code] * rep_c)

            if desc:
                desc_tokens.append(desc.lower())
                mm = LEVEL_RE.search(desc)
                if mm:
                    lv = int(mm.group(1))
                    lvl_list.extend([lv] * qty)

            if line > 0:
                line_totals.append(line)

        out["codes_doc_qty"] = " ".join(qty_tokens)
        out["codes_doc_cost"] = " ".join(cost_tokens) if cost_tokens else out["codes_doc_qty"]
        out["desc_doc"] = " ".join(desc_tokens)

        out["qty_sum"] = int(sum(code_qty.values()))
        out["n_unique_codes"] = int(len(code_qty))

        if line_totals:
            arr = np.array(line_totals, dtype=np.float32)
            out["line_sum"] = float(arr.sum())
            out["line_mean"] = float(arr.mean())
            out["line_max"] = float(arr.max())
            out["line_std"] = float(arr.std())

            tot = float(out["pdf_total_parsed"]) if (np.isfinite(out["pdf_total_parsed"]) and out["pdf_total_parsed"] > 0) else float(arr.sum())
            if tot > 0:
                out["cost_concentration"] = float(out["line_max"] / tot)
                share = arr / tot
                out["hhi"] = float(np.sum(share * share))

        if lvl_list:
            out["ed_level_avg"] = float(np.mean(lvl_list))
            out["ed_level_high_ratio"] = float(np.mean([1.0 if lv >= 4 else 0.0 for lv in lvl_list]))
    else:
        codes = CODE_TOKEN_RE.findall(text)
        for c in codes:
            code_qty[c] += 1
        out["codes_doc_qty"] = " ".join(codes)
        out["codes_doc_cost"] = out["codes_doc_qty"]
        out["desc_doc"] = ""
        out["qty_sum"] = int(sum(code_qty.values()))
        out["n_unique_codes"] = int(len(code_qty))

    out["code_qty_dict"] = dict(code_qty)
    out["code_cost_dict"] = dict(code_cost)
    out["pdf_ok"] = 1 if len(out["codes_doc_qty"]) > 0 else 0
    return out


def load_receipts(patient_ids: List[int]) -> pd.DataFrame:
    cache = {"version": CACHE_VERSION, "data": {}}
    if RECEIPT_CACHE.exists():
        try:
            cache = joblib.load(RECEIPT_CACHE)
        except Exception:
            cache = {"version": CACHE_VERSION, "data": {}}
    if not isinstance(cache, dict) or cache.get("version") != CACHE_VERSION:
        cache = {"version": CACHE_VERSION, "data": {}}

    data = cache.get("data", {})
    missing = [pid for pid in patient_ids if str(pid) not in data]
    logging.info(f"Receipt cache | hit={len(patient_ids)-len(missing):,} miss={len(missing):,} version={CACHE_VERSION}")

    for pid in tqdm(missing, desc="Parsing receipts", ncols=100):
        data[str(pid)] = parse_receipt_pdf(int(pid))

    cache["data"] = data
    if missing:
        try:
            joblib.dump(cache, RECEIPT_CACHE, compress=3)
        except Exception as e:
            logging.warning(f"Receipt cache save failed: {e}")

    rows = [data.get(str(pid), parse_receipt_pdf(int(pid))) for pid in patient_ids]
    return pd.DataFrame(rows)


# =========================
# CODE PIVOT FEATURES
# =========================
def _sanitize_code(code: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(code))


def add_code_pivots(df: pd.DataFrame, codes: List[str], total_col: str) -> pd.DataFrame:
    dict_qty = df["code_qty_dict"].tolist()
    dict_cost = df["code_cost_dict"].tolist()
    total = df[total_col].astype(float).values
    n = len(df)

    for code in codes:
        col = _sanitize_code(code)
        qty_arr = np.fromiter((d.get(code, 0) if isinstance(d, dict) else 0 for d in dict_qty), dtype=np.float32, count=n)
        cost_arr = np.fromiter((d.get(code, 0.0) if isinstance(d, dict) else 0.0 for d in dict_cost), dtype=np.float32, count=n)
        df[f"qty_{col}"] = qty_arr
        df[f"cost_{col}"] = cost_arr
        df[f"share_{col}"] = cost_arr / (total + 1e-6)
    return df


# =========================
# TARGET ENCODING (fold-safe)
# =========================
def build_code_te_maps(
    df: pd.DataFrame,
    idx_tr: np.ndarray,
    y: np.ndarray,
    group_key: Optional[np.ndarray],
    alpha: float,
) -> Tuple[Dict[str, float], float, Optional[Dict[Tuple[str, str], float]], Optional[Dict[str, float]]]:
    y_tr = y[idx_tr]
    global_med = float(np.median(y_tr))

    bucket = defaultdict(list)
    for i in idx_tr:
        d = df.at[i, "code_qty_dict"]
        if not isinstance(d, dict):
            continue
        for c in d.keys():
            bucket[c].append(float(y[i]))

    code_map = {}
    for c, vals in bucket.items():
        n = len(vals)
        med = float(np.median(vals))
        code_map[c] = (n * med + alpha * global_med) / (n + alpha)

    grp_code_map = None
    grp_med = None

    if group_key is not None:
        grp_code_map = {}
        grp_med = {}
        g_tr = group_key[idx_tr].astype(str)

        for g in np.unique(g_tr):
            m = (g_tr == g)
            grp_med[g] = float(np.median(y_tr[m])) if np.any(m) else global_med

        bucket_gc = defaultdict(list)
        for i in idx_tr:
            g = str(group_key[i])
            d = df.at[i, "code_qty_dict"]
            if not isinstance(d, dict):
                continue
            for c in d.keys():
                bucket_gc[(g, c)].append(float(y[i]))

        for (g, c), vals in bucket_gc.items():
            n = len(vals)
            med = float(np.median(vals))
            base = grp_med.get(g, global_med)
            grp_code_map[(g, c)] = (n * med + alpha * base) / (n + alpha)

    return code_map, global_med, grp_code_map, grp_med


def patient_te_features(
    df: pd.DataFrame,
    indices: np.ndarray,
    code_map: Dict[str, float],
    global_med: float,
    grp_code_map: Optional[Dict[Tuple[str, str], float]],
    grp_med: Optional[Dict[str, float]],
    group_key: Optional[np.ndarray],
    total_col: str,
) -> np.ndarray:
    totals = df[total_col].astype(float).values
    use_group = (TE_USE_GROUP and grp_code_map is not None and group_key is not None and grp_med is not None)
    out = np.zeros((len(indices), 10 if use_group else 5), dtype=np.float32)

    for r, i in enumerate(indices):
        d_qty = df.at[i, "code_qty_dict"]
        d_cost = df.at[i, "code_cost_dict"]
        if not isinstance(d_qty, dict):
            d_qty = {}
        if not isinstance(d_cost, dict):
            d_cost = {}

        tot = float(totals[i])
        codes = list(d_qty.keys())
        if not codes:
            base = global_med
            out[r, :5] = base
            if out.shape[1] > 5:
                out[r, 5:] = base
            continue

        vals = []
        w_qty = []
        w_cost = []
        for c in codes:
            v = float(code_map.get(c, global_med))
            vals.append(v)
            w_qty.append(float(d_qty.get(c, 0)))
            w_cost.append(float(d_cost.get(c, 0.0)) / tot if tot > 0 else 0.0)

        vals_arr = np.array(vals, dtype=np.float32)
        w_qty_arr = np.array(w_qty, dtype=np.float32)
        w_cost_arr = np.array(w_cost, dtype=np.float32)

        vals_sorted = np.sort(vals_arr)[::-1]
        top3 = vals_sorted[:3]
        mean_v = float(np.mean(vals_arr))
        max_v = float(vals_sorted[0])
        top3_mean = float(np.mean(top3))
        wq = float(np.average(vals_arr, weights=w_qty_arr)) if np.sum(w_qty_arr) > 0 else mean_v
        wc = float(np.average(vals_arr, weights=w_cost_arr)) if np.sum(w_cost_arr) > 0 else mean_v

        out[r, 0] = mean_v
        out[r, 1] = max_v
        out[r, 2] = top3_mean
        out[r, 3] = wq
        out[r, 4] = wc

        if use_group:
            g = str(group_key[i])
            base_g = float(grp_med.get(g, global_med))
            vals_g = [float(grp_code_map.get((g, c), base_g)) for c in codes]
            vals_g = np.array(vals_g, dtype=np.float32)
            vals_g_sorted = np.sort(vals_g)[::-1]
            top3_g = vals_g_sorted[:3]
            out[r, 5] = float(np.mean(vals_g))
            out[r, 6] = float(vals_g_sorted[0])
            out[r, 7] = float(np.mean(top3_g))
            out[r, 8] = out[r, 5] - out[r, 0]
            out[r, 9] = out[r, 6] - out[r, 1]

    return out


# =========================
# PREPROCESSORS
# =========================
def build_preprocessor(num_cols: List[str], cat_cols: List[str]) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", make_ohe())]), cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def fit_vectorizers_and_transform(
    tr_text_qty: np.ndarray, va_text_qty: np.ndarray, te_text_qty: np.ndarray,
    tr_text_cost: np.ndarray, va_text_cost: np.ndarray, te_text_cost: np.ndarray,
) -> Tuple[csr_matrix, csr_matrix, csr_matrix]:
    vec_qty = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b(?:[A-Z]\d{4}|\d{5})\b",
        lowercase=False,
        ngram_range=TFIDF_NGRAM_QTY,
        max_features=TFIDF_MAX_QTY,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )
    vec_cost = TfidfVectorizer(
        analyzer="word",
        token_pattern=r"(?u)\b(?:[A-Z]\d{4}|\d{5})\b",
        lowercase=False,
        ngram_range=TFIDF_NGRAM_COST,
        max_features=TFIDF_MAX_COST,
        min_df=TFIDF_MIN_DF,
        max_df=TFIDF_MAX_DF,
        sublinear_tf=True,
        norm="l2",
        dtype=np.float32,
    )

    def _safe_fit_transform(vec, tr, va, te):
        try:
            Xtr = vec.fit_transform(tr.astype(str))
            Xva = vec.transform(va.astype(str))
            Xte = vec.transform(te.astype(str))
            return ensure_csr_float32_int32(Xtr), ensure_csr_float32_int32(Xva), ensure_csr_float32_int32(Xte)
        except ValueError:
            return csr_matrix((len(tr), 0), dtype=np.float32), csr_matrix((len(va), 0), dtype=np.float32), csr_matrix((len(te), 0), dtype=np.float32)

    Xtr_q, Xva_q, Xte_q = _safe_fit_transform(vec_qty, tr_text_qty, va_text_qty, te_text_qty)
    Xtr_c, Xva_c, Xte_c = _safe_fit_transform(vec_cost, tr_text_cost, va_text_cost, te_text_cost)

    Xtr = ensure_csr_float32_int32(hstack([Xtr_q, Xtr_c], format="csr"))
    Xva = ensure_csr_float32_int32(hstack([Xva_q, Xva_c], format="csr"))
    Xte = ensure_csr_float32_int32(hstack([Xte_q, Xte_c], format="csr"))
    return Xtr, Xva, Xte


# =========================
# CALIBRATION
# =========================
def shrunk_group_median(resid: np.ndarray, group: np.ndarray, alpha: float) -> Tuple[Dict[str, float], float]:
    g = group.astype(str)
    global_med = float(np.median(resid))
    out = {}
    for key in np.unique(g):
        m = (g == key)
        r = resid[m]
        if r.size == 0:
            continue
        med = float(np.median(r))
        n = float(r.size)
        out[str(key)] = (n * med + alpha * global_med) / (n + alpha)
    return out, global_med


def apply_bias(pred: np.ndarray, group: np.ndarray, bias: Dict[str, float], default_bias: float) -> np.ndarray:
    g = group.astype(str)
    p = pred.astype(np.float64).copy()
    for i in range(len(p)):
        p[i] += bias.get(str(g[i]), default_bias)
    return p.astype(np.float32)


# =========================
# MAIN
# =========================
def run_iteration11(diagnostic: bool = False) -> Dict[str, Any]:
    setup_logging()
    logging.info(f"XGBoost version: {getattr(xgb,'__version__','unknown')}")
    logging.info("=== Iteration 11: Leakage-safe TE + robust receipt parsing + XGB GPU ===")
    logging.info(f"diagnostic={diagnostic} | DEVICE={DEVICE}")

    train = pd.read_csv(TRAIN_CSV)
    test  = pd.read_csv(TEST_CSV)
    logging.info(f"Loaded train={train.shape} test={test.shape}")

    if PATIENTS_CSV.exists():
        pat = pd.read_csv(PATIENTS_CSV)
        if "patient_id" in pat.columns:
            train = train.merge(pat, on="patient_id", how="left")
            test  = test.merge(pat, on="patient_id", how="left")
            logging.info(f"Merged patients.csv: train={train.shape} test={test.shape}")
    else:
        logging.warning("patients.csv not found; continuing with ed_cost_*.csv only.")

    # receipts parse
    all_ids = pd.concat([train["patient_id"], test["patient_id"]], ignore_index=True).astype(int).tolist()
    rec = load_receipts(all_ids)
    train = train.merge(rec, on="patient_id", how="left")
    test  = test.merge(rec, on="patient_id", how="left")

    # ensure docs strings
    for df in [train, test]:
        for c in ["codes_doc_qty", "codes_doc_cost", "desc_doc"]:
            df[c] = df[c].fillna("").astype(str)
        df["pdf_insurance"] = df["pdf_insurance"].fillna("missing").astype(str)
        df["pdf_zip3"] = df["pdf_zip3"].fillna("missing").astype(str)

    # totals sanity: spec says parsed TOTAL matches prior cost; use prior cost as authoritative total
    for df in [train, test]:
        df["pdf_total_used"] = df["prior_ed_cost_5y_usd"].astype(float)
        df["total_absdiff_prior"] = np.abs(df["pdf_total_parsed"].astype(float) - df["prior_ed_cost_5y_usd"].astype(float))
        df["total_parse_ok"] = ((df["total_absdiff_prior"] <= 0.01) & np.isfinite(df["pdf_total_parsed"].astype(float))).astype(int)
        df["line_absdiff_total"] = np.abs(df["line_sum"].astype(float) - df["prior_ed_cost_5y_usd"].astype(float))

    logging.info(
        "Receipt sanity | parse_ok_rate(train)={:.3f} | total_parse_ok_rate(train)={:.3f} | "
        "line_absdiff_total_med(train)={:.3f} | max_absdiff_total(train)={:.3f}".format(
            float(train["parse_ok"].mean()),
            float(train["total_parse_ok"].mean()),
            float(train["line_absdiff_total"].median()),
            float(train["line_absdiff_total"].max()),
        )
    )

    # choose pivot codes (from TRAIN frequency)
    code_freq = defaultdict(int)
    for d in train["code_qty_dict"].tolist():
        if isinstance(d, dict):
            for c in d.keys():
                code_freq[c] += 1
    codes_sorted = sorted(code_freq.items(), key=lambda x: (-x[1], x[0]))
    pivot_codes = [c for c, _ in codes_sorted[:TOP_CODE_PIVOTS]]
    logging.info(f"Top code pivots: K={len(pivot_codes)} | unique_codes_in_train={len(code_freq)}")

    train = add_code_pivots(train, pivot_codes, total_col="pdf_total_used")
    test  = add_code_pivots(test,  pivot_codes, total_col="pdf_total_used")

    # derived tabular
    for df in [train, test]:
        df["prior_cost_per_visit"] = df["prior_ed_cost_5y_usd"] / (df["prior_ed_visits_5y"] + 1.0)
        df["log_prior_cost"] = np.log1p(df["prior_ed_cost_5y_usd"].astype(float))
        df["log_prior_visits"] = np.log1p(df["prior_ed_visits_5y"].astype(float))
        df["log_pdf_total_used"] = np.log1p(df["pdf_total_used"].astype(float))

        if "insurance" in df.columns:
            df["ins_mismatch"] = ((df["insurance"].astype(str) != df["pdf_insurance"].astype(str)) & (df["pdf_ok"] == 1)).astype(int)
        else:
            df["ins_mismatch"] = 0
        if "zip3" in df.columns:
            df["zip_mismatch"] = ((df["zip3"].astype(str) != df["pdf_zip3"].astype(str)) & (df["pdf_ok"] == 1)).astype(int)
        else:
            df["zip_mismatch"] = 0

    if diagnostic:
        return {
            "train_parse_ok_rate": float(train["parse_ok"].mean()),
            "train_total_parse_ok_rate": float(train["total_parse_ok"].mean()),
            "train_line_absdiff_total_max": float(train["line_absdiff_total"].max()),
            "unique_codes_in_train": int(len(code_freq)),
            "pivot_codes": pivot_codes[:20],
            "nonempty_codes_doc_qty_train": int((train["codes_doc_qty"].str.len() > 0).sum()),
            "nonempty_codes_doc_qty_test": int((test["codes_doc_qty"].str.len() > 0).sum()),
        }

    y = train["ed_cost_next3y_usd"].values.astype(np.float32)

    # stratification: chronic + insurance + cost/visit bins
    if "insurance" in train.columns:
        ins = train["insurance"].astype(str)
    else:
        ins = train["pdf_insurance"].astype(str)

    cost_bin = pd.qcut(train["prior_ed_cost_5y_usd"], q=6, duplicates="drop").astype(str)
    vis_bin  = pd.qcut(train["prior_ed_visits_5y"], q=3, duplicates="drop").astype(str)
    strat = (train["primary_chronic"].astype(str) + "_" + ins + "_" + cost_bin + "_" + vis_bin).values

    # identify columns for tabular preprocessor
    text_cols = {"codes_doc_qty", "codes_doc_cost", "desc_doc"}
    dict_cols = {"code_qty_dict", "code_cost_dict"}
    exclude = {"patient_id", "ed_cost_next3y_usd"} | text_cols | dict_cols

    cat_cols: List[str] = []
    base_cat = ["primary_chronic", "sex", "insurance", "zip3", "pdf_insurance", "pdf_zip3"]
    for c in base_cat:
        if c in train.columns:
            cat_cols.append(c)

    for c in train.columns:
        if c in exclude or c in cat_cols:
            continue
        if train[c].dtype == "object" or str(train[c].dtype).startswith("string"):
            cat_cols.append(c)

    num_cols: List[str] = []
    for c in train.columns:
        if c in exclude or c in cat_cols:
            continue
        if pd.api.types.is_numeric_dtype(train[c]):
            num_cols.append(c)

    for c in cat_cols:
        train[c] = train[c].astype("string").fillna("missing")
        test[c] = test[c].astype("string").fillna("missing")

    logging.info(f"Tabular cols | num={len(num_cols)} cat={len(cat_cols)}")

    # CV splits
    splits = []
    for rep, seed in enumerate(REPEAT_SEEDS):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(train)), strat)):
            splits.append((rep, fold, seed, tr_idx, va_idx))
    logging.info(f"Total folds: {len(splits)} | device={DEVICE}")

    oof_sum = np.zeros(len(train), dtype=np.float64)
    oof_cnt = np.zeros(len(train), dtype=np.float64)
    te_sum  = np.zeros(len(test), dtype=np.float64)
    n_models = 0

    params = dict(
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        device=DEVICE,
        n_estimators=N_ESTIMATORS,
        learning_rate=LR,
        max_depth=MAX_DEPTH,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_lambda=REG_LAMBDA,
        min_child_weight=MIN_CHILD_WEIGHT,
        gamma=GAMMA,
        n_jobs=-1,
        verbosity=1,
        early_stopping_rounds=EARLY_STOPPING,
    )

    for rep, fold, seed, tr_idx, va_idx in tqdm(splits, desc="CV", ncols=110):
        tag = f"rep={rep} fold={fold}"
        t0 = time.time()

        df_tr = train.iloc[tr_idx].reset_index(drop=True)
        df_va = train.iloc[va_idx].reset_index(drop=True)
        df_te = test.reset_index(drop=True)

        y_tr = y[tr_idx]
        y_va = y[va_idx]

        # ------------------
        # Fold-safe Target Encoding
        # ------------------
        idx_tr_local = np.arange(len(df_tr))
        group_tr = (df_tr["primary_chronic"].astype(str) + "_" +
                    (df_tr["insurance"].astype(str) if "insurance" in df_tr.columns else df_tr["pdf_insurance"].astype(str))).values if TE_USE_GROUP else None
        group_va = (df_va["primary_chronic"].astype(str) + "_" +
                    (df_va["insurance"].astype(str) if "insurance" in df_va.columns else df_va["pdf_insurance"].astype(str))).values if TE_USE_GROUP else None
        group_te = (df_te["primary_chronic"].astype(str) + "_" +
                    (df_te["insurance"].astype(str) if "insurance" in df_te.columns else df_te["pdf_insurance"].astype(str))).values if TE_USE_GROUP else None

        code_map, global_med, grp_code_map, grp_med = build_code_te_maps(
            df_tr, idx_tr_local, y_tr, group_key=group_tr, alpha=TE_ALPHA
        )

        te_tr = patient_te_features(df_tr, np.arange(len(df_tr)), code_map, global_med, grp_code_map, grp_med, group_tr, "pdf_total_used")
        te_va = patient_te_features(df_va, np.arange(len(df_va)), code_map, global_med, grp_code_map, grp_med, group_va, "pdf_total_used")
        te_te = patient_te_features(df_te, np.arange(len(df_te)), code_map, global_med, grp_code_map, grp_med, group_te, "pdf_total_used")

        X_te_tr = ensure_csr_float32_int32(csr_matrix(te_tr.astype(np.float32)))
        X_te_va = ensure_csr_float32_int32(csr_matrix(te_va.astype(np.float32)))
        X_te_te = ensure_csr_float32_int32(csr_matrix(te_te.astype(np.float32)))

        # ------------------
        # Tabular preprocessor (fold fit)
        # ------------------
        pre = build_preprocessor(num_cols, cat_cols)

        X_tr_tab = pre.fit_transform(df_tr)
        X_va_tab = pre.transform(df_va)
        X_te_tab = pre.transform(df_te)

        X_tr_tab = ensure_csr_float32_int32(X_tr_tab.tocsr() if hasattr(X_tr_tab, "tocsr") else csr_matrix(X_tr_tab, dtype=np.float32))
        X_va_tab = ensure_csr_float32_int32(X_va_tab.tocsr() if hasattr(X_va_tab, "tocsr") else csr_matrix(X_va_tab, dtype=np.float32))
        X_te_tab = ensure_csr_float32_int32(X_te_tab.tocsr() if hasattr(X_te_tab, "tocsr") else csr_matrix(X_te_tab, dtype=np.float32))

        # ------------------
        # TF-IDF (fold fit)
        # ------------------
        X_tr_txt, X_va_txt, X_te_txt = fit_vectorizers_and_transform(
            df_tr["codes_doc_qty"].values, df_va["codes_doc_qty"].values, df_te["codes_doc_qty"].values,
            df_tr["codes_doc_cost"].values, df_va["codes_doc_cost"].values, df_te["codes_doc_cost"].values,
        )

        X_tr = ensure_csr_float32_int32(hstack([X_tr_tab, X_te_tr, X_tr_txt], format="csr"))
        X_va = ensure_csr_float32_int32(hstack([X_va_tab, X_te_va, X_va_txt], format="csr"))
        X_te = ensure_csr_float32_int32(hstack([X_te_tab, X_te_te, X_te_txt], format="csr"))

        logging.info(f"{tag} | X shapes | tr={X_tr.shape} va={X_va.shape} nnz(tr)={X_tr.nnz:,}")

        model = XGBRegressor(**params, random_state=10000 * rep + seed + fold)

        fit_sig = inspect.signature(model.fit).parameters
        fit_kwargs = {"eval_set": [(X_va, y_va)]}
        if "verbose" in fit_sig:
            fit_kwargs["verbose"] = VERBOSE_EVERY
        elif "verbose_eval" in fit_sig:
            fit_kwargs["verbose_eval"] = VERBOSE_EVERY

        logging.info(f"{tag} | fit START")
        model.fit(X_tr, y_tr, **fit_kwargs)
        logging.info(f"{tag} | fit END | best_iteration={getattr(model,'best_iteration',None)}")

        p_va = model.predict(X_va).astype(np.float32)
        p_te = model.predict(X_te).astype(np.float32)

        if CLIP_AT_ZERO:
            p_va = np.maximum(p_va, 0.0)
            p_te = np.maximum(p_te, 0.0)

        oof_sum[va_idx] += p_va
        oof_cnt[va_idx] += 1.0
        te_sum += p_te
        n_models += 1

        fold_mae = mae(y_va, p_va)
        logging.info(f"{tag} | fold_MAE={fold_mae:.3f} | fold_time={time.time()-t0:.1f}s")

    oof = (oof_sum / np.maximum(oof_cnt, 1.0)).astype(np.float32)
    pred_test = (te_sum / max(1, n_models)).astype(np.float32)

    raw_mae = mae(y, oof)
    logging.info(f"OOF MAE (raw): {raw_mae:.4f}")

    resid = (y.astype(np.float64) - oof.astype(np.float64))

    best = (raw_mae, "none", pred_test)

    # chronic calibration
    bias_c, def_c = shrunk_group_median(resid, train["primary_chronic"].astype(str).values, alpha=TE_ALPHA)
    oof_c = apply_bias(oof, train["primary_chronic"].astype(str).values, bias_c, def_c)
    if CLIP_AT_ZERO:
        oof_c = np.maximum(oof_c, 0.0)
    mae_c = mae(y, oof_c)
    logging.info(f"OOF MAE (chronic calib): {mae_c:.4f}")
    if mae_c < best[0]:
        pred_c = apply_bias(pred_test, test["primary_chronic"].astype(str).values, bias_c, def_c)
        if CLIP_AT_ZERO:
            pred_c = np.maximum(pred_c, 0.0)
        best = (mae_c, "chronic", pred_c)

    # chronic+insurance calibration
    if "insurance" in train.columns and "insurance" in test.columns:
        key_tr = (train["primary_chronic"].astype(str) + "_" + train["insurance"].astype(str)).values
        key_te = (test["primary_chronic"].astype(str) + "_" + test["insurance"].astype(str)).values
        bias_ci, def_ci = shrunk_group_median(resid, key_tr, alpha=TE_ALPHA)
        oof_ci = apply_bias(oof, key_tr, bias_ci, def_ci)
        if CLIP_AT_ZERO:
            oof_ci = np.maximum(oof_ci, 0.0)
        mae_ci = mae(y, oof_ci)
        logging.info(f"OOF MAE (chronic+ins calib): {mae_ci:.4f}")
        if mae_ci < best[0]:
            pred_ci = apply_bias(pred_test, key_te, bias_ci, def_ci)
            if CLIP_AT_ZERO:
                pred_ci = np.maximum(pred_ci, 0.0)
            best = (mae_ci, "chronic+insurance", pred_ci)

    chosen_mae, chosen_name, final_pred = best
    logging.info(f"Chosen calibration: {chosen_name} | OOF MAE={chosen_mae:.4f}")

    sub = pd.DataFrame({
        "patient_id": test["patient_id"].astype(int).values,
        "ed_cost_next3y_usd": final_pred.astype(np.float32),
    })
    sub.to_csv(SUBMISSION_PATH, index=False)
    logging.info(f"Saved submission: {SUBMISSION_PATH}")

    return {
        "oof_mae_raw": float(raw_mae),
        "oof_mae_best": float(chosen_mae),
        "calibration": chosen_name,
        "n_models": int(n_models),
        "submission_path": str(SUBMISSION_PATH),
    }


In [59]:
res = run_iteration11(diagnostic=True)
res

2026-02-14 05:16:29,154 | INFO | XGBoost version: 3.0.0
2026-02-14 05:16:29,156 | INFO | === Iteration 11: Leakage-safe TE + robust receipt parsing + XGB GPU ===
2026-02-14 05:16:29,156 | INFO | diagnostic=True | DEVICE=cuda
2026-02-14 05:16:29,176 | INFO | Loaded train=(2000, 5) test=(2000, 4)
2026-02-14 05:16:29,187 | INFO | Merged patients.csv: train=(2000, 9) test=(2000, 8)
2026-02-14 05:16:29,190 | INFO | Receipt cache | hit=0 miss=4,000 version=111
Parsing receipts: 100%|████████████████████████████████████████| 4000/4000 [00:12<00:00, 309.09it/s]
2026-02-14 05:16:51,065 | INFO | Receipt sanity | parse_ok_rate(train)=1.000 | total_parse_ok_rate(train)=1.000 | line_absdiff_total_med(train)=0.000 | max_absdiff_total(train)=0.002
2026-02-14 05:16:51,067 | INFO | Top code pivots: K=18 | unique_codes_in_train=18


{'train_parse_ok_rate': 1.0,
 'train_total_parse_ok_rate': 1.0,
 'train_line_absdiff_total_max': 0.0024999999986903276,
 'unique_codes_in_train': 18,
 'pivot_codes': ['99292',
  '99291',
  '36556',
  '36620',
  '31500',
  '92950',
  '71045',
  '99281',
  '85025',
  '87070',
  '99282',
  '99283',
  '74177',
  '99285',
  '84484',
  '99284',
  'G0378',
  '70450'],
 'nonempty_codes_doc_qty_train': 2000,
 'nonempty_codes_doc_qty_test': 2000}

In [60]:
res = run_iteration11(diagnostic=False)
res


2026-02-14 05:16:56,317 | INFO | XGBoost version: 3.0.0
2026-02-14 05:16:56,318 | INFO | === Iteration 11: Leakage-safe TE + robust receipt parsing + XGB GPU ===
2026-02-14 05:16:56,318 | INFO | diagnostic=False | DEVICE=cuda
2026-02-14 05:16:56,324 | INFO | Loaded train=(2000, 5) test=(2000, 4)
2026-02-14 05:16:56,329 | INFO | Merged patients.csv: train=(2000, 9) test=(2000, 8)
2026-02-14 05:16:56,493 | INFO | Receipt cache | hit=4,000 miss=0 version=111
Parsing receipts: 0it [00:00, ?it/s]
2026-02-14 05:17:04,740 | INFO | Receipt sanity | parse_ok_rate(train)=1.000 | total_parse_ok_rate(train)=1.000 | line_absdiff_total_med(train)=0.000 | max_absdiff_total(train)=0.002
2026-02-14 05:17:04,742 | INFO | Top code pivots: K=18 | unique_codes_in_train=18
2026-02-14 05:17:04,808 | INFO | Tabular cols | num=81 cat=6
2026-02-14 05:17:04,817 | INFO | Total folds: 10 | device=cuda
CV:   0%|                                                                              | 0/10 [00:00<?, ?it/s]2026

[0]	validation_0-mae:1365.23244
[100]	validation_0-mae:544.20477
[200]	validation_0-mae:477.63994
[300]	validation_0-mae:476.29458
[400]	validation_0-mae:479.79778
[434]	validation_0-mae:481.38519


2026-02-14 05:17:08,385 | INFO | rep=0 fold=0 | fit END | best_iteration=234
2026-02-14 05:17:08,409 | INFO | rep=0 fold=0 | fold_MAE=475.164 | fold_time=3.6s
CV:  10%|███████                                                               | 1/10 [00:03<00:32,  3.59s/it]2026-02-14 05:17:09,009 | INFO | rep=0 fold=1 | X shapes | tr=(1600, 2063) va=(400, 2063) nnz(tr)=135,072
2026-02-14 05:17:09,011 | INFO | rep=0 fold=1 | fit START


[0]	validation_0-mae:1434.32055
[100]	validation_0-mae:549.02341
[200]	validation_0-mae:465.23034
[300]	validation_0-mae:453.54644
[400]	validation_0-mae:451.73603
[500]	validation_0-mae:450.34306
[600]	validation_0-mae:450.38913
[700]	validation_0-mae:450.32701
[713]	validation_0-mae:450.43369


2026-02-14 05:17:13,080 | INFO | rep=0 fold=1 | fit END | best_iteration=514
2026-02-14 05:17:13,114 | INFO | rep=0 fold=1 | fold_MAE=450.066 | fold_time=4.7s
CV:  20%|██████████████                                                        | 2/10 [00:08<00:33,  4.25s/it]2026-02-14 05:17:13,634 | INFO | rep=0 fold=2 | X shapes | tr=(1600, 2063) va=(400, 2063) nnz(tr)=134,935
2026-02-14 05:17:13,636 | INFO | rep=0 fold=2 | fit START


...

[0]	validation_0-mae:1444.97405
[100]	validation_0-mae:541.27993
[200]	validation_0-mae:451.20102
[300]	validation_0-mae:440.57042
[400]	validation_0-mae:437.94305
[500]	validation_0-mae:436.50584
[600]	validation_0-mae:436.18852
[700]	validation_0-mae:436.23446
[764]	validation_0-mae:436.74945


2026-02-14 05:17:45,851 | INFO | rep=1 fold=3 | fit END | best_iteration=564
2026-02-14 05:17:45,891 | INFO | rep=1 fold=3 | fold_MAE=435.686 | fold_time=5.7s
CV:  90%|███████████████████████████████████████████████████████████████       | 9/10 [00:41<00:04,  4.87s/it]2026-02-14 05:17:46,495 | INFO | rep=1 fold=4 | X shapes | tr=(1600, 2063) va=(400, 2063) nnz(tr)=134,793
2026-02-14 05:17:46,497 | INFO | rep=1 fold=4 | fit START


[0]	validation_0-mae:1396.24317
[100]	validation_0-mae:543.24644
[200]	validation_0-mae:471.22915
[300]	validation_0-mae:465.16558
[400]	validation_0-mae:464.75848
[500]	validation_0-mae:464.95608
[569]	validation_0-mae:465.09267


2026-02-14 05:17:50,057 | INFO | rep=1 fold=4 | fit END | best_iteration=369
2026-02-14 05:17:50,110 | INFO | rep=1 fold=4 | fold_MAE=463.937 | fold_time=4.2s
CV: 100%|█████████████████████████████████████████████████████████████████████| 10/10 [00:45<00:00,  4.53s/it]
2026-02-14 05:17:50,113 | INFO | OOF MAE (raw): 452.5401
2026-02-14 05:17:50,118 | INFO | OOF MAE (chronic calib): 448.8062
2026-02-14 05:17:50,131 | INFO | OOF MAE (chronic+ins calib): 448.3251
2026-02-14 05:17:50,135 | INFO | Chosen calibration: chronic+insurance | OOF MAE=448.3251
2026-02-14 05:17:50,146 | INFO | Saved submission: D:\AgentDs\agent_ds_healthcare\submission_ICHI_V1.csv


{'oof_mae_raw': 452.5401306152344,
 'oof_mae_best': 448.3250732421875,
 'calibration': 'chronic+insurance',
 'n_models': 10,
 'submission_path': 'D:\\AgentDs\\agent_ds_healthcare\\submission_ICHI_V1.csv'}

# New Iter

# Submission

In [62]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("Healthcare", 2, "D:/AgentDs/agent_ds_healthcare/submission_ICHI_V1.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. Move on to Healthcare Challenge 3!")


🚀 Submitting predictions...
✅ Prediction submitted successfully!
📊 Score: 469.7176 (MAE)
✅ Validation passed
✅ Submission successful!
   📊 Score: 469.7176
   📏 Metric: MAE
   ✔️  Validation: Passed

🎯 Next steps:
   1. Try incorporating relevant information outside this table!
   2. Move on to Healthcare Challenge 3!
